In [ ]:
# general imports
import os
import scipy.stats as stats
import itertools
from itertools import product, combinations_with_replacement
import pickle
from unidecode import unidecode
import pandas as pd
import json
import re
import warnings
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)

# plotting imports
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import kaleido
import nbformat

# local imports
from utils.unicode_utils import apply_greek_labels_heatmap, introduce_greek_letters, replace_greek_letters, apply_greek_labels
from utils.helper_functions import get_common_index, leiden_clustering
from utils.plotting_utils import cm_to_inches, setup_plotting_environment, plot_overview_labels_along_y_axis, create_sankey_diagram_strength

setup_plotting_environment()

# ------- CRITICAL -------
alt_flag = False # use an alternative cell type set (more fine-grained)
write_SI_tables_flag = True # whether to write tables with SI values (if False, only plots will be generated)
if not alt_flag:
    from utils.color_constants import CELL_TYPE_COLORS, ALL_CELL_TYPES
else:
    from utils.color_constants import CELL_TYPE_COLORS_ALT, ALL_CELL_TYPES_ALT
    ALL_CELL_TYPES = ALL_CELL_TYPES_ALT
    CELL_TYPE_COLORS = CELL_TYPE_COLORS_ALT
# ------- CRITICAL -------

# set up folders
input_data_folder = f"../../data/main_screen/"
os.makedirs(input_data_folder, exist_ok=True)

output_folder = f"../../outputs/main_screen/"
os.makedirs(output_folder, exist_ok=True)

if not alt_flag:
    base_plot_folder = os.path.join(output_folder, "plots")
    os.makedirs(base_plot_folder, exist_ok=True)
    intermediate_output_folder = os.path.join(output_folder, "intermediate_outputs")
    os.makedirs(intermediate_output_folder, exist_ok=True)
    output_tables_folder = os.path.join(output_folder, "output_tables")
    os.makedirs(output_tables_folder, exist_ok=True)
    overall_output_table_name = os.path.join(output_tables_folder, "SI_tables.xlsx")
if alt_flag:
    base_plot_folder = os.path.join(output_folder, "plots_alt")
    os.makedirs(base_plot_folder, exist_ok=True)
    intermediate_output_folder = os.path.join(output_folder, "intermediate_outputs_alt")
    os.makedirs(intermediate_output_folder, exist_ok=True)
    output_tables_folder = os.path.join(output_folder, "output_tables_alt")
    os.makedirs(output_tables_folder, exist_ok=True)
    overall_output_table_name = os.path.join(output_tables_folder, "SI_tables_alt.xlsx")

def write_to_SI_table(df, sheet_name, mode="a"):
    if not os.path.exists(overall_output_table_name):
        mode = "w"
    else:
        mode = "a"
    with pd.ExcelWriter(
        overall_output_table_name, mode=mode, engine="openpyxl",
        if_sheet_exists="replace" if mode == "a" else None) as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

# 1 - Load and inspect the data

In [ ]:
plot_folder = os.path.join(base_plot_folder, "1_data_loading")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
# info on cytokines
cytokine_info = pd.read_excel("../../annotations/cytokine_info.xlsx", index_col=0, sheet_name="all_cytokines")
gene_to_cytokine = cytokine_info.set_index("gene")['gene_to_cytokine'].dropna().to_dict()
cytokine_to_gene = {v: k for k, v in gene_to_cytokine.items()}
ALL_CYTOKINES_ORDER = cytokine_info['gene_to_cytokine'].dropna().tolist()

cytokine_name_to_level_1_family = cytokine_info.reset_index().set_index("name")["family_level_1"].to_dict()
cytokine_name_to_level_2_family = cytokine_info.reset_index().set_index("name")["family_level_2"].to_dict()
cytokine_name_to_functional = cytokine_info.reset_index().set_index("name")["functional_grouping"].to_dict()

# gene-level names (e.g. IL-32, C3, C5, LT-alpha-beta) are aliases of cytokine_info names
gene_level_alias = {alias: name for name, alias in cytokine_info["gene_to_cytokine"].dropna().items() if alias != name}
for name_map in [cytokine_name_to_level_1_family, cytokine_name_to_level_2_family, cytokine_name_to_functional]:
    name_map.update({alias: name_map[name] for alias, name in gene_level_alias.items()})

cytokine_level_1_order = list(dict.fromkeys(cytokine_name_to_level_1_family.values()))
cytokine_level_2_order = list(dict.fromkeys(cytokine_name_to_level_2_family.values()))
level2_to_level1 = cytokine_info.set_index("family_level_2")["family_level_1"].to_dict()

# tested subset
cytokine_info_parse = cytokine_info[cytokine_info["parse_cytokine"]]

# ordering
cytokine_info_order = pd.read_excel("../../annotations/cytokine_info.xlsx", index_col=0, sheet_name="suggested_order")
CYTOKINE_ORDER = list(cytokine_info_order["name"])
CYTOKINE_ORDER_WITH_PBS = ["PBS"] + CYTOKINE_ORDER
SINGLE_FAMILY_ORDER = cytokine_info_order["DEG_plot_family"].unique()

In [ ]:
if write_SI_tables_flag:
    curr_info = cytokine_info_parse.loc[CYTOKINE_ORDER].copy().reset_index()
    curr_info['name'] = curr_info['name'].apply(introduce_greek_letters)
    curr_info = curr_info[['name', 'gene', 'DEG_plot_family', 'receptor gene']]
    curr_info = curr_info.rename(columns={
        'name': 'Cytokine',
        'gene': 'Gene(s)',
        'DEG_plot_family': 'Family',
        'receptor gene': 'Receptor gene(s)'
    })
    write_to_SI_table(curr_info, sheet_name="cytokine_screen")

    curr_info = cytokine_info.copy().reset_index()
    curr_info['name'] = curr_info['name'].apply(introduce_greek_letters)
    curr_info = curr_info[['name', 'gene', 'family_level_1', 'family_level_2']]
    curr_info = curr_info.rename(columns={
        'name': 'Cytokine',
        'gene': 'Gene(s)',
        'family_level_1': 'Family',
        'family_level_2': 'Subfamily'
    })
    write_to_SI_table(curr_info, sheet_name="all_cytokines")

### Compare used concentrations to the literature

In [ ]:
used_concentrations = pd.read_excel("../../annotations/concentration_info.xlsx", index_col=0)['concentration (ng/mL)']
used_concentrations.index = [replace_greek_letters(c) for c in used_concentrations.index]
used_concentrations = used_concentrations.reset_index()
used_concentrations.columns = ["cytokine", "concentration (ng/mL)"]

In [ ]:
from pyCytED._paths import content_path

CytED = pd.read_parquet(content_path('CytED.parquet'))
CytED = CytED[CytED['experimental_concentration'] != ""].copy()
CytED = CytED[CytED['experimental_system_type'].str.contains("in vitro")]

In [ ]:
def extract_entries(s):
    if not 'entries' in s:
        return []
    try:
        obj = json.loads(s.replace("'", '"'))
    except Exception:
        return []  # or None
    entries = obj.get("entries")
    if not isinstance(entries, dict):
        return []
    # return list of dicts; add source_id later
    return [
        {"entity": v.get("entity"), "concentration": v.get("concentration"), "unit": v.get("unit")}
        for v in entries.values()
        if isinstance(v, dict)
    ]

tmp = CytED[["source_id", "experimental_concentration"]].copy()
tmp["entries_list"] = tmp["experimental_concentration"].map(extract_entries)
out = tmp.explode("entries_list", ignore_index=True)

In [ ]:
def replace_cytokines(s):
    if s == "Thrombopoietin":
        return "TPO"
    elif s == "Erythropoietin":
        return "EPO"
    elif s == "Prolactin":
        return "PRL"
    elif s == "IFN-alpha":
        return "IFN-alpha1"
    elif s == 'TGF-beta':
        return "TGF-beta1"
    else:
        return s

cyted_concentrations = pd.concat([out[["source_id"]], pd.json_normalize(out["entries_list"])], axis=1).dropna(subset=["entity", "concentration", "unit"], how="all")
cyted_concentrations['entity'] = cyted_concentrations['entity'].apply(replace_cytokines)
cyted_concentrations = cyted_concentrations[cyted_concentrations['entity'].isin(CYTOKINE_ORDER)]
cyted_concentrations['unit'] = cyted_concentrations['unit'].str.strip().str.lower()

In [ ]:
unit_counts = cyted_concentrations.groupby(['entity', 'unit']).size().rename('n').reset_index()
unit_counts['fraction'] = unit_counts['n'] / unit_counts.groupby('entity')['n'].transform('sum')

# most frequent unit per cytokine
top_unit = unit_counts.sort_values('n', ascending=False).drop_duplicates('entity').set_index('entity')
nm_fraction = unit_counts[unit_counts['unit'] == 'nm'].set_index('entity')['fraction']
nm_fraction[nm_fraction > 0.1]

In [ ]:
molecular_weight = {
    "IL-17D": 20.3,
    "C3a": 9.1,
    "C5a": 8.3,
    "Decorin": 36.3,
    "IL-16": 12.4,
    "Leptin": 16.0,
    "IGF-1": 7.6,
    "IL-26": 17.6,
    "IL-8": 8.4,
    "PRL": 22.9,
    "OX40L": 15.4,
    "IL-36Ra": 16.8,
}

cyted_concentrations['concentration'] = pd.to_numeric(cyted_concentrations['concentration'], errors='coerce')
is_molar = cyted_concentrations['unit'].isin(['nm', 'nmol/l']) & cyted_concentrations['entity'].isin(molecular_weight)
cyted_concentrations.loc[is_molar, 'concentration'] *= cyted_concentrations.loc[is_molar, 'entity'].map(molecular_weight)
cyted_concentrations.loc[is_molar, 'unit'] = 'ng/ml'

In [ ]:
cyted_concentrations = cyted_concentrations[cyted_concentrations['unit'] == 'ng/ml']
cyted_concentrations = cyted_concentrations.drop_duplicates()
cyted_concentrations['concentration'] = cyted_concentrations['concentration'].astype(float)
cyted_concentrations = cyted_concentrations[cyted_concentrations['concentration'] > 0].copy()  

In [ ]:
def pretty(p):
    if abs(p) >= 10:
        return f"{p:.0f}"
    return f"{p:.2f}".rstrip("0").rstrip(".")

used_concentrations['cytokine'] = used_concentrations['cytokine'].apply(lambda x: replace_greek_letters(x))

cytokines_parse = set(used_concentrations['cytokine'])
cytokines_cyted = set(cyted_concentrations["entity"])
missing = cytokines_parse - cytokines_cyted
print(cytokines_parse - cytokines_cyted)

In [ ]:
os.makedirs(os.path.join(plot_folder, "percentiles"), exist_ok=True)

percentiles_by_cytokine = []
for cytokine in CYTOKINE_ORDER:
    cytokine = replace_cytokines(cytokine)    
    if cytokine in missing:
        continue
    df = cyted_concentrations[(cyted_concentrations["entity"] == cytokine)].copy()
    df["concentration"] = df["concentration"].astype(float)
    vals = df["concentration"].to_numpy()
    
    used_conc = used_concentrations[
        used_concentrations['cytokine'] == cytokine]['concentration (ng/mL)'].iloc[0]
    if len(vals) < 2:
        print(f"{cytokine} - not enough data points to determine percentile (only {len(vals)} entries), used concentration: {used_conc} ng/ml")
        continue

    w = 1 / df["source_id"].map(df["source_id"].value_counts()).to_numpy()
    c = df["concentration"].to_numpy()
    percentile = 100 * (w[c < used_conc].sum() + 0.5 * w[c == used_conc].sum()) / w.sum()
    percentiles_by_cytokine.append({"Cytokine": cytokine, "Percentile": percentile})
    
    vals = np.log10(vals)
    plt.figure(figsize=cm_to_inches((7, 4)))
    plt.hist(vals, weights=w, bins=30, color="lightblue", edgecolor="black")
    plt.axvline(np.log10(used_conc), color="red", linestyle="--", label=f"({pretty(percentile)}th perc.)")
    plt.legend(loc="upper left", fontsize=7)
    plt.xlabel("Concentration (log10 ng/ml)")
    plt.ylabel("Number of publications")
    plt.title(cytokine)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, "percentiles", f"{cytokine}_percentile_histogram.png"), bbox_inches="tight", dpi=300)
    plt.close()
    
percentiles_by_cytokine = pd.DataFrame(percentiles_by_cytokine)
percentiles_by_cytokine = percentiles_by_cytokine.sort_values("Percentile").set_index('Cytokine')

In [ ]:
fig, ax = plt.subplots(figsize=cm_to_inches((18, 3)))
ax.scatter(percentiles_by_cytokine.index, percentiles_by_cytokine["Percentile"], s=5)
plt.plot([-0.5, len(percentiles_by_cytokine)-0.5], [50, 50], color="red", linestyle="--", linewidth=0.5)

ax.set_ylabel("Literature percentile\nof used conc.")
ax.set_ylim(0, 100)
ax.set_xlim(-0.5, len(percentiles_by_cytokine)-0.5)
ax.tick_params(axis='x', labelsize=5.5)

apply_greek_labels_heatmap(ax)
plt.tick_params(axis='x', length=0)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "cytokine_concentration_percentiles.svg"), dpi=300)
plt.show()

## 1.1 - Process pseudobulks

In [ ]:
from utils.cell_type_mapping import get_relevant_cell_types
relevant_cell_types, relevant_cell_type_conversion, relevant_cell_to_pseudobulk = get_relevant_cell_types(alt_flag)

In [ ]:
import anndata as ad
def load_pseudobulks(path, relevant_cell_types, relevant_cell_type_conversion, cytokines=None, cell_types=None, pool_wells=True):
    # Merge individual pseudobulk files
    if cell_types is None:
        cell_types = relevant_cell_types
    if not set(cell_types).issubset(relevant_cell_types):
        raise ValueError("Requested cell types must be in relevant_cell_types.")

    pseudobulk_files = sorted(file for file in os.listdir(path) if not file.startswith('.') and '_pbulk_' in file and file.endswith('_sum.h5ad'))
    raw_cell_types = sorted(relevant_cell_type_conversion, key=len, reverse=True)

    all_dfs = []
    all_counts_by_condition = []
    for file in pseudobulk_files:
        stem = file.split('_pbulk_', 1)[1].removesuffix('_sum.h5ad')
        for raw_cell_type in raw_cell_types:
            if stem.startswith(raw_cell_type + '_'):
                break
        else:
            continue

        cell_type = relevant_cell_type_conversion[raw_cell_type]
        cytokine = stem[len(raw_cell_type) + 1:]
        if cell_type not in cell_types:
            continue
        if cytokines is not None and cytokine not in cytokines:
            continue

        adata = ad.read_h5ad(os.path.join(path, file))
        df = adata.to_df()
        if pool_wells:
            df.index = pd.MultiIndex.from_tuples([(cell_type, cytokine, donor) for donor in adata.obs['donor']])
            df.index.names = ['cell_type', 'cytokine', 'donor']
        else:
            if adata.obs['bc1_well'].astype(str).str.contains(',').any():
                raise ValueError(f"{file}: wells are already pooled; use the original per-well files.")
            df.index = pd.MultiIndex.from_tuples([(cell_type, cytokine, donor, well) for donor, well in zip(adata.obs['donor'], adata.obs['bc1_well'])])
            df.index.names = ['cell_type', 'cytokine', 'donor', 'well']

        counts = adata.obs['psbulk_n_cells'].copy()
        counts.index = df.index

        df = df.groupby(level=df.index.names, sort=False).sum()
        counts = counts.groupby(level=counts.index.names, sort=False).sum()
        all_dfs.append(df)
        all_counts_by_condition.append(counts)

    if not all_dfs:
        raise ValueError(f"No matching pseudobulks found in {path}.")

    count_df_raw = pd.concat(all_dfs, axis=0, verify_integrity=True).fillna(0).T
    all_counts_by_condition = pd.concat(all_counts_by_condition)
    all_counts_by_condition = all_counts_by_condition.reindex(count_df_raw.columns)

    return count_df_raw, all_counts_by_condition

if not os.path.exists(os.path.join(input_data_folder, 'pseudobulks_main.parquet')):
    sum_gene_count_df_raw_by_donor, all_counts_by_condition = load_pseudobulks(os.path.join(input_data_folder, 'pseudobulks'), relevant_cell_types, relevant_cell_type_conversion)
    sum_gene_count_df_raw_by_donor.to_parquet(os.path.join(input_data_folder, 'pseudobulks_main.parquet'))
    all_counts_by_condition.to_frame(name="cell_count").to_parquet(os.path.join(input_data_folder, 'cell_counts_main.parquet'))
else:
    sum_gene_count_df_raw_by_donor = pd.read_parquet(os.path.join(input_data_folder, 'pseudobulks_main.parquet'))
    all_counts_by_condition = pd.read_parquet(os.path.join(input_data_folder, 'cell_counts_main.parquet'))['cell_count']

sum_gene_count_df_raw_by_donor = sum_gene_count_df_raw_by_donor[ALL_CELL_TYPES]
all_counts_by_condition = all_counts_by_condition[ALL_CELL_TYPES]

## add data from a follow-up screen

In [ ]:
input_data_folder_review = "../../data/review_screen/"
if not os.path.exists(os.path.join(input_data_folder_review, 'pseudobulks_review.parquet')):
    count_df_review, cell_count_review = load_pseudobulks(os.path.join(input_data_folder_review, "pseudobulks"), relevant_cell_types, relevant_cell_type_conversion, cell_types=ALL_CELL_TYPES)
    count_df_review.to_parquet(os.path.join(input_data_folder_review, 'pseudobulks_review.parquet'))
    cell_count_review.to_frame(name="cell_count").to_parquet(os.path.join(input_data_folder_review, 'cell_counts_review.parquet'))
else:
    count_df_review = pd.read_parquet(os.path.join(input_data_folder_review, 'pseudobulks_review.parquet'))
    cell_count_review = pd.read_parquet(os.path.join(input_data_folder_review, 'cell_counts_review.parquet'))['cell_count']

In [ ]:
review_replace_list = ['IL-18', 'TNF-alpha', 'TGF-beta1', 'C3a']

# Rename follow-up conditions
treatments = count_df_review.columns.get_level_values('cytokine').unique()
treatment_conversion = {treat: treat if treat in review_replace_list else treat + "_review" for treat in treatments}
review_treatments = [treatment_conversion[treat] for treat in treatments if treat not in review_replace_list]
full_review_list = review_replace_list + review_treatments

count_df_review = count_df_review.rename(columns=treatment_conversion, level='cytokine')
cell_count_review = cell_count_review.rename(index=treatment_conversion, level='cytokine')

In [ ]:
# Replace the four main-screen conditions and merge expression data on shared genes
mask = sum_gene_count_df_raw_by_donor.columns.get_level_values('cytokine').isin(review_replace_list)
sum_gene_count_df_raw_by_donor = sum_gene_count_df_raw_by_donor.loc[:, ~mask]
sum_gene_count_df_raw_by_donor = pd.concat([sum_gene_count_df_raw_by_donor, count_df_review], axis=1, join='inner', verify_integrity=True)
sum_gene_count_df_raw_by_donor = sum_gene_count_df_raw_by_donor[ALL_CELL_TYPES]

# Apply the same replacement to cell counts
mask = all_counts_by_condition.index.get_level_values('cytokine').isin(review_replace_list)
all_counts_by_condition = all_counts_by_condition[~mask]
all_counts_by_condition = pd.concat([all_counts_by_condition, cell_count_review], verify_integrity=True)
all_counts_by_condition_unfiltered = all_counts_by_condition.copy()
all_counts_by_condition = all_counts_by_condition[ALL_CELL_TYPES]

In [ ]:
# everything present for the reviews
full_review_list = review_replace_list + review_treatments

## aggregation

In [ ]:
exclude_cell_types = ['HSPC']
aggregate_attribution = {
    "B cell": "B cell",
    "CD4 T cell": "T cell",
    "CD8 T cell": "T cell",
    "Treg": "T cell",
    "NKT": "T cell",
    "MAIT": "T cell",
    "NK CD56low": "NK cell",
    "NK CD56hi": "NK cell",
    "CD14 Mono": "Mono/cDC",
    "CD16 Mono": "Mono/cDC",
    "cDC": "Mono/cDC",
}

# create aggregate cell counts
cell_count_agg = all_counts_by_condition[~all_counts_by_condition.index.get_level_values(0).isin(exclude_cell_types)].copy()
cell_count_agg = cell_count_agg.rename(index=aggregate_attribution, level=0)
cell_count_agg = cell_count_agg.groupby(level=[0, 1, 2]).sum()

In [ ]:
from utils.processing_utils import get_filter_by_counts, normalize_df_to_rpm

count_df_raw_agg = sum_gene_count_df_raw_by_donor.loc[:, ~sum_gene_count_df_raw_by_donor.columns.get_level_values("cell_type").isin(exclude_cell_types)].copy()
count_df_raw_agg.columns = pd.MultiIndex.from_tuples([(aggregate_attribution[i[0]], i[1], i[2]) for i in count_df_raw_agg.columns])
count_df_raw_agg = count_df_raw_agg.fillna(0).T.groupby(level=[0, 1, 2]).sum().T
count_df_raw_agg.columns.names = ['cell_type', 'cytokine', 'donor']

# normalize to rpm
mask_average_calc = get_filter_by_counts(cell_count_agg, plot=False)
mask_average_calc = mask_average_calc[mask_average_calc.index.isin(count_df_raw_agg.columns)]
count_df_raw_agg = count_df_raw_agg.astype(float)
count_df_raw_agg.loc[:, mask_average_calc] = np.nan
count_df_norm_agg = normalize_df_to_rpm(count_df_raw_agg, along_axis=0, pseudocount_initial=0, pseudocount_after=0.5)

In [ ]:
from utils.plotting_utils import test_pairs, annotate_pair_results

counts_per_cell = count_df_raw_agg.sum(axis=0) / cell_count_agg
counts_per_cell = counts_per_cell.reset_index()
counts_per_cell = counts_per_cell.rename({0: "counts"}, axis=1)
counts_per_cell['counts'] = np.log10(counts_per_cell['counts'])

mean_UMIs_per_cell = 10**counts_per_cell.groupby("cell_type")['counts'].mean()
mean_UMIs_per_cell = mean_UMIs_per_cell/mean_UMIs_per_cell.median()

plt.figure(figsize=cm_to_inches((7, 4)))
order = ['B cell', 'Mono/cDC', 'NK cell', 'T cell']

sns.boxplot(
    data=counts_per_cell,
    x="cell_type",
    y="counts",
    flierprops={
        "marker": "o",
        "markersize": 3,
        "markerfacecolor": "black",
        "markeredgecolor": "black",
    },
    # s=2,
    # color='black',
    order=order
)
plt.xlabel('')
plt.ylabel('log10(mean UMIs per cell)')

pairs = [
    ("B cell", "Mono/cDC"),
    ("B cell", "NK cell"),
    ("Mono/cDC", "NK cell"),
    ("Mono/cDC", "T cell"),
    ("NK cell", "T cell")
]

test_results = test_pairs(counts_per_cell, x='cell_type', y='counts', pairs=pairs, alternative='two-sided')
annotate_pair_results(plt.gca(), results=test_results, order=order, loc='outside', margin=0.1, gap=0.1)

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "UMIs_per_cell.svg"))
plt.show()

## filtering

In [ ]:
from utils.processing_utils import get_filter_by_counts

mask_average_calc = get_filter_by_counts(all_counts_by_condition, cell_type_colors=CELL_TYPE_COLORS)

In [ ]:
# subset to only conditions present in the data (e.g. some cytokines are missing for some cell types)
mask_average_calc = mask_average_calc.reindex(sum_gene_count_df_raw_by_donor.columns)

# filter these out by setting to nan (these will be ignored in the average calculation)
sum_gene_count_df_raw_by_donor = sum_gene_count_df_raw_by_donor.astype(float)
cols_to_nan = sum_gene_count_df_raw_by_donor.columns[mask_average_calc.fillna(False).to_numpy()]
sum_gene_count_df_raw_by_donor.loc[:, cols_to_nan] = np.nan

# normalize to rpm
sum_gene_count_df_norm_by_donor = normalize_df_to_rpm(sum_gene_count_df_raw_by_donor, along_axis=0)

In [ ]:
# average across donors
average_sum_gene_count_df = sum_gene_count_df_raw_by_donor.T.groupby(level=[0, 1]).mean().T.loc[:, ALL_CELL_TYPES]
average_norm_sum_gene_count_df = sum_gene_count_df_norm_by_donor.T.groupby(level=[0, 1]).mean().T.loc[:, ALL_CELL_TYPES]

In [ ]:
if alt_flag:
    sum_gene_count_name = "pbulk_average_sum_gene_count_df_alt.parquet"
    norm_sum_gene_count_name = "pbulk_average_norm_sum_gene_count_df_alt.parquet"
    norm_sum_gene_count_name_std = "pbulk_std_norm_sum_gene_count_df_alt.parquet"
else:
    sum_gene_count_name = "pbulk_average_sum_gene_count_df.parquet"
    norm_sum_gene_count_name = "pbulk_average_norm_sum_gene_count_df.parquet"
    norm_sum_gene_count_name_std = "pbulk_std_norm_sum_gene_count_df.parquet"

if not os.path.exists(os.path.join(output_tables_folder, sum_gene_count_name)):
    average_sum_gene_count_df.to_parquet(os.path.join(output_tables_folder, sum_gene_count_name))
    average_norm_sum_gene_count_df.to_parquet(os.path.join(output_tables_folder, norm_sum_gene_count_name))

In [ ]:
# separate out the control/PBS data
control_norm_gene_count_df = average_norm_sum_gene_count_df.loc[:, (slice(None), "PBS")]
control_norm_gene_count_df.columns = control_norm_gene_count_df.columns.droplevel(1) # drop the "PBS" from the index

control_raw_gene_count_df = average_sum_gene_count_df.loc[:, (slice(None), "PBS")]
control_raw_gene_count_df.columns = control_raw_gene_count_df.columns.droplevel(1) # drop the "PBS" from the index

control_norm_gene_count_df_by_donor = sum_gene_count_df_norm_by_donor.loc[:, (slice(None), "PBS", slice(None))]
control_norm_gene_count_df_by_donor.columns = control_norm_gene_count_df_by_donor.columns.droplevel(1) # drop the "PBS" from the index

control_raw_gene_count_df_by_donor = sum_gene_count_df_raw_by_donor.loc[:, (slice(None), "PBS", slice(None))]
control_raw_gene_count_df_by_donor.columns = control_raw_gene_count_df_by_donor.columns.droplevel(1) # drop the "PBS" from the index

if not os.path.exists(os.path.join(output_tables_folder, f"control_norm_gene_count_df{'_alt' if alt_flag else ''}.parquet")):
    control_norm_gene_count_df.to_parquet(os.path.join(output_tables_folder, f"control_norm_gene_count_df{'_alt' if alt_flag else ''}.parquet"))
    control_raw_gene_count_df.to_parquet(os.path.join(output_tables_folder, f"control_raw_gene_count_df{'_alt' if alt_flag else ''}.parquet"))
    control_raw_gene_count_df_by_donor.to_parquet(os.path.join(output_tables_folder, f"control_raw_gene_count_df_by_donor{'_alt' if alt_flag else ''}.parquet"))
    control_norm_gene_count_df_by_donor.to_parquet(os.path.join(output_tables_folder, f"control_norm_gene_count_df_by_donor{'_alt' if alt_flag else ''}.parquet"))

In [ ]:
questionable_genes = set(pd.read_csv(os.path.join(input_data_folder, "questionable_genes.csv"))['gene'])
unfiltered_genes = average_norm_sum_gene_count_df.index

inclusive_filters_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    df = average_sum_gene_count_df[cell_type]
    df = df[df.max(axis=1) >= 20]
    df = df[~df.index.isin(questionable_genes)]
    print(cell_type, len(df))
    inclusive_filters_by_cell_type[cell_type] = df.index

In [ ]:
# Calculate PBMC-wide count and log2fc values
from utils.processing_utils import get_raw_log2fc_values

# calculate total expression
counts_all_agg_raw = sum_gene_count_df_raw_by_donor.T.groupby(level=[1,2]).sum().T
counts_all_agg_raw.columns = pd.MultiIndex.from_tuples(
    [('all', c[0], c[1]) for c in counts_all_agg_raw.columns]
)
counts_all_agg_raw.columns.names = ['cell_type', 'cytokine', 'donor']
counts_all_agg_norm = normalize_df_to_rpm(counts_all_agg_raw, along_axis=0, pseudocount_initial=0)

# make sure to use the correct reference values for each
_, log2fcs_main, _ = get_raw_log2fc_values(
    counts_all_agg_norm.loc[:, ~counts_all_agg_norm.columns.get_level_values('cytokine').isin(full_review_list)],
    counts_all_agg_raw.loc[:, ~counts_all_agg_raw.columns.get_level_values('cytokine').isin(full_review_list)],
    require_both_counts=False
)
_, log2fcs_review, _ = get_raw_log2fc_values(
    counts_all_agg_norm.loc[:, counts_all_agg_norm.columns.get_level_values('cytokine').isin(full_review_list)],
    counts_all_agg_raw.loc[:, counts_all_agg_raw.columns.get_level_values('cytokine').isin(full_review_list)],
    baseline_label='PBS_review',
    require_both_counts=False
)
log2fcs_all_agg = pd.concat([log2fcs_main, log2fcs_review], axis=1)

# get rid of the 'all' helper
counts_all_agg_norm = counts_all_agg_norm['all']
counts_all_agg_raw = counts_all_agg_raw['all']
log2fcs_all_agg = log2fcs_all_agg['all']

p = os.path.join(output_tables_folder, 'pbmc_weighted_log2fc.parquet')
if not os.path.exists(p):
    log2fcs_all_agg.to_parquet(p)

In [ ]:
# shorten donor names and do some bookkeeping
cols = list(sum_gene_count_df_norm_by_donor.columns.levels[2])
new_cols = [c.replace("Donor", "D") if isinstance(c, str) else c for c in cols]
sum_gene_count_df_norm_by_donor.columns = sum_gene_count_df_norm_by_donor.columns.set_levels(new_cols, level=2)

cols = list(sum_gene_count_df_raw_by_donor.columns.levels[2])
new_cols = [c.replace("Donor", "D") if isinstance(c, str) else c for c in cols]
sum_gene_count_df_raw_by_donor.columns = sum_gene_count_df_raw_by_donor.columns.set_levels(new_cols, level=2)

idx = all_counts_by_condition.index
new_idx = idx.set_levels([idx.levels[0], idx.levels[1],
                          [c.replace("Donor", "D") if isinstance(c, str) else c for c in idx.levels[2]]], level=[0, 1, 2])
all_counts_by_condition.index = new_idx

idx = all_counts_by_condition_unfiltered.index
new_idx = idx.set_levels([idx.levels[0], idx.levels[1],
                          [c.replace("Donor", "D") if isinstance(c, str) else c for c in idx.levels[2]]], level=[0, 1, 2])
all_counts_by_condition_unfiltered.index = new_idx

In [ ]:
counts_per_cell_detailed = sum_gene_count_df_raw_by_donor.sum(axis=0).div(all_counts_by_condition.reindex(sum_gene_count_df_raw_by_donor.columns)).replace([np.inf, -np.inf], np.nan)
counts_per_cell_detailed = counts_per_cell_detailed.where(counts_per_cell_detailed > 0)
mean_UMIs_per_cell_detailed = 10**np.log10(counts_per_cell_detailed).groupby(level="cell_type").mean()
mean_UMIs_per_cell_detailed /= mean_UMIs_per_cell_detailed.median()

In [ ]:
# filter per donor to get only expressed genes
mask = sum_gene_count_df_raw_by_donor < 10
sum_gene_count_df_norm_by_donor_filter = sum_gene_count_df_norm_by_donor.copy()
sum_gene_count_df_norm_by_donor_filter[mask] = np.nan

sum_gene_count_df_norm_by_donor_pbs = sum_gene_count_df_norm_by_donor_filter.loc[:, (slice(None), "PBS", slice(None))].droplevel(1, axis=1)
sum_gene_count_df_norm_by_donor_pbs = sum_gene_count_df_norm_by_donor_pbs.dropna(axis=1, how='all')

In [ ]:
from utils.processing_utils import get_raw_log2fc_values
output_path_per_donor = os.path.join(output_tables_folder, 'raw_log2fc_values_by_donor.parquet')
output_path_mean = os.path.join(output_tables_folder, 'raw_log2fc_values.parquet')
output_path_std = os.path.join(output_tables_folder, 'raw_log2fc_values_std.parquet')

if not os.path.exists(output_path_per_donor) | not os.path.exists(output_path_mean) | not os.path.exists(output_path_std):
    raw_log2fc_values_by_donor, raw_log2fc_values, raw_log2fc_values_std = get_raw_log2fc_values(
        sum_gene_count_df_norm_by_donor.loc[:, ~sum_gene_count_df_norm_by_donor.columns.get_level_values(1).isin(full_review_list)],
        sum_gene_count_df_raw_by_donor.loc[:, ~sum_gene_count_df_raw_by_donor.columns.get_level_values(1).isin(full_review_list)],
        require_both_counts=False
    )

    rlfc_donor_review, rlfc_review, rlfc_review_std = get_raw_log2fc_values(
        sum_gene_count_df_norm_by_donor.loc[:, sum_gene_count_df_norm_by_donor.columns.get_level_values(1).isin(full_review_list)],
        sum_gene_count_df_raw_by_donor.loc[:, sum_gene_count_df_raw_by_donor.columns.get_level_values(1).isin(full_review_list)],
        baseline_label='PBS_review',
        require_both_counts=False
    )

    raw_log2fc_values_by_donor = pd.concat([raw_log2fc_values_by_donor, rlfc_donor_review], axis=1)
    raw_log2fc_values = pd.concat([raw_log2fc_values, rlfc_review], axis=1)
    raw_log2fc_values_std = pd.concat([raw_log2fc_values_std, rlfc_review_std], axis=1)

    raw_log2fc_values_by_donor.to_parquet(output_path_per_donor)
    raw_log2fc_values.to_parquet(output_path_mean)
    raw_log2fc_values_std.to_parquet(output_path_std)
else:
    raw_log2fc_values_by_donor = pd.read_parquet(output_path_per_donor)
    raw_log2fc_values = pd.read_parquet(output_path_mean)
    raw_log2fc_values_std = pd.read_parquet(output_path_std)

### Per-well PBS pseudobulk data

In [ ]:
sum_gene_count_df_raw_by_well_pbs, all_counts_by_condition_by_well_pbs = load_pseudobulks(os.path.join(input_data_folder, 'pseudobulks'), relevant_cell_types, relevant_cell_type_conversion, cytokines=["PBS"], pool_wells=False)
sum_gene_count_df_raw_by_well_pbs = sum_gene_count_df_raw_by_well_pbs.xs("PBS", level="cytokine", axis=1)
all_counts_by_condition_by_well_pbs = all_counts_by_condition_by_well_pbs.xs("PBS", level="cytokine")

In [ ]:
# pseudobulking across donors could be skewed if there are very few cells in a given condition
mask_low_count = all_counts_by_condition_by_well_pbs < 10
very_low_count = all_counts_by_condition_by_well_pbs < 5
mask_low_fraction = all_counts_by_condition_by_well_pbs < 0.2 * all_counts_by_condition_by_well_pbs.groupby(level=0).transform("median")
mask_average_calc = (mask_low_count & mask_low_fraction) | very_low_count

g = mask_average_calc[ALL_CELL_TYPES].groupby(level=[0,1]).sum()
plt.figure(figsize=cm_to_inches((8, 4)))
bottom = np.zeros(len(g.index.get_level_values(1).unique()))
for cell_type in g.index.get_level_values(0).unique():
    donor_vals = g.xs(cell_type, level=0)
    plt.bar(donor_vals.index, donor_vals, bottom=bottom, color=CELL_TYPE_COLORS[cell_type], label=cell_type)
    bottom += donor_vals
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2)
plt.xticks(rotation=90)
plt.xlabel("Donor")
plt.ylabel("Low cell counts")
plt.show()

In [ ]:
# subset to only conditions present in the data (e.g. some cytokines are missing for some cell types)
mask_average_calc = mask_average_calc[mask_average_calc.index.isin(sum_gene_count_df_raw_by_well_pbs.columns)]

# filter these out by setting to nan (these will be ignored in the average calculation)
sum_gene_count_df_raw_by_well_pbs.loc[:, mask_average_calc] = np.nan

# normalize to rpm
sum_gene_count_df_norm_by_well_pbs = normalize_df_to_rpm(sum_gene_count_df_raw_by_well_pbs, along_axis=0)

# filter per well to get only expressed genes
mask = sum_gene_count_df_raw_by_well_pbs < 10
sum_gene_count_df_norm_by_well_pbs_filter = sum_gene_count_df_norm_by_well_pbs.copy()
sum_gene_count_df_norm_by_well_pbs_filter[mask] = np.nan

cols = list(sum_gene_count_df_norm_by_well_pbs_filter.columns.levels[1])
new_cols = [c.replace("Donor", "D") if isinstance(c, str) else c for c in cols]
sum_gene_count_df_norm_by_well_pbs_filter.columns = sum_gene_count_df_norm_by_well_pbs_filter.columns.set_levels(new_cols, level=1)

## 1.2 - Load DEGs

In [ ]:
deg_processed_path = os.path.join(input_data_folder, 'edger_results_main.parquet')
if not os.path.exists(deg_processed_path):
    deg_input_path = "/Volumes/Extreme Pro/parse_data/fig_1"

    edger_results = pd.read_csv(os.path.join(deg_input_path, "05_degs", "exp_1_DEGs_all_unfiltered.csv"))
    keys = ['experiment', 'cell_type', 'cytokine', 'variable']
    support_columns = ['n_wells_tested', 'n_wells_de', 'n_wells_de_up', 'n_wells_de_down', 'wells_de', 'de_in_at_least_4_wells', 'de_in_at_least_4_wells_same_direction']
    pbs_well_results = pd.read_csv(os.path.join(deg_input_path, "06_degs_per_pbs_well", "exp_1_DEGs_aggregated_across_pbs_wells.csv"), usecols=keys + support_columns)

    edger_results = edger_results.merge(pbs_well_results, on=keys, how='left', validate='one_to_one')
    edger_results = edger_results.rename(columns={'variable': 'gene', 'n_wells_de': 'pbs_well'})

    # map cell types
    mapped_cell_types = edger_results['celltype'].map(relevant_cell_type_conversion)
    unmapped = edger_results.loc[mapped_cell_types.isna(), 'celltype'].value_counts(dropna=False)
    if not unmapped.empty:
        print("Unmapped cell types (number of rows):")
        print(unmapped.to_string())
        raise ValueError("Update relevant_cell_type_conversion before continuing.")
    edger_results['cell_type'] = mapped_cell_types
    edger_results.to_parquet(deg_processed_path)
else:
    edger_results = pd.read_parquet(deg_processed_path)

In [ ]:
# set p-values for genes that fail the well consistency test to 1
# don't do this for the log_fc, creates confusing results
mask = ~edger_results['de_in_at_least_4_wells_same_direction'].fillna(False)
edger_results.loc[mask, "p_value"] = 1
edger_results.loc[mask, "adj_p_value"] = 1

In [ ]:
deg_processed_path = os.path.join(input_data_folder_review, 'edger_results_review.parquet')
if not os.path.exists(deg_processed_path):
    deg_input_path = "/Volumes/Extreme Pro/parse_data/fig_1"
    edger_review = pd.read_csv(os.path.join(deg_input_path, "05_degs", "exp_2_DEGs_all_unfiltered.csv"))
    edger_review = edger_review.rename(columns={'variable': 'gene'})

    # Map cell types and report failures
    mapped_cell_types = edger_review['celltype'].map(relevant_cell_type_conversion)
    unmapped = edger_review.loc[mapped_cell_types.isna(), 'celltype'].value_counts(dropna=False)
    if not unmapped.empty:
        print("Unmapped review cell types (number of rows):")
        print(unmapped.to_string())
        raise ValueError("Update relevant_cell_type_conversion before continuing.")
    edger_review['cell_type'] = mapped_cell_types

    # Separate review results from the original main-screen comparisons
    edger_review['cytokine'] = edger_review['cytokine'] + "_review"
    edger_review['pbs_well'] = np.nan

    edger_review.to_parquet(deg_processed_path)
else:
    edger_review = pd.read_parquet(deg_processed_path)

In [ ]:
edger_results = pd.concat([edger_results, edger_review], ignore_index=True)

In [ ]:
# remove questionable genes from edger results
mask = edger_results['gene'].isin(questionable_genes)
edger_results.loc[mask, ["log_fc", "p_value", "adj_p_value"]] = np.nan

In [ ]:
# not a valid way to calculate errors for a multi-factor design, but can approximate
# edger_results['SE_logFC'] = edger_results['log_fc'].abs() / np.sqrt(edger_results['F']+0.01)
# edger_results['adjusted_fc'] = edger_results.apply(shrink_FCs, axis=1)

# plt.figure(figsize=(4, 3))
# plt.hist(edger_results["adjusted_fc"], bins = np.arange(-4, 4, 0.1))
# plt.axvline(x=0.5, color='red', linestyle='--')
# plt.axvline(x=-0.5, color='red', linestyle='--')
# plt.show()

# len(edger_results[edger_results["log_fc"].abs() > 0.5]) / len(edger_results)

In [ ]:
from utils.processing_utils import create_edger_val_df
edger_df_log2FC = create_edger_val_df(edger_results, "log_fc")

cell_types = ['CD4 T cell', 'CD14 Mono']
cytokines = ['IFN-beta', 'IL-15', 'TNF-alpha', 'IL-32-beta']
fig, axes = plt.subplots(2, 4, figsize=cm_to_inches((12, 8)), sharex = True, sharey = True)
axes = axes.flatten()

i = 0
for cell_type in cell_types:
    for cytokine in cytokines:
        ax = axes[i]
        i+=1

        x = edger_df_log2FC.loc[:, (cell_type, cytokine)].dropna()
        y = edger_df_log2FC.loc[:, (cell_type, cytokine+"_review")].dropna()
        x, y = get_common_index(x ,y)
        ax.scatter(x, y, s=3, alpha=0.5, edgecolor="none")
        r = np.corrcoef(x,y)[0, 1]
        ax.set_title(f"{cell_type}, {introduce_greek_letters(cytokine)}\nr={r:.2f}")
        ax.set_xlabel('log2fc_original')
        ax.set_ylabel('log2fc_review')

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, '1_original_review_comp.png'), dpi=300)

### Decide what to retain between the original and the review screen

In [ ]:
exchange_for_review = ['C3a', 'TGF-beta1', 'IL-18', 'TNF-alpha']
edger_results = edger_results[~edger_results['cytokine'].isin(exchange_for_review)]

# rename
rename_dict = {c+"_review": c for c in exchange_for_review}
edger_results['cytokine'] = edger_results['cytokine'].apply(lambda x: rename_dict.get(x, x))

# throw out others
edger_results = edger_results[~edger_results['cytokine'].str.contains("_review")]

In [ ]:
edger_df_padj = create_edger_val_df(edger_results, "adj_p_value")
edger_df_log2CPM = create_edger_val_df(edger_results, "logCPM")
edger_df_log2FC = create_edger_val_df(edger_results, "log_fc")

In [ ]:
if not os.path.exists(os.path.join(output_tables_folder, "EdgeR_Log2FC.parquet")):
    edger_df_log2FC.to_parquet(os.path.join(output_tables_folder, "EdgeR_Log2FC.parquet"))
if not os.path.exists(os.path.join(output_tables_folder, "EdgeR_Padj.parquet")):
    edger_df_padj.to_parquet(os.path.join(output_tables_folder, "EdgeR_Padj.parquet"))

### Sanity check results

In [ ]:
cell_types = ['CD4 T cell', 'CD14 Mono']
cytokines = ['IFN-beta', 'IL-15', 'TNF-alpha', 'TGF-beta1']
fig, axes = plt.subplots(2, 4, figsize=cm_to_inches((12, 8)), sharex = True, sharey = True)
axes = axes.flatten()

i = 0
for cell_type in cell_types:
    df = average_sum_gene_count_df[cell_type]
    for cytokine in cytokines:
        ax = axes[i]
        i+=1

        x = edger_df_log2FC.loc[:, (cell_type, cytokine)].dropna()
        y = raw_log2fc_values.loc[:, (cell_type, cytokine)].dropna()
        x, y = get_common_index(x ,y)

        ax.scatter(x, y, s=3, alpha=0.5, edgecolor="none")
        ax.plot([-6, 6], [-6, 6], ls='--', lw=0.5, color='black')
        r = np.corrcoef(x,y)[0, 1]
        ax.set_title(f"{cell_type}, {introduce_greek_letters(cytokine)}\nr={r:.2f}")
        ax.set_xlabel('log2fc_edger')
        ax.set_ylabel('log2fc_raw')

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, '1_raw_edger_comp.png'), dpi=300)
plt.show()

## 1.3 - Get gene filters

In [ ]:
from utils.processing_utils import get_protein_coding_genes, get_sex_gene_list
protein_coding_genes = get_protein_coding_genes(annotation_folder="../../annotations")
sex_gene_list = get_sex_gene_list(input_data_folder)

## 1.4 - Overview of significance and fold change distribution

None of these plots are particularly interesting, so they're not in the actual publication

In [ ]:
edger_df_log2FC_filtered = edger_df_log2FC[edger_df_padj < 0.05]

positive_values = (edger_df_log2FC_filtered > 0.5).sum(axis=0)
negative_values = (edger_df_log2FC_filtered < -0.5).sum(axis=0)
total_values = positive_values + negative_values

plt.figure(figsize=cm_to_inches((5, 3)))
plt.hist(edger_df_log2FC_filtered.values.flatten(), bins=np.arange(-3, 3, 0.05))

# Calculate total positive and negative values
total_positive = np.sum(positive_values)
total_negative = np.sum(negative_values)

# Add text annotations for counts
plt.text(0.05, 0.95, f"log2FC < -0.5:\n{total_negative}", 
         transform=plt.gca().transAxes, 
         verticalalignment='top', 
         horizontalalignment='left',
         fontsize=6)

plt.text(0.95, 0.95, f"log2FC > 0.5:\n{total_positive}", 
         transform=plt.gca().transAxes, 
         verticalalignment='top', 
         horizontalalignment='right',
         fontsize=6)

plt.xlabel("Log2 Fold Change")
plt.ylabel("Count")
plt.title("Distribution of significant log2 fold changes")

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"1.4_edger_df_log2FC_filtered_distribution.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
edger_df_log2FC_filtered = edger_df_log2FC[edger_df_padj < 0.05]

# Create subplots for each cell type
if not alt_flag:
    fig, axes = plt.subplots(3, 5, figsize=cm_to_inches((12, 7)))
else:
    fig, axes = plt.subplots(4, 5, figsize=cm_to_inches((12, 7)))
fig.tight_layout(pad=2.0)

axes = axes.flatten()
for i, cell_type in enumerate(ALL_CELL_TYPES):
    # Filter data for this cell type
    cell_type_data = edger_df_log2FC_filtered.xs(cell_type, level=0, axis=1)
    
    # Calculate positive and negative values for this cell type
    positive_values_cell = (cell_type_data > 0.5).sum(axis=0)
    negative_values_cell = (cell_type_data < -0.5).sum(axis=0)
    
    # Calculate totals
    total_positive_cell = np.sum(positive_values_cell)
    total_negative_cell = np.sum(negative_values_cell)
    
    # Plot histogram
    n, bins, patches = axes[i].hist(cell_type_data.values.flatten(), bins=np.arange(-3, 3, 0.05), density=True, color=CELL_TYPE_COLORS[cell_type])
    
    # Add labels
    axes[i].set_xlabel(f"log$_{2}$FC")
    axes[i].set_ylabel("density")
    axes[i].set_ylim(0, max(n) * 1.25)
    # Add title in the upper left corner
    axes[i].text(0.05, 0.95, f"{cell_type}", transform=axes[i].transAxes, 
                 va='top', ha='left', fontsize=7)
    axes[i].label_outer()
    
    # Set tick length to 0 for inner plots
    axes[i].tick_params(length=0)
    
    # Remove y ticks
    axes[i].set_yticks([])
    
# Delete unused plots
for i in range(len(ALL_CELL_TYPES), len(axes)):
    fig.delaxes(axes[i])
    
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"1.4_edger_df_log2FC_filtered_distribution_by_cell_type.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
positive_values = positive_values[total_values > 10]
negative_values = negative_values[total_values > 10]
ratio = positive_values / (positive_values + negative_values)
ratio_mean = ratio.groupby(level=1).mean()
ratio_std = ratio.groupby(level=1).std()
ratio_mean = ratio_mean.sort_values(ascending=False)
ratio_std = ratio_std.loc[ratio_mean.index]

# Create a bar plot of the ratio of positive to total DEGs for each cytokine with error bars
plt.figure(figsize=cm_to_inches((18, 4)))
x_pos = range(len(ratio_mean))
bars = plt.bar(x_pos, ratio_mean.values, yerr=ratio_std.values, capsize=2, error_kw={'elinewidth': 1})
# Apply Greek letters to x-tick labels
x_labels = [introduce_greek_letters(lbl) for lbl in ratio_mean.index]
plt.xticks(x_pos, x_labels, rotation=90, fontsize=5.5)
plt.yticks(fontsize=5.5)
plt.ylabel(r'mean($\frac{\mathrm{DEG}(log2FC>0.5)}{\mathrm{DEG}(log2FC>0.5) + \mathrm{DEG}(log2FC<-0.5)}$)')

plt.ylim(0, 1)
plt.xlim(-1, len(ratio_mean))  # Reduce space on left and right of x-axis
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"1.4_ratio_of_positive_to_total_DEGs.{format}"), bbox_inches="tight", dpi=300)
plt.show()

## 1.5 - Basic dataset inspection (check top up- or downregulated genes)

In [ ]:
edger_df_log2FC_protein = edger_df_log2FC[edger_df_log2FC.index.isin(protein_coding_genes)]

In [ ]:
cell_type = "CD4 T cell"
cytokine = "IL-32-beta"

top_up = edger_df_log2FC_protein[cell_type, cytokine].dropna().sort_values(ascending=False).head(15)
top_down = edger_df_log2FC_protein[cell_type, cytokine].dropna().sort_values(ascending=True).head(15)

In [ ]:
top_down

In [ ]:
del edger_df_log2FC_protein

# 2 - Summarize results

In [ ]:
plot_folder = os.path.join(base_plot_folder, "2_result_summary")
os.makedirs(plot_folder, exist_ok=True)

## 2.1 - Receptor expression

In [ ]:
def get_complex_expression(average_control_expression, cytokine, cytokine_info, complex_column, verbose=False, insert_minimal=True):
    """Insert minimal is dangerous when using noisy input data (e.g., per donor HSPC)."""
    complex_genes_all = str(cytokine_info.loc[cytokine, complex_column])
    if pd.isna(complex_genes_all) or complex_genes_all == "nan" or complex_genes_all.strip() == "":
        if verbose:
            print(f"No complex genes found for {cytokine}")
        return None
    
    all_complex_expression_vals = []
    complex_variants = complex_genes_all.split(";")
    for complex_variant in complex_variants:
        complex_genes = complex_variant.split(",")
        complex_genes = [gene.strip().replace(" ", "") for gene in complex_genes]
        all_present = all([gene in average_control_expression.index for gene in complex_genes])
        if not all_present:
            # skip if any of the complex genes are missing to begin with
            if verbose:
                print(f"Not all complex genes found for {cytokine} variant {complex_variant}:" + 
                      f" missing {[gene for gene in complex_genes if gene not in average_control_expression.index]}")
            continue
        # all subunits must be expressed to trigger a response    
        complex_expression = average_control_expression.loc[complex_genes, :].min(axis=0, skipna=False) 
        all_complex_expression_vals.append(complex_expression)
    if len(all_complex_expression_vals) > 0: # any of the alternative complexes can trigger a response
        complex_expression = pd.concat(all_complex_expression_vals, axis=1).max(axis=1, skipna=True)
    else:
        if insert_minimal:
            complex_expression = pd.Series(1, index=average_control_expression.columns)
        else:
            complex_expression = None
    return complex_expression

receptor_expression_by_cytokine = {}
for cytokine in CYTOKINE_ORDER:
    receptor_expression = get_complex_expression(control_norm_gene_count_df, cytokine, cytokine_info_parse, 'receptor gene', verbose=True)
    if receptor_expression is not None:
        receptor_expression_by_cytokine[cytokine] = receptor_expression
receptor_expression_by_cytokine = pd.DataFrame(receptor_expression_by_cytokine).T

In [ ]:
# Receptor expression by donor, if needed
receptor_expression_by_cytokine_and_donor = {}
for cytokine in CYTOKINE_ORDER:
    receptor_by_donor = {}
    for donor in sum_gene_count_df_norm_by_donor_pbs.columns.levels[1]:
        expression = sum_gene_count_df_norm_by_donor_pbs.loc[:, (slice(None), donor)].droplevel(1, axis=1)
        receptor_expression = get_complex_expression(
            expression,
            cytokine,
            cytokine_info,
            'receptor gene',
            verbose=False,
            insert_minimal=False # we average over donors later, so this can be misleading
            )
        if receptor_expression is not None:
            receptor_by_donor[donor] = receptor_expression
    receptor_expression_by_cytokine_and_donor[cytokine] = pd.DataFrame(receptor_by_donor)
    receptor_expression_by_cytokine_and_donor[cytokine] = receptor_expression_by_cytokine_and_donor[cytokine].reindex(
        index=ALL_CELL_TYPES, columns=sum_gene_count_df_norm_by_donor_pbs.columns.levels[1])
    
# Convert the nested dictionary structure to a proper DataFrame
receptor_expression_by_cytokine_and_donor_flat = {}
for cytokine, df in receptor_expression_by_cytokine_and_donor.items():
    for donor in df.columns:
        for cell_type in df.index:
            receptor_expression_by_cytokine_and_donor_flat[(cytokine, donor, cell_type)] = df.loc[cell_type, donor]

# Create MultiIndex DataFrame from the flattened dictionary
multi_index = pd.MultiIndex.from_tuples(list(receptor_expression_by_cytokine_and_donor_flat.keys()), 
                                       names=['cytokine', 'donor', 'cell_type'])
receptor_expression_series = pd.Series(list(receptor_expression_by_cytokine_and_donor_flat.values()), 
                                     index=multi_index)
receptor_expression_by_cytokine_and_donor = receptor_expression_series.unstack(['cytokine', 'donor'])
receptor_expression_by_cytokine_and_donor.drop("IL-32-beta", level=0, inplace=True, axis=1)

In [ ]:
# check whether receptor expression is dominated by outliers
receptor_mean_by_cytokine = receptor_expression_by_cytokine_and_donor.T.groupby(level=0).mean().T.fillna(1)
receptor_median_by_cytokine = receptor_expression_by_cytokine_and_donor.T.groupby(level=0).median().T.fillna(1)
mean_over_median = receptor_mean_by_cytokine / receptor_median_by_cytokine
mean_over_median = mean_over_median.melt(var_name="cytokine", value_name="mean_over_median", ignore_index=False).reset_index()
mean_over_median.sort_values(by="mean_over_median", ascending=False, inplace=True)

# top outliers all seem biologically plausible and are probably real
mean_over_median

In [ ]:
receptor_expression_by_cytokine_raw = receptor_expression_by_cytokine.copy() # just in case we process it after?

# generate the relative receptor expression
receptor_expression_by_cytokine = receptor_expression_by_cytokine[ALL_CELL_TYPES]
receptor_expression_by_cytokine_relative = receptor_expression_by_cytokine.div(receptor_expression_by_cytokine.max(axis=1), axis=0)

# we should have found receptors for all cytokines except for IL-32-beta (which doesn't really have one)
for cytokine in CYTOKINE_ORDER:
    if not cytokine in receptor_expression_by_cytokine_relative.index:
        print(f"No receptor expression found for {cytokine}")

In [ ]:
if write_SI_tables_flag:
    df = receptor_expression_by_cytokine.copy()
    df = df.loc[[c for c in CYTOKINE_ORDER if c in df.index], :].reset_index()
    df['index'] = df['index'].apply(introduce_greek_letters)
    df = df.rename(columns={'index': 'Cytokine'})
    write_to_SI_table(df, "receptor_expression")

In [ ]:
from utils.parameters import ReceptorExpression
ReceptorParameters = ReceptorExpression()

plt.figure(figsize=cm_to_inches((5, 7)))
ax = plt.gca()
log_counts = np.log2(receptor_expression_by_cytokine.clip(1))

offset = 0.5
for i, cell_type in enumerate(ALL_CELL_TYPES[::-1]):
    data = log_counts[cell_type]
    counts, bins = np.histogram(data, bins=50, range=(0, 10))
    
    # Normalize counts to have similar heights
    if counts.max() > 0:
        counts = counts / counts.max() * 0.4
    
    # Plot histogram with vertical offset
    plt.bar(bins[:-1], counts, width=bins[1]-bins[0], bottom=i*offset, 
            alpha=0.7, label=cell_type if i == 0 else "", color=CELL_TYPE_COLORS[cell_type])
    
    # Add cell type label
    plt.text(-0.2, i*offset + 0.2, cell_type, ha='right', va='center', fontsize=6)
    
    # Add a horizontal line at the bottom of each histogram instead of a box
    plt.plot([0, 10], [i*offset, i*offset], color='black', linestyle='-', linewidth=0.5)

plt.plot([np.log2(ReceptorParameters.present_threshold), np.log2(ReceptorParameters.present_threshold)], [0, len(ALL_CELL_TYPES)*offset], color='red', linestyle='--', linewidth=1, label='Expression threshold')
plt.xlabel(r'log$_{2}$(receptor expression)')
plt.yticks([])
plt.xlim(0, 10)
plt.ylim(0, len(ALL_CELL_TYPES)*offset)

plt.tight_layout()
# # Save the figure
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2.1_receptor_expression_dist.{format}"), bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
fig = plt.figure(figsize=cm_to_inches((6, 20)))
ax = sns.heatmap(np.log10(receptor_expression_by_cytokine_relative.clip(0.01, 1)), 
                cmap="coolwarm", 
                square=True, 
                cbar_kws={"shrink": 0.4, "label": r"$\log_{10}$(relative receptor expression)"})

ax.set_yticks(np.arange(len(receptor_expression_by_cytokine_relative.index)) + 0.5)
ax.set_yticklabels([cytokine for cytokine in receptor_expression_by_cytokine_relative.index], fontsize=5.5)
apply_greek_labels_heatmap(ax)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis='y', length=0)
ax.tick_params(axis='x', rotation=90, labelsize=5.5, length=0)

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2.1_receptor_expression_by_cytokine_relative_log.{format}"), 
                bbox_inches="tight", 
                dpi=300)
plt.show()

In [ ]:
max_receptor_expression = receptor_expression_by_cytokine[ALL_CELL_TYPES].max(axis=1)

plt.figure(figsize=cm_to_inches((1, 22.85)))
max_receptor_expression = max_receptor_expression[::-1]
plt.barh(y=max_receptor_expression.index, width=np.log10(max_receptor_expression.values), color="black", height=0.7)
plt.ylim(-0.5, len(max_receptor_expression)-0.5)

# get the current figure and draw it (to get apply_greek_labels_heatmap to work for this plot)
fig = plt.gcf()
fig.canvas.draw()
apply_greek_labels_heatmap(plt.gca())

plt.xlim(0, 3)
plt.xlabel(r"$\log_{10}$(max. cpm)", fontsize=6)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2.1_maximum_receptor_expression_by_cytokine.{format}"), bbox_inches="tight", dpi=300)
plt.show()

## 2.2 -  Calculate the response_magnitude

### Euclidean distances

In [ ]:
from utils.response_magnitude import compute_response_euclidean_distance, winsorize_and_normalize_distances

all_raw_euclidean_distances_main = {}
for cell_type in ALL_CELL_TYPES:
    distances = {}
    
    # get the inclusive filter and average normalized log2 data
    all_inclusive_filter = inclusive_filters_by_cell_type[cell_type] # doesn't change a lot
    average_norm_filter = np.log2(average_norm_sum_gene_count_df.loc[all_inclusive_filter, :] + 1)
    
    for cytokine in [c for c in CYTOKINE_ORDER if not c in review_replace_list]:
        ref = 'PBS'
        if cytokine == ref:
            continue
        distance = compute_response_euclidean_distance(
            average_norm_filter, cell_type, cytokine, min_genes=1000, length_norm=True, print_min_gene_warning=True, control=ref
            )
        distances[cytokine] = distance
        
    distances_series = pd.Series(distances)
    all_raw_euclidean_distances_main[cell_type] = distances_series
all_raw_euclidean_distances_main = pd.DataFrame(all_raw_euclidean_distances_main)

In [ ]:
all_raw_euclidean_distances_review = {}
for cell_type in ALL_CELL_TYPES:
    distances = {}
    
    # get the inclusive filter and average normalized log2 data
    all_inclusive_filter = inclusive_filters_by_cell_type[cell_type] # doesn't change a lot
    average_norm_filter = np.log2(average_norm_sum_gene_count_df.loc[all_inclusive_filter, :] + 1)
    
    for cytokine in full_review_list:
        ref = 'PBS_review'
        if len(average_norm_filter[cell_type][cytokine].dropna()) < 1000:
            continue
        if cytokine == ref:
            continue
        distance = compute_response_euclidean_distance(
            average_norm_filter, cell_type, cytokine, min_genes=1000, length_norm=True, print_min_gene_warning=True, control=ref
            )
        distances[cytokine] = distance
        
    distances_series = pd.Series(distances)
    all_raw_euclidean_distances_review[cell_type] = distances_series
all_raw_euclidean_distances_review = pd.DataFrame(all_raw_euclidean_distances_review)

In [ ]:
def deming_regression(x, y, lambda_=1.0):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x_mean = x.mean()
    y_mean = y.mean()

    sxx = np.mean((x - x_mean)**2)
    syy = np.mean((y - y_mean)**2)
    sxy = np.mean((x - x_mean) * (y - y_mean))

    slope = (
        syy - lambda_ * sxx
        + np.sqrt((syy - lambda_ * sxx)**2 + 4 * lambda_ * sxy**2)
    ) / (2 * sxy)

    intercept = y_mean - slope * x_mean

    return slope, intercept

correction_factors = []
all_raw_euclidean_distances = {}
all_normalized_euclidean_distances = {}
for cell in ALL_CELL_TYPES:
    dists_main = all_raw_euclidean_distances_main[cell].copy()
    dists_review = all_raw_euclidean_distances_review[cell].copy()

    shared = [treat for treat in dists_main.index
            if treat+"_review" in dists_review.index
            and pd.notna(dists_main[treat]) and pd.notna(dists_review[treat+"_review"])]
    dists_main_filter = dists_main.loc[shared]
    dists_review_filter = dists_review.copy()
    dists_review_filter.index = [i.replace("_review", "") for i in dists_review.index]
    dists_review_filter = dists_review_filter.loc[shared]
    slope, intercept = deming_regression(dists_main_filter,dists_review_filter,lambda_=1)

    # plt.figure(figsize=cm_to_inches((4,3)))
    # plt.scatter(dists_main_filter, dists_review_filter)
    # min_val, max_val = dists_main_filter.min(), dists_main.max()
    
    # x = np.arange(0, 1, 0.01)
    # plt.plot(x, slope*x+intercept)
    # plt.plot([min_val, max_val], [min_val, max_val], color='black')
    # plt.title(f"slope: {slope:.2f}, intercept: {intercept:.2f}")
    # plt.show()

    print(f"Batch correcting {cell} with slope: {slope:.2f}, intercept: {intercept:.2f}")
    dists_review_corrected = (dists_review - intercept) / slope
    dists_review_corrected = dists_review_corrected.loc[review_replace_list]

    distances_both = pd.concat([dists_main, dists_review_corrected])
    assert(len(distances_both) == len(CYTOKINE_ORDER))
    distances_both = distances_both.loc[CYTOKINE_ORDER]
    all_raw_euclidean_distances[cell] = distances_both
    
    # winsorize and normalize
    distances_both = winsorize_and_normalize_distances(distances_both)
    all_normalized_euclidean_distances[cell] = distances_both

    correction_factors.append({
        "cell": cell,
        "slope": slope,
        "intercept": intercept
    })
all_raw_euclidean_distances = pd.DataFrame(all_raw_euclidean_distances)
all_normalized_euclidean_distances = pd.DataFrame(all_normalized_euclidean_distances)
correction_factors = pd.DataFrame(correction_factors)

### Compare with DEG number

In [ ]:
significant_results = pd.DataFrame(index=ALL_CELL_TYPES, columns=CYTOKINE_ORDER, data=0)
for cell_line in ALL_CELL_TYPES:
    padj_df = edger_df_padj[cell_line]
    log2fc_df = edger_df_log2FC[cell_line]
    num_significant = ((padj_df < 0.05) & (log2fc_df.abs() > 0.25)).sum(axis=0)
    significant_results.loc[cell_line, CYTOKINE_ORDER] = num_significant[CYTOKINE_ORDER].values

In [ ]:
plt.figure(figsize=cm_to_inches((5, 4)))
for cell_type in ALL_CELL_TYPES:
    rel_sig = winsorize_and_normalize_distances(significant_results.loc[cell_type] + 1)
    r = np.corrcoef(all_normalized_euclidean_distances.loc[CYTOKINE_ORDER, cell_type], rel_sig.values)
    plt.scatter(all_normalized_euclidean_distances.loc[CYTOKINE_ORDER, cell_type],
                rel_sig.values,
                label=f"{cell_type}: r={r[0, 1]:.2f}",
                color=CELL_TYPE_COLORS[cell_type],
                s = 2.5,
                alpha=0.75)
    
plt.xlabel("response magnitude\n(Euclidean)")
plt.ylabel("relative number of DEGs")
plt.tight_layout()
plt.legend(loc=[1,0], fontsize=6, ncol=1)
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_norm_dist_vs_DEGs.{format}"), bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=cm_to_inches((5, 4)))
cell_type = 'CD4 T cell'
rel_sig = winsorize_and_normalize_distances(
    significant_results.loc[cell_type] + 1
    )
r = np.corrcoef(
    all_normalized_euclidean_distances.loc[CYTOKINE_ORDER, cell_type],
            rel_sig.values)
plt.scatter(all_normalized_euclidean_distances.loc[CYTOKINE_ORDER, cell_type],
            rel_sig.values,
            label=f"{cell_type}: r={r[0, 1]:.2f}",
            color=CELL_TYPE_COLORS[cell_type],
            s = 2.5,
            alpha=0.75)
    
plt.xlabel("response magnitude\n(Euclidean)")
plt.ylabel("relative number of DEGs")
plt.tight_layout()
plt.legend(loc=[1,0], fontsize=6, ncol=1)
plt.show()

### Calculate Euclidean per donor first, then average (to see if that changes a lot)

Also, will use the results for the per-donor analysis

In [ ]:
from utils.response_magnitude import compute_response_euclidean_distance
path_main_distances_per_donor = os.path.join(output_tables_folder, "all_raw_euclidean_distances_by_donor_main.parquet")
path_review_distances_per_donor = os.path.join(output_tables_folder, "all_raw_euclidean_distances_by_donor_review.parquet")

if not os.path.exists(path_main_distances_per_donor):
    all_raw_euclidean_distances_by_donor_main = {}
    all_raw_euclidean_distances_by_donor_review = {}
    for donor in sum_gene_count_df_norm_by_donor_filter.columns.levels[2]:
        print(f"Processing donor {donor}")
        all_distances_raw = {}
        all_distances_raw_review = {}
        df = sum_gene_count_df_norm_by_donor_filter.loc[:, (slice(None), slice(None), donor)].copy().droplevel(2, axis=1)
        df_log = np.log2(df + 1)
        if np.isinf(df.values).any():
            df_log = np.log2(df + 1) # weirdness ...

        for cell_type in ALL_CELL_TYPES:
            # get the inclusive filter and average normalized log2 data
            all_inclusive_filter = inclusive_filters_by_cell_type[cell_type]
            average_norm_filter = df_log.loc[all_inclusive_filter, :]

            # main screen
            distances = {}
            for cytokine in [c for c in CYTOKINE_ORDER if not c in review_replace_list]:
                if cytokine in average_norm_filter[cell_type].columns:
                    ref = 'PBS'
                    if cytokine == ref:
                        continue

                    distance = compute_response_euclidean_distance(
                        average_norm_filter, cell_type, cytokine,
                        dropna=True, min_genes=1000, control=ref,
                        print_min_gene_warning=False, length_norm=True)
                    if np.isnan(distance):
                        continue
                    distances[cytokine] = distance
            all_distances_raw[cell_type] = pd.Series(distances)

            # review screen
            distances = {}
            for cytokine in full_review_list:
                if cytokine in average_norm_filter[cell_type].columns:
                    ref = 'PBS_review'
                    if cytokine == ref:
                        continue
                    if len(average_norm_filter[cell_type][cytokine].dropna()) < 1000:
                        continue
                    try:
                        distance = compute_response_euclidean_distance(
                            average_norm_filter, cell_type, cytokine,
                            dropna=True, min_genes=1000, control=ref,
                            print_min_gene_warning=False, length_norm=True)
                    except:
                        distance = np.nan
                    if np.isnan(distance):
                        continue
                    distances[cytokine] = distance
            all_distances_raw_review[cell_type] = pd.Series(distances)

        all_raw_euclidean_distances_by_donor_main[donor] = pd.DataFrame(all_distances_raw)
        all_raw_euclidean_distances_by_donor_review[donor] = pd.DataFrame(all_distances_raw_review)
        
    all_raw_euclidean_distances_by_donor_main = pd.concat(all_raw_euclidean_distances_by_donor_main, axis=1)
    all_raw_euclidean_distances_by_donor_review = pd.concat(all_raw_euclidean_distances_by_donor_review, axis=1)
    
    all_raw_euclidean_distances_by_donor_main.to_parquet(path_main_distances_per_donor)
    all_raw_euclidean_distances_by_donor_review.to_parquet(path_review_distances_per_donor)
else:
    all_raw_euclidean_distances_by_donor_main = pd.read_parquet(path_main_distances_per_donor)
    all_raw_euclidean_distances_by_donor_review = pd.read_parquet(path_review_distances_per_donor)

In [ ]:
donor_correction_factors = []
for cell in ALL_CELL_TYPES:
    all_pairs = []

    for donor in all_raw_euclidean_distances_by_donor_review.columns.get_level_values(0).unique():
        dists_main = all_raw_euclidean_distances_by_donor_main[donor][cell].copy().dropna()
        dists_review = all_raw_euclidean_distances_by_donor_review[donor][cell].copy().dropna()
        dists_review.index = [i.replace("_review", "") for i in dists_review.index]

        shared = dists_main.index.intersection(dists_review.index)
        if len(shared) == 0:
            continue

        pair_df = pd.DataFrame({
            "main": dists_main.loc[shared],
            "review": dists_review.loc[shared]
        })
        all_pairs.append(pair_df)

    all_pairs = pd.concat(all_pairs).dropna()
    slope, intercept = deming_regression(all_pairs["main"], all_pairs["review"], lambda_=1)

    donor_correction_factors.append({
        "cell": cell,
        "slope": slope,
        "intercept": intercept
    })

donor_correction_factors = pd.DataFrame(donor_correction_factors).set_index("cell")

In [ ]:
all_raw_euclidean_distances_by_donor = {}
all_normalized_euclidean_distances_by_donor = {}
for donor in all_raw_euclidean_distances_by_donor_review.columns.get_level_values(0).unique():
    for cell in ALL_CELL_TYPES:
        dists_main = all_raw_euclidean_distances_by_donor_main[donor][cell].copy()
        dists_review = all_raw_euclidean_distances_by_donor_review[donor][cell].copy()

        slope = donor_correction_factors.loc[cell, "slope"]
        intercept = donor_correction_factors.loc[cell, "intercept"]

        dists_review_corrected = (dists_review - intercept) / slope
        dists_review_corrected = dists_review_corrected.loc[review_replace_list]

        distances_both = pd.concat([dists_main, dists_review_corrected])
        distances_both = distances_both.loc[CYTOKINE_ORDER].dropna().astype(float)

        all_raw_euclidean_distances_by_donor[(cell, donor)] = distances_both
        if len(distances_both) >= 50:
            all_normalized_euclidean_distances_by_donor[(cell, donor)] = winsorize_and_normalize_distances(distances_both)
        else:  # too few cytokines for a meaningful per-donor normalization
            all_normalized_euclidean_distances_by_donor[(cell, donor)] = distances_both * np.nan
all_raw_euclidean_distances_by_donor = pd.DataFrame(all_raw_euclidean_distances_by_donor)
all_normalized_euclidean_distances_by_donor = pd.DataFrame(all_normalized_euclidean_distances_by_donor)

In [ ]:
euclidean_mean_raw_across_donors = all_raw_euclidean_distances_by_donor.T.groupby(level=0).mean().T
euclidean_mean_norm_across_donors = all_normalized_euclidean_distances_by_donor.T.groupby(level=0).mean().T

In [ ]:
cell_type = "HSPC"
x = all_raw_euclidean_distances[cell_type].dropna()
y = euclidean_mean_raw_across_donors[cell_type].dropna()
x,y = get_common_index(x,y)

plt.figure(figsize=cm_to_inches((6,  6)))
plt.scatter(x, y, s=5)
r = stats.pearsonr(x, y)
plt.xlabel("Euclidean distance (average)")
plt.ylabel("Euclidean distance (by donor)")
plt.title(f"{cell_type}\nPearson r={r.statistic:.2f}")
plt.show()

df = pd.DataFrame({
    "base": x,
    "across_donors": y
})
df['diff'] = df['base'] - df['across_donors']
df.sort_values(by='diff')

for cell_type in ALL_CELL_TYPES:
    x = all_raw_euclidean_distances[cell_type].dropna()
    y = euclidean_mean_raw_across_donors[cell_type].dropna()
    x,y = get_common_index(x,y)
    r = stats.pearsonr(x, y)
    print(f"{cell_type}: r={r.statistic:.2f}")

### Comparison with rms log2fc calc

In brief: Looks overall similar but is more vulnerable to noise; correlates worse with DEGs; use the original definition for the most part

In [ ]:
# from utils.response_magnitude import winsorize_and_normalize_distances

# rms_dist_raw_by_donor = np.sqrt(
#     raw_log2fc_values_by_donor.pow(2).mean(axis=0, skipna=True)
# ).dropna()

# rms_dist_normalized_by_donor = (
#     rms_dist_raw_by_donor
#     .groupby(level=["donor", "cell_type"])
#     .transform(winsorize_and_normalize_distances)
# )

# rms_dist_raw_mean = rms_dist_raw_by_donor.groupby(level=['cytokine', 'cell_type']).mean()
# rms_dist_norm_mean = rms_dist_raw_mean.groupby('cell_type').transform(winsorize_and_normalize_distances)

# rms_dist_raw_mean = rms_dist_raw_mean.reset_index().pivot(columns='cell_type', index='cytokine', values=0).loc[CYTOKINE_ORDER, ALL_CELL_TYPES]
# rms_dist_norm_mean = rms_dist_norm_mean.reset_index().pivot(columns='cell_type', index='cytokine', values=0).loc[CYTOKINE_ORDER, ALL_CELL_TYPES]

# # HSPCs are pretty much the worst case scenario
# cell_type = "CD8 T cell"
# plt.figure(figsize=cm_to_inches((6,  6)))
# plt.scatter(all_raw_euclidean_distances[cell_type], rms_dist_raw_mean[cell_type], s=5)
# r = stats.pearsonr(all_raw_euclidean_distances[cell_type], rms_dist_raw_mean[cell_type])
# plt.xlabel("Euclidean distance (average)")
# plt.ylabel("Euclidean distance (by donor)")
# plt.title(f"{cell_type}\nPearson r={r.statistic:.2f}")
# plt.show()

# cell_type = 'CD14 Mono'
# df = pd.DataFrame({
#     "base": all_normalized_euclidean_distances[cell_type],
#     "rms": rms_dist_norm_mean[cell_type]
# })
# df['diff'] = df['base'] - df['rms']
# df.sort_values(by='diff')

### 2.3 - Identify strong responses

In [ ]:
# doesn't matter whether normalized or raw distances are used here
all_normalized_distances_used = all_normalized_euclidean_distances[ALL_CELL_TYPES].copy()

In [ ]:
# Percentile-based (deprecated)
def convert_to_percentile_along_axis(df, axis, granularity=5):
    """Convert values to percentile ranks along specified axis with given granularity.
    
    Args:
        df: pandas DataFrame to convert
        axis: 0 for columns, 1 for rows 
        granularity: Size of percentile bins (e.g. 5 means bins of 0-5%, 5-10%, etc)
        
    Returns:
        DataFrame with values converted to percentile ranks
    """
    # Calculate percentile rank along axis
    percentile_df = df.rank(axis=axis, pct=True) * 100
    
    # Round to nearest granularity value
    percentile_df = (percentile_df / granularity).round() * granularity
    
    return percentile_df

all_normalized_distances_percentile = convert_to_percentile_along_axis(all_normalized_distances_used, axis=0, granularity=5)
# calculate the magntiude threshold for each cell type
magnitude_threshold_by_cell_type_percentile = {}
for cell_type in ALL_CELL_TYPES:
    # get the mean of the low percentile cytokines
    low_percentile = all_normalized_distances_percentile[cell_type] < 35
    low_percentile_cytokines = low_percentile.index[low_percentile]
    low_percentile_mean = all_normalized_distances_used[cell_type][low_percentile_cytokines].mean()
    
    # set the threshold to 3x the mean of the low percentile cytokines
    magnitude_threshold_by_cell_type_percentile[cell_type] = 3*low_percentile_mean
    
strong_impact_by_cell_type_percentile = {}
for cell_type in ALL_CELL_TYPES:
    strong_impact_by_cell_type_percentile[cell_type] = all_normalized_distances_used[cell_type] > magnitude_threshold_by_cell_type_percentile[cell_type]
strong_impact_by_cell_type_percentile = pd.DataFrame(strong_impact_by_cell_type_percentile, index=all_normalized_distances_used.index)

In [ ]:
# z-score based (use for final publication)
from scipy.stats import norm
z_95 = norm.ppf(0.95)

low_receptor_expression = (receptor_expression_by_cytokine < ReceptorParameters.present_threshold).sum(axis=1)
low_receptor_expression = low_receptor_expression[low_receptor_expression == len(ALL_CELL_TYPES)].index
print(f"Number of cytokines with low receptor expression across all cell types: {len(low_receptor_expression)}")
distance_z_scores_by_cell_type = {}

magnitude_threshold_by_cell_type_zscore = {}
for cell_type in ALL_CELL_TYPES:
    weak_distances = all_normalized_distances_used[cell_type].loc[low_receptor_expression]
    mean_weak = weak_distances.mean()
    std_weak = weak_distances.std()
    z_scores = (all_normalized_distances_used[cell_type] - mean_weak) / std_weak
    distance_z_scores_by_cell_type[cell_type] = z_scores
    magnitude_threshold_by_cell_type_zscore[cell_type] = z_95 * std_weak + mean_weak
    
distance_z_scores_by_cell_type = pd.DataFrame(distance_z_scores_by_cell_type)
strong_impact_by_cell_type_zscore = distance_z_scores_by_cell_type > z_95

### Compare the two measures

In [ ]:
both_active = strong_impact_by_cell_type_percentile & strong_impact_by_cell_type_zscore
only_percentile = strong_impact_by_cell_type_percentile & ~strong_impact_by_cell_type_zscore
only_zscore = ~strong_impact_by_cell_type_percentile & strong_impact_by_cell_type_zscore
neither_active = ~strong_impact_by_cell_type_percentile & ~strong_impact_by_cell_type_zscore

plt.figure(figsize=cm_to_inches((9, 4)))
bottom = np.zeros(len(ALL_CELL_TYPES))
for category, color in zip(["measures agree", "only percentile", "only zscore", "neither_active"], ["green", "orange", "blue", "grey"]):
    if category == "measures agree":
        counts = both_active.sum()
    elif category == "only percentile":
        counts = only_percentile.sum()
    elif category == "only zscore":
        counts = only_zscore.sum()
    else:
        counts = neither_active.sum()
        
    plt.bar(ALL_CELL_TYPES, counts, bottom=bottom, color=color, label=category)
    bottom += counts
    
plt.ylabel("Number of Cytokines")
plt.xticks(rotation=90)
plt.legend(loc=[1,0], fontsize=6)
plt.tick_params(axis='x', rotation=90, labelsize=6)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"2_magnitude_threshold_comparison.png"), dpi=300)
plt.show()

In [ ]:
# Create a figure with subplots arranged in a grid
n_cell_types = len(ALL_CELL_TYPES)
n_rows = 3 if not alt_flag else 4
n_cols = 6
fig, axes = plt.subplots(n_rows, n_cols, figsize=cm_to_inches((17, 7)))

# Flatten axes array for easier iteration
axes_flat = axes.flatten()

for idx, cell_type in enumerate(ALL_CELL_TYPES):
    ax = axes_flat[idx]
    ax.hist(all_normalized_distances_used[cell_type].values, bins=30, color='tab:blue', alpha=1)
    ax.axvline(magnitude_threshold_by_cell_type_zscore[cell_type], color="red", linestyle="--", label="z-score threshold (95%)" if idx == 0 else "")
    if idx % n_cols == 0:  # Only show y-label on leftmost plots
        ax.set_ylabel("cytokine count")
    else:
        ax.set_yticks([])
    if idx >= len(ALL_CELL_TYPES) - n_cols:  # Only show x-label on bottom plots
        ax.set_xlabel("response magnitude")
        # ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1])
        # ax.set_xticklabels(['0', '0.2', '0.4', '0.6', '0.8', '1'])
    ax.set_title(cell_type, fontsize=7)
    ax.set_ylim(0, 30)
    if idx == 0:
        ax.legend(fontsize=6)

# Remove any empty subplots
for idx in range(len(ALL_CELL_TYPES), len(axes_flat)):
    fig.delaxes(axes_flat[idx])

plt.tight_layout()

# Save the figure
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_distance_distributions.{format}"), bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# what to use downstream
strong_impact_by_cell_type = strong_impact_by_cell_type_zscore
magnitude_threshold_by_cell_type = magnitude_threshold_by_cell_type_zscore

In [ ]:
all_normalized_distances_used.to_csv(os.path.join(output_tables_folder, "2_response_magnitudes.csv"))
strong_impact_by_cell_type.to_csv(os.path.join(output_tables_folder, "2_strong_impact.csv"))
pd.Series(magnitude_threshold_by_cell_type).to_csv(os.path.join(output_tables_folder, "2_response_magnitude_thresholds.csv"))

In [ ]:
if write_SI_tables_flag:
    df = all_normalized_distances_used.reset_index().copy()
    df = df.rename(columns={"index": "Cytokine"})
    df['Cytokine'] = df['Cytokine'].apply(introduce_greek_letters)
    write_to_SI_table(df, sheet_name="response_magnitudes")

    df = strong_impact_by_cell_type.reset_index().copy()
    df = df.rename(columns={"index": "Cytokine"})
    df['Cytokine'] = df['Cytokine'].apply(introduce_greek_letters)
    write_to_SI_table(df, sheet_name="strong_impact")

## 2.4 - Create overview plots for DEGs and the response magnitude

In [ ]:
# Create a figure for histograms of cell type counts per condition/cell type
donor_independent_counts = all_counts_by_condition.groupby(level=[0,1]).sum()
log_counts = np.log10(donor_independent_counts + 1)

plt.figure(figsize=cm_to_inches((5, 7)))
offset = 0.5
for i, cell_type in enumerate(ALL_CELL_TYPES[::-1]):
    data = log_counts.loc[cell_type]
    data = data[data.index != "PBS"].values
    
    counts, bins = np.histogram(data, bins=100, range=(1, 5))
    if counts.max() > 0:
        counts = counts / counts.max() * 0.4
    
    plt.bar(bins[:-1], counts, width=bins[1]-bins[0], bottom=i*offset, 
            alpha=0.7, label=cell_type if i == 0 else "", color=CELL_TYPE_COLORS[cell_type])
    
    plt.text(1.3, i*offset + 0.2, cell_type, ha='right', va='center', fontsize=6)
    plt.plot([1.5, 5], [i*offset, i*offset], color='black', linestyle='-', linewidth=0.5)

plt.xlabel(r'log$_{10}$(cell type # by per stim. + 1)')
plt.yticks([])
plt.xlim(1.5, 5)
plt.ylim(0, len(ALL_CELL_TYPES)*offset)

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_count_distributions.{format}"), bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# Create a figure for histograms of log10(significant genes+1) per cell type
log_significant = np.log10(significant_results + 1)

plt.figure(figsize=cm_to_inches((5, 7)))
offset = 0.5
for i, cell_type in enumerate(ALL_CELL_TYPES[::-1]):
    data = log_significant.loc[cell_type].values
    counts, bins = np.histogram(data, bins=20, range=(0, 4))
    
    if counts.max() > 0:
        counts = counts / counts.max() * 0.4
    
    plt.bar(bins[:-1], counts, width=bins[1]-bins[0], bottom=i*offset, 
            alpha=0.7, label=cell_type if i == 0 else "", color=CELL_TYPE_COLORS[cell_type])
    
    plt.text(-0.2, i*offset + 0.2, cell_type, ha='right', va='center', fontsize=6)
    plt.plot([0, 4], [i*offset, i*offset], color='black', linestyle='-', linewidth=0.5)

plt.xlabel(r'log$_{10}$(DEG # by per stim. + 1)')
plt.yticks([])
plt.xlim(0, 4)
plt.ylim(0, len(ALL_CELL_TYPES)*offset)

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_DEG_distributions.{format}"), bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# How are cell type counts and DEGs related?
plt.figure(figsize=cm_to_inches((4, 3.2)))

log_counts_cells = np.log10(donor_independent_counts + 1).groupby(level=0).mean()
log_significant = np.log10(significant_results + 1).mean(axis=1)
log_counts_cells, log_significant = get_common_index(log_counts_cells, log_significant)
for cell_type in ALL_CELL_TYPES:
    plt.scatter(
        log_counts_cells[cell_type], log_significant[cell_type],
        color=CELL_TYPE_COLORS[cell_type], s=15, edgecolors="none", alpha=0.75)

plt.xlabel(r'mean(log$_{10}$(cell type # by per stim.))')
plt.ylabel(r'mean(log$_{10}$(DEG #+ 1))')
plt.xlim(2, 5)
plt.ylim(0, 3)

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_deg_vs_count.{format}"), bbox_inches="tight", dpi=300)
plt.show()

### If using finer-grained cell types, check how similar responses are

In [ ]:
if alt_flag:
    means_by_cell_type = {}
    stds_by_cell_type = {}
    for base_cell_type in ALL_CELL_TYPES:
        corrwith_vals = {}
        for cell_type in ALL_CELL_TYPES:
            cytokines_strong_base = strong_impact_by_cell_type.index[strong_impact_by_cell_type[base_cell_type]]
            cytokines_strong_cell = strong_impact_by_cell_type.index[strong_impact_by_cell_type[cell_type]]
            common_cytokines = cytokines_strong_base.intersection(cytokines_strong_cell)

            if len(common_cytokines) < 3:
                continue

            # subset the two DataFrames
            df_base = edger_df_log2FC[base_cell_type][common_cytokines]
            df_cell = edger_df_log2FC[cell_type][common_cytokines]

            # align them and compute correlations columnwise
            corr_series = df_base.corrwith(df_cell, method='pearson')
            corrwith_vals[cell_type] = corr_series
        corrwith_vals = pd.DataFrame(corrwith_vals)
        means_by_cell_type[base_cell_type] = corrwith_vals.mean()
        stds_by_cell_type[base_cell_type] = corrwith_vals.std()
    means_by_cell_type = pd.DataFrame(means_by_cell_type)   
    means_by_cell_type = means_by_cell_type.loc[ALL_CELL_TYPES, ALL_CELL_TYPES]    
    
    plt.figure(figsize=cm_to_inches((12, 12)))
    sns.heatmap(means_by_cell_type, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, annot_kws={"size":5.5},
                cbar_kws={"label": "mean Pearson correlation of log2FC\nin cell types with strong impact", "shrink": 0.5},
                square=True)
    plt.tick_params(axis='both', length=0)
    plt.savefig(os.path.join(plot_folder, f"2_correlation_of_strong_impact_log2FC_mean.{format}"), bbox_inches="tight", dpi=300)
    plt.show()

    plt.figure(figsize=cm_to_inches((12, 12)))
    sns.heatmap(all_normalized_distances_used.corr(), annot=True, annot_kws={"size":5.5},
                fmt=".2f", cmap="viridis", vmin=0.5, vmax=1,
                cbar_kws={"label": "mean Pearson correlation of response magnitudes", "shrink": 0.5},
                square=True)
    plt.tick_params(axis='both', length=0)
    plt.savefig(os.path.join(plot_folder, f"2_correlation_of_response_magnitude.{format}"), bbox_inches="tight", dpi=300)
    plt.show()

### Create main overview figures

In [ ]:
data_long = significant_results.reset_index().rename(columns={'index': 'cell_type'})
data_long = data_long.melt(id_vars=['cell_type'], var_name='cytokine', value_name='DEGs')
data_long['cytokine'] = pd.Categorical(data_long['cytokine'], categories=CYTOKINE_ORDER, ordered=True)
data_long = data_long.sort_values(by='cytokine', ascending=False)

In [ ]:
all_normalized_distances_long = all_normalized_distances_used.reset_index().rename(columns={'index': 'cytokine'})
all_normalized_distances_long = all_normalized_distances_long.melt(id_vars=['cytokine'], var_name='cell_type', value_name='Distance')

# clip at 0 and 1 (should basically already be the case, just make sure that minor numerical issues don't cause problems)
all_normalized_distances_long["Distance"] = all_normalized_distances_long["Distance"].clip(lower=0, upper=1)

all_raw_distances_long = all_raw_euclidean_distances.reset_index().rename(columns={'index': 'cytokine'})
all_raw_distances_long = all_raw_distances_long.melt(id_vars=['cytokine'], var_name='cell_type', value_name='Distance_Raw')

# merge
data_long = data_long.merge(all_normalized_distances_long, on=["cytokine", "cell_type"])

In [ ]:
data_long["max"] = data_long.groupby("cell_type")["DEGs"].transform("max")

# clip at at minimum 1% so that the dot size doesn't become too small
data_long["DEGs_clip"] = data_long.apply(lambda x: min(max(x["DEGs"], x["max"]/101.0), x["max"]), axis=1)
data_long["DEG_rel"] = data_long["DEGs_clip"] / data_long["max"]

# show the response magnitude as quintiles (color saturation)
data_long["distance_quintile"] = (pd.qcut(data_long["Distance"], q=5, labels=False) + 1) / 5

# calculate label for DEGs
labels = [
    "<0.01",
    "0.01-0.02",
    "0.02-0.05",
    "0.05-0.10",
    "0.10-0.25",
    "0.25-0.50",
    "0.50-1.00"
]
edges = [-np.inf]
for lab in labels: # basically, take the above and convert to numerical bins
    nums = re.findall(r"[\d.]+", lab)
    if lab.startswith("<"):
        upper = float(nums[0])
        edges.append(upper)
    else:
        low, high = nums[:2]
        edges.append(float(high))
bins = np.array(edges)
data_long["DEG_rel_binned"] = pd.cut(
    data_long["DEG_rel"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

max_size = 25
total_labels = len(labels)
size_mapping = {lab: ((i + 1) / total_labels)**2 * max_size for i, lab in enumerate(labels)}
data_long["size"] = data_long["DEG_rel_binned"].map(size_mapping)

In [ ]:
fig, ax = plt.subplots(figsize=cm_to_inches((3.8, 20)))
for cell_type in ALL_CELL_TYPES:
    subset = data_long[data_long['cell_type'] == cell_type]
    ax.scatter(
        subset['cell_type'], subset['cytokine'],
        s=subset["size"],
        marker='H', c=[CELL_TYPE_COLORS[cell_type]], edgecolor='k',
        alpha=subset['distance_quintile'],
        linewidth=0.5
    )

# generate legend for size
size_marker_legend_elements = []
size_marker_legend_elements.append(
        plt.scatter([], [], 
                   s=0,
                   marker='H',
                   color='gray',
                   edgecolor='k',
                   label=f'DEG fraction')
    )
for lab in labels:
    size_marker_legend_elements.append(
        plt.scatter([], [], 
                   s=size_mapping[lab],
                   marker='H',
                   color='gray',
                   edgecolor='k',
                   label=f'{lab}')
    )

# Family labels
x_offset_level1 = -6  
vals_per_label_level1 = cytokine_info_order["DEG_plot_family"].value_counts()
plot_overview_labels_along_y_axis(
    fig, ax, SINGLE_FAMILY_ORDER, vals_per_label_level1, x_offset_level1,
    len(subset['cytokine']),
    fontsize=6, fontweight="bold", color="black", linewidth=1,
    rotation=0)

# Need to draw to make greek conversion work
fig.canvas.draw()
ax.set_ylim(-1, len(CYTOKINE_ORDER))
apply_greek_labels_heatmap(ax)

plt.xticks(rotation=90, fontsize=5.5)
plt.yticks(fontsize=5.5)
plt.tick_params(length=0, axis='both')
plt.legend(
    handles=size_marker_legend_elements,
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    borderaxespad=0,
    fontsize=5.5,
    ncol=2
)

for format in ["png", "svg"]:
    plt.savefig(
        os.path.join(
            plot_folder,
            f"2_relative_DEG_numbers_by_cell_type.{format}"
            ),
        bbox_inches="tight", dpi=300)
plt.show()

#### Supplementary barplots (total DEGs, strong impact)

In [ ]:
# plot strong impact by cytokine
strong_impact_by_cytokine = strong_impact_by_cell_type.sum(axis=1).iloc[::-1]

plt.figure(figsize=cm_to_inches((1, 20)))
plt.barh(strong_impact_by_cytokine.index, strong_impact_by_cytokine.values, color="black")
plt.ylim(-1, len(CYTOKINE_ORDER))

# get the current figure and draw it (to get apply_greek_labels_heatmap to work)
fig = plt.gcf()
fig.canvas.draw()
apply_greek_labels_heatmap(plt.gca())
plt.xticks([0, len(ALL_CELL_TYPES)], [0, len(ALL_CELL_TYPES)], rotation=0, fontsize=6)
plt.xlim(0, len(ALL_CELL_TYPES)+1)
plt.title("strong\nimpact", fontsize=7)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_strong_impact_by_cytokine.{format}"), bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# plot the max DEGs per cell type
max_DEGs_by_cell_type = data_long.groupby("cell_type")["DEGs"].max()
max_DEGs_by_cell_type = max_DEGs_by_cell_type[ALL_CELL_TYPES]

plt.figure(figsize=cm_to_inches((4.75, 1.3)))
plt.bar(max_DEGs_by_cell_type.index, max_DEGs_by_cell_type.values, color="black", width=0.7)
plt.xticks([])
plt.ylabel("max.\nDEGs")
plt.xlim(-0.75, len(ALL_CELL_TYPES)-0.25)
plt.yticks([0, 7000], ["0", r"7000"])
plt.ylim(0, 7000)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_max_DEGs_by_cell_type.{format}"), bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# Make a more detailed impact plot
strong_impact_by_cell_type_long = strong_impact_by_cell_type.melt(
    var_name="cell_type", value_name="strong_impact", ignore_index=False).reset_index().rename({"index": "cytokine"}, axis=1)
data_long = pd.merge(data_long, strong_impact_by_cell_type_long, on=["cell_type", "cytokine"])

In [ ]:
fig, ax = plt.subplots(figsize=cm_to_inches((4.7, 20)))
for cell_type in ALL_CELL_TYPES:
    subset = data_long[data_long['cell_type'] == cell_type]
    ax.scatter(
        subset['cell_type'], subset['cytokine'],
        s = 10 * subset['strong_impact'].astype(int),
        marker='s', c="black", edgecolor="black", #c=[CELL_TYPE_COLORS[cell_type]], edgecolor='k',
        linewidth=0,
    )

# Family labels
x_offset_level1 = -6  
vals_per_label_level1 = cytokine_info_order["DEG_plot_family"].value_counts()
plot_overview_labels_along_y_axis(fig, ax, SINGLE_FAMILY_ORDER, vals_per_label_level1, x_offset_level1,
                                  len(subset['cytokine']),
                                  fontsize=6, fontweight="bold", color="black", linewidth=1,
                                  rotation=0)

fig.canvas.draw()
ax.set_ylim(-1, len(CYTOKINE_ORDER))
apply_greek_labels_heatmap(ax)
plt.xticks(rotation=90, fontsize=6)
plt.yticks(fontsize=5.5)

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2.4_strong_impact_details.{format}"), bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=cm_to_inches((4.7, 20)))
for cell_type in ALL_CELL_TYPES:
    subset = data_long[data_long['cell_type'] == cell_type]
    ax.scatter(
        subset['cell_type'], subset['cytokine'],
        s = 10 * subset['Distance'],
        marker='s', c="black", edgecolor="black",
        linewidth=0,
    )

# Family labels
x_offset_level1 = -6
vals_per_label_level1 = cytokine_info_order["DEG_plot_family"].value_counts()
plot_overview_labels_along_y_axis(fig, ax, SINGLE_FAMILY_ORDER, vals_per_label_level1, x_offset_level1,
                                  len(subset['cytokine']),
                                  fontsize=6, fontweight="bold", color="black", linewidth=1,
                                  rotation=0)

fig.canvas.draw()
ax.set_ylim(-1, len(CYTOKINE_ORDER))
apply_greek_labels_heatmap(ax)
plt.xticks(rotation=90, fontsize=6)
plt.yticks(fontsize=5.5)

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2.4_response_magnitude_by_size.{format}"), bbox_inches="tight", dpi=300)
plt.show()

# 3 - receptor expression and cytokine impact, consensus responses

In [ ]:
plot_folder = os.path.join(base_plot_folder, "3_receptor_cytokine_consensus")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
# get the subset of distances relevant for later plotting
all_normalized_distances_used_subset = all_normalized_distances_used.loc[
    receptor_expression_by_cytokine_relative.index, receptor_expression_by_cytokine_relative.columns
    ]

# generate melted dataframes for the scatter plot
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative.melt(ignore_index=False).rename(
    columns={"value": "relative_expression"})
all_normalized_distances_used_flat = all_normalized_distances_used_subset.melt(ignore_index=False).rename(
    columns={"value": "distance"}).rename(columns={"variable": "cell_type"})
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative_flat.reset_index().merge(
    all_normalized_distances_used_flat.reset_index(),
    on=["index", "cell_type"]).set_index("index")

# add the raw receptor expression
raw_expression = receptor_expression_by_cytokine_raw.melt(ignore_index=False)
raw_expression = raw_expression.rename(columns={"value": "raw_expression", "variable": "cell_type"})
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative_flat.merge(
    raw_expression, 
    left_on=["index", "cell_type"],
    right_on=[raw_expression.index, "cell_type"]
)

# add the relative number of DEGs
deg_count = data_long[["cell_type", "cytokine", "DEG_rel"]].set_index("cytokine").rename(columns={"cell_type": "cell_type"})
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative_flat.merge(
    deg_count, 
    left_on=["index", "cell_type"],
    right_on=[deg_count.index, "cell_type"]
)

In [ ]:
excluded_cytokines = []
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=cm_to_inches((12, 10)), constrained_layout=True)
axes = axes.flatten()

# Collect legend labels
legend_labels = []
primary_cell_types = []
for i, cell_type in enumerate(ALL_CELL_TYPES):
    ax = axes[i]
    df_subset = receptor_expression_by_cytokine_relative_flat[receptor_expression_by_cytokine_relative_flat["cell_type"] == cell_type].copy()
    df_subset['raw_expression_log2'] = np.log2(df_subset['raw_expression'])
    df_subset = df_subset[~df_subset["index"].isin(excluded_cytokines)].copy()
    
    r = np.corrcoef(df_subset["raw_expression_log2"],
                    df_subset["distance"])[0, 1]
    scatter = ax.scatter(df_subset["raw_expression_log2"],
                         df_subset["distance"], s=3,
                         color=CELL_TYPE_COLORS[cell_type])
    
    # for plotting only (useful ones to plot labels for)
    df_subset_very_low = df_subset[
        (df_subset["raw_expression"] < ReceptorParameters.present_threshold) & (df_subset["distance"] > 1.5 * magnitude_threshold_by_cell_type[cell_type])]
    label = df_subset_very_low.sort_values(by="distance", ascending=False).head(3)["index"]
    
    # Add text labels for all cytokines in 'label'
    for idx, cytokine in enumerate(label):
        ax.annotate(
            introduce_greek_letters(cytokine),
            xy=(df_subset_very_low[df_subset_very_low["index"] == cytokine]["raw_expression_log2"].values[0],
                df_subset_very_low[df_subset_very_low["index"] == cytokine]["distance"].values[0]),
            xytext=(0, 0),  # offset for annotation text position
            textcoords='offset points',
            fontsize=5.5,
            color='black'
        )
    
    # Collect the label for the legend
    legend_labels.append((scatter, f"{cell_type} (r = {r:.2f})"))
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlim(-0.3, 10)
    
    ax.axhline(y=magnitude_threshold_by_cell_type[cell_type], linestyle='--', color='gray', alpha=0.75)
    ax.axvline(x=np.log2(ReceptorParameters.present_threshold), linestyle='--', color='gray', alpha=0.75)
    
    if i % 4 == 0:  # First column
        ax.set_ylabel("response magnitude")
    if i >= 8:  # Last row
        ax.set_xlabel(r"$\log_{2}$(receptor cpm)")
    ax.label_outer()

# Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Plot the legend once
handles, labels = zip(*legend_labels)
fig.legend(handles, labels, loc="lower right", fontsize=6, ncol=2)

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_scatter_receptor_vs_distance.{format}"), bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
def characterize_response_type(row):
    receptor = row["raw_expression"]
    response = row["distance"]
    
    classification = 'missing classification'
    if response >= 1.5 * magnitude_threshold_by_cell_type[row["cell_type"]]:
        if receptor < ReceptorParameters.present_threshold:
            classification = f"low receptor (<{ReceptorParameters.present_threshold}), clear response (>=1.5x thresh.)"
        elif receptor >= ReceptorParameters.present_threshold:
            classification = f"expressed receptor (>={ReceptorParameters.present_threshold}), strong response (>=1x thresh.)"
        else:
            classification = 'Error'
    elif response >= 1 * magnitude_threshold_by_cell_type[row["cell_type"]]:
        if receptor >= ReceptorParameters.present_threshold:
            classification = f"expressed receptor (>={ReceptorParameters.present_threshold}), strong response (>=1x thresh.)"
        elif receptor < ReceptorParameters.present_threshold:
            classification = f"low receptor (<{ReceptorParameters.present_threshold}), intermediate response (>=1x & <1.5x thresh.)"
    elif response < 1 * magnitude_threshold_by_cell_type[row["cell_type"]]:
        if receptor >= ReceptorParameters.high_threshold:
            classification = f"high receptor (>={ReceptorParameters.high_threshold}), weak response (<1x thresh.)"
        elif receptor < ReceptorParameters.present_threshold:
            classification = f"low receptor (<{ReceptorParameters.present_threshold}), weak response (<1x thresh.)"
        else:
            classification = f"intermediate receptor (>={ReceptorParameters.present_threshold} & <{ReceptorParameters.high_threshold}), weak response (<1x thresh.)"
    else:
        classification = f"Error"
    return classification

classifications = {
    f'low receptor (<{ReceptorParameters.present_threshold}), weak response (<1x thresh.)': 'lightgray',
    f'low receptor (<{ReceptorParameters.present_threshold}), intermediate response (>=1x & <1.5x thresh.)': 'salmon',
    f'low receptor (<{ReceptorParameters.present_threshold}), clear response (>=1.5x thresh.)': 'red',
    f'intermediate receptor (>={ReceptorParameters.present_threshold} & <{ReceptorParameters.high_threshold}), weak response (<1x thresh.)': '#E0CFA6',
    f'high receptor (>={ReceptorParameters.high_threshold}), weak response (<1x thresh.)': 'gold',
    f'expressed receptor (>={ReceptorParameters.present_threshold}), strong response (>=1x thresh.)': 'lightseagreen', 
}
classifications_shorthand = {
    f'low receptor (<{ReceptorParameters.present_threshold}), weak response (<1x thresh.)': 'low rec., weak resp.',
    f'low receptor (<{ReceptorParameters.present_threshold}), intermediate response (>=1x & <1.5x thresh.)': 'low rec., int. resp.',
    f'low receptor (<{ReceptorParameters.present_threshold}), clear response (>=1.5x thresh.)': 'low rec., clear resp.',
    f'intermediate receptor (>={ReceptorParameters.present_threshold} & <{ReceptorParameters.high_threshold}), weak response (<1x thresh.)': 'int. rec., weak resp.',
    f'high receptor (>={ReceptorParameters.high_threshold}), weak response (<1x thresh.)': 'high rec., weak resp.',
    f'expressed receptor (>={ReceptorParameters.present_threshold}), strong response (>=1x thresh.)': 'expressed rec., strong resp.', 
}

receptor_expression_by_cytokine_relative_flat['classification'] = receptor_expression_by_cytokine_relative_flat.apply(characterize_response_type, axis=1)
rec_expr_reshaped = receptor_expression_by_cytokine_relative_flat.pivot(index="index", columns="cell_type", values="classification")
rec_expr_reshaped = rec_expr_reshaped.loc[[c for c in CYTOKINE_ORDER if c in rec_expr_reshaped.index], ALL_CELL_TYPES]

interesting = [
    f'low receptor (<{ReceptorParameters.present_threshold}), clear response (>=1.5x thresh.)',
    f'high receptor (>={ReceptorParameters.high_threshold}), weak response (<1x thresh.)',
    f'expressed receptor (>={ReceptorParameters.present_threshold}), strong response (>=1x thresh.)'
    ]

mask = rec_expr_reshaped.isin(interesting).any(axis=1)
rec_expr_reshaped_filter = rec_expr_reshaped[mask]
rec_expr_reshaped = rec_expr_reshaped.map(lambda x: classifications_shorthand[x])

In [ ]:
from matplotlib.colors import ListedColormap

# map to integeres
labels = list(classifications.keys())
class_to_int = {lab: i for i, lab in enumerate(labels)}
colors = [classifications[lab] for lab in labels]
cmap = ListedColormap(colors)

int_matrix = rec_expr_reshaped_filter.replace(class_to_int)
int_matrix = int_matrix.apply(pd.to_numeric, errors="coerce")

# split into panels
n = 3
row_indices = list(range(int_matrix.shape[0]))
chunk_size = (len(row_indices) + n - 1) // n
row_idx_chunks = [row_indices[i:i + chunk_size] for i in range(0, len(row_indices), chunk_size)]

# plot
fig, axes = plt.subplots(1, len(row_idx_chunks), figsize=cm_to_inches((14, 10)))
axes = np.atleast_1d(axes)

for ax, row_idx in zip(axes, row_idx_chunks):
    submat = int_matrix.iloc[row_idx, :]
    mask = submat.isna()

    sns.heatmap(submat, cmap=cmap, cbar=False, linewidths=0.5, linecolor='gray', square=True, ax=ax, mask=mask, vmin=0, vmax=len(labels)-1)

    ax.tick_params(length=0)
    ax.set_ylabel("")
    ax.set_xlabel("")
    apply_greek_labels_heatmap(ax)

handles = [mpatches.Patch(color=classifications[label], label=label) for label in labels]
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02))

plt.subplots_adjust(wspace=0.5)
plt.savefig(os.path.join(plot_folder,"3_cytokine_response_classification_heatmap.svg"),dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
cytokine_subset = receptor_expression_by_cytokine_relative_flat[receptor_expression_by_cytokine_relative_flat['index'] == 'IL-12']

plt.figure(figsize=cm_to_inches((4, 5)))
plt.barh(cytokine_subset['cell_type'], cytokine_subset['raw_expression'], color='tab:blue', edgecolor='black')
plt.axvline(x=ReceptorParameters.present_threshold, linestyle='--', color='red', alpha=0.75)
plt.gca().invert_yaxis()
plt.tick_params(axis='y', length=0)
plt.xlabel("IL-12 receptor cpm")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "3_IL-12_receptor_expression_by_cell_type.png"), dpi=300)
plt.show()

In [ ]:
outlier_matrix = rec_expr_reshaped == 'low rec., clear resp.'
outlier_matrix = outlier_matrix.astype(int)
outlier_matrix = outlier_matrix[outlier_matrix.max(axis=1) > 0]

plt.figure(figsize=cm_to_inches((8, 6)))
ax = sns.heatmap(outlier_matrix, cmap="rocket", cbar=True, linewidths=0.5, linecolor='gray', square=True)
plt.xlabel("")
plt.ylabel("")
plt.title("Potential secondary responses")
ax.tick_params(axis='x', which='both', bottom=False, top=False)
ax.tick_params(axis='y', which='both', left=False, right=False)
ax = apply_greek_labels_heatmap(ax)
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_likely_secondary_responses.{format}"), dpi=300)
plt.show()

In [ ]:
outlier_matrix = rec_expr_reshaped == 'high rec., weak resp.'
outlier_matrix = outlier_matrix.astype(int)
outlier_matrix = outlier_matrix[outlier_matrix.max(axis=1) > 0]

plt.figure(figsize=cm_to_inches((8, 10)))
ax = sns.heatmap(outlier_matrix, cmap="rocket", cbar=True, linewidths=0.5, linecolor='gray', square=True)
plt.xlabel("")
plt.ylabel("")
plt.title("Strong receptor-weak response")
ax.tick_params(axis='x', which='both', bottom=False, top=False)
ax.tick_params(axis='y', which='both', left=False, right=False)
ax = apply_greek_labels_heatmap(ax)
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_strong_receptor_weak_response.{format}"), dpi=300)
plt.show()

In [ ]:
strong_sum = strong_impact_by_cell_type.sum(axis=1)

outlier_matrix = rec_expr_reshaped == 'high rec., weak resp.'
outlier_matrix = outlier_matrix.astype(int)
outlier_matrix = outlier_matrix[outlier_matrix.max(axis=1) > 0].sum(axis=1).to_frame().rename(columns={0: "count"})
outlier_matrix["strong_impact"] = strong_sum.loc[outlier_matrix.index]
outlier_matrix = outlier_matrix[outlier_matrix['count'] > outlier_matrix["strong_impact"]]

plt.figure(figsize=cm_to_inches((6, 7)))
plt.scatter(outlier_matrix['strong_impact'], outlier_matrix['count'], alpha=0)  # invisible points as placeholders

# Add text labels at each point with vertical spacing for duplicates
point_dict = {}
for idx, row in outlier_matrix.iterrows():
    key = (row['strong_impact'], row['count'])
    if key not in point_dict:
        point_dict[key] = []
    point_dict[key].append(idx)
for coords, labels in point_dict.items():
    x, y = coords
    if len(labels) == 1:
        plt.text(x, y, introduce_greek_letters(labels[0]), fontsize=6, 
                 ha='center', va='center')
    else:
        # Multiple labels at same point - space them vertically
        offset = 0.5  # Vertical spacing between labels
        total_height = offset * (len(labels) - 1)
        start_y = y - total_height/2
        
        for i, label in enumerate(labels):
            plt.text(x, start_y + i*offset, introduce_greek_letters(label), 
                     fontsize=6, ha='center', va='center')

plt.xlim(-0.9, 5.9)
plt.ylim(0, 14)
plt.xlabel('# strong impact')
plt.ylabel('# strong receptor - weak response')
plt.grid(True, alpha=0.3)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_failed-responses.{format}"), dpi=300)
plt.show()

## 3.2 - Expected literature responses

In [ ]:
plot_folder = os.path.join(base_plot_folder, "3.2_individual_literature_responses")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
# manually look up
strongest_response_in_literature = {}
upregulated_genes = pd.DataFrame(columns=["gene", "cell_type", "cytokine", "response_type"])

def evaluate_cytokine(current_cytokine, strongest_response_in_literature):
    data_long_cytokine = data_long[data_long["cytokine"] == current_cytokine].copy().set_index("cell_type")
    print("There should be strong response in the following cell types:")
    print(strongest_response_in_literature[current_cytokine])
    print("The relative DEGs and Magnitudes are:")
    print(data_long_cytokine[["DEG_rel", "Distance"]])

def print_regulated_genes(upregulated_genes, current_cytokine):
    # Get genes for this cytokine
    cytokine_genes = upregulated_genes[upregulated_genes["cytokine"] == current_cytokine]
    
    # Group by cell type
    for cell_type, cell_genes in cytokine_genes.groupby("cell_type"):
        print(f"{cell_type}")
        print("--------------------------------")
        
        for _, row in cell_genes.iterrows():
            gene = row["gene"]
            if gene in edger_df_log2FC.index:
                fc = edger_df_log2FC.loc[gene, (cell_type, current_cytokine)]
                pval = edger_df_padj.loc[gene, (cell_type, current_cytokine)]
                
                print(f"The fold change for {gene} in response to {current_cytokine} is {fc}")
                print(f"The p-value for {gene} in response to {current_cytokine} is {pval:e}" \
                    if not pd.isna(pval) else f"The p-value for {gene} in response to {current_cytokine} is NaN")
                print("----")
            else:
                if gene in unfiltered_genes:
                    print(f"The gene {gene} is not in padj, but is in the list of unfiltered genes.")
                else:
                    print(f"The gene {gene} is not in padj, and is not in the list of unfiltered genes.")

In [ ]:
# --------------------------------------------------
# IL-3: 10.3389/fimmu.2024.1411047
current_cytokine = "IL-3"
print(f"Processing {current_cytokine}")
print("""CD123 (IL3RA) is mainly expressed on hematopoietic cells, including hematopoietic stem and progenitor cells (HSPCs),
basophils, eosinophils, mast cells, non-classical monocytes, macrophages, human plasmacytoid dendritic cells (pDCs)
and activated T and B cells (...) In steady-state, murine monocytes and human classical
monocytes do not express the receptor for IL-3. Only human intermediate and non-classical monocytes express it.""")

strongest_response_in_literature[current_cytokine] = ["HSPC", "CD14 Mono", "CD16 Mono"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

"""IL-3 stimulates the growth and
differentiation of HSPCs from bone marrow cultures and in naive or
sublethally irradiated mice into a large range of cell lineages,
including basophils, neutrophils, eosinophils, macrophages"""

"""IL-3 synergizes with IL-4 to enhance CCL17 expression in CD14+ monocytes """
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'CCL3', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'}, 
    {'gene': 'CCL7', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'},
    {'gene': 'CCL17', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'}
])])

print("upregulated genes")
# print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# IL-32-beta: 10.1002/JLB.3MR0620-550R
current_cytokine = "IL-32-beta"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["NK CD56low", "NK CD56hi", "CD14 Mono", "CD16 Mono", "cDC"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# pick some upregulated cytokines
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'IL1B', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'inflammatory'},
    {'gene': 'IL6', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'inflammatory'}
])])

# print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# M/GM-CSF: 10.1016_j.immuni.2016.10.026
current_cytokine = "GM-CSF"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["CD14 Mono", "CD16 Mono", "cDC"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# Croxford (2015) - Immunity - The Cytokine GM-CSF (pick a subset of genes for plotting)
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'PYCARD', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'phagocytosis'},
    {'gene': 'CLEC7A', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'phagocytosis'},
    {'gene': 'CCL17', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'},
    {'gene': 'CCL24', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'},
])])
print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# TPO: 10.1080/09537104.2021.1925102
current_cytokine = "TPO"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["HSPC"] 
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# de Graaf (2011) - Cell cycle - Thrombopoietin and hematopoietic stem cells
# these genes aren't really mentioned directly, so don't plot this
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'CDKN1C', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'quiescence'},
    {'gene': 'CDKN2D', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'quiescence'},
    {'gene': 'HOXB4', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'stemness'},
    {'gene': 'HOXA5', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'stemness'},
    {'gene': 'HOXA9', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'stemness'},
    {'gene': 'HOXA10', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'stemness'},
    {'gene': 'BCL2L1', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'survival'}
])])

# print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# IL-7: 10.3389/fimmu.2021.747324
current_cytokine = "IL-7"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["CD4 T cell", "CD8 T cell", "MAIT", "NKT", "Treg"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# IL-7: 10.3389/fimmu.2021.747324
# Don't add CD14 Monos here to avoid clutter
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'BCL2', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'survival'},
    {'gene': 'PIM1', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'survival'},
    {'gene': 'BCL2', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell' if not alt_flag else 'CD4 Naive', 'response_type': 'survival'},
    {'gene': 'PIM1', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell' if not alt_flag else 'CD4 Naive', 'response_type': 'survival'},
#    {'gene': 'IFNG', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytokine'},
])])

# print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# CD40L: 10.1111/j.1600-065X.2009.00782.x
current_cytokine = "CD40L"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["B cell", "cDC", "CD14 Mono", "CD16 Mono"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# all of these work, but I'll pick two to actually plot
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'NFKB2', 'cytokine': current_cytokine, 'cell_type': 'B cell' if not alt_flag else 'B Naive', 'response_type': 'NF-kappaB'},
    {'gene': 'TRAF1', 'cytokine': current_cytokine, 'cell_type': 'B cell' if not alt_flag else 'B Naive', 'response_type': 'NF-kappaB'},
])])

# print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# IL-4: 10.12703/r/10-71
current_cytokine = "IL-4"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["B cell", "CD4 T cell", "CD8 T cell", "MAIT", "NKT", "Treg", "NK CD56low", "NK CD56hi", "CD14 Mono", "CD16 Mono", "cDC"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# only plot B cell responses for this one
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'FCER2', 'cytokine': current_cytokine, 'cell_type': 'B cell' if not alt_flag else 'B Naive', 'response_type': 'isotype'}, # isotype switch, IgE receptor
    {'gene': 'IGHE', 'cytokine': current_cytokine, 'cell_type': 'B cell' if not alt_flag else 'B Naive', 'response_type': 'isotype'}, # isotype switch, IgG3
    {'gene': 'IL4I1', 'cytokine': current_cytokine, 'cell_type': 'B cell' if not alt_flag else 'B Naive', 'response_type': 'other'}, ## of recent interest
    {'gene': 'IL4R', 'cytokine': current_cytokine, 'cell_type': 'B cell' if not alt_flag else 'B Naive', 'response_type': 'other'}, ## induce its own receptor
])])

# print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# IL-12: 10.17179/excli2020-3104
current_cytokine = "IL-12"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["NK CD56low", "NK CD56hi", "CD4 T cell", "CD8 T cell", "MAIT", "NKT", "Treg"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'IFNG', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytokine'},
    {'gene': 'TNF', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytokine'},
    {'gene': 'IFNG', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell' if not alt_flag else "CD4 Naive", 'response_type': 'cyto. prod.'},
    {'gene': 'GZMB', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytotoxicity'},
    {'gene': 'PRF1', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytotoxicity'}
])])

# print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
strongest_response_in_literature["IFN-beta"] = ALL_CELL_TYPES

count_significant_responses = 0
expected_responses = pd.DataFrame(index=strongest_response_in_literature.keys(), columns=ALL_CELL_TYPES, data=0)
for cytokine in strongest_response_in_literature.keys():
    for cell_type in strongest_response_in_literature[cytokine]:
        if cell_type in ALL_CELL_TYPES:
            expected_responses.loc[cytokine, cell_type] = 1
            if strong_impact_by_cell_type.loc[cytokine, cell_type]:
                count_significant_responses += 1
        else:
            print(f"The cell type {cell_type} is not in the list of all cell types.")

In [ ]:
no_significant_responses_subset = strong_impact_by_cell_type.loc[expected_responses.index, expected_responses.columns][ALL_CELL_TYPES]
count_significant_responses / expected_responses.sum().sum()

In [ ]:
ordered_index = [i for i in CYTOKINE_ORDER if i in expected_responses.index]
expected_responses = expected_responses.loc[ordered_index, :]
all_distances_mean_subset = all_normalized_distances_used.loc[expected_responses.index, expected_responses.columns][ALL_CELL_TYPES]

In [ ]:
plt.figure(figsize=cm_to_inches((5,3.8)))
ax = sns.heatmap(all_distances_mean_subset, vmin=0.0, vmax=1, cmap="coolwarm",
            cbar_kws={"ticks": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0], "label": "response magnitude", "shrink": 0.6}, square=True)

ax = plt.gca()

for cytokine_index, cytokine in enumerate(expected_responses.index):
    row = expected_responses.loc[cytokine]

    # go through contiguous runs of same-color boxes
    prev_color = None
    run_start = None

    for i, cell_type in enumerate(expected_responses.columns):
        if row[cell_type] == 1:
            is_strong = bool(strong_impact_by_cell_type.loc[cytokine, cell_type])
            color = "limegreen" if is_strong else "coral"
        else:
            color = None

        # check if we continue a run
        if color == prev_color and color is not None:
            # still in same-color run
            pass
        else:
            # close previous run
            if prev_color is not None:
                rect = plt.Rectangle(
                    (run_start, cytokine_index),
                    width=i - run_start,
                    height=1,
                    fill=False,
                    edgecolor=prev_color,
                    linewidth=1.3,
                    clip_on=False,
                    ls="--",
                    zorder=2 if prev_color == "lightgreen" else 3
                )
                ax.add_patch(rect)

            # start new run
            if color is not None:
                run_start = i
            else:
                run_start = None

        prev_color = color

    # close final run at row end
    if prev_color is not None and run_start is not None:
        rect = plt.Rectangle(
            (run_start, cytokine_index),
            width=len(expected_responses.columns) - run_start,
            height=1,
            fill=False,
            edgecolor=prev_color,
            linewidth=1.3,
            clip_on=False,
            ls="--",
            zorder=2 if prev_color == "lightgreen" else 3
        )
        ax.add_patch(rect)
            
# # Add legend for green box
# # Create custom legend patch for green box
# green_box = plt.Rectangle((0,0), 1, 1, fill=False, edgecolor="lightgreen", linewidth=2)
# plt.gca().add_artist(plt.legend([green_box], ['expected response'], loc='center left', bbox_to_anchor=(0.5, 0.5), handlelength=1, handletextpad=1))

apply_greek_labels_heatmap(ax)
plt.gca().tick_params(length=0)

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_expected_strength_responses_literature.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
plt.figure(figsize=cm_to_inches((5,3.8)))
receptor_expression_by_cytokine_relative_plot = receptor_expression_by_cytokine_relative.copy()
receptor_expression_by_cytokine_relative_plot.loc["IL-32-beta"] = np.nan
ax = sns.heatmap(np.log10(receptor_expression_by_cytokine_relative_plot.loc[expected_responses.index, expected_responses.columns].clip(0.01)), 
                cmap="coolwarm", 
                square=True, 
                cbar_kws={"shrink": 0.6, "label": r"$\log_{10}$(rel. receptor expr.)"})

ax = plt.gca()

for cytokine_index, cytokine in enumerate(expected_responses.index):
    row = expected_responses.loc[cytokine]

    # go through contiguous runs of same-color boxes
    prev_color = None
    run_start = None

    for i, cell_type in enumerate(expected_responses.columns):
        if row[cell_type] == 1:
            is_strong = bool(strong_impact_by_cell_type.loc[cytokine, cell_type])
            color = "limegreen" if is_strong else "coral"
        else:
            color = None

        # check if we continue a run
        if color == prev_color and color is not None:
            # still in same-color run
            pass
        else:
            # close previous run
            if prev_color is not None:
                rect = plt.Rectangle(
                    (run_start, cytokine_index),
                    width=i - run_start,
                    height=1,
                    fill=False,
                    edgecolor=prev_color,
                    linewidth=1.3,
                    clip_on=False,
                    ls="--",
                    zorder=2 if prev_color == "lightgreen" else 3
                )
                ax.add_patch(rect)

            # start new run
            if color is not None:
                run_start = i
            else:
                run_start = None

        prev_color = color

    # close final run at row end
    if prev_color is not None and run_start is not None:
        rect = plt.Rectangle(
            (run_start, cytokine_index),
            width=len(expected_responses.columns) - run_start,
            height=1,
            fill=False,
            edgecolor=prev_color,
            linewidth=1.3,
            clip_on=False,
            ls="--",
            zorder=2 if prev_color == "lightgreen" else 3
        )
        ax.add_patch(rect)

apply_greek_labels_heatmap(ax)
plt.gca().tick_params(length=0)

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_receptor_expression_subset.{format}"), 
                bbox_inches="tight", 
                dpi=300)

In [ ]:
pair = ['IL-4', 'IL-13']
plt.figure(figsize=cm_to_inches((6,4)))
subset = receptor_expression_by_cytokine_relative.loc[pair, ALL_CELL_TYPES]
plt.bar(x=np.arange(len(ALL_CELL_TYPES)) - 0.2, height=subset.loc[pair[0]], width=0.4, label=introduce_greek_letters(pair[0]), color="steelblue")
plt.bar(x=np.arange(len(ALL_CELL_TYPES)) + 0.2, height=subset.loc[pair[1]], width=0.4, label=introduce_greek_letters(pair[1]), color="coral")
plt.xticks(ticks=np.arange(len(ALL_CELL_TYPES)), labels=[introduce_greek_letters(ct) for ct in ALL_CELL_TYPES], rotation=90, fontsize=6)
plt.tick_params(axis='x', length=0)
plt.ylabel("relative receptor expression")
plt.tight_layout()
plt.legend(loc="upper right", fontsize=6)
plt.savefig(os.path.join(plot_folder, f"3_receptor_expression_{pair[0]}_{pair[1]}.svg"), dpi=300, bbox_inches="tight")

plt.figure(figsize=cm_to_inches((6,4)))
subset = all_normalized_distances_used.loc[pair, ALL_CELL_TYPES]
plt.bar(x=np.arange(len(ALL_CELL_TYPES)) - 0.2, height=subset.loc[pair[0]], width=0.4, label=introduce_greek_letters(pair[0]), color="steelblue")
plt.bar(x=np.arange(len(ALL_CELL_TYPES)) + 0.2, height=subset.loc[pair[1]], width=0.4, label=introduce_greek_letters(pair[1]), color="coral")
plt.xticks(ticks=np.arange(len(ALL_CELL_TYPES)), labels=[introduce_greek_letters(ct) for ct in ALL_CELL_TYPES], rotation=90, fontsize=6)
plt.tick_params(axis='x', length=0)
plt.ylabel("response magnitude")
plt.tight_layout()
plt.legend(loc="upper right", fontsize=6)
plt.savefig(os.path.join(plot_folder, f"3_response_magnitude_{pair[0]}_{pair[1]}.svg"), dpi=300, bbox_inches="tight")

In [ ]:
upregulated_genes = upregulated_genes.reset_index(drop=True)
for index, row in upregulated_genes.iterrows():
    if row["gene"] in edger_df_log2FC.index:
        # could plot the edgeR values, but that makes the correspondence between individual dots and the overall value strange
        fc = raw_log2fc_values.loc[row["gene"], (row["cell_type"], row["cytokine"])]
        upregulated_genes.loc[index, "fc"] = fc
        
        # pval = edger_df_padj.loc[row["gene"], (row["cell_type"], row["cytokine"])]
        # upregulated_genes.loc[index, "pval"] = pval
upregulated_genes.dropna(inplace=True)
upregulated_genes["cyto_response"] = upregulated_genes["cytokine"] + "\n" + upregulated_genes["response_type"]

In [ ]:
if not alt_flag:
    cell_types = ["CD14 Mono", "CD4 T cell", "NK CD56hi", "B cell"]
else:
    cell_types = ["B Naive", "CD4 Naive", "NK CD56hi", "CD14 Mono"]

fig = plt.figure(figsize=cm_to_inches((6, 6)))
outer = fig.add_gridspec(2, 1)
top = outer[0].subgridspec(1, 2, width_ratios=[3, 1])
bottom = outer[1].subgridspec(1, 2, width_ratios=[1, 1])
axes = np.array([
    fig.add_subplot(top[0, 0]),
    fig.add_subplot(top[0, 1]),
    fig.add_subplot(bottom[0, 0]),
    fig.add_subplot(bottom[0, 1]),
])

for ax, cell_type in zip(axes, cell_types):
    df_subset = upregulated_genes[upregulated_genes["cell_type"] == cell_type].copy()
    categories = list(df_subset["cyto_response"].unique())

    group_gap = 0.5
    bar_width = 0.8
    xticks = []
    xticklabels = []
    current_x = 0
    y_max = df_subset['fc'].max()

    for category in categories:
        sub = df_subset[df_subset["cyto_response"] == category].copy()
        cytokine = sub['cytokine'].iloc[0]
        n = len(sub)
        xs = np.arange(current_x, current_x + n)

        bars = ax.bar(xs, sub["fc"], color='none', width=bar_width)
        for bar, gene, val in zip(bars, sub["gene"], sub["fc"]):
            y = bar.get_height()
            offset = 0.03 if y >= 0 else -0.03
            donor_vals = raw_log2fc_values_by_donor[cell_type][cytokine].loc[gene]
            x_center = bar.get_x() + bar.get_width() / 2
            jitter = np.linspace(-0.12, 0.12, len(donor_vals))
            ax.scatter(x_center + jitter, donor_vals, s=2, color=CELL_TYPE_COLORS[cell_type], zorder=3)

        xticks.append(xs.mean())
        xticklabels.append(category)
        current_x = xs[-1] + 1 + group_gap

    ax.set_xticks(xticks)
    ax.set_xticklabels([introduce_greek_letters(c) for c in xticklabels], rotation=45, fontsize=6)
    ax.set_title(cell_type, fontsize=7)
    ax.set_xlabel("")
    ax.set_ylabel(r"log$_2$(fold change)" if ax is axes[0] or ax is axes[2] else "")
    
    ax.tick_params(axis="y", labelsize=6)
    ax.tick_params(axis="x", labelsize=6, length=0, rotation=45)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    #ax.set_ylim(0, y_max*1.5)
    
    from matplotlib.ticker import MaxNLocator
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

plt.tight_layout()

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder,f"3_individual_upregulated_genes_by_cytokine_barplots.{format}"),bbox_inches="tight",dpi=300)
plt.show()

## 4 - Donor variability analysis

In [ ]:
from utils.helper_functions import get_upper_triangle, delete_diagonal
plot_folder = os.path.join(base_plot_folder, "4_donor_variability")
os.makedirs(plot_folder, exist_ok=True)

### 4.0 - Inspect Basic metadata

In [ ]:
donor_metadata = pd.read_excel("../../annotations/donor_metadata.xlsx")
donor_metadata["Donor number"] = donor_metadata["Donor number"].str.replace(" ", "")
donor_metadata.index = donor_metadata["Donor number"]
donor_metadata["age"] = 2024 - donor_metadata["Birth year"]
donor_metadata = donor_metadata.sort_values(by="Sex")
DONOR_NAMES = donor_metadata.index.tolist()
DONOR_NAMES = [d.replace("Donor", "D") for d in DONOR_NAMES]

In [ ]:
donor_metadata["Donor number"] = donor_metadata["Donor number"].str.replace("Donor", "D")
donor_metadata['Weight (kg)'] = donor_metadata['Weight (lbs)']*0.453592
donor_metadata['Height (m)'] = donor_metadata['Height (in)']*2.54/100
donor_metadata["BMI"] = donor_metadata["Weight (kg)"] / donor_metadata['Height (m)']**2

In [ ]:
DONOR_GROUPS = {
    'interferon': ["D1", "D3", "D4", "D10"],
    'non-interferon': ["D5", "D7", "D8", "D9", "D11", "D12"],
    'other': ['D6', 'D2'] # intermediate/variable, IL-32-like condition
}
DONOR_COLOR_SCHEME = {
    "interferon": "royalblue",
    "non-interferon": "forestgreen",
    "other": "tab:gray"
}
DONOR_GROUPS_DEMOGRAPHIC = {
    "D1": "older\nfemale",
    "D3": "older\nfemale", 
    "D4": "older\nfemale",
    "D7": "younger\nfemale",
    "D8": "younger\nfemale",
    "D12": "younger\nfemale", 
    "D2": "younger\nmale",
    "D5": "younger\nmale",
    "D6": "younger\nmale", 
    "D9": "younger\nmale",
    "D10": "younger\nmale",
    "D11": "younger\nmale"
}
DONOR_ORDER = list(DONOR_GROUPS_DEMOGRAPHIC.keys())
DONOR_ORDER_IFN_GROUPED = ['D1', 'D3', 'D4', 'D10', 'D5', 'D7', 'D8', 'D9', 'D11', 'D12', 'D6', 'D2']
DONOR_COLOR_DIRECT = {
    donor: DONOR_COLOR_SCHEME['interferon'] if donor in DONOR_GROUPS['interferon'] else \
        DONOR_COLOR_SCHEME['non-interferon'] if donor in DONOR_GROUPS['non-interferon'] else \
        DONOR_COLOR_SCHEME['other'] for donor in DONOR_ORDER_IFN_GROUPED
}
DONOR_COLOR_SCHEME_D2 = {d: 'black' for d in DONOR_NAMES}
DONOR_COLOR_SCHEME_D2["D2"] = 'tab:red'
BAR_COLOR_STANDARD = [
    DONOR_COLOR_DIRECT[donor] for donor in DONOR_ORDER_IFN_GROUPED
    ]

In [ ]:
fig = plt.figure(figsize=cm_to_inches((3.7,4.3)))
plt.barh(
    donor_metadata["Donor number"].iloc[::-1],
    donor_metadata['age'].iloc[::-1],
    edgecolor="black",
    color="skyblue",
    height=0.7,
    linewidth=0.5)
plt.xlim(25, 80)
plt.xlabel("age (years)")
plot_overview_labels_along_y_axis(
    fig,
    plt.gca(),
    labels=['older\nfemale', 'younger\nfemale', 'younger\nmale'],
    vals_per_label=pd.Series(DONOR_GROUPS_DEMOGRAPHIC).value_counts(),
    x_offset=-1.2,
    no_y_vals=len(DONOR_NAMES),
    fontsize=6.5
)
plt.tick_params(axis='y', length=0)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, 'donor_ages_bar.svg'))

# fig = plt.figure(figsize=cm_to_inches((3.7,4.3)))
# plt.barh(
#     donor_metadata["Donor number"].iloc[::-1],
#     donor_metadata['BMI'].iloc[::-1],
#     edgecolor="black",
#     color="skyblue",
#     height=0.7)
# plt.xlim(20, 40)
# plt.xlabel("BMI")
# plot_overview_labels_along_y_axis(
#     fig,
#     plt.gca(),
#     labels=['older\nfemale', 'younger\nfemale', 'younger\nmale'],
#     vals_per_label=pd.Series(DONOR_GROUPS_DEMOGRAPHIC).value_counts(),
#     x_offset=13,
#     no_y_vals=len(DONOR_NAMES),
#     fontsize=6.5    
# )
# plt.tick_params(axis='y', length=0)
# plt.tight_layout()
# plt.savefig(os.path.join(plot_folder, '4_donor_bmi_bar.svg'))
# plt.show()

### Analyze donor cell type abundance

In [ ]:
# Replace "Donor" with "D" in index level 2
all_counts_by_condition.index = all_counts_by_condition.index.set_levels(
    [level.str.replace("Donor", "D") if i == 2 else level 
     for i, level in enumerate(all_counts_by_condition.index.levels)],
    level=[0, 1, 2]
)

all_counts_by_condition_summarized = all_counts_by_condition.groupby(level=[0, 2]).sum()
all_counts_by_condition_summarized = all_counts_by_condition_summarized.swaplevel(0, 1)

# Prepare data for stacked barplot
plot_data = all_counts_by_condition_summarized.unstack(level=0, fill_value=0)
plot_data = plot_data.reindex(DONOR_NAMES, axis=1)  # Order donors
plot_data = plot_data.reindex(ALL_CELL_TYPES, axis=0)  # Order cell types
plot_data = plot_data / plot_data.sum()

# Create stacked barplot of cell type counts by donor
fig, ax = plt.subplots(figsize=cm_to_inches((8, 4.3)))
# Create stacked barplot
bottom = np.zeros(len(DONOR_NAMES))
for cell_type in ALL_CELL_TYPES:
    if cell_type in plot_data.index:
        values = plot_data.loc[cell_type, DONOR_NAMES[::-1]].values
        ax.barh(DONOR_NAMES[::-1], values, left=bottom, 
               color=CELL_TYPE_COLORS[cell_type], label=cell_type, height=0.8)
        bottom += values

plot_overview_labels_along_y_axis(
    fig,
    plt.gca(),
    labels=['older\nfemale', 'younger\nfemale', 'younger\nmale'],
    vals_per_label=pd.Series(DONOR_GROUPS_DEMOGRAPHIC).value_counts(),
    color='black',
    x_offset=-0.4,
    no_y_vals=len(DONOR_NAMES),
    fontsize=6.5
)
plt.tick_params(axis='y', length=0)
ax.set_xlabel('fraction of cells')
plt.tight_layout()
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=6, ncol=1)
plt.savefig(os.path.join(plot_folder, "4_cell_type_proportions.svg"))
plt.show()

In [ ]:
cell_type_order = [
    "B cell",
    "CD4 T cell",
    "CD8 T cell",
    "Treg",
    "MAIT",
    "NKT",
    "NK CD56hi",
    "NK CD56low",
    "pDC",
    "cDC",
    "CD14 Mono",
    "CD16 Mono",
    "HSPC",
    "ILC",
    "granulocyte",
    "plasmablast"
]


# If all_counts_by_condition is a Series with MultiIndex (cell_type, ...):
g = all_counts_by_condition_unfiltered.groupby(level=0)

vals = g.median()
mad = g.apply(lambda s: np.mean(np.abs(s - s.median())))

vals = vals.reindex(cell_type_order)
mad  = mad.reindex(cell_type_order)

vals = vals.iloc[::-1]
mad  = mad.iloc[::-1]

plt.figure(figsize=cm_to_inches((4, 6)))
plt.barh(vals.index, vals.values, xerr=mad.values, color="skyblue", edgecolor="black", height=0.7, ecolor="black", capsize=0)

plt.xscale("log")
plt.axvline(x=10, color="red", linestyle="--", linewidth=1, label='minimum median cell count')
plt.xlabel("median cell count per condition\n(cell type-cytokine-donor)") 
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "4_median_cell_counts.svg"))
plt.show()

In [ ]:
# mean counts per condition across cell types
all_counts_by_condition.groupby(level=[1, 2]).sum().mean()

In [ ]:
all_counts_by_condition_norm = all_counts_by_condition.unstack().fillna(0)
all_counts_by_condition_norm = 100 * all_counts_by_condition_norm / all_counts_by_condition_norm.sum()
all_counts_by_condition_norm = all_counts_by_condition_norm.stack()

In [ ]:
zero_counts = {}
means_by_cell_type = {}
stds_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    df = all_counts_by_condition_unfiltered[cell_type][CYTOKINE_ORDER]
    mean = df.groupby(level=1).mean()
    std = df.groupby(level=1).std()
    zero_counts[cell_type] = (df == 0).sum()
    means_by_cell_type[cell_type] = mean
    stds_by_cell_type[cell_type] = std
means_by_cell_type = pd.DataFrame(means_by_cell_type).loc[DONOR_NAMES]
stds_by_cell_type = pd.DataFrame(stds_by_cell_type).loc[DONOR_NAMES]

In [ ]:
from matplotlib.ticker import ScalarFormatter

fig, axes = plt.subplots(6, 2, figsize=cm_to_inches((9, 12)), sharex=True)
axes = axes.flatten()
for i, col in enumerate(ALL_CELL_TYPES):
    means = means_by_cell_type[col]
    stds = stds_by_cell_type[col]
    ax = axes[i]
    ax.bar(means.index, means.values, yerr=stds.values, capsize=2)
    ax.set_title(col, fontsize=6)
    if i % 4 == 0:
        ax.set_ylabel("mean cell count")
    if i < 12:
        ax.tick_params(axis='x', length=0)
    ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    ax.yaxis.get_major_formatter().set_scientific(True)
    ax.yaxis.get_major_formatter().set_powerlimits((0, 0))

plt.subplots_adjust(hspace=0.75)
plt.savefig(os.path.join(plot_folder, "4_mean_cell_counts_by_cytokine.svg"), bbox_inches="tight")
plt.show()

#### Cell type abundance analysis

In [ ]:
import anndata as ad
import pertpy as pt

counts = all_counts_by_condition.copy().reset_index()
counts.columns = ["cell_type", "cytokine", "donor", "count"]
counts = counts.pivot(index=['cytokine', 'donor'], columns='cell_type', values='count').reset_index()
relevant_cols = ['cytokine', 'donor'] + ALL_CELL_TYPES
counts = counts[relevant_cols].fillna(0)

obs = counts[["cytokine", "donor"]].astype(str)
obs["screen"] = np.where(obs["cytokine"].isin(full_review_list), "review", "main")
obs.loc[obs["cytokine"] == "PBS_review", "cytokine"] = "PBS"
obs = obs.astype("category")
obs.index = obs.index.astype(str)
obs["cytokine"] = obs["cytokine"].cat.reorder_categories(
    ["PBS"] + [c for c in obs["cytokine"].cat.categories if c != "PBS"])
obs["screen"] = obs["screen"].cat.reorder_categories(["main", "review"])
data_all = ad.AnnData(X=counts[ALL_CELL_TYPES].to_numpy(dtype=float), obs=obs,
                      var=pd.DataFrame(index=ALL_CELL_TYPES))

sccoda = pt.tl.Sccoda()
sim_results_all = {}
for reference_cell_type in ["CD4 T cell"]:
    print(f"Analyzing {reference_cell_type}.")
    path = os.path.join(output_tables_folder, f"sccoda_effects_pertpy_{reference_cell_type.replace(' ', '').lower()}.parquet")
    if os.path.exists(path):
        sim_results_all[reference_cell_type] = pd.read_parquet(path)
        continue

    adata_curr = sccoda.prepare(data_all.copy(), formula="cytokine + donor + screen", reference_cell_type=reference_cell_type)
    sccoda.run_nuts(adata_curr, num_samples=5000, num_warmup=1000, rng_key=1)
    effects = sccoda.get_effect_df(adata_curr)
    effects.to_parquet(path)
    sim_results_all[reference_cell_type] = effects

In [ ]:
for reference_cell_type, results in sim_results_all.items():
    eff = results.reset_index()
    eff = eff[eff["Covariate"].str.startswith("cytokine")].copy()
    eff["cytokine"] = eff["Covariate"].str.replace("cytokine[T.", "").str.replace("]", "")
    eff = eff[~eff["cytokine"].str.endswith("_review")]

    lfc = eff.pivot(index="cytokine", columns="Cell Type", values="log2-fold change")[ALL_CELL_TYPES]
    credible = eff.pivot(index="cytokine", columns="Cell Type", values="Final Parameter")[ALL_CELL_TYPES] != 0
    keep = (credible & (lfc.abs() > 0.5)).any(axis=1)
    order = [c for c in CYTOKINE_ORDER if c in lfc.index and keep[c]]
    lfc, credible = lfc.loc[order].T, credible.loc[order].T

    fig, ax = plt.subplots(figsize=cm_to_inches((7, 4)))
    sns.heatmap(lfc, cmap="coolwarm", center=0, vmin=-1.5, vmax=1.5, ax=ax,
                cbar_kws={"label": "log2FC (expected cell fraction)", "shrink": 1})
    ys, xs = np.where(credible.values)
    
    ax.text(len(order)/2, 1.5, 'reference cell type', ha='center')
    
    ax.scatter(xs + 0.5, ys + 0.5, s=4, color="black")
    ax.set_xticks(np.arange(len(order)) + 0.5)
    ax.set_xticklabels([introduce_greek_letters(c) for c in order], fontsize=5.5)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(length=0, labelsize=6)
    plt.savefig(os.path.join(plot_folder, f"4_sccoda_effects_heatmap_{reference_cell_type.replace(' ', '').lower()}.svg"), bbox_inches="tight")
    plt.show()

### 4.1 - Donor baseline characterization

#### Clustering (all wells)

In [ ]:
from utils.processing_utils import get_raw_log2fc_to_condition_median
def filter_to_meaningful_genes(gene_list):
    gene_list = [gene for gene in gene_list if not gene in sex_gene_list] # remove sex-specific genes
    gene_list = [gene for gene in gene_list if not "ENSG" in gene] # remove genes that might not be meaningful (and possibly sex-specific, since unannotated)
    gene_list = [gene for gene in gene_list if gene in protein_coding_genes] # filter to protein-coding genes for ease of interpretation
    return gene_list

donor_pbs_ratio_to_median = get_raw_log2fc_to_condition_median(
    count_df = sum_gene_count_df_norm_by_donor.loc[:, (slice(None), "PBS", slice(None))],
    count_df_raw = sum_gene_count_df_raw_by_donor.loc[:, (slice(None), "PBS", slice(None))]
)
gene_list = filter_to_meaningful_genes(donor_pbs_ratio_to_median.index)
donor_pbs_ratio_to_median = donor_pbs_ratio_to_median[
    donor_pbs_ratio_to_median.index.isin(gene_list)
].droplevel(1, axis=1)

In [ ]:
# plot some of the ratios
for cell_type in ["CD4 T cell", "CD14 Mono"]:
    df = donor_pbs_ratio_to_median[cell_type][DONOR_NAMES]

    for donors in [("D1", "D3"), ("D3", "D4"), ("D3", "D8"), ("D3", "D10"), ("D5", "D12")]:
        donor1 = donors[0]
        donor2 = donors[1]
        series1, series2 = get_common_index(df[donor1], df[donor2])
        plt.figure(figsize=cm_to_inches((2.15, 1.85)))
        plt.scatter(series1, series2, alpha=1, s=2, rasterized=True, edgecolors="none", color=CELL_TYPE_COLORS[cell_type])

        r = np.corrcoef(series1, series2)[0,1]
        plt.text(-4, 3, f"r={r:.2f}", fontsize=6)
        plt.xlabel(f'baseline log2FC {donor1}')
        plt.ylabel(f'baseline log2FC {donor2}')
        min_x, max_x = series1.min(), series1.max()
        min_y, max_y = series2.min(), series2.max()
        plt.xlim(min_x - 1, max_x + 1)
        plt.ylim(min_y - 1, max_y + 1)
        plt.tick_params(axis='x', length=0)
        plt.tick_params(axis='y', length=0)
        plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=4))
        plt.gca().yaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=4))

        # Can figure out unexpected strong outliers here        
        # Label the outliers
        # for gene in outliers:
        #     plt.annotate(gene, (series1[gene], series2[gene]), 
        #                 xytext=(1, 1), textcoords='offset points', fontsize=6)
        
        plt.savefig(os.path.join(plot_folder, f"pbs_ratio_to_median_scatter_{cell_type}_{donor1}_{donor2}.svg"), dpi=400)
        plt.show()

In [ ]:
# plot inflammation-related genes of interest
ISGs = ["IFIT1", "IFIT2", "IFIT3", "RSAD2", "OAS1", "MX1", "TNFSF10", "IL1RN"]
for cell_type in ["B cell", "CD4 T cell", "NK CD56low", "CD14 Mono"]:
    
    vals = donor_pbs_ratio_to_median[cell_type]

    plt.figure(figsize=cm_to_inches((4, 3.2)))
    for i, (name, comm) in enumerate(DONOR_GROUPS.items()):
        vals_donors = vals.loc[ISGs, comm]
        for j, gene in enumerate(ISGs):
            x_pos = j# + (i - 0.5) * 0.35  # slight shift for different communities
            gene_vals = vals_donors.loc[gene].dropna()
            plt.scatter([x_pos] * len(gene_vals), gene_vals, 
                    color=DONOR_COLOR_SCHEME[name], alpha=1, s=10, 
                    label=name if j == 0 else "", edgecolors='black', linewidth=0.5)
            
        for donor in vals_donors.columns:
            plt.plot(ISGs, vals_donors, 
                    color=DONOR_COLOR_DIRECT[donor], alpha=0.5, linewidth=0.5)

    plt.tick_params(axis='x', length=0)
    plt.xticks(range(len(ISGs)), ISGs, rotation=90, fontsize=6)
    plt.ylabel(f"baseline log2FC", fontsize=6)
    #plt.legend(fontsize=6, loc="upper left")
    plt.tight_layout()
    plt.title(cell_type)
    plt.savefig(os.path.join(plot_folder, f"4_gene_expression_dots_donor_baseline_information_{cell_type}.svg"), dpi=300)

In [ ]:
from pyCytED.gene_set import load_cytosig_signature, run_decoupler
cytosig_signature = load_cytosig_signature(original=True)
cytokines_of_interest = ['IFN1', 'IFNG', 'TNFA', 'IL6', 'IL1B', 'IL2',
                         'IL4', 'IL10', 'TGFB1', 'GMCSF', 'IL17A', 'IL12', 'IL21', 'CD40L']

cytosig_activities = {}
for cell_type in ["B cell", "CD4 T cell", "NK CD56low", "CD14 Mono"]:
    data = donor_pbs_ratio_to_median[cell_type].copy().T
    cyto_acts, cyto_padj = run_decoupler(
        data,
        cytosig_signature
    )
    cytosig_activities[cell_type] = cyto_acts.copy()

    plt.figure(figsize=cm_to_inches((6, 6)))
    sns.heatmap(
        cyto_acts[cytokines_of_interest],
        cmap='viridis',
        center=0,
        cbar_kws={"label": "CytoSig activity", "shrink": 0.6},
        square=True,
    )
    plt.xticks(rotation=90, ha='center')
    plt.yticks(rotation=0)
    plt.title(cell_type)
    plt.tick_params(axis='both', length=0)
    apply_greek_labels_heatmap(plt.gca())
    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, f"4_cytosig_decoupler_heatmap_{cell_type.replace(' ', '_').lower()}.svg"), dpi=300)
    plt.show()

In [ ]:
sd = pd.DataFrame({ct: acts.std() for ct, acts in cytosig_activities.items()})
rng = np.random.default_rng(0)

fig, ax = plt.subplots(figsize=cm_to_inches((4.4, 3.2)))
for i, ct in enumerate(sd.columns):
    x = pd.Series(i + rng.uniform(-0.18, 0.18, len(sd)),index=sd.index,)

    ax.scatter(x, sd[ct],s=7, color="0.65", alpha=0.6, linewidths=0, zorder=2)
    ax.scatter(x["IFN1"], sd.loc["IFN1", ct],s=22, color="tab:red", edgecolors="black", linewidths=0.5,label="IFN1" if i == 0 else None, zorder=3)
    ax.scatter(x["IFNG"], sd.loc["IFNG", ct],s=22, color="tab:blue", edgecolors="black", linewidths=0.5,label="IFNG" if i == 0 else None, zorder=3)
ax.set_xticks(np.arange(len(sd.columns)), sd.columns, rotation=30)
ax.set(
    xlim=(-0.7, len(sd.columns) - 0.3),
    ylim=(0, sd.max().max() * 1.2),
    ylabel="SD of CytoSig activity\nacross donors",
)

ax.grid(axis="y", color="0.92", linewidth=0.5, zorder=0)
ax.tick_params(axis="both", length=0)
ax.tick_params(axis="both", length=0)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"4_cytosig_SD.svg"), dpi=300)
plt.show()

In [ ]:
# also plot the normalized expression values as a sanity check
for cell_type in ["B cell", "CD4 T cell", "NK CD56low", "CD14 Mono"]:
    genes = ["IFIT1", "IFIT2", "IFIT3", "RSAD2", "OAS1", "MX1", "TNFSF10", "IL1RN"]
    vals = sum_gene_count_df_norm_by_donor[cell_type]["PBS"]

    plt.figure(figsize=cm_to_inches((4.5, 3.2)))
    for i, (name, comm) in enumerate(DONOR_GROUPS.items()):
        vals_donors = vals.loc[genes, comm]
        for j, gene in enumerate(genes):
            x_pos = j + (i - 0.5) * 0.35  # slight shift for different communities
            gene_vals = vals_donors.loc[gene].dropna()
            plt.scatter([x_pos] * len(gene_vals), gene_vals, 
                    color=DONOR_COLOR_SCHEME[name], alpha=1, s=10, 
                    label=name if j == 0 else "", edgecolors='black', linewidth=0.5)

    plt.tick_params(axis='x', length=0)
    plt.xticks(range(len(genes)), genes, rotation=90, fontsize=6)
    plt.ylabel(f"log2(donor PBS cpm)", fontsize=6)
    #plt.legend(fontsize=6, loc="upper left")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, f"4_gene_expression_dots_donor_baseline_information_raw_{cell_type}.svg"), dpi=300)
    plt.show()

### Redo clustering analysis per well

In [ ]:
# compare PBS ratios to median
sum_gene_count_df_norm_pbs_by_well_median = sum_gene_count_df_norm_by_well_pbs_filter.T.groupby(level=0).median().T

donor_pbs_ratio_to_median_by_well = {}
for cell_type in ALL_CELL_TYPES:
    median_df = sum_gene_count_df_norm_pbs_by_well_median[cell_type].copy()
    
    # buncha filtering
    curr_filter = inclusive_filters_by_cell_type[cell_type] # genes passing basic expression filter
    curr_filter = filter_to_meaningful_genes(curr_filter) # filter to meaningful genes
    
    for donor in DONOR_NAMES:
        for well in ["H7", "H8", "H9", "H10", "H11", "H12"]:
            df = sum_gene_count_df_norm_by_well_pbs_filter.loc[curr_filter, (cell_type, donor, well)]
            df_filter, median_df_filter = get_common_index(df, median_df)
            ratio = np.log2(df_filter / median_df_filter)
            donor_pbs_ratio_to_median_by_well[f"{cell_type}_{donor}_{well}"] = ratio

# Convert to DataFrame with three-level column index
donor_pbs_ratio_to_median_by_well_df = pd.DataFrame(donor_pbs_ratio_to_median_by_well)

# Split the column names to create MultiIndex
cell_type_donor_well_triples = [col.split('_', 2) for col in donor_pbs_ratio_to_median_by_well_df.columns]
multi_index = pd.MultiIndex.from_tuples(cell_type_donor_well_triples, names=['cell_type', 'donor', 'well'])
donor_pbs_ratio_to_median_by_well_df.columns = multi_index

In [ ]:
cell_type = "CD4 T cell"
data = donor_pbs_ratio_to_median_by_well_df[cell_type].copy().T
data.index = [f"{donor}_{well}" for donor, well in data.index]
cyto_acts, cyto_padj = run_decoupler(
    data,
    cytosig_signature
)
cyto_acts.index = pd.MultiIndex.from_tuples([tuple(idx.split('_')) for idx in cyto_acts.index], names=['donor', 'well'])

cytokine = 'IFN1'
plt.figure(figsize=cm_to_inches((6, 4)))
sns.scatterplot(
    y=cyto_acts[cytokine].values,
    x=cyto_acts.index.get_level_values('donor'),
    hue=cyto_acts.index.get_level_values('well'),
    palette='viridis',
    size=5,
    edgecolor='black',
    linewidth=0.5
)
plt.legend(title='well', bbox_to_anchor=(1.25, 1), fontsize=6)
plt.xlabel("")
plt.ylabel(f"CytoSig activity")
plt.tick_params(axis='x', length=0)
plt.title(f"{introduce_greek_letters(cytokine)} activity in {cell_type}")
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore", message="vert: bool will be deprecated", category=PendingDeprecationWarning)

cell_types = ["CD4 T cell", "CD14 Mono", "B cell", "NK CD56low"]
activities_by_donor = {donor: [] for donor in DONOR_NAMES}
for cell_type in cell_types:
    data = donor_pbs_ratio_to_median_by_well_df[cell_type].copy().T
    data.index = [f"{donor}_{well}" for donor, well in data.index]
    cyto_acts, cyto_padj = run_decoupler(
        data,
        cytosig_signature
    )
    cyto_acts.index = pd.MultiIndex.from_tuples([tuple(idx.split('_')) for idx in cyto_acts.index], names=['donor', 'well'])
    cyto_acts = cyto_acts['IFN1']
    activities_by_donor = {donor: activities_by_donor[donor] + cyto_acts[cyto_acts.index.get_level_values('donor') == donor].values.tolist() for donor in DONOR_NAMES}
activities_by_donor = pd.DataFrame(activities_by_donor)
activities_by_donor = activities_by_donor[DONOR_ORDER_IFN_GROUPED]
colors = [DONOR_COLOR_DIRECT[d] for d in DONOR_ORDER_IFN_GROUPED]

plt.figure(figsize=cm_to_inches((6, 3.2)))
sns.boxplot(data=activities_by_donor, palette=colors, flierprops={'markerfacecolor': 'black', 'markersize': 1})

# cell type x donor, mean over the 6 PBS wells
well_means = activities_by_donor.groupby(np.repeat(cell_types, 6)).mean()  
all_group_1_acts = well_means[DONOR_GROUPS['interferon']].values.flatten()
all_group_2_acts = well_means[DONOR_GROUPS['non-interferon']].values.flatten()
stat, pval = mannwhitneyu(all_group_1_acts, all_group_2_acts, alternative="two-sided")

plt.plot([0, 3], [37.5, 37.5], color='black', linewidth=0.75)
plt.plot([1.5, 1.5], [37.5, 45], color='black', linewidth=0.75)
plt.plot([1.5, 2.5], [45, 45], color='black', linewidth=0.75)
plt.plot([4, 9], [37.5, 37.5], color='black', linewidth=0.75)
plt.plot([7.5, 7.5], [37.5, 45], color='black', linewidth=0.75)
plt.plot([6.5, 7.5], [45, 45], color='black', linewidth=0.75)
plt.text(4.5, 42.5, r"log$_{10}$(p)"+f"={np.log10(pval):.1f}", ha='center', fontsize=6)

plt.text(0.15, 0.55, f"n={len(all_group_1_acts)}", transform=plt.gca().transAxes)
plt.text(0.55, 0.75, f"n={len(all_group_2_acts)}", transform=plt.gca().transAxes)
plt.tick_params(axis='x', length=0)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.ylabel(introduce_greek_letters("CytoSig IFN1 activity"))
plt.savefig(os.path.join(plot_folder, f"4_ifnbeta_activity_by_donor_boxplot.svg"), dpi=300)
plt.show()

### Other per-donor analysis

In [ ]:
# plot a table of top and bottom genes by donor
top_10_by_donor = {}
bottom_10_by_donor = {}
for donor in DONOR_NAMES:
    vals = donor_pbs_ratio_to_median['CD4 T cell'][donor].dropna()
    vals = vals[~vals.index.str.contains('orf')]
    top_10_by_donor[donor] = vals.sort_values(ascending=False).head(10)
    bottom_10_by_donor[donor] = vals.sort_values(ascending=True).head(10)

# Create a visualization of top and bottom genes by donor
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=cm_to_inches((14.5, 6)), sharex=True)

# Prepare data for plotting
n_genes = 10  # Top/bottom 10 genes

# Top genes subplot
top_data = []
for donor in DONOR_NAMES:
    top_genes = top_10_by_donor[donor]
    for i, (gene, value) in enumerate(top_genes.items()):
        top_data.append({'donor': donor, 'gene': gene, 'value': value, 'rank': i})

top_df = pd.DataFrame(top_data)

# Create heatmap data for top genes
top_pivot = top_df.pivot_table(index='rank', columns='donor', values='value', fill_value=0)
top_gene_names = {}
for donor in DONOR_NAMES:
    top_genes = top_10_by_donor[donor]
    for i, gene in enumerate(top_genes.index):
        if i not in top_gene_names:
            top_gene_names[i] = []
        top_gene_names[i].append(gene)

# Bottom genes subplot
bottom_data = []
for donor in DONOR_NAMES:
    bottom_genes = bottom_10_by_donor[donor]
    for i, (gene, value) in enumerate(bottom_genes.items()):
        bottom_data.append({'donor': donor, 'gene': gene, 'value': value, 'rank': i})

bottom_df = pd.DataFrame(bottom_data)
bottom_pivot = bottom_df.pivot_table(index='rank', columns='donor', values='value', fill_value=0)

top_pivot = top_pivot.reindex(columns=DONOR_NAMES)
bottom_pivot = bottom_pivot.reindex(columns=DONOR_NAMES)

# Plot top genes
im1 = ax1.imshow(top_pivot.values, cmap='Reds', aspect='auto')
ax1.set_yticks(range(n_genes))
ax1.set_yticklabels([f"#{i+1}" for i in range(n_genes)])
ax1.set_title('Top 10 up- and downregulated genes by donor (CD4 T cell)', fontsize=7.5)
ax1.set_ylabel('Gene rank', fontsize=7)
ax1.tick_params(length=0)

# Add gene names as text annotations
for i in range(n_genes):
    for j, donor in enumerate(DONOR_NAMES):
        gene = top_10_by_donor[donor].index[i]
        value = top_10_by_donor[donor].iloc[i]
        ax1.text(j, i, f'{gene}', ha='center', va='center', 
                fontsize=5, color='white' if value > top_pivot.values.max()*0.7 else 'black')

# Plot bottom genes
im2 = ax2.imshow(bottom_pivot.values, cmap='Blues_r', aspect='auto')
ax2.set_yticks(range(n_genes))
ax2.set_yticklabels([f"#{i+1}" for i in range(n_genes)])
ax2.set_xticks(range(len(DONOR_NAMES)))
ax2.set_xticklabels(DONOR_NAMES, rotation=0)
ax2.set_ylabel('Gene rank', fontsize=7)
ax2.tick_params(length=0)

# Add gene names as text annotations
for i in range(n_genes):
    for j, donor in enumerate(DONOR_NAMES):
        gene = bottom_10_by_donor[donor].index[i]
        value = bottom_10_by_donor[donor].iloc[i]
        ax2.text(j, i, f'{gene}', ha='center', va='center', 
                fontsize=5, color='white' if abs(value) > abs(bottom_pivot.values.min())*0.7 else 'black')

# Add colorbars
cbar1 = plt.colorbar(im1, ax=ax1, shrink=1)
cbar1.set_label('log2(baseline FC)', fontsize=6)
cbar2 = plt.colorbar(im2, ax=ax2, shrink=1)
cbar2.set_label('log2(baseline FC)', fontsize=6)

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"top_bottom_genes_by_donor_CD4T.svg"), dpi=300)
plt.show()

In [ ]:
# do the donors cluster into groups?
cluster_cell_type = ALL_CELL_TYPES

# Within each cell type, correlate every donor–well expression profile
well_corrs_by_type = pd.concat({ct: donor_pbs_ratio_to_median_by_well_df[ct].corr(min_periods=100) for ct in cluster_cell_type})

# Average each well-pair correlation across cell types
well_corrs = well_corrs_by_type.groupby(level=[1, 2]).mean()

# Put rows and columns in the same order
well_corrs = well_corrs.reindex(columns=well_corrs.index)
well_corrs = (well_corrs + well_corrs.T) / 2

# Cluster wells using their correlations
well_communities, _ = leiden_clustering(well_corrs, resolution_parameter=0.01, invert_distance=False, seed=0)

# Give each donor–well pair its community label
well_labels = pd.Series({sample: f"comm {i+1}" for i, community in enumerate(well_communities) for sample in community}, name="community")
well_labels.index.names = ["donor", "well"]

well_counts = pd.crosstab(well_labels.index.get_level_values("donor"), well_labels).reindex(DONOR_NAMES, fill_value=0)
donor_assignment = well_counts.idxmax(axis=1).where(well_counts.max(axis=1) >= 5, "unassigned").rename("assignment")
well_counts = well_counts.loc[DONOR_ORDER_IFN_GROUPED]

In [ ]:
resolutions = 10**np.arange(-3, 0, 0.25)
sample_ids = {sample: i for i, sample in enumerate(well_corrs.index)}
well_membership, donor_assignment, community_counts = {}, {}, {}

for resolution in resolutions:
    communities, _ = leiden_clustering(well_corrs, resolution_parameter=resolution, invert_distance=False, seed=0)
    labels = {}
    for community in communities:
        community_id = min(sample_ids[sample] for sample in community)
        labels.update({sample: community_id for sample in community})

    labels = pd.Series(labels).reindex(well_corrs.index)
    labels.index.names = ["donor", "well"]
    counts = pd.crosstab(labels.index.get_level_values("donor"), labels)
    counts = counts.reindex(DONOR_ORDER_IFN_GROUPED, fill_value=0)

    well_membership[resolution] = labels
    donor_assignment[resolution] = counts.idxmax(axis=1).where(counts.max(axis=1) >= 5)
    community_counts[resolution] = len(communities)

well_membership = pd.DataFrame(well_membership)
well_order = [sample for donor in DONOR_ORDER_IFN_GROUPED for sample in well_membership.index if sample[0] == donor]
well_membership = well_membership.reindex(well_order)
donor_assignment = pd.DataFrame(donor_assignment)
colour_ids = {v: i for i, v in enumerate(np.unique(well_membership.values))}
cmap = plt.get_cmap("tab20", max(2, len(colour_ids))).copy()
cmap.set_bad("lightgrey")

values = well_membership.replace(colour_ids).to_numpy(dtype=float).T[::-1, :]
fig, ax = plt.subplots(figsize=cm_to_inches((5.5, 3)))
ax.imshow(values, aspect="auto", interpolation="nearest",cmap=cmap, vmin=0, vmax=max(1, len(colour_ids) - 1))

donors = well_membership.index.get_level_values("donor")
positions = pd.Series(np.arange(len(donors)), index=donors).groupby(level=0, sort=False)
ax.set_xticks(positions.mean(), positions.mean().index, rotation=90)
for boundary in positions.max().iloc[:-1]:
    ax.axvline(boundary + 0.5, color="white", lw=0.5)

ticks = np.arange(0, len(resolutions), 4)
ax.set_yticks(ticks, [rf"$10^{{{np.log10(resolutions[i]):.0f}}}$" for i in ticks][::-1])
ax.set_ylabel("Leiden resolution", fontsize=7)
ax.tick_params(axis="both", length=0, labelsize=6, pad=1)

plt.tight_layout(pad=0.2)
plt.savefig(os.path.join(plot_folder, "4_community_membership.svg"))
plt.show()

In [ ]:
# Stacked bargraph for donor assignment
setup_plotting_environment()
fig, ax = plt.subplots(figsize=cm_to_inches((6, 3)))
x = np.arange(len(well_counts))
bottom = np.zeros(len(well_counts))
colors = plt.get_cmap("tab20").colors
for i, community in enumerate(well_counts.columns):
    heights = well_counts[community].to_numpy()
    ax.bar(x, heights, bottom=bottom, width=0.8, color=colors[i % len(colors)], label=community, edgecolor="black", linewidth=0.3)
    bottom += heights
ax.set(xticks=x, xticklabels=well_counts.index, xlabel="", ylabel="PBS wells")
ax.tick_params(axis="x", length=0, rotation=90)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "4_pbs_community membership.svg")) 
plt.show()

In [ ]:
# Mean corr plot
fig = plt.figure(figsize=cm_to_inches((6,5)))
well_corrs_mean = well_corrs.mask(np.eye(len(well_corrs), dtype=bool))

# Average over wells on both axes, leaving one row/column per donor.
well_corrs_mean = well_corrs.mask(np.eye(len(well_corrs), dtype=bool))
well_corrs_mean = well_corrs_mean.groupby(level="donor").mean().T.groupby(level="donor").mean().T.reindex(index=DONOR_NAMES, columns=DONOR_NAMES)
well_corrs_mean = well_corrs_mean.loc[DONOR_ORDER_IFN_GROUPED][DONOR_ORDER_IFN_GROUPED]
sns.heatmap(well_corrs_mean, vmin=-1, vmax=1, cmap='coolwarm',
            cbar_kws={"label": f'Mean Pearson r of baseline log2FC', "shrink": 0.735}, square=True)

plot_overview_labels_along_y_axis(
    fig, plt.gca(),
    labels=["Other", "Group 2", "Group 1"],
    vals_per_label={"Group 1": 4, "Group 2": 6, "Other": 2},
    x_offset=-2.5,
    no_y_vals=len(well_corrs_mean),
    fontsize=6,
)

plt.tick_params(axis='x', which='both', bottom=False, top=False, length=0)
plt.tick_params(axis='y', which='both', left=False, right=False, length=0)
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "4_pbs_ratio_to_median_corr.svg")) 
plt.show()

In [ ]:
# # calculate mean corr within groups
# for name, group in DONOR_GROUPS.items():
#     corrs_group = get_upper_triangle(corrs.loc[group][group])
#     print(f"mean corr within {name}: {corrs_group.stack().mean():.3f}")
#     print(f"std corr within {name}: {corrs_group.stack().std():.3f}")

In [ ]:
mean_expr_group1_pbs = sum_gene_count_df_norm_by_donor_pbs.loc[:, (slice(None), DONOR_GROUPS['interferon'])].T.groupby(level=0).mean().T
mean_expr_group2_pbs = sum_gene_count_df_norm_by_donor_pbs.loc[:, (slice(None), DONOR_GROUPS['non-interferon'])].T.groupby(level=0).mean().T
mean_ratio_group1_group2 = mean_expr_group1_pbs / mean_expr_group2_pbs

In [ ]:
from matplotlib.font_manager import FontProperties
import gseapy as gp
msigdb = gp.Msigdb()
hallmark = msigdb.get_gmt(category="h.all", dbver="2026.1.Hs")
ISGs = (set(hallmark["HALLMARK_INTERFERON_ALPHA_RESPONSE"]) | set(hallmark["HALLMARK_INTERFERON_GAMMA_RESPONSE"]))

cell_type = "CD4 T cell"
curr_filter = inclusive_filters_by_cell_type[cell_type]
curr_filter = filter_to_meaningful_genes(curr_filter)

curr_df = np.log2(mean_ratio_group1_group2[cell_type].loc[curr_filter].dropna()).to_frame()
curr_df['raw_group1'] = mean_expr_group1_pbs[cell_type].loc[curr_filter].dropna()
curr_df['raw_group2'] = mean_expr_group2_pbs[cell_type].loc[curr_filter].dropna()
curr_df = curr_df.rename(columns={cell_type: "log2(group1/group2)"})
curr_df = curr_df.sort_values(by="log2(group1/group2)", ascending=False).head(20)

plt.figure(figsize=cm_to_inches((6, 3)))
plt.bar(curr_df.index, curr_df["log2(group1/group2)"])

ax = plt.gca()
ax.set_xticks(range(len(curr_df.index)))
ax.set_xticklabels(curr_df.index, rotation=90, fontsize=6)

for tick in ax.get_xticklabels():
    tick.set_fontfamily("DejaVu Sans")
    tick.set_fontsize(6)
    tick.set_fontweight("bold" if tick.get_text() in ISGs else "normal")
    tick.set_rotation(90)
    tick.set_rotation_mode("anchor")
    tick.set_horizontalalignment("right")
    tick.set_verticalalignment("center")

plt.ylabel("log2(group1/group2)")
plt.tick_params(axis='x', length=0)
plt.ylim(2, 4)
plt.xlim(-1, 20)
plt.title("Top 20 differential genes", fontsize=6)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"group1_group2_ratio_barplot_{cell_type}.svg"), dpi=300)
plt.show()

### IL-32 analysis

In [ ]:
plt.figure(figsize=cm_to_inches((3.8,3.2)))

# plot all other donors
for donor in DONOR_NAMES:
    if donor == "D2":
        continue
    series_donor = donor_pbs_ratio_to_median['CD14 Mono'][donor]
    series_edger = edger_df_log2FC['CD14 Mono']['IL-32-beta']
    series_donor, series_edger = get_common_index(series_donor, series_edger)    
    plt.scatter(series_donor.iloc[::5], series_edger.iloc[::5], alpha=0.5, s=1, color="black", edgecolors="none", rasterized=True)

# plot donor D2 in color
series_donor = donor_pbs_ratio_to_median['CD14 Mono']['D2']
series_edger = edger_df_log2FC['CD14 Mono']['IL-32-beta']
series_donor, series_edger = get_common_index(series_donor, series_edger)    
plt.scatter(series_donor, series_edger, alpha=0.5, s=1, color=DONOR_COLOR_SCHEME_D2["D2"], edgecolors="none", rasterized=True)

# labeling
plt.xlabel(r'baseline log2FC')
plt.xlim(-10, 10)
plt.ylim(-10, 10)
plt.xticks([-10, -5, 0, 5, 10])
plt.ylabel(introduce_greek_letters('consensus log2FC(IL-32-beta)'))
r = np.corrcoef(series_donor, series_edger)[0,1]
plt.text(-9,7,f"r={r:.2f}", color=DONOR_COLOR_SCHEME_D2["D2"])
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "D2_CD14_Mono_scatter.svg"), dpi=400)
plt.show()

In [ ]:
all_corrs = {}
for donor in DONOR_NAMES:
    df_donor = donor_pbs_ratio_to_median.loc[:, (slice(None), donor)].droplevel(1, axis=1)
    df_edger = edger_df_log2FC.loc[:, (slice(None), 'IL-32-beta')].droplevel(1, axis=1)
    corr = df_edger.corrwith(df_donor, method="pearson")[ALL_CELL_TYPES]
    all_corrs[donor] = corr
all_corrs = pd.DataFrame(all_corrs)

plt.figure(figsize=cm_to_inches((3.8, 4.7)))
for donor_name in DONOR_NAMES:
    plt.scatter(
        all_corrs.loc[ALL_CELL_TYPES[::-1], donor_name],
        np.arange(len(ALL_CELL_TYPES)),
        alpha=1,
        s=3,
        color=DONOR_COLOR_SCHEME_D2[donor_name],
        label="D2" if donor_name == "D2" else None,
        edgecolors="none",
        rasterized=True
        )
plt.yticks(np.arange(len(ALL_CELL_TYPES)), ALL_CELL_TYPES[::-1], fontsize=6)
plt.tick_params(axis='y', length=0)
plt.xlabel(introduce_greek_letters('r(baseline log2FC, consensus log2FC)'))
plt.xticks([-0.5,0,0.5,1])
plt.title(introduce_greek_letters("IL-32-beta"), fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "donor_il32_corr_by_cell_type.svg"), dpi=400)
plt.show()

In [ ]:
all_corrs = []
for donor in DONOR_NAMES:
    df_donor = donor_pbs_ratio_to_median_by_well_df.loc[:, (slice(None), donor)].droplevel(1, axis=1)
    df_edger = edger_df_log2FC.loc[:, (slice(None), 'IL-32-beta')].droplevel(1, axis=1)

    for cell_type in ALL_CELL_TYPES:
        df_donor_cell = df_donor[cell_type].dropna()
        df_edger_cell = df_edger[cell_type].dropna()
        common_idx = df_donor_cell.index.intersection(df_edger_cell.index)
        for col in df_donor_cell.columns:
            donor_series = df_donor_cell[col].loc[common_idx]
            edger_series = df_edger_cell.loc[common_idx]
            corr = donor_series.corr(edger_series, method="pearson")
            all_corrs.append({
                "donor": donor,
                "cell_type": cell_type,
                "well": col,
                "correlation": corr
            })
all_corrs = pd.DataFrame(all_corrs)
all_corrs = all_corrs.set_index(["donor", "cell_type", "well"])["correlation"]

plt.figure(figsize=cm_to_inches((3.8, 4.7)))
for donor_name in DONOR_NAMES:
    for well in ["H7", "H8", "H9", "H10", "H11", "H12"]:
        plt.scatter(
            all_corrs.loc[(donor_name, ALL_CELL_TYPES[::-1], well)],
            np.arange(len(ALL_CELL_TYPES)),
            alpha=1,
            s=3,
            color=DONOR_COLOR_SCHEME_D2[donor_name],
            label="D2" if donor_name == "D2" else None,
            edgecolors="none",
            rasterized=True
            )
        
plt.yticks(np.arange(len(ALL_CELL_TYPES)), ALL_CELL_TYPES[::-1], fontsize=6)
plt.tick_params(axis='y', length=0)
plt.xlabel(introduce_greek_letters('r(baseline log2FC, consensus log2FC)'))
plt.xticks([-0.5,0,0.5,1])
plt.title(introduce_greek_letters("IL-32-beta"), fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "donor_il32_corr_by_cell_type_and_well.svg"), dpi=400)
plt.show()

In [ ]:
# HLA-DRA, CIITA
expression = sum_gene_count_df_norm_by_donor_pbs['CD14 Mono'].loc[['HLA-DRA', 'HLA-DRB1', 'CIITA']]
expression = expression.div(expression.max(axis=1), axis=0)[DONOR_ORDER]

plt.figure(figsize=cm_to_inches((4.5, 3.2)))
plt.scatter(expression.columns, expression.loc['HLA-DRA'], label='HLA-DRA', s=5)
plt.scatter(expression.columns, expression.loc['HLA-DRB1'], label='HLA-DRB1', s=5)
plt.scatter(expression.columns, expression.loc['CIITA'], label='CIITA', s=5)
plt.xticks(rotation=90)
plt.tick_params(axis='x', length=0)
plt.ylim(0, 1.05)
plt.ylabel("normalized expression")
plt.title("CD14 Mono")
plt.tight_layout()
plt.legend(fontsize=7, loc=[1.05,0])
plt.savefig(os.path.join(plot_folder, f"hla_dra_ciita_expression_cd14_mono.svg"), dpi=300)
plt.show()

## 4.2 - response magnitudes by donor

#### variability of relative response strengths
variability by cell type is here defined as the standard deviation of the response magnitude relative to the maximum response magnitude

In [ ]:
used_by_donor_distances = all_raw_euclidean_distances_by_donor.copy()

In [ ]:
cell_type = "CD4 T cell"
donor_pairs = [("D5", "D7"), ("D5", "D3")]
plt.figure(figsize=cm_to_inches((3.4,3)))
symbols = ['o', 's']
colors = ['black', "lightgray"]
for i, donor_pair in enumerate(donor_pairs):
    donor1 = donor_pair[0]
    donor2 = donor_pair[1]
    
    vals1 = used_by_donor_distances.loc[:, (cell_type, donor1)]
    vals2 = used_by_donor_distances.loc[:, (cell_type, donor2)]

    r = np.corrcoef(vals1, vals2)[0, 1]
    plt.scatter(vals1, vals2, s=5, edgecolors="none", color=colors[i], label=f"{donor1}-{donor2}\nr={r:.2f}", marker=symbols[i])
    
    # for debugging only
    # Add tiny text labels for each point
    # for j, cytokine in enumerate(vals1.index):
    #     plt.text(vals1.iloc[j], vals2.iloc[j], cytokine, fontsize=2, ha='center', va='center', alpha=0.7)
    
plt.xlabel("response magnitude")
plt.ylabel("response magnitude")
plt.xticks([0.1, 0.45, 0.8])
plt.yticks([0.1, 0.45, 0.8])
plt.legend(fontsize=6)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "response_mag_donor_donor.svg"))
plt.show()

In [ ]:
all_variability_by_cell_type = {}

for cell_type in ALL_CELL_TYPES:
    strong_impact = strong_impact_by_cell_type[cell_type].copy()
    distances_cell_type = used_by_donor_distances[cell_type]
    distances_cell_type = distances_cell_type.div(distances_cell_type.max(axis=1).clip(lower=0.5), axis=0)
    response_strength_variability = distances_cell_type.std(axis=1)
    strong_impact = strong_impact.loc[strong_impact]
    response_strength_variability = response_strength_variability[strong_impact.index]
    all_variability_by_cell_type[cell_type] = response_strength_variability

all_variability_by_cell_type = pd.DataFrame(all_variability_by_cell_type)
all_variability_by_cell_type = all_variability_by_cell_type.reindex(CYTOKINE_ORDER, fill_value=0)
all_variability_by_cell_type = all_variability_by_cell_type.iloc[::-1]

In [ ]:
# make a horizontal bar plot
plt.figure(figsize=cm_to_inches((1, 20)))
mean = all_variability_by_cell_type.mean(axis=1).values
std = all_variability_by_cell_type.std(axis=1).values
min_val = mean[mean > 0].min()
max_val = mean.max()

plt.barh(all_variability_by_cell_type.index, mean, color="black")
plt.errorbar(mean, range(len(mean)), xerr=std, fmt='none', color='lightgray', alpha=1, capsize=0, capthick=0.5)

plt.ylim(-1, len(CYTOKINE_ORDER))

fig = plt.gcf()
fig.canvas.draw()
apply_greek_labels_heatmap(plt.gca())
plt.title("donor\nresponse\nstrength\nvariability", fontsize=7)
plt.xlim(min_val-0.1*(max_val-min_val), max_val+0.1*(max_val-min_val))
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"4.2_response_strength_variability.{format}"), bbox_inches="tight", dpi=300)
plt.show()

#### Pearson r of response magnitudes between donors (per cell type)

In [ ]:
records = []
for cell_type in ALL_CELL_TYPES:
    corr = used_by_donor_distances[cell_type].corr()
    donors = corr.index
    for i, d1 in enumerate(donors):
        for j, d2 in enumerate(donors):
            if i == j: # skip diagonal
                continue
            records.append({
                "cell_type": cell_type,
                "donor1": d1,
                "donor2": d2,
                "correlation": corr.loc[d1, d2]
            })
long_df = pd.DataFrame(records)
long_df = long_df.drop_duplicates(subset=["cell_type", "donor1", "donor2"])

In [ ]:
fig, axes = plt.subplots(len(DONOR_NAMES), len(DONOR_NAMES), 
                         figsize=cm_to_inches((5, 10)),
                         sharex=True, sharey=True)

for i, d1 in enumerate(DONOR_ORDER_IFN_GROUPED):
    for j, d2 in enumerate(DONOR_ORDER_IFN_GROUPED):
        ax = axes[i, j]

        if j > i:   # upper triangle blank
            ax.axis("off")
            continue
        elif i == j:   # diagonal blank
            ax.axis("off")
            continue

        # get canonical (sorted) donor pair
        dmin, dmax = sorted([d1, d2])
        subset = long_df[(long_df["donor1"] == dmin) & (long_df["donor2"] == dmax)]
        if len(subset) == 0:
            subset = long_df[(long_df["donor1"] == dmax) & (long_df["donor2"] == dmin)]

        # mean line
        mean = subset["correlation"].mean()
        ax.plot([-0.3, 0.3], [mean, mean], color="black", linestyle="-", linewidth=1.5)

        # scatter with jitter
        for ct, row in subset.groupby("cell_type"):
            jitter = np.random.uniform(-0.2, 0.2, size=len(row))
            ax.scatter(
                jitter, 
                row["correlation"],
                color=CELL_TYPE_COLORS.get(ct, "grey"),
                s=10,
                alpha=0.8,
                edgecolors="none"
            )

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlim(-0.5, 0.5)
        ax.set_ylim(0, 1)
        if j == 0:
            ax.set_ylabel(d1)
        if i == len(DONOR_NAMES)-1:
            ax.set_xlabel(d2, rotation=0)

fig.text(-0.05, 0.5, 'Pearson r of response magnitudes [0, 1]', va='center', rotation=90, fontsize=7)

plt.tight_layout()
plt.subplots_adjust(wspace=0.075, hspace=0.05)
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"4_correlation_of_response_magnitudes.{format}"), dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, 
    figsize=cm_to_inches((2.35, 3.0)),
    sharex=False, sharey=False)

donor_group_1 = DONOR_GROUPS['interferon']
donor_group_2 = DONOR_GROUPS['non-interferon']
group_name_1 = ["IFN"]
group_name_2 = ["non-IFN"]
for i, groups in enumerate(itertools.product([donor_group_1, donor_group_2], [donor_group_1, donor_group_2])):
    vals = long_df[long_df['donor1'].isin(groups[0])]
    vals = vals[vals['donor2'].isin(groups[1])]
    mean = vals['correlation'].mean()
    
    row = i % 2
    col = i // 2
    ax = axes[row, col]
    ax.plot([-0.3, 0.3], [mean, mean], color="black", linestyle="-", linewidth=1.5)

    # scatter with jitter
    mean_by_cell_type = vals.groupby("cell_type")['correlation'].mean()
    for ct, val in mean_by_cell_type.items():
        jitter = np.random.uniform(-0.2, 0.2)
        ax.scatter(jitter, val, color=CELL_TYPE_COLORS[ct], s=10, alpha=0.8, edgecolors="none")

    ax.set_xticks([])
    ax.set_xlim(-0.5, 0.5)
    ax.set_ylim(0, 1)
    
    if i > 1:  # Right plots
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position("right")
        ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
        ax.set_yticklabels(['0', '0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=6)
    else:
        ax.yaxis.set_ticks_position("none")        # no ticks at all
        ax.set_yticks([])                          # enforce no ticks
        ax.set_yticklabels([])                     # enforce no labels
    
    # Set group names based on which groups are being compared
    if i % 2 == 1:
        if groups[0] == donor_group_1:
            x_label = group_name_1[0]
        else:
            x_label = group_name_2[0]
        ax.set_xlabel(x_label)
    
    if i < 2:
        if groups[1] == donor_group_1:
            y_label = group_name_1[0]
        else:
            y_label = group_name_2[0]
        ax.set_ylabel(y_label)
        
plt.savefig(os.path.join(plot_folder, "Response_mag_by_group.svg"))
plt.show()

In [ ]:
vals = long_df[long_df['donor1'].isin(donor_group_1)]
vals = vals[vals['donor2'].isin(donor_group_1)]
mean = vals['correlation'].mean()
std = vals['correlation'].std()
print(f"mean corr. in group 1: {mean:.3f} ± {std:.3f}")

vals = long_df[long_df['donor1'].isin(donor_group_2)]
vals = vals[vals['donor2'].isin(donor_group_2)]
mean = vals['correlation'].mean()
std = vals['correlation'].std()
print(f"mean corr. in group 2: {mean:.3f} ± {std:.3f}")

In [ ]:
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

stats_by_cell_type = []
for cell_type in ["B cell", "CD4 T cell", "NK CD56low", "CD14 Mono"]:
    print(cell_type)
    cell_df = used_by_donor_distances[cell_type]
    
    # divide by the 95th percentile to put the response magnitudes of differen cell types on the same scale
    percentile = np.percentile(cell_df.values.flatten(), 95)
    cell_df = cell_df / percentile
    
    # use low receptor expression cytokines to estimate a noise floor (different donors have different cell numbers)
    assert len(low_receptor_expression) == 20
    noise = cell_df.loc[low_receptor_expression]
    cell_df = cell_df - noise.mean()

    group_1_df = cell_df[donor_group_1]
    group_2_df = cell_df[donor_group_2]

    group_1_distances = group_1_df.mean(axis=1)
    group_2_distances = group_2_df.mean(axis=1)
    effects = group_1_distances - group_2_distances

    # use the mean across groups as an independent filter for pre-selection
    distances_cell_type = all_normalized_distances_used[cell_type]
    distances_cell_type = distances_cell_type.sort_values(ascending=False)
    pvals = {}
    for cytokine in distances_cell_type.head(20).index:
        g1 = group_1_df.loc[cytokine].values
        g2 = group_2_df.loc[cytokine].values
        stat, p = ttest_ind(g1, g2, equal_var=False)
        pvals[cytokine] = p
    pvals = pd.Series(pvals)
    q = multipletests(pvals.values, method='fdr_bh')[1]
    qs = pd.Series(q, index=pvals.index)

    stats_by_cell_type.append(pd.DataFrame({
        "p_value": pvals,
        "q_value": qs,
        "effect_size": effects,
        "cell_type": cell_type
    }))

stats_by_cell_type = pd.concat(stats_by_cell_type, axis=0).reset_index().rename(columns={"index": "cytokine"}).dropna()
effects_by_cell_type = stats_by_cell_type.pivot(index="cytokine", columns="cell_type", values="effect_size")
qs_by_cell_type = stats_by_cell_type.pivot(index="cytokine", columns="cell_type", values="q_value")

In [ ]:
# Prepare data for plotting
mean_vals_series = effects_by_cell_type.mean(axis=1).sort_values()
mean_vals_data = mean_vals_series.to_numpy()

std_vals_series = effects_by_cell_type.std(axis=1)
std_vals_series = std_vals_series[mean_vals_series.index]
std_vals_data = std_vals_series.to_numpy()

# Calculate y1 and y2 for the shaded area
y1_data = mean_vals_data - std_vals_data
y2_data = mean_vals_data + std_vals_data

In [ ]:
from utils.unicode_utils import apply_greek_labels
plt.figure(figsize=cm_to_inches((6.5,3.5)))

x_plot_indices = np.arange(len(mean_vals_series))

# Plot mean line using NumPy arrays
plt.plot(x_plot_indices, mean_vals_data, '-', markersize=2, color="black")

# Plot standard deviation as shaded area
plt.fill_between(x_plot_indices, 
                 y1_data,
                 y2_data,
                 alpha=0.3,
                 color='gray',
                 edgecolors="none")

for cell_i, cell_type in enumerate(effects_by_cell_type.columns):
    series = effects_by_cell_type[cell_type].loc[mean_vals_series.index]
    plt.scatter(x_plot_indices, series.to_numpy(), s=8, 
                color=CELL_TYPE_COLORS[cell_type], alpha=1, label=f"{cell_type}", zorder=2, edgecolor="none")

    qs = qs_by_cell_type[cell_type].loc[mean_vals_series.index]
    significant = qs < 0.05
    plt.scatter(x_plot_indices[significant], [0.9 - 0.1 * cell_i]*len(x_plot_indices[significant]), s=12, 
                color=CELL_TYPE_COLORS[cell_type], alpha=1, marker="*")

# visually separate out the stars
plt.plot([-1, len(x_plot_indices)], [0.525, 0.525], color='black', linewidth=0.5, ls='--')
plt.plot([-1, len(x_plot_indices)], [0, 0], color='black', linewidth=0.5)

plt.xticks(x_plot_indices, mean_vals_series.index, rotation=90, ha='center', fontsize=6)
plt.ylabel('response magnitude above\nnoise floor (group 1-group 2)')
plt.margins(x=0.01) # remove whitespace on left and right

plt.xlim(-0.5,len(x_plot_indices) - 0.5)
plt.ylim(-0.7, 1)  # Set y-axis limits
plt.gca().set_yticks([-0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6])

apply_greek_labels(plt.gca())
plt.legend(loc="lower right", ncol=2, fontsize=6)
plt.tick_params(axis='x', length=0)
plt.tick_params(axis='y', length=0)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "4.2_group1_group2_discrepancy_long.svg"))
plt.show()

In [ ]:
plt.figure(figsize=cm_to_inches((1.0, 3.7)))
cell_type_means = long_df.groupby("cell_type")['correlation'].mean()[ALL_CELL_TYPES].iloc[::-1]
errorbar = long_df.groupby("cell_type")['correlation'].std()[ALL_CELL_TYPES].iloc[::-1]
colors = [CELL_TYPE_COLORS.get(ct, "grey") for ct in cell_type_means.index]
plt.barh(cell_type_means.index, cell_type_means.values, color=colors)
plt.errorbar(cell_type_means.values, cell_type_means.index, xerr=errorbar.values, fmt='none', color='black',
             alpha=1, capsize=1.5, capthick=0.5, linewidth=1)
plt.xticks(fontsize=6)

plt.ylim(-0.75, len(cell_type_means)-0.25)
plt.xlim(0, 1)
plt.xlabel("Pearson r")
plt.ylabel("")
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"4.2_correlation_of_response_magnitudes_by_cell_type.{format}"), dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=cm_to_inches((1.0, 3.7)))
donor_means = long_df.groupby("donor2")['correlation'].mean()[DONOR_ORDER_IFN_GROUPED].iloc[::-1]
errorbar = long_df.groupby("donor2")['correlation'].std()[DONOR_ORDER_IFN_GROUPED].iloc[::-1]
plt.barh(donor_means.index, donor_means.values, color="gray")
plt.errorbar(donor_means.values, donor_means.index, xerr=errorbar.values, fmt='none', color='black',
             alpha=1, capsize=1.5, capthick=0.5, linewidth=1)
plt.xticks(fontsize=6)

plt.ylim(-0.75, len(donor_means)-0.25)
plt.xlim(0, 1)
plt.xlabel("Pearson r")
plt.ylabel("")
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"4.2_correlation_of_response_magnitudes_by_donor.{format}"), dpi=300)
plt.show()

## 4.3 - Donor-specific fold-change analysis

In [ ]:
# an example of consensus vs donor-specific correlations
cell_type = "CD4 T cell"
cytokine = "IFN-beta"
fc_edger = edger_df_log2FC[cell_type][cytokine].copy().dropna()
fc_donors = raw_log2fc_values_by_donor[cell_type, cytokine].dropna()

common_index = fc_donors.index.intersection(fc_edger.index)
fc_donors = fc_donors.loc[common_index]
fc_edger = fc_edger.loc[common_index]

plt.figure(figsize=cm_to_inches((5, 5)))
plt.scatter(fc_edger, fc_donors['D8'], s=3, color="tab:red", edgecolors="none", alpha=0.7,
            label="D8, r={:.2f}".format(np.corrcoef(fc_edger, fc_donors['D8'])[0,1]), rasterized=True)
plt.scatter(fc_edger, fc_donors['D1'], s=3, color="tab:blue", edgecolors="none", alpha=0.7,
            label="D1, r={:.2f}".format(np.corrcoef(fc_edger, fc_donors['D1'])[0,1]), rasterized=True)
plt.scatter(fc_edger, fc_donors['D3'], s=3, color="tab:olive", edgecolors="none", alpha=0.7,
            label="D3, r={:.2f}".format(np.corrcoef(fc_edger, fc_donors['D3'])[0,1]), rasterized=True)
plt.xlabel(introduce_greek_letters(f'edgeR log2FC'))
plt.ylabel('donor log2FC')
plt.tight_layout()
plt.legend(fontsize=6)
plt.savefig(os.path.join(plot_folder, f"scatter_highlight_{cell_type}_{cytokine}.svg"), dpi=400)
plt.show()

In [ ]:
# correlation between donor-specific fc and edger fc
donor_corrs_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    donor_corrs_by_cytokine = {}
    for cytokine in CYTOKINE_ORDER:
        # get both
        fc_edger = edger_df_log2FC[cell_type][cytokine].copy()
        fc_donors = raw_log2fc_values_by_donor[cell_type, cytokine]

        # correlate
        correlations = fc_donors.corrwith(fc_edger, method="pearson")
        donor_corrs_by_cytokine[cytokine] = correlations
    donor_corrs_by_cytokine = pd.DataFrame(donor_corrs_by_cytokine).T
    donor_corrs_by_cell_type[cell_type] = donor_corrs_by_cytokine

In [ ]:
# mean of the donor-specific - edgeR correlations by cell type
mean_corrs_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    # calculate mean corr for strong impact cytokines only (not entirely necessary but probably more meaningful)
    strong_impact = strong_impact_by_cell_type[cell_type]
    strong_impact = strong_impact[strong_impact].index
    corrs = donor_corrs_by_cell_type[cell_type].loc[strong_impact]
    mean_corrs_by_cell_type[cell_type] = corrs.mean(axis=1)
    
mean_corrs_by_cell_type = pd.DataFrame(mean_corrs_by_cell_type)
mean_corrs_by_cell_type = mean_corrs_by_cell_type.dropna(axis=0, how="all")

In [ ]:
mean = mean_corrs_by_cell_type.mean(axis=1)
std = mean_corrs_by_cell_type.std(axis=1)

mean = mean.reindex(CYTOKINE_ORDER[::-1]).fillna(0)
std = std.reindex(CYTOKINE_ORDER[::-1]).fillna(0)

# make a horizontal bar plot
plt.figure(figsize=cm_to_inches((1, 20)))
min_val = mean[mean > 0].min()
max_val = mean.max()

plt.barh(mean.index, mean, color="black")
plt.errorbar(mean, range(len(mean)), xerr=std, fmt='none', color='lightgray', alpha=1, capsize=0, capthick=0.5)

plt.ylim(-1, len(CYTOKINE_ORDER))

fig = plt.gcf()
fig.canvas.draw()
apply_greek_labels_heatmap(plt.gca())
plt.title("mean corr", fontsize=7)
plt.xlim(min_val-0.1*(max_val-min_val), max_val+0.1*(max_val-min_val))
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"4.2_mean_corr_variability.{format}"), bbox_inches="tight", dpi=300)
plt.show()

#### Find outlier donors for particular cytokine - cell type combinations

In [ ]:
WIDTH_FACTOR_DONOR_OUTLIERS = 0.35

donor_corr_outliers_by_cell_type = {}
threshold_by_cell_type = {}
top8_median = {}
for cell_type in ALL_CELL_TYPES:
    outliers_by_cytokine = {}
    threshold_by_cytokine = {}
    for cytokine in donor_corrs_by_cell_type[cell_type].index:
        median = donor_corrs_by_cell_type[cell_type].loc[cytokine].sort_values(ascending=False).dropna()
        if len(median) < 8:
            print(f"Insufficient donor values for {cell_type}, {cytokine}")
            median = np.nan
        else:
            median = median.head(8).median()
        top8_median[f"{cell_type}_{cytokine}"] = median
        
        non_outlier_range = WIDTH_FACTOR_DONOR_OUTLIERS * max(median, 0.6)
        threshold_by_cytokine[cytokine] = median - non_outlier_range
        
        dev_from_median = donor_corrs_by_cell_type[cell_type].loc[cytokine] - median
        outliers = dev_from_median < -non_outlier_range
        # only compute for strong impact cytokines - somewhat meaningless otherwise
        outliers = outliers & strong_impact_by_cell_type.loc[cytokine, cell_type]
        outliers_by_cytokine[cytokine] = outliers
        
    threshold_by_cell_type[cell_type] = pd.Series(threshold_by_cytokine)
    outliers_by_cytokine = pd.DataFrame(outliers_by_cytokine)
    donor_corr_outliers_by_cell_type[cell_type] = outliers_by_cytokine
threshold_by_cell_type = pd.DataFrame(threshold_by_cell_type)
    
# Build a DataFrame cytokine x cell_type with counts
corr_outliers_by_cell_type_and_cytokine = []
corr_outliers_by_cell_type_and_donor = {}
for cell_type in ALL_CELL_TYPES:
    corr_outliers_by_cell_type_and_donor[cell_type] = donor_corr_outliers_by_cell_type[cell_type].sum(axis=1)
    curr_outliers = donor_corr_outliers_by_cell_type[cell_type].sum(axis=0)
    corr_outliers_by_cell_type_and_cytokine.append(curr_outliers.rename(cell_type))

corr_outliers_by_cell_type_and_cytokine = pd.concat(corr_outliers_by_cell_type_and_cytokine, axis=1).fillna(0)
corr_outliers_by_cell_type_and_donor = pd.DataFrame(corr_outliers_by_cell_type_and_donor).fillna(0)

# Ensure cytokine order matches CYTOKINE_ORDER
corr_outliers_by_cell_type_and_cytokine = corr_outliers_by_cell_type_and_cytokine.reindex(CYTOKINE_ORDER)    

In [ ]:
# visually verify for a specific cell type to make sure this algorithm works as expected
cell_type = "CD14 Mono"

fig, ax = plt.subplots(figsize=cm_to_inches((4, 10)))
corrs = donor_corrs_by_cell_type[cell_type].copy()
strong_impact = strong_impact_by_cell_type[cell_type].copy()
strong_impact = strong_impact[strong_impact].index
corrs = corrs.loc[strong_impact, DONOR_ORDER_IFN_GROUPED]

sns.heatmap(corrs, vmin=0, vmax=1, cmap="viridis", cbar=False)
ax.tick_params(axis='y', length=0, labelsize=5.5)
ax.tick_params(axis='x', length=0, labelsize=5.5)
ax.set_yticks(np.arange(0.5, len(corrs.index)))
ax.set_yticklabels(corrs.index, rotation=0)
apply_greek_labels_heatmap(ax)

outliers_curr = donor_corr_outliers_by_cell_type[cell_type].T.copy()
for cytokine, donor in outliers_curr.stack().index:
    if outliers_curr.loc[cytokine, donor]:
        cytokine_idx = corrs.index.get_loc(cytokine)
        donor_idx = corrs.columns.get_loc(donor)
        ax.add_patch(plt.Rectangle((donor_idx, cytokine_idx), 1, 1, fill=False, edgecolor='red', lw=0.8))

plt.title(cell_type, fontsize=7)
plt.show()

In [ ]:
# visualize the top8 median correlations for only the cytokines with at least one strong impact
# this basically shows that most donors agree
top8_median = pd.Series(top8_median)
top8_median.index = top8_median.index.str.split("_", expand=True)
top8_median = top8_median.unstack()

any_strong = strong_impact_by_cell_type.any(axis=1)
any_strong = any_strong[any_strong].index
top8_median=top8_median[any_strong]
plt.figure(figsize=cm_to_inches((18,2)))
for cell_type in ALL_CELL_TYPES:
    strong_impact = strong_impact_by_cell_type[cell_type].copy()
    top8_strong = top8_median.loc[cell_type]
    top8_strong = top8_strong[top8_strong.index.isin(strong_impact[strong_impact].index)]
    top8_strong = top8_strong.reindex(any_strong)
    plt.scatter(top8_strong.index, top8_strong.values, s=5, color=CELL_TYPE_COLORS[cell_type], edgecolors="none")
plt.tick_params(axis='x', length=0)
plt.xlim(-0.5, len(any_strong)-0.5)
plt.xticks(rotation=90, fontsize=5.5)
plt.yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0], fontsize=6)
apply_greek_labels_heatmap(plt.gca())
plt.ylim(0, 1)
plt.show()

In [ ]:
size_scale = 0.5  # same scale used everywhere for s=count*size_scale
fig, (ax_main, ax_leg) = plt.subplots(1, 2, figsize=cm_to_inches((6, 4)), gridspec_kw={'width_ratios': [5, 2]}, constrained_layout=True)

# scatterplot
for i, donor in enumerate(DONOR_NAMES):
    for j, cell_type in enumerate(ALL_CELL_TYPES[::-1]):
        count = corr_outliers_by_cell_type_and_donor.loc[donor, cell_type]
        if count > 0:
            ax_main.scatter(i, j, s=count * size_scale, alpha=1, c=CELL_TYPE_COLORS[cell_type], edgecolors="none")

ax_main.set_yticks(range(len(ALL_CELL_TYPES)))
ax_main.set_yticklabels(ALL_CELL_TYPES[::-1], rotation=0)
ax_main.set_xticks(range(len(DONOR_NAMES)))
ax_main.set_xticklabels(DONOR_NAMES, rotation=90)
ax_main.tick_params(length=0)

# legend
example_counts = [1, 5, 10, 25, 50, 100]
for k, count in enumerate(reversed(example_counts)):
    y = k
    ax_leg.scatter(0, y, s=count * size_scale, alpha=1, c='gray', edgecolors='none')
    ax_leg.text(0.6, y, str(count), va='center', ha='left', fontsize=6)
ax_leg.set_title('outlier count', fontsize=7, pad=6)
ax_leg.set_xlim(-0.5, 2.5)
ax_leg.set_ylim(-0.5, len(example_counts) - 0.5)
ax_leg.axis('off')

for fmt in ['png', 'svg']:
    fig.savefig(os.path.join(plot_folder, f"corr_outliers_by_donor_x_cell_type_with_legend.{fmt}"),dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=cm_to_inches((18, 5)))

offset = 0
yticks = []
yticklabels = []

for index, cell_type in enumerate(ALL_CELL_TYPES[::-1]):
    curr_outliers = corr_outliers_by_cell_type_and_cytokine[cell_type]
    curr_outliers = curr_outliers.reindex(CYTOKINE_ORDER).fillna(0)

    # plot bars with vertical offset
    plt.bar(curr_outliers.index, curr_outliers.values, bottom=offset, color=CELL_TYPE_COLORS[cell_type], width=0.8, label=cell_type)
    plt.axhline(y=offset,linewidth=0.5,color="black")
    plt.text(-2-index%2, offset+max(0,max(curr_outliers.values)/2-1), f"{max(curr_outliers.values)}", fontsize=5.5)

    # track midpoint for yticks
    yticks.append(offset + curr_outliers.max() / 2)
    yticklabels.append(cell_type)

    # increment offset for next cell type
    offset += curr_outliers.max() + 0.5   # +1 for spacing

plt.xticks(rotation=90, fontsize=5.5)
plt.yticks([])# yticks, yticklabels)
plt.ylabel("outlier donors", labelpad=15)
plt.xlim(-1, len(CYTOKINE_ORDER))
apply_greek_labels_heatmap(plt.gca())

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"stacked_barplot_donor_outliers.{format}"))
plt.show()

In [ ]:
# mask things without strong impact
corr_outliers_by_cell_type_and_cytokine[~strong_impact_by_cell_type] = np.nan
# remove cytokines without any strong impact
keep = strong_impact_by_cell_type.drop("HSPC", axis=1).any(axis=1)
keep = keep[keep].index
corr_outliers_by_cell_type_and_cytokine = corr_outliers_by_cell_type_and_cytokine.loc[keep]

# remove HSPCs (too frail to detect outliers reliably)
mean = corr_outliers_by_cell_type_and_cytokine.drop("HSPC", axis=1).mean(axis=1)
std = corr_outliers_by_cell_type_and_cytokine.drop("HSPC", axis=1).std(axis=1)

plt.figure(figsize=cm_to_inches((12, 3)))

plt.bar(mean.index, mean, color="black")
plt.errorbar(mean.index, mean, yerr=std, color="grey", fmt='none')

plt.xticks(rotation=90, fontsize=5.5)
plt.ylabel("mean # of outlier donors per cell type", labelpad=15)
plt.xlim(-1, len(mean))
plt.ylim(0, 6)
apply_greek_labels_heatmap(plt.gca())

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"stacked_barplot_donor_outliers_mean.{format}"))
plt.show()

In [ ]:
mean = mean[mean > 0]
std = std.loc[mean.index]

# parameters
n_panels = 5
n = len(mean)
chunk = int(np.ceil(n / n_panels))

# split into chunks
mean_chunks = [mean.iloc[i*chunk:(i+1)*chunk] for i in range(n_panels)]
std_chunks  = [std.iloc[i*chunk:(i+1)*chunk]  for i in range(n_panels)]

# reverse all but the last panel for readability; keep the last as-is
for i in range(n_panels):
    mean_chunks[i] = mean_chunks[i].iloc[::-1]
    std_chunks[i]  = std_chunks[i].iloc[::-1]
panels = list(zip(mean_chunks, std_chunks))

fig, axes = plt.subplots(1, n_panels, figsize=cm_to_inches((9, 3)), sharex=True)
axes = axes.flatten()
# ensure equal y scaling for subplots
ymax = max(len(m) for m, _ in panels)
for i, (m, s) in enumerate(panels):
    ax = axes[i]

    # Calculate top padding (for top alignment)
    pad = ymax - len(m)
    y_pos = np.arange(pad, pad + len(m))

    # Draw bars and errors
    ax.barh(y_pos, m.values, color="black", height=0.6)
    ax.errorbar(m.values, y_pos, xerr=s.values, color="grey", fmt='none')

    # Add yticks and labels (blank for padding)
    ax.set_yticks(np.arange(ymax))
    tick_labels = [""] * pad + list(m.index)
    ax.set_yticklabels(tick_labels, fontsize=5.5)

    if i == 2:
        ax.set_xlabel("outlier donors, mean across cell types with strong impact", labelpad=5)
    ax.set_ylim(-0.5, ymax - 0.5)
    
    apply_greek_labels_heatmap(ax)
    ax.set_xlim(0, 6)
    ax.set_xticks(range(0, 7, 2))
    ax.set_xticklabels([str(t) for t in range(0, 7, 2)], rotation=0)
    ax.tick_params(axis='both', length=0, rotation=0)

plt.subplots_adjust(wspace=1.5)
for fmt in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"barplot_donor_outliers_split{n_panels}.{fmt}"),
                bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=cm_to_inches((5, 1.75)))
plt.bar(DONOR_NAMES, corr_outliers_by_cell_type_and_donor.drop(columns="HSPC").sum(axis=1)[DONOR_NAMES], color="black", width=0.8)
plt.xlim(-0.5, len(DONOR_NAMES)-0.5)
plt.ylim(0, 200)
plt.ylabel("total outliers\nby donor")
plt.tick_params(axis='x', length=0, rotation=90)
plt.tight_layout()
for format in ['png', 'svg']:
    plt.savefig(os.path.join(plot_folder, f"corr_outliers_by_donor_x_cell_type_barplot.{format}"), dpi=300)
plt.show()

In [ ]:
major_cell_types = ["B cell", "CD4 T cell", "NK CD56low", "CD14 Mono"]
outlier_fractions = pd.DataFrame(index=DONOR_NAMES)

total_outliers = 0
total_evaluated = 0
for ct in major_cell_types:
    corrs = donor_corrs_by_cell_type[ct].reindex(columns=DONOR_NAMES)
    strong = strong_impact_by_cell_type[ct].reindex(corrs.index).fillna(False)
    threshold = threshold_by_cell_type[ct].reindex(corrs.index)

    eligible = corrs.notna()
    eligible.loc[~(strong & threshold.notna()), :] = False

    flags = donor_corr_outliers_by_cell_type[ct].T.reindex(
        index=corrs.index, columns=DONOR_NAMES
    ).fillna(False)

    outlier_fractions[ct] = (
        (flags & eligible).sum(axis=0)
        / eligible.sum(axis=0).replace(0, np.nan)
    )

    total_outliers += (flags & eligible).to_numpy().sum()
    total_evaluated += eligible.to_numpy().sum()

fig, ax = plt.subplots(figsize=cm_to_inches((7.5, 2.1)))
x = np.arange(len(DONOR_NAMES))
width = 0.19
for i, ct in enumerate(major_cell_types):
    ax.bar(x + (i - 1.5) * width,100 * outlier_fractions[ct],width=width, color=CELL_TYPE_COLORS[ct], label=ct,)

ax.set_xticks(x, DONOR_NAMES, rotation=90)
ax.set_xlim(-0.5, len(DONOR_NAMES) - 0.5)
ax.set_ylim(bottom=0)
ax.set_ylabel("% outlier\nresponses")
ax.tick_params(axis="x", length=0)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, bbox_to_anchor=(0.05, 1), loc="upper left", fontsize=6, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"fraction_outliers_by_donor_and_cell_type.{format}"), dpi=300)
plt.show()

overall_fraction = total_outliers / total_evaluated
print(f"{total_outliers}/{total_evaluated} responses were outliers ({overall_fraction:.1%})")

### Get correlation plots between different donors

In [ ]:
def get_donor_fc_vals_corr(cytokine, cell_type, fc_vals):
    """Get the correlation matrix of donor raw FC values for a given cytokine and cell type."""
    vals = fc_vals[cell_type, cytokine]
    relevant_donor_names = [donor for donor in DONOR_NAMES if donor in vals.columns]
    vals_corr = vals.corr().loc[relevant_donor_names, relevant_donor_names]
    return vals_corr

def get_donor_fc_scatterplot(
        cytokine1,
        cytokine2,
        cell_type,
        fc_vals,
        donor1,
        donor2,
        display_counts=False,
        display_r=True,
        savepath=None,
        display_plot=True,
        fixed_limits=None,
        small_plot=False
    ):
    """Get a scatterplot comparing raw FC values between two donors and two cytokines for a given cell type."""
    vals1 = fc_vals[cell_type, cytokine1]
    vals2 = fc_vals[cell_type, cytokine2]
    x = vals1[donor1].copy().dropna()
    y = vals2[donor2].copy().dropna()
    x,y = get_common_index(x,y)
    r = np.corrcoef(x, y)[0, 1]
    rho = stats.spearmanr(x, y)[0]
    
    if not fixed_limits:
        min_x = min(x.min(), y.min()) - 0.5
        max_x = max(x.max(), y.max()) + 0.5
    else:
        min_x = fixed_limits[0]
        max_x = fixed_limits[1]

    if small_plot:
        size = (2.6, 2.6)
    else:
        size = (3.5, 3.5)
    plt.figure(figsize=cm_to_inches(size))
    plt.scatter(x, y, s=1, edgecolors="none", rasterized=True, alpha=1)
    if not small_plot:
        plt.plot([min_x, max_x], [min_x, max_x], color="black", linestyle="--")

    if display_counts:
        count_x_cond = all_counts_by_condition.loc[(cell_type, cytokine1, donor1)]
        count_x_pbs = all_counts_by_condition.loc[(cell_type, "PBS", donor1)]
        count_y_cond = all_counts_by_condition.loc[(cell_type, cytokine2, donor2)]
        count_y_pbs = all_counts_by_condition.loc[(cell_type, "PBS", donor2)]
        # Add text annotations for counts
        plt.text(0.05, 0.95, f"{donor1}\n{cytokine1} {count_x_cond:.0f}\n(PBS/6) {count_x_pbs/6:.0f}", 
                transform=plt.gca().transAxes, fontsize=6, verticalalignment='top')
        plt.text(0.95, 0.05, f"{donor2}\n{cytokine2} {count_y_cond:.0f}\n(PBS/6) {count_y_pbs/6:.0f}", 
                transform=plt.gca().transAxes, fontsize=6, verticalalignment='bottom', 
                horizontalalignment='right') 

    if display_r:
        plt.text(0.05, 0.95, f"r = {r:.2f}", transform=plt.gca().transAxes, fontsize=6, verticalalignment='top')
        #plt.text(0.05, 0.80, f"ρ = {rho:.2f}", transform=plt.gca().transAxes, fontsize=6, verticalalignment='top')

    plt.xlim(min_x, max_x)
    plt.ylim(min_x, max_x)
    plt.xticks([min_x, 0, max_x])
    plt.yticks([min_x, 0, max_x])
    
    plt.tick_params(axis='x', length=0)
    plt.tick_params(axis='y', length=0)

    if cytokine1 == cytokine2:
        if not small_plot:
            plt.title(introduce_greek_letters(f"{cell_type}: {cytokine1}"), fontsize=7)
        plt.xlabel(f"log2FC ({donor1})")
        plt.ylabel(f"log2FC ({donor2})")
    else:
        if not small_plot:
            plt.title(f"{cell_type}", fontsize=7)
        plt.xlabel(introduce_greek_letters(f"log2FC ({cytokine1}, {donor1})"))
        plt.ylabel(introduce_greek_letters(f"log2FC ({cytokine2}, {donor2})"))

    plt.tight_layout()
    if savepath is not None:
        for format in [".png", ".svg"]:
            plt.savefig(savepath+format, dpi=300)
    if display_plot:
        plt.show()
    plt.close()
    

def get_donor_fc_scatterplot_to_edgeR(
        cytokine_donor,
        cytokine_edger,
        cell_type,
        fc_vals,
        donor,
        display_r=True,
        savepath=None,
        display_plot=True,
        fixed_limits=None,
        small_plot=False,
        ax = None,
    ): 
    """Get a scatterplot comparing raw FC values for a given donor and cytokine to edgeR log2FC values for a given cytokine and cell type."""
    x = fc_vals[cell_type][cytokine_donor][donor].copy().dropna()
    y = edger_df_log2FC[cell_type][cytokine_edger].copy().dropna()
        
    x,y = get_common_index(x,y)
    r = np.corrcoef(x, y)[0, 1]
    rho = stats.spearmanr(x, y)[0]
    
    if not fixed_limits:
        min_x = min(x.min(), y.min()) - 0.5
        max_x = max(x.max(), y.max()) + 0.5
    else:
        min_x = fixed_limits[0]
        max_x = fixed_limits[1]

    if small_plot:
        size = (2.6, 2.6)
    else:
        size = (3.5, 3.5)
        
    if ax is None:
        output_flag = True
        fig, ax = plt.subplots(figsize=cm_to_inches(size))
    else:
        output_flag = False
    ax.scatter(x, y, s=1, edgecolors="none", rasterized=True, alpha=1)
    if not small_plot:
        ax.plot([min_x, max_x], [min_x, max_x], color="black", linestyle="--")

    if display_r:
        ax.text(0.05, 0.95, f"r = {r:.2f}", transform=ax.transAxes, fontsize=6, verticalalignment='top')

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_x, max_x)
    ax.set_xticks([min_x, 0, max_x])
    ax.set_yticks([min_x, 0, max_x])
    
    ax.tick_params(axis='x', length=0)
    ax.tick_params(axis='y', length=0)

    if cytokine_donor == cytokine_edger:
        if not small_plot:
            ax.set_title(introduce_greek_letters(f"{cell_type}: {cytokine_donor}"), fontsize=7)
        ax.set_xlabel(f"log2FC ({donor})")
        ax.set_ylabel(f"consensus log2FC")
    else:
        if not small_plot:
            ax.set_title(f"{cell_type}", fontsize=7)
        ax.set_xlabel(introduce_greek_letters(f"log2FC ({cytokine_donor}, {donor})"))
        ax.set_ylabel(introduce_greek_letters(f"consensus log2FC ({cytokine_edger})"))

    if output_flag:
        plt.tight_layout()
        if savepath is not None:
            for fmt in [".png", ".svg"]:
                plt.savefig(savepath+fmt, dpi=300)
        if display_plot:
            plt.show()
        plt.close()
    

cell_type = "CD4 T cell"
cytokine1 = "IL-2"
cytokine2 = "IL-2"
donor1 = "D5"
donor2 = "D7"
get_donor_fc_scatterplot(cytokine1, cytokine2, cell_type, raw_log2fc_values_by_donor, donor1, donor2, small_plot=True, fixed_limits=[-4, 4])

cell_type = "CD8 T cell"
cytokine_donor = "IL-32-beta"
cytokine_edger = "IL-10"
donor = "D3"
get_donor_fc_scatterplot_to_edgeR(cytokine_donor, cytokine_edger, cell_type, raw_log2fc_values_by_donor, donor, small_plot=True, fixed_limits=[-4, 4])

In [ ]:
def get_donor_corr_heatmap(cytokine, cell_type, fc_vals,
                     del_colorbar=False, annot_heatmap=True,
                     communities=None, community_colors=None,
                     savepath=None, display_plot=True, hide_ylabel=False):
    """Get a heatmap of donor correlations for a given cytokine and cell type, with barplots of response magnitude, raw response strength, and edger correlation outliers."""
    # get values for the heatmap
    vals = fc_vals[cell_type][cytokine]
    if communities is not None:
        comms_flat = [i for x in communities for i in x]
        relevant_donor_names = comms_flat
    else:
        relevant_donor_names = [donor for donor in DONOR_NAMES if donor in vals.columns]
    vals_corr = vals.corr().loc[relevant_donor_names, relevant_donor_names]
    vals_corr = vals_corr.dropna(how='all', axis=0).dropna(how='all', axis=1)
    relevant_donor_names = vals_corr.columns.tolist()
    
    # get values for the top barplot
    response_magnitudes = all_normalized_euclidean_distances_by_donor.loc[cytokine, (cell_type, relevant_donor_names)].values
    
    # values for bottom barplot are calculated below
    
    # number of data points
    n = len(relevant_donor_names)

    fig = plt.figure(figsize=cm_to_inches((5.5, 6)), constrained_layout=False)
    gs  = fig.add_gridspec(3, 2, height_ratios=[1, 4, 1], width_ratios=[1, 0.05], hspace=0.1, wspace=0)

    ax_bar  = fig.add_subplot(gs[0, 0])
    # ax_bar2 = fig.add_subplot(gs[1, 0])
    ax_heat = fig.add_subplot(gs[1, 0])
    cax     = fig.add_subplot(gs[1, 1])
    ax_bar2 = fig.add_subplot(gs[2, 0])

    # --- Heatmap ---
    hm = sns.heatmap(
        vals_corr,
        ax=ax_heat,
        annot=annot_heatmap,
        cmap="coolwarm",
        fmt=".2f",
        square=True,
        cbar=True,
        cbar_ax=cax,
        vmin=-1,
        vmax=1,
        cbar_kws={"label": "Pearson r of donorlog2FC", },
        annot_kws={"size": 5.5}
    )
    
    # Draw black rectangles around communities
    if communities is not None:
        for comm_i, community in enumerate(communities):
            # Find the indices of this community in the current plot
            community_indices = [relevant_donor_names.index(donor) for donor in community if donor in relevant_donor_names]
            min_idx = min(community_indices)
            max_idx = max(community_indices)
            
            # Draw rectangle around the community block
            if community_colors is not None:
                color = community_colors[comm_i]
            else:
                color = 'black'
            rect = plt.Rectangle((min_idx, min_idx), max_idx - min_idx + 1, max_idx - min_idx + 1, 
                                fill=False, edgecolor=color, linewidth=2, clip_on=False)
            ax_heat.add_patch(rect)

    #ax_heat.set_xticklabels(ax_heat.get_xticklabels(), rotation=90, ha='center')
    ax_heat.set_xticklabels([])
    ax_heat.tick_params(axis='both', which='both', length=0)
    ax_heat.tick_params(axis='x', pad=2)
    ax_heat.tick_params(axis='y', pad=2)
    ax_heat.set_ylabel("")
    ax_heat.set_xlabel("")
    
    if communities is None:
        plot_overview_labels_along_y_axis(fig, ax_heat,
                                        ["Male", "Female"],
                                        {"Female": 6, "Male": 6},
                                        x_offset=-2.5,
                                        no_y_vals=12,
                                        fontsize=6, fontweight="bold", color="black", linewidth=1,
                                        rotation=90)

    # --- Bars ---
    bar_centers = np.arange(n) + 0.5
    ax_bar.bar(
        bar_centers,
        response_magnitudes,
        width=0.75,
        color="darkgray",
        align="center",
        edgecolor="black",
        linewidth=1
    )
    ax_bar.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    ax_bar.tick_params(axis='y', which='both', bottom=False, top=False, labelbottom=False, length=0)
    if hide_ylabel:
        ax_bar.set_ylabel("")
    else:
        ax_bar.set_ylabel('Relative\nResponse\nMagnitude')
    ax_bar.set_ylim(0, 1)
    ax_bar.set_yticks([0, 0.7])
    ax_bar.set_xlim(0, n)
    for ax in (ax_bar,  ax_heat):
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    # bottom barplot
    vals = donor_corrs_by_cell_type[cell_type].loc[cytokine]
    vals = vals[relevant_donor_names]
    median = vals.sort_values(ascending=False).head(8).median()
    non_outlier_range = WIDTH_FACTOR_DONOR_OUTLIERS * max(median, 0.6)
    ax_bar2.axhspan(median - non_outlier_range, 1, alpha=0.3, facecolor='gray', edgecolor="none", linewidth=0)
    x0, x1 = -0.5, len(vals)-0.5
    ax_bar2.hlines([median - non_outlier_range, 1], x0, x1, colors='black', linewidth=0.5)
    ax_bar2.bar(vals.index, vals, edgecolor="black", color="skyblue",width=0.7, zorder=3)
    min_y = 0 #max(min(vals.min(), 0.5)-0.1,0)
    max_y = 1
    ax_bar2.set_xlim(-0.5, n-0.5)
    ax_bar2.set_ylim(min_y, max_y)
    ax_bar2.set_xticks(range(len(vals)))
    ax_bar2.set_xticklabels(vals.index, rotation=90)
    ax_bar2.tick_params(axis='y', which='both', bottom=False, top=False, labelbottom=False, length=0)
    
    if hide_ylabel:
        ax_bar2.set_ylabel("")
    else:
        ax_bar2.set_ylabel('Pearson r\nto\noverall\nlog2FC')

    # --- Manual Alignment after Drawing ---
    # First, force a draw of the canvas to finalize the initial layout
    fig.canvas.draw()

    # Get the final positions of the axes in figure coordinates (values from 0 to 1)
    pos_heat = ax_heat.get_position()
    pos_bar = ax_bar.get_position()
    # pos_bar2 = ax_bar2.get_position()
    pos_bar3 = ax_bar2.get_position()

    # Set the bar plot's position to match the heatmap's horizontal extent.
    # The format is [left, bottom, width, height].
    ax_bar.set_position([pos_heat.x0, pos_bar.y0, pos_heat.width, pos_bar.height])
    # ax_bar2.set_position([pos_heat.x0, pos_bar2.y0, pos_heat.width, pos_bar2.height])
    ax_bar2.set_position([pos_heat.x0, pos_bar3.y0, pos_heat.width, pos_bar3.height])
    
    # Set overall figure title
    fig.suptitle(f'{introduce_greek_letters(cytokine)} on {cell_type}', fontsize=7, y=0.95)
    if del_colorbar:
        cax.set_visible(False)
    
    if savepath is not None:
        for format in ["png", "svg"]:
            plt.savefig(os.path.join(savepath, f"corr_heatmap_{cytokine}_{cell_type}.{format}"), dpi=300, bbox_inches="tight")
    if display_plot:
        plt.show()
    plt.close()

get_donor_corr_heatmap("IL-2", "NK CD56low", raw_log2fc_values_by_donor, annot_heatmap=False, del_colorbar=True, hide_ylabel=True)
get_donor_corr_heatmap("IL-32-beta", "CD8 T cell", raw_log2fc_values_by_donor, annot_heatmap=False, del_colorbar=True, hide_ylabel=True)

In [ ]:
cytokine = "IL-10"
cell_type = "CD4 T cell"
donor_combs = [
    ("D1", "D3"),
    ("D1", "D7"),
    ("D7", "D9")
]
for donor_comb in donor_combs:
    get_donor_fc_scatterplot(
        cytokine,
        cytokine,
        cell_type,
        raw_log2fc_values_by_donor,
        donor_comb[0],
        donor_comb[1],
        small_plot=True,
        fixed_limits=[-5,5],
        savepath=os.path.join(plot_folder, f"scatter_log2fc_{donor_comb[0]}_{donor_comb[1]}")
    )

In [ ]:
cell_type = "CD4 T cell"

limits_and_labels = {
    "IFN-beta": {"limits": [-5,5], "labels": [-4,0,4]},
    "IL-10": {"limits": [-4,4], "labels": [-3,0,3]},
}

for cytokine in ["IFN-beta", "IL-10"]:
    fig, axes = plt.subplots(1,12,figsize=cm_to_inches((18,2.4)))
    for i, donor in enumerate(DONOR_NAMES):
        ax = axes[i]
        get_donor_fc_scatterplot_to_edgeR(
            cytokine,
            cytokine,
            cell_type,
            raw_log2fc_values_by_donor,
            donor,
            small_plot=True,
            fixed_limits=limits_and_labels[cytokine]['limits'],
            ax = ax
            )
        ax.set_xticks(limits_and_labels[cytokine]['labels'])
        ax.set_yticks(limits_and_labels[cytokine]['labels'])
        ax.label_outer()
        ax.set_xlabel(donor)
    plt.figtext(0.5, -0.1, f"log2FC donor", ha='center', va='bottom', fontsize=6)    
    plt.figtext(0.5, 0.95, introduce_greek_letters(cytokine), ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    savepath=os.path.join(plot_folder, f"scatter_log2fc_{cytokine}_to_overall")
    plt.savefig(savepath+".svg", dpi=400)
    plt.show()

In [ ]:
cell_type_cytokine_pairs = [
    ("CD4 T cell", "IL-10"),
    ("CD4 T cell", "IFN-beta"),
    ("CD4 T cell", "IL-15"),
    ("NK CD56low", "IL-15"),
    ("CD8 T cell", "VEGF"),
    ("CD14 Mono", "IL-32-beta"),
    ("CD4 T cell", "IL-32-beta"),
    ("CD8 T cell", "IL-32-beta"),
    ("B cell", "CD40L")
]

for index, (cell_type, cytokine) in enumerate(cell_type_cytokine_pairs):    
    get_donor_corr_heatmap(
        cytokine,
        cell_type,
        raw_log2fc_values_by_donor,
        annot_heatmap=False,
        del_colorbar=index!=0,
        hide_ylabel=index!=0,
        savepath=plot_folder,
        communities=[DONOR_GROUPS['non-interferon'], DONOR_GROUPS['interferon'], DONOR_GROUPS['other']],
        community_colors=[DONOR_COLOR_SCHEME['non-interferon'], DONOR_COLOR_SCHEME['interferon'], DONOR_COLOR_SCHEME['other']]
        )

In [ ]:
# %%capture output
all_donor_correlation_dfs_across_cell_types = {}
for cell_type in ALL_CELL_TYPES:
    print(cell_type)

    all_correlations = {}
    for cytokine in CYTOKINE_ORDER:
        vals_corr = get_donor_fc_vals_corr(cytokine, cell_type, raw_log2fc_values_by_donor)
        # Store only the upper triangle of the correlation matrix
        n = len(vals_corr)
        upper_triangle_corrs = []
        for i in range(n):
            for j in range(i+1, n):
                upper_triangle_corrs.append(vals_corr.iloc[i, j])
        all_correlations[cytokine] = upper_triangle_corrs

    mean_correlations = {}
    std_correlations = {}
    mean_response_magnitude = {}
    std_response_magnitude = {}
    for cytokine in CYTOKINE_ORDER:
        mean_correlations[cytokine] = np.nanmean(all_correlations[cytokine])
        std_correlations[cytokine] = np.nanstd(all_correlations[cytokine])
        mean_response_magnitude[cytokine] = np.nanmean(all_normalized_euclidean_distances_by_donor.loc[cytokine, (cell_type, slice(None))])
        std_response_magnitude[cytokine] = np.nanstd(all_normalized_euclidean_distances_by_donor.loc[cytokine, (cell_type, slice(None))])
        
    mean_correlations = pd.Series(mean_correlations)
    std_correlations = pd.Series(std_correlations)
    mean_response_magnitude = pd.Series(mean_response_magnitude)
    std_response_magnitude = pd.Series(std_response_magnitude)    
    all_donor_correlation_dfs_across_cell_types[cell_type] = pd.DataFrame({
        "mean_correlations": mean_correlations,
        "std_correlations": std_correlations,
        "mean_response_magnitude": mean_response_magnitude,
        "std_response_magnitude": std_response_magnitude
    })

    # ------------- DIAGNOSTIC PLOTS -------------------
    # # ----------------------------------------------------
    # plt.figure(figsize=cm_to_inches((4.5, 3.5)))
    # plt.scatter(mean_correlations, mean_response_magnitude, s=5, color='tab:blue')
    # r = np.corrcoef(mean_correlations, mean_response_magnitude)[0, 1]

    # # find outliers
    # slope, intercept, r_value, p_value, std_err = stats.linregress(mean_correlations, mean_response_magnitude)
    # fitted_line = slope * mean_correlations + intercept

    # # Calculate residuals
    # residuals = mean_response_magnitude - fitted_line

    # # Find the strongest outlier (largest absolute residual)
    # strongest_outlier_idx = np.abs(residuals).idxmax()
    # strongest_outlier_name = strongest_outlier_idx

    # # Highlight and label the strongest outlier
    # outlier_x = mean_correlations[strongest_outlier_idx]
    # outlier_y = mean_response_magnitude[strongest_outlier_idx]
    # plt.scatter(outlier_x, outlier_y, color='tab:blue', s=20, zorder=5)
    # plt.annotate(strongest_outlier_name, (outlier_x, outlier_y), 
    #             xytext=(2, 2), textcoords='offset points', 
    #             fontsize=6, ha='left', va='bottom')

    # plt.xlim(0, 1)
    # plt.ylim(0, 1.05)
    # plt.text(0.1, 0.85, f'r = {r:.2f}', fontsize=7)
    # plt.xlabel('Mean Pearson r of donor log2FC')
    # plt.ylabel('Mean response magnitude')
    # plt.title(cell_type)
    # plt.tight_layout()
    # for format in ["png", "svg"]:
    #     plt.savefig(os.path.join(plot_folder, f"2.4_mean_correlations_vs_mean_response_magnitude_{cell_type}.{format}"), dpi=300)
        
    # ----------------------------------------------------
    # plt.figure(figsize=cm_to_inches((4.5, 3.5)))
    # plt.scatter(std_correlations, std_response_magnitude, s=5, color='tab:blue')
    # r = np.corrcoef(std_correlations, std_response_magnitude)[0, 1]

    # # find outliers
    # slope, intercept, r_value, p_value, std_err = stats.linregress(std_correlations, std_response_magnitude)
    # fitted_line = slope * std_correlations + intercept

    # # Calculate residuals
    # residuals = std_response_magnitude - fitted_line

    # # Find the strongest outlier (largest absolute residual)
    # strongest_outlier_idx = np.abs(residuals).idxmax()
    # strongest_outlier_name = strongest_outlier_idx

    # # Plot the fitted line
    # plt.plot(std_correlations, fitted_line, 'r--', alpha=0.7, linewidth=1)

    # # Highlight and label the strongest outlier
    # outlier_x = std_correlations[strongest_outlier_idx]
    # outlier_y = std_response_magnitude[strongest_outlier_idx]
    # plt.scatter(outlier_x, outlier_y, color='tab:blue', s=20, zorder=5)
    # plt.annotate(strongest_outlier_name, (outlier_x, outlier_y), 
    #              xytext=(2, 2), textcoords='offset points', 
    #              fontsize=6, ha='left', va='bottom')

    # plt.xlim(0, 1.1*max(std_correlations))
    # plt.ylim(0, 1.1*max(std_response_magnitude))
    # plt.text(0.1*max(std_correlations), 0.85*max(std_response_magnitude), f'r = {r:.2f}', fontsize=7)
    # plt.xlabel('Mean std of Pearson r of donor-specificlog2FC')
    # plt.ylabel('Std response magnitude')
    # plt.title(cell_type)
    # plt.tight_layout()
    # for format in ["png", "svg"]:
    #     plt.savefig(os.path.join(plot_folder, f"2.4_std_correlations_vs_std_response_magnitude_{cell_type}.{format}"), dpi=300)

    # ----------------------------------------------------
    # # Create a list to store data for boxplot
    # boxplot_data = []
    # cytokine_labels = []

    # # Extract data from all_correlations dictionary
    # for cytokine, correlations in all_correlations.items():
    #     strong_impact = strong_impact_by_cell_type.loc[cytokine, cell_type]
    #     if not strong_impact:
    #         continue
    #     boxplot_data.append(correlations)
    #     cytokine_labels.append(cytokine)

    # # Create boxplot
    # plt.figure(figsize=(12, 6))
    # plt.boxplot(boxplot_data, labels=cytokine_labels)
    # plt.xticks(rotation=90, ha='center')
    # plt.ylabel('Correlation Values')
    # plt.xlabel('Cytokines')
    # plt.title('Distribution of Correlations by Cytokine')
    # plt.tight_layout()
    # plt.show()   

In [ ]:
# the stronger the response, the more correlation we see (makes sense)
plt.figure(figsize=cm_to_inches((4, 3.8)))
all_x_vals = []
all_y_vals = []
all_r_vals = []
for index, cell_type in enumerate(ALL_CELL_TYPES):
    x_vals = all_donor_correlation_dfs_across_cell_types[cell_type]["mean_correlations"]
    y_vals = all_donor_correlation_dfs_across_cell_types[cell_type]["mean_response_magnitude"]

    x_vals = x_vals.dropna()
    y_vals = y_vals.dropna()
    x_vals, y_vals = get_common_index(x_vals, y_vals)
    all_x_vals.extend(x_vals)
    all_y_vals.extend(y_vals)

    plt.scatter(x_vals, y_vals, s=5, color=CELL_TYPE_COLORS[cell_type], alpha=0.75, edgecolors="none")

    r = np.corrcoef(x_vals, y_vals)[0, 1]
    all_r_vals.append(r)
    
plt.xlim(0, 0.8)
plt.ylim(0, 1.05)
plt.title(f'mean r = {np.mean(all_r_vals):.2f}', fontsize=7)
plt.xlabel('Mean Pearson r of donor log2FC')
plt.ylabel('Mean response magnitude')
#plt.legend(fontsize=7, loc="lower right")
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"4.3_mean_correlations_vs_mean_response_magnitude.{format}"), dpi=300)
plt.show()

## 4.4 - Detect sub-structure in the donor responses

In [ ]:
# Create more robuse new log2FC values for sub-structure detection
new_log2fc_main, _, _ = get_raw_log2fc_values(
    sum_gene_count_df_norm_by_donor.loc[:, ~sum_gene_count_df_norm_by_donor.columns.get_level_values(1).isin(full_review_list)],
    sum_gene_count_df_raw_by_donor.loc[:, ~sum_gene_count_df_raw_by_donor.columns.get_level_values(1).isin(full_review_list)],
    require_both_counts=True,
    min_counts=20
)
new_log2fc_review, _, _ = get_raw_log2fc_values(
    sum_gene_count_df_norm_by_donor.loc[:, sum_gene_count_df_norm_by_donor.columns.get_level_values(1).isin(full_review_list)],
    sum_gene_count_df_raw_by_donor.loc[:, sum_gene_count_df_raw_by_donor.columns.get_level_values(1).isin(full_review_list)],
    baseline_label='PBS_review',
    require_both_counts=True,
    min_counts=20
)
new_log2fc = pd.concat([new_log2fc_main, new_log2fc_review], axis=1)
del new_log2fc_main
del new_log2fc_review

In [ ]:
import warnings
warnings.filterwarnings("error", message="Mean of empty slice", category=RuntimeWarning)

plot_folder_heatmaps = os.path.join(plot_folder, "heatmaps_interesting_substructure")
os.makedirs(plot_folder_heatmaps, exist_ok=True)

def get_leiden_communities(cytokine, cell_type, fc_vals, resolution=0.5):
    distance_matrix = get_donor_fc_vals_corr(cytokine, cell_type, fc_vals).astype(float)
    # Get the distance matrix.
    df = distance_matrix.copy()
    # Remove NaN donors
    df = df.dropna(how='all', axis=0).dropna(how='all', axis=1)
    # Ensure that the diagonal is zero (i.e., no self-loops)
    np.fill_diagonal(df.values, 0)
    all_communities, _ = leiden_clustering(df, resolution, False, False)
    return all_communities

def evaluate_community_structure(cytokine, cell_type, all_donor_fc_vals, comms):
    # look into within vs across community ratio means
    vals_corr = get_donor_fc_vals_corr(cytokine, cell_type, all_donor_fc_vals)
    vals_corr = vals_corr.dropna(how='all', axis=0).dropna(how='all', axis=1)
    vals_corr = delete_diagonal(vals_corr)
    corr_comms = {}
    for comm_comb in itertools.product(comms, comms):
        name1 = "_".join(comm_comb[0])
        name2 = "_".join(comm_comb[1])
        corr = vals_corr.loc[comm_comb[0], comm_comb[1]]
        mean = np.nan if len(corr.dropna(how='all', axis=0).dropna(how='all', axis=1))==0 else np.nanmean(corr)
        corr_comms[(name1, name2)] = mean

    # Convert to DataFrame
    comm_names = ["_".join(comm) for comm in comms]
    corr_comms_df = pd.DataFrame(index=comm_names, columns=comm_names)
    for (name1, name2), value in corr_comms.items():
        corr_comms_df.loc[name1, name2] = value

    return corr_comms_df

def find_interesting_substructure(all_donor_fc_vals, margin, combos=None, make_plots=True):
    all_community_means = {}
    all_community_stds = {}
    all_community_members = {}
    all_r_vals_to_other_cytokines_by_comm = {}
    all_r_vals_to_other_cytokines_by_donor = {}
    for cytokine_idx, cytokine in enumerate(CYTOKINE_ORDER):
        if combos is None:
            print(f"Process cytokine {cytokine_idx+1}/{len(CYTOKINE_ORDER)}: {cytokine}")
            strong_impact = strong_impact_by_cell_type.loc[cytokine, ALL_CELL_TYPES]
            strong_impact = strong_impact[strong_impact].index
        else:
            strong_impact = [ct for c, ct in combos if c == cytokine]
        for cell_type in strong_impact:
            print(cell_type)
            
            if f"{cytokine}_{cell_type}" in all_community_means: # what was I dong here? avoiding running this twice?
                continue

            resolution_vals = [0.2, 0.3, 0.4, 0.6, 0.8]
            success = False
            for resolution_val in resolution_vals:
                # are there communities of sufficient size?
                print("Attempting: ", resolution_val)
                try:
                    comms = get_leiden_communities(cytokine, cell_type, all_donor_fc_vals, resolution_val)
                except Exception as e:
                    print(f"Error processing cytokine {cytokine} for cell type {cell_type}: {e}")
                    continue
                
                # Only communities of size >= 3 are eligible
                comms_pass = [comm for comm in comms if len(comm) >= 3]

                # Require at least two eligible communities
                if len(comms_pass) < 2:
                    print("Fewer than two communities of size 3 or more.")
                    continue

                # are they interesting?
                corr_comms_df = evaluate_community_structure(cytokine, cell_type, all_donor_fc_vals, comms_pass)
                keep = corr_comms_df.index[np.isfinite(np.diag(corr_comms_df.to_numpy(dtype=float)))]
                corr_comms_df = corr_comms_df.loc[keep, keep]
                if len(corr_comms_df) < 2:
                    continue
                    
                # get the margin
                margin_to_self = corr_comms_df - np.diag(corr_comms_df)
                current_margin_threshold = max(0.2, margin * corr_comms_df.max().max())

                interesting_communities = (margin_to_self < -current_margin_threshold).any(axis=0)
                interesting_communities = interesting_communities[interesting_communities].index

                # continue if the communities don't look interesting
                if len(interesting_communities) < 2:
                    print("Insufficient correlation substructure between communities.")
                    continue
                
                success = True
                final_res_val = resolution_val
                break
            if not success:
                continue
            print("Succeeded with: ", final_res_val)
            
            # ------------------------------------------------------------------
            # if yes, evaluate similarity to other cytokines
            raw_ratio_vals = all_donor_fc_vals[cell_type][cytokine]
            mean_by_community = {}
            std_by_community = {}
            community_members = {}
            for i, comm in enumerate(comms_pass):
                name = "_".join(comm)
                community_members[name] = comm
                comm_vals = raw_ratio_vals[comm]
                mean_by_community[name] = comm_vals.mean(axis=1, skipna=True) # get mean and std of ratio vals
                std_by_community[name] = comm_vals.std(axis=1, skipna=True)
            
            # see which other cytokines are most similar to the cytokine of interest (by community)
            r_vals_to_other_cytokines = {}
            for comm, cytokine_curr in itertools.product(mean_by_community.keys(), CYTOKINE_ORDER):
                mean_by_community_donor = mean_by_community[comm].dropna()
                edger_lfc = edger_df_log2FC[(cell_type, cytokine_curr)].dropna()
                mean_by_community_donor, edger_lfc = get_common_index(mean_by_community_donor, edger_lfc)
                if (len(mean_by_community_donor) < 50):
                    continue
                r = np.corrcoef(mean_by_community_donor, edger_lfc)[0, 1]
                r_vals_to_other_cytokines[(comm, cytokine_curr)] = r

            r_vals_to_other_cytokines = pd.Series(r_vals_to_other_cytokines)
            all_community_means[f"{cytokine}_{cell_type}"] = mean_by_community
            all_community_stds[f"{cytokine}_{cell_type}"] = std_by_community
            all_community_members[f"{cytokine}_{cell_type}"] = community_members
            all_r_vals_to_other_cytokines_by_comm[f"{cytokine}_{cell_type}"] = r_vals_to_other_cytokines
            
            # see which other cytokines are most similar to the cytokine of interest (by donor)
            r_vals_to_other_cytokines = {}
            for donor in DONOR_NAMES:
                if donor not in raw_ratio_vals:
                    continue
                for cytokine_curr in CYTOKINE_ORDER:
                    fc_vals = raw_ratio_vals[donor].dropna()
                    edger_lfc = edger_df_log2FC[(cell_type, cytokine_curr)].dropna()
                    fc_vals, edger_lfc = get_common_index(fc_vals, edger_lfc)
                    if len(fc_vals) < 50:
                        continue
                    r_vals_to_other_cytokines[(donor, cytokine_curr)] = np.corrcoef(fc_vals, edger_lfc)[0, 1]
            all_r_vals_to_other_cytokines_by_donor[f"{cytokine}_{cell_type}"] = r_vals_to_other_cytokines
            
            if make_plots:
                get_donor_corr_heatmap(cytokine, cell_type, all_donor_fc_vals,
                                    del_colorbar=True, annot_heatmap=False, communities=comms,
                                    savepath=plot_folder_heatmaps, display_plot=False)

    return all_community_means, all_community_stds, all_community_members, all_r_vals_to_other_cytokines_by_comm, all_r_vals_to_other_cytokines_by_donor


# this is the margin that we want between within vs across community correlations
MARGIN_THRESHOLD_FOR_SUBSTRUCTURE = 0.3

# Check if all_community_means exists in locals
if not os.path.exists(os.path.join(intermediate_output_folder, "donor_analysis_all_community_means.pkl")):
    all_community_means, all_community_stds, all_community_members, all_r_vals_to_other_cytokines_by_comm, all_r_vals_to_other_cytokines_by_donor = \
    find_interesting_substructure(new_log2fc, MARGIN_THRESHOLD_FOR_SUBSTRUCTURE)
            
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_community_means.pkl"), "wb") as f:
        pickle.dump(all_community_means, f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_community_stds.pkl"), "wb") as f:
        pickle.dump(all_community_stds, f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_community_members.pkl"), "wb") as f:
        pickle.dump(all_community_members, f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_r_vals_to_other_cytokines_by_comm.pkl"), "wb") as f:
        pickle.dump(all_r_vals_to_other_cytokines_by_comm, f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_r_vals_to_other_cytokines_by_donor.pkl"), "wb") as f:
        pickle.dump(all_r_vals_to_other_cytokines_by_donor, f)
else:
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_community_means.pkl"), "rb") as f:
        all_community_means = pickle.load(f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_community_stds.pkl"), "rb") as f:
        all_community_stds = pickle.load(f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_community_members.pkl"), "rb") as f:
        all_community_members = pickle.load(f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_r_vals_to_other_cytokines_by_comm.pkl"), "rb") as f:
        all_r_vals_to_other_cytokines_by_comm = pickle.load(f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_r_vals_to_other_cytokines_by_donor.pkl"), "rb") as f:
        all_r_vals_to_other_cytokines_by_donor = pickle.load(f)

# restore warnings    
warnings.resetwarnings()
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)

In [ ]:
# null model: one PBS well vs the pooled other five wells of the same donor
wells = ["H7", "H8", "H9", "H10", "H11", "H12"]
raw_w = sum_gene_count_df_raw_by_well_pbs.copy()
raw_w.columns = raw_w.columns.set_levels([d.replace("Donor", "D") for d in raw_w.columns.levels[1]], level=1)

null_lfc = {}
for (cell_type, donor, w) in raw_w.columns:
    if cell_type not in ALL_CELL_TYPES:
        continue
    rest = raw_w.loc[:, (cell_type, donor, [x for x in wells if x != w])].sum(axis=1, min_count=1)
    pair = pd.concat([raw_w[(cell_type, donor, w)], rest], axis=1)
    if pair.isna().all().any():  # masked well (too few cells)
        continue
    norm = normalize_df_to_rpm(pair, along_axis=0)
    lfc = np.log2(norm.iloc[:, 0]) - np.log2(norm.iloc[:, 1])
    null_lfc[(cell_type, f"null_{w}", donor)] = lfc.where((pair.iloc[:, 0] >= 20) & (pair.iloc[:, 1] >= 20))
null_lfc = pd.DataFrame(null_lfc)
null_lfc.columns.names = ["cell_type", "cytokine", "donor"]

null_combos = [(f"null_{w}", ct) for ct in ALL_CELL_TYPES for w in wells]
null_results = find_interesting_substructure(null_lfc, MARGIN_THRESHOLD_FOR_SUBSTRUCTURE,
                                             combos=null_combos, make_plots=False)
n_null = len(null_results[2])
n_real, n_tested = len(all_community_members), int(strong_impact_by_cell_type.values.sum())
print(f"null: {n_null}/{len(null_combos)} PBS-well pseudo-conditions with substructure ({n_null/len(null_combos):.1%})")
print(f"real: {n_real}/{n_tested} strong-impact conditions ({n_real/n_tested:.1%})")
print({k: list(v.keys()) for k, v in null_results[2].items()}) # which donors split, if any

In [ ]:
communities = [DONOR_GROUPS['non-interferon'], DONOR_GROUPS['interferon'], DONOR_GROUPS['other']]
cytokine = "IL-32-beta"
cell_type = "CD8 T cell"
get_donor_corr_heatmap(
    cytokine, cell_type, new_log2fc,
    del_colorbar=True, annot_heatmap=False, communities=communities,
    savepath=plot_folder, display_plot=True)

In [ ]:
internal_by_combo = {}
cross_by_combo = {}
minority_comm_by_combo = {}
majority_comm_by_combo = {}
for cytokine, cell_type in [key.split("_") for key in all_community_members.keys()]:
    comms_pass = [comm for comm in all_community_members[f"{cytokine}_{cell_type}"].values() if len(comm) > 1]

    # look into within vs across community ratio means
    vals_corr = delete_diagonal(get_donor_fc_vals_corr(cytokine, cell_type, new_log2fc))
    corr_comms = {}
    for comm_comb in itertools.product(comms_pass, comms_pass):
        name1 = "_".join(comm_comb[0])
        name2 = "_".join(comm_comb[1])
        corr = vals_corr.loc[comm_comb[0], comm_comb[1]]
        mean = np.nanmean(corr.values.flatten())
        corr_comms[(name1, name2)] = mean
        std = np.nanstd(corr.values.flatten())

    # convert to DataFrame
    comm_names = ["_".join(comm) for comm in comms_pass]
    corr_comms_df = pd.DataFrame(index=comm_names, columns=comm_names)
    corr_comms_df_std = pd.DataFrame(index=comm_names, columns=comm_names)
    for (name1, name2), value in corr_comms.items():
        corr_comms_df.loc[name1, name2] = value

    # get the margin
    margin_to_self = corr_comms_df - np.diag(corr_comms_df)
    max_margin_row, max_margin_col = margin_to_self.stack().idxmin()
    
    internal_corr_1 = corr_comms_df.loc[max_margin_row, max_margin_row]
    internal_corr_2 = corr_comms_df.loc[max_margin_col, max_margin_col]
    internal_min = min(internal_corr_1, internal_corr_2)
    cross_corr = corr_comms_df.loc[max_margin_row, max_margin_col]

    internal_by_combo[(cytokine, cell_type)] = internal_min
    cross_by_combo[(cytokine, cell_type)] = cross_corr
    
    len_1 = max_margin_row.count("_") + 1
    len_2 = max_margin_col.count("_") + 1
    smaller_comm = max_margin_row if len_1 < len_2 else max_margin_col
    larger_comm = max_margin_col if len_1 < len_2 else max_margin_row
    minority_comm_by_combo[(cytokine, cell_type)] = smaller_comm
    majority_comm_by_combo[(cytokine, cell_type)] = larger_comm

In [ ]:
counts_minority_by_donor = pd.Series(0, index=DONOR_NAMES)
for combo in minority_comm_by_combo.keys():
    donors = minority_comm_by_combo[combo].split("_")
    counts_minority_by_donor[donors] += 1
counts_minority_by_donor = counts_minority_by_donor.iloc[::-1]

plt.figure(figsize=cm_to_inches((3.5, 5)))
counts_minority_by_donor.plot(kind="barh", color="salmon", edgecolor="black", width=0.7)
plt.xlabel("Count in smaller community\nfor substantial substructure")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"4.4_minority_community_counts_by_donor.svg"), dpi=300)
plt.show()

counts_minority_by_donor_and_cytokine = pd.DataFrame(0, index=DONOR_NAMES, columns=CYTOKINE_ORDER)
for combo in minority_comm_by_combo.keys():
    donors = minority_comm_by_combo[combo].split("_")
    counts_minority_by_donor_and_cytokine.loc[donors, combo[0]] += 1
counts_minority_by_donor_and_cytokine = counts_minority_by_donor_and_cytokine.loc[:, counts_minority_by_donor_and_cytokine.sum(axis=0) > 0]
counts_minority_by_donor_and_cytokine = counts_minority_by_donor_and_cytokine.iloc[::-1]

# plt.figure(figsize=cm_to_inches((3.5, 5)))
# bottom = np.zeros(len(DONOR_NAMES))
# colormap = plt.get_cmap("tab20").colors
# for i, cytokine in enumerate(counts_minority_by_donor_and_cytokine.columns):
#     values = counts_minority_by_donor_and_cytokine[cytokine].values
#     plt.barh(counts_minority_by_donor_and_cytokine.index, values, left=bottom,
#              label=introduce_greek_letters(cytokine), color=colormap[i])
#     bottom += values
# plt.xlabel("Count in smaller community\nfor substantial substructure")
# plt.tight_layout()
# plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=6, ncol=2)
# plt.savefig(os.path.join(plot_folder, f"4.4_minority_community_counts_by_donor_stacked.svg"), dpi=300)

In [ ]:
cross_by_combo = pd.Series(cross_by_combo)
internal_by_combo = pd.Series(internal_by_combo)
overall_by_combo = pd.DataFrame({
    "cross": cross_by_combo,
    "internal": internal_by_combo
})
overall_by_combo['diff'] = overall_by_combo['internal'] - overall_by_combo['cross']
overall_by_combo['label'] = overall_by_combo.index.map(lambda x: f"{x[0]}_{x[1]}")

In [ ]:
# Positions along x, with gaps between cytokine groups.
cytokines = [l.split("_")[0] for l in overall_by_combo["label"]]
cell_types = [l.split("_")[1] for l in overall_by_combo["label"]]

gap = 0.6
x_positions = []
extra_space = 0
for i, cyto in enumerate(cytokines):
    if i > 0 and cyto != cytokines[i - 1]:
        extra_space += gap
    x_positions.append(i + extra_space)

fig, ax = plt.subplots(figsize=cm_to_inches((14, 2.5)))

ax.bar(x_positions, overall_by_combo["internal"],
       color="lightseagreen", edgecolor="black", label="min within groups")
ax.bar(x_positions, overall_by_combo["cross"],
       color="coral", edgecolor="black", label="between groups")

ax.set_xticks(x_positions, cell_types, rotation=90, fontsize=6)

# Find the center of each cytokine block.
unique_cytokines, group_centers = [], []
for i, cyto in enumerate(cytokines):
    if i == 0 or cyto != cytokines[i - 1]:
        unique_cytokines.append(cyto)
        end = i
        while end + 1 < len(cytokines) and cytokines[end + 1] == cyto:
            end += 1
        group_centers.append((x_positions[i] + x_positions[end]) / 2)

# Cytokine labels below the cell-type labels.
for i, (cyto, center) in enumerate(zip(unique_cytokines, group_centers)):
    ax.text(
        center, -0.75 - 0.12 * (i % 2), introduce_greek_letters(cyto),
        ha="center", va="top", fontsize=6,
        transform=ax.get_xaxis_transform(), clip_on=False,
    )

ax.set_xlim(-1, x_positions[-1] + 1)
ax.tick_params(axis="x", length=0)
ax.set_ylabel("Pearson r")
ax.legend(fontsize=6, loc="upper right")
plt.savefig(
    os.path.join(plot_folder, "4.4_internal_vs_cross_community_correlation.svg"),
    dpi=300, bbox_inches="tight",
)
plt.show()

In [ ]:
# # --- Prepare adjusted y positions with gaps between cytokine groups ---
# cytokines = [l.split('_')[0] for l in overall_by_combo['label']]
# cell_types = [l.split('_')[1] for l in overall_by_combo['label']]

# gap = 0.6  # controls spacing between cytokine groups
# y_positions = []
# extra_space = 0
# for i, cyto in enumerate(cytokines):
#     if i > 0 and cyto != cytokines[i - 1]:
#         extra_space += gap
#     y_positions.append(i + extra_space)

# plt.figure(figsize=cm_to_inches((5, 10)))

# # Plot bars at adjusted y positions
# plt.barh(y_positions, overall_by_combo['internal'], color="lightseagreen", edgecolor="black", label="min within groups")
# plt.barh(y_positions, overall_by_combo['cross'], color="coral", edgecolor="black", label="between groups")

# # --- Custom y-axis labeling ---
# plt.yticks(y_positions, cell_types)
# ax = plt.gca()

# # Compute group centers for cytokine labels
# unique_cytokines = []
# group_centers = []
# for i, cyto in enumerate(cytokines):
#     if i == 0 or cyto != cytokines[i - 1]:
#         unique_cytokines.append(cyto)
#         start = i
#         end = i
#         while end + 1 < len(cytokines) and cytokines[end + 1] == cyto:
#             end += 1
#         start_y = y_positions[start]
#         end_y = y_positions[end]
#         group_centers.append((start_y + end_y) / 2)

# # Draw cytokine names (rotated, centered)
# for i, (cyto, center) in enumerate(zip(unique_cytokines, group_centers)):
#     ax.text(-0.35-0.1*(i%2), center, introduce_greek_letters(cyto),
#             ha='right', va='center', rotation=90, fontsize=6,
#             transform=ax.get_yaxis_transform())
# ax.tick_params(axis='y', length=0)
# plt.legend(fontsize=7, loc="lower right")
# plt.xlabel("Pearson r")
# plt.savefig(os.path.join(plot_folder, f"4.4_internal_vs_cross_community_correlation.svg"), dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
# a couple more examples of scatterplots
cytokine = "IL-1-alpha"
cell_type = "CD4 T cell"
donor_combs = [
    ("D5", "D6"),
    ("D5", "D4"),
    ("D3", "D4")
]
for donor_comb in donor_combs:
    get_donor_fc_scatterplot(
        cytokine,
        cytokine,
        cell_type,
        new_log2fc,
        donor_comb[0],
        donor_comb[1],
        small_plot=True,
        fixed_limits=[-4,4],
        savepath=os.path.join(plot_folder_heatmaps, f"scatter_log2fc_{donor_comb[0]}_{donor_comb[1]}")
    )

In [ ]:
interesting_substructure = pd.DataFrame(index=CYTOKINE_ORDER, columns=ALL_CELL_TYPES, data=0)
for key in all_community_means.keys():
    cytokine, cell_type = key.split("_")
    interesting_substructure.loc[cytokine, cell_type] += 1
interesting_substructure = interesting_substructure.loc[:, interesting_substructure.sum(axis=0) > 0]
interesting_substructure = interesting_substructure[interesting_substructure.sum(axis=1) > 0]

plt.figure(figsize=cm_to_inches((6,4)))
for i, cytokine in enumerate(interesting_substructure.index):
    for j, cell_type in enumerate(interesting_substructure.columns[::-1]):
        count = interesting_substructure.loc[cytokine, cell_type]
        if count > 0:
            plt.scatter(i, j, s=count*10, alpha=1, c=CELL_TYPE_COLORS[cell_type], edgecolors=None)

plt.yticks(range(len(interesting_substructure.columns)), interesting_substructure.columns[::-1], rotation=0)
plt.xticks(range(len(interesting_substructure.index)), interesting_substructure.index, rotation=90)
apply_greek_labels(plt.gca())
plt.xlim(-1, len(interesting_substructure.index))
plt.ylim(-1, len(interesting_substructure.columns))
plt.tick_params(length=0)
plt.title("substantial response sub-structure", fontsize=6.5)
plt.tight_layout()
for format in ['png', 'svg']:
    plt.savefig(os.path.join(plot_folder, f"4.4_interesting_substructure_counts.{format}"), dpi=300)
plt.show()

In [ ]:
def format_community_name(name):
    donors = name.split("_")
    text = ""
    
    # Determine block dimensions based on number of donors
    num_donors = len(donors)
    if num_donors <= 4:
        cols = 2
    elif num_donors <= 9:
        cols = 3
    else:
        cols = 4
    
    for i, donor in enumerate(donors):
        text += donor + " "
        # Add newline when reaching end of row (but not on last item)
        if (i + 1) % cols == 0 and i != num_donors - 1:
            text += "\n"
    
    return text

for key in all_community_means.keys():
    cytokine, cell_type = key.split("_")
    minor_community = minority_comm_by_combo[(cytokine, cell_type)]
    major_community = majority_comm_by_combo[(cytokine, cell_type)]
    df = all_r_vals_to_other_cytokines_by_comm[f'{cytokine}_{cell_type}'].unstack()
    df = df.loc[[major_community, minor_community], :]
    all_cytokines_to_display = [cytokine]
    for i in range(len(df.index)):
        subset = abs(df).iloc[i].sort_values(ascending=False)
        subset = subset[subset.index != cytokine]
        subset = subset[subset>0.3]
        cytokines_to_display = [c for c in subset.head(3).index if c not in all_cytokines_to_display]
        all_cytokines_to_display.extend(cytokines_to_display)
    plt.figure(figsize=cm_to_inches((3.5,2.0)))
    sns.heatmap(df[all_cytokines_to_display], cbar=True, cmap='coolwarm', vmin=-1, vmax=1,
                cbar_kws={"label": "Pearson r to consensus log2FC"}, annot=True, fmt=".2f", annot_kws={"size": 5, "rotation": 90}, square=False)
    comm_names = df.index
    plt.yticks(np.arange(len(comm_names))+0.5, [format_community_name(name) for name in comm_names], rotation=0, fontsize=6)
    apply_greek_labels_heatmap(plt.gca())
    plt.tick_params(length=0)
    plt.title(f"{introduce_greek_letters(cytokine)} on {cell_type}", fontsize=7)
    for fmt in ['png', 'svg']:
        plt.savefig(os.path.join(plot_folder_heatmaps, f"4.4_corr_to_other_cytokines_{cytokine}_{cell_type}.{fmt}"), dpi=300)
    plt.close()

In [ ]:
# compare response to IL-32-beta in CD4 T cells to overall response to IL-10 by donor subgroup
cell_types = ["CD8 T cell"]
for cell_type in cell_types:
    cytokine_base = "IL-32-beta"
    cytokine_comp = "IL-10"
    colors = {'D7_D8_D12_D2_D5_D6_D9_D11': "forestgreen", "D1_D3_D4_D10": "royalblue"}
    plt.figure(figsize=cm_to_inches((3.4,3.2)))
    current_means = all_community_means[f'{cytokine_base}_{cell_type}'].copy()
    for i, comm in enumerate(current_means.keys()):
        current_mean = current_means[comm].dropna()
        edger_lfc = edger_df_log2FC[(cell_type, cytokine_comp)].dropna()
        current_mean, edger_lfc = get_common_index(current_mean, edger_lfc)
        r = np.corrcoef(current_mean, edger_lfc)[0, 1]
        plt.scatter(current_mean, edger_lfc, edgecolors="none", alpha=0.5, s=1, color=colors[comm], rasterized=True,
                    label=f"r={r:.2f}")
    plt.xlabel(introduce_greek_letters(f"mean log2FC ({cytokine_base})"), fontsize=6)
    plt.ylabel(f"consensus log2FC ({cytokine_comp})", fontsize=6)
    plt.xticks([-2, 0, 2])
    plt.tight_layout()
    plt.legend(loc='upper left')
    plt.savefig(os.path.join(plot_folder_heatmaps, f"4.4_scatter_across_cyto_{cell_type}_{cytokine_base}_{cytokine_comp}.svg"), dpi=400)
    plt.show()

In [ ]:
main_cell_types =  ['B cell', 'CD4 T cell', 'NK CD56low', 'CD14 Mono']

pairwise_corrs = []
for cytokine in CYTOKINE_ORDER:
    for cell_type in main_cell_types:
        corr = get_donor_fc_vals_corr(cytokine, cell_type, new_log2fc)
        corr = corr.rename_axis("donor1", axis=0).rename_axis("donor2", axis=1).reset_index().melt(id_vars="donor1", var_name="donor2", value_name="corr")
        corr = corr[corr['donor1'] < corr['donor2']]
        corr['cytokine'] = cytokine
        corr['cell_type'] = cell_type
        pairwise_corrs.append(corr)
pairwise_corrs = pd.concat(pairwise_corrs, ignore_index=True)

In [ ]:
df = pairwise_corrs.copy()

donor_groups = {donor: group for group, donors in DONOR_GROUPS.items() for donor in donors}
df["group1"] = df["donor1"].map(donor_groups)
df["group2"] = df["donor2"].map(donor_groups)

# Keep the IFN/non-IFN comparison; investigate D2/D6 separately.
groups = ["interferon", "non-interferon"]
df = df[df["group1"].isin(groups) & df["group2"].isin(groups)].copy()
df["comparison"] = np.select(
    [
        df["group1"].eq("interferon") & df["group2"].eq("interferon"),
        df["group1"].eq("non-interferon") & df["group2"].eq("non-interferon"),
    ],
    ["within IFN", "within non-IFN"],
    default="between groups",
)

# Restrict the overview to strong responses.
strong = strong_impact_by_cell_type.stack().rename("strong")
strong.index.names = ["cytokine", "cell_type"]
df = df.join(strong, on=["cytokine", "cell_type"])
df = df[df["strong"].fillna(False)].copy()

summary = df.groupby(["cell_type", "cytokine", "comparison"])["corr"].median().unstack("comparison")
summary["difference"] = summary["between groups"] - summary["within non-IFN"]

fig, axes = plt.subplots(1, len(main_cell_types), figsize=cm_to_inches((10, 3.25)), sharex=True, sharey=True)
for cell_type_i, cell_type in enumerate(main_cell_types):
    ax = axes[cell_type_i]
    sub = summary.xs(cell_type, level="cell_type")
    r = np.corrcoef(sub["within non-IFN"], sub['between groups'])[0, 1]
    ax.scatter(sub["within non-IFN"], sub["between groups"],s=10, color=CELL_TYPE_COLORS[cell_type], edgecolors="none")
    ax.text(0.05, 0.9, f"r={r:.2f}")
    ax.plot([-1, 1], [-1, 1], "--", color="grey", linewidth=1)
    ax.set_title(cell_type, fontsize=6)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel('within non-IFN r')
    if cell_type_i == 0:
        ax.set_ylabel('between-group r')

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "4_association_r_within_group.svg"))
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(main_cell_types), figsize=cm_to_inches((9, 3.75)))
for cell_type_i, cell_type in enumerate(main_cell_types):
    ax = axes[cell_type_i]
    
    sub = summary.xs(cell_type, level="cell_type").copy()
    sub["deficit"] = sub["within non-IFN"] - sub["between groups"]
    top = sub.dropna(subset=["deficit"]).nlargest(10, "deficit")
    y = np.arange(len(top))

    ax.barh(y, top["deficit"], color=CELL_TYPE_COLORS[cell_type])
    ax.set_yticks(y, top.index)
    ax.invert_yaxis()
    ax.axvline(0, color="grey", linewidth=0.8)
    ax.set_xlim(0, 0.5)
    ax.set_title(cell_type, fontsize=6)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis='y', length=0)
    if cell_type_i == 1:
        ax.set_xlabel('Median r: within non-IFN - between-group')
    apply_greek_labels_heatmap(ax)
    
plt.tight_layout()
plt.show()

In [ ]:
df = pairwise_corrs.dropna(subset=["corr"]).copy()

# Include only cytokine–cell-type combinations with strong responses.
strong = strong_impact_by_cell_type.stack().rename("strong")
strong.index.names = ["cytokine", "cell_type"]
df = df.join(strong, on=["cytokine", "cell_type"])
df = df[df["strong"].fillna(False)]

baseline = df.groupby("cell_type")["corr"].transform("mean")
observed = df["corr"]

predictions = {
    "Cytokine means": df.groupby(["cell_type", "cytokine"])["corr"].transform("mean"),
    "Donor-pair means": df.groupby(["cell_type", "donor1", "donor2"])["corr"].transform("mean"),
}

fig, axes = plt.subplots(1, 2, figsize=cm_to_inches((7, 3.5)), sharex=True, sharey=True)
limit = observed.abs().max() * 1.05

for ax, (title, predicted) in zip(axes, predictions.items()):
    ax.scatter(predicted, observed, s=2, alpha=0.12, color="#3F6B8A", edgecolors="none", rasterized=True)
    ax.plot([-limit, limit], [-limit, limit], "--", color="0.5", lw=0.6)

    r2 = np.corrcoef(observed - baseline, predicted - baseline)[0, 1] ** 2
    ax.text(
        0.05, 1.0, f"partial $r^2$:\n{r2:.2f}",
        transform=ax.transAxes, va="top", fontsize=6,
    )
    ax.set(title=title, xlim=(-0.2, 1), ylim=(-0.2, 1), xlabel="Predicted correlation")
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylabel("Observed correlation")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, '4_explanatory_r2.svg'), dpi=400)
plt.show()

In [ ]:
df = pairwise_corrs.dropna(subset=["corr"]).copy()

# Remove self-pairs and symmetric duplicates.
df[["donor1", "donor2"]] = np.sort(df[["donor1", "donor2"]].astype(str).to_numpy(), axis=1)
df = df[df["donor1"] != df["donor2"]].drop_duplicates(["cytokine", "cell_type", "donor1", "donor2"])

# Restrict the overview to strong responses.
strong = strong_impact_by_cell_type.stack().rename("strong")
strong.index.names = ["cytokine", "cell_type"]
df = df.join(strong, on=["cytokine", "cell_type"])
df = df[df["strong"].fillna(False)].copy()

ifn = DONOR_GROUPS["interferon"]
non_ifn = DONOR_GROUPS["non-interferon"]
labels = ["Overall", "IFN", "non-IFN", "IFN to\nnon-IFN", "D2 to rest", "D6 to rest"]
rng = np.random.default_rng(0)

fig, axes = plt.subplots(1, len(main_cell_types), figsize=cm_to_inches((12, 3.5)), sharey=True)
for cell_type_i, cell_type in enumerate(main_cell_types):
    ax = axes[cell_type_i]
    sub = df[df["cell_type"] == cell_type]
    a, b = sub["donor1"], sub["donor2"]

    masks = [
        pd.Series(True, index=sub.index),
        a.isin(ifn) & b.isin(ifn),
        a.isin(non_ifn) & b.isin(non_ifn),
        (a.isin(ifn) & b.isin(non_ifn)) |
        (a.isin(non_ifn) & b.isin(ifn)),
        a.eq("D2") | b.eq("D2"),
        a.eq("D6") | b.eq("D6"),
    ]

    for i, mask in enumerate(masks):
        values = sub.loc[mask, "corr"].to_numpy()
        ax.scatter(i + rng.uniform(-0.23, 0.23, len(values)), values, s=2, alpha=0.2, color=CELL_TYPE_COLORS[cell_type], edgecolors="none", rasterized=True)
        if len(values):
            ax.plot([i - 0.25, i + 0.25], [np.median(values)] * 2,color="black", linewidth=1.5)

    ax.set_xticks(range(len(labels)), labels, rotation=90, ha="center")
    ax.tick_params(axis='x', length=0)
    ax.set_title(rf"{cell_type}, $\tilde{{r}}={sub['corr'].median():.2f}$", fontsize=6)
    ax.set_ylim(-0.2, 1)
    ax.set_xlim(-0.5, 5.5)
    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Pairwise\ndonor Pearson r")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "4_donor_donor_corr_by_cell_type.svg"), dpi=400)
plt.show()

# global plot
sub = df.copy()
a, b = sub["donor1"], sub["donor2"]
masks = [
    pd.Series(True, index=sub.index),
    a.isin(ifn) & b.isin(ifn),
    a.isin(non_ifn) & b.isin(non_ifn),
    (a.isin(ifn) & b.isin(non_ifn)) |
    (a.isin(non_ifn) & b.isin(ifn)),
    a.eq("D2") | b.eq("D2"),
    a.eq("D6") | b.eq("D6"),
]
values_by_group = [
    sub.loc[mask, "corr"].dropna().to_numpy()
    for mask in masks
]

fig, ax = plt.subplots(figsize=cm_to_inches((4, 4)))
bp = ax.boxplot(
    values_by_group, positions=np.arange(len(labels)),
    widths=0.6, patch_artist=True, showfliers=False,
    medianprops={"color": "black", "linewidth": 1.2},
    whiskerprops={"linewidth": 0.6},
    capprops={"linewidth": 0.6},
    boxprops={"linewidth": 0.6},
)
for box in bp["boxes"]:
    box.set_facecolor("tab:gray")
    box.set_alpha(0.5)

ax.set_xticks(range(len(labels)), labels, rotation=90, ha="center")
ax.tick_params(axis="x", length=0)
ax.set_ylim(-0.2, 1)
ax.set_xlim(-0.5, len(labels) - 0.5)
ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
ax.spines[["top", "right"]].set_visible(False)
ax.set_ylabel("Pairwise donor Pearson r")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "4_donor_donor_corr_global.svg"), dpi=400)
plt.show()

# 5 - Average cytokine production response across cell types

Here, we
1) Get the average and max over all cytokines.
2) Filter to genes where the average expression is at least 5, and there is one condition with both UMIs >= 20 and raw counts >= 20.
3) Filter the *mean* average expression to these columns.
4) Calculate the relative rpm across cell types

The resultant gene filter (expressed_genes_by_cell_type_dict) is later used to calculate correlation values between expression patterns.

In [ ]:
plot_folder = os.path.join(base_plot_folder, "5_average_cytokine_production_by_cell_type")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
# subset to main screen cytokines
average_norm_sum_gene_count_df = average_norm_sum_gene_count_df.loc[
    :, average_norm_sum_gene_count_df.columns.get_level_values(level='cytokine').isin(CYTOKINE_ORDER_WITH_PBS)
].copy()
average_sum_gene_count_df = average_sum_gene_count_df.loc[
    :, average_sum_gene_count_df.columns.get_level_values(level='cytokine').isin(CYTOKINE_ORDER_WITH_PBS)
].copy()

In [ ]:
# Which genes are expressed in which cell type?
# This is used later for XYZ
sum_over_all_cytokines_mean = average_norm_sum_gene_count_df.T.groupby(level='cell_type').mean().T
detected_in_same_condition = ((average_norm_sum_gene_count_df >= 20) & (average_sum_gene_count_df >= 20)).T.groupby(level='cell_type').any().T
expressed_genes_by_cell_type_df = (sum_over_all_cytokines_mean >= 5) & detected_in_same_condition
expressed_genes_by_cell_type_dict = {
    cell_type: list(expressed_genes_by_cell_type_df[cell_type][expressed_genes_by_cell_type_df[cell_type]].index) for cell_type in expressed_genes_by_cell_type_df.columns
    }

In [ ]:
cpm_by_donor = normalize_df_to_rpm(sum_gene_count_df_raw_by_donor, along_axis=0, pseudocount_initial=0, pseudocount_after=0)
average_cpm_no_pseudocount = cpm_by_donor.T.groupby(level=[0, 1]).mean().T.reindex(columns=average_sum_gene_count_df.columns)
del cpm_by_donor

# Which cytokines are expressed in which cell type?
cytokine_expression_df_all_details = []
cytokine_expression_df_substantial = []
for cytokine in cytokine_info.index:
    CPMs = get_complex_expression(average_cpm_no_pseudocount, cytokine, cytokine_info, "gene", insert_minimal=False)
    if not CPMs is None:
        CPMs.name = cytokine
        cytokine_expression_df_all_details.append(CPMs)

    UMIs = get_complex_expression(average_sum_gene_count_df, cytokine, cytokine_info, "gene", insert_minimal=False)
    if not UMIs is None:
        UMIs.name = cytokine
        cytokine_expression_df_substantial.append(UMIs)
    
cytokine_expression_df_all_details = pd.concat(cytokine_expression_df_all_details, axis=1)
cytokine_expression_df_all_PBS = cytokine_expression_df_all_details.xs("PBS", level="cytokine", axis=0)
cytokine_expression_df_mean = cytokine_expression_df_all_details.groupby(level=["cell_type"]).mean()
cytokine_expression_df_top5 = cytokine_expression_df_all_details.groupby(level="cell_type").agg(lambda s: s.nlargest(5).mean())
cytokine_expression_df_max = cytokine_expression_df_all_details.groupby(level=["cell_type"]).max()
cytokine_expression_df_substantial = pd.concat(cytokine_expression_df_substantial, axis=1)

In [ ]:
# test robustness of filter choice
expressed_cytokines_by_filter_combination = {}

for max_val in [5, 10, 20, 30, 50, 100]:
    for mean_val in [1, 2, 4, 8, 16, 32]:
        filter_name = f"mean>{mean_val}_max>{max_val}"
        # Filter to genes where the maximum average expression (across cell types) is at least 4 and the maximum maximum expression (across cell types) at least 20
        cytokine_expression_df_filtered_mean = cytokine_expression_df_mean.loc[:, cytokine_expression_df_mean.loc[ALL_CELL_TYPES].max(axis=0) >= mean_val]
        cytokine_expression_df_filtered_max = cytokine_expression_df_max.loc[:, cytokine_expression_df_max.loc[ALL_CELL_TYPES].max(axis=0) >= max_val]

        # get the intersection of the two column sets
        columns_mean = cytokine_expression_df_filtered_mean.columns
        columns_max = cytokine_expression_df_filtered_max.columns
        columns_intersection = columns_mean.intersection(columns_max)
        expressed_cytokines_by_filter_combination[filter_name] = len(columns_intersection)

expressed_cytokines_by_filter_combination = pd.DataFrame.from_dict(
    expressed_cytokines_by_filter_combination, orient='index', columns=['num_cytokines']
    ).reset_index()
expressed_cytokines_by_filter_combination[['mean_filter', 'max_filter']] = expressed_cytokines_by_filter_combination['index'].str.extract(r'mean>(\d+)_max>(\d+)').astype(int)
expressed_cytokines_by_filter_combination = expressed_cytokines_by_filter_combination.pivot(index='mean_filter', columns='max_filter', values='num_cytokines')

plt.figure(figsize=cm_to_inches((6,5)))
sns.heatmap(expressed_cytokines_by_filter_combination, annot=True, fmt="d", cmap="YlGnBu", 
            cbar_kws={"label": "number of cytokines expresseed in >=1 cell type"}, square=True)
plt.xlabel("max expression filter")
plt.ylabel("mean expression filter")
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"5_expressed_cytokines_by_filter_combination.{format}"), bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# Get the actual expression filter for all cytokines and our tested (parse) cytokines
cytokine_detected_in_same_condition = ((cytokine_expression_df_all_details >= 20) & (cytokine_expression_df_substantial >= 20)).groupby(level='cell_type').any()
# expressed_cytokines_by_cell_type_all_df = (cytokine_expression_df_mean >= 5) & cytokine_detected_in_same_condition
expressed_cytokines_by_cell_type_all_df = cytokine_detected_in_same_condition

expressed_cytokines_by_cell_type_all = {}
for cell_type in expressed_cytokines_by_cell_type_all_df.index:
    series = expressed_cytokines_by_cell_type_all_df.loc[cell_type]
    expressed_cytokines_by_cell_type_all[cell_type] = [cyto for cyto in series.index if series[cyto]]
    
expressed_cytokines_by_cell_type_parse = {}
for cell_type in expressed_cytokines_by_cell_type_all.keys():
    curr_list = expressed_cytokines_by_cell_type_all[cell_type]
    expressed_cytokines_by_cell_type_parse[cell_type] = [cyto for cyto in curr_list if cyto in CYTOKINE_ORDER]

In [ ]:
# --------- EXPRESSION PER DONOR (UNUSED) -------------
# cytokine_expression_df_all_details_by_donor = []
# cytokine_expression_df_substantial_by_donor = []
# for cytokine in cytokine_info.index:
#     for donor in sum_gene_count_df_norm_by_donor.columns.get_level_values("donor").unique():
#         expression = sum_gene_count_df_norm_by_donor.loc[:, (slice(None), CYTOKINE_ORDER_WITH_PBS, donor)].droplevel(2, axis=1)
#         cytokine_expression = get_complex_expression(
#             expression, cytokine, cytokine_info, "gene", insert_minimal=False
#             )
#         if not cytokine_expression is None:
#             cytokine_expression.name = (cytokine, donor)
#             cytokine_expression_df_all_details_by_donor.append(cytokine_expression)
#     # s = get_complex_expression(substantial_detection, cytokine, cytokine_info, "gene", insert_minimal=False)
#     # if not s is None:
#     #     s.name = cytokine
#     #     cytokine_expression_df_substantial_by_donor.append(s)
    
# cytokine_expression_df_all_details_by_donor = pd.concat(cytokine_expression_df_all_details_by_donor, axis=1)
# cytokine_expression_df_all_details_by_donor.columns.names = ['response', 'donor']
# cytokine_expression_df_all_details_by_donor = cytokine_expression_df_all_details_by_donor.stack('donor').loc[ALL_CELL_TYPES]
# cytokine_expression_df_substantial = pd.concat(cytokine_expression_df_substantial, axis=1)

In [ ]:
# get a table for individual conditions
cytokine_max_indices = cytokine_expression_df_all_details.idxmax(axis=0)
cytokine_max_values = cytokine_expression_df_all_details.max(axis=0)

cytokine_max_values = cytokine_max_values[cytokine_max_values > 40] # only plot interesting cytokines with large maximum expression values
cytokine_max_values = cytokine_max_values[cytokine_max_values.index.isin(cytokine_info.index[cytokine_info['gene_to_cytokine'].notna()])] # avoid duplicate for LT
cytokine_max_indices = cytokine_max_indices.loc[cytokine_max_values.index]

# Create a DataFrame for better display
cytokine_max_indices_df = pd.DataFrame({
    'produced cytokine': cytokine_max_indices.index,
    'cell-stim': cytokine_max_indices.values,
    'mean cpm': cytokine_max_values.values
})
cytokine_max_indices_df['cell type'] = cytokine_max_indices_df['cell-stim'].apply(lambda x: x[0])
cytokine_max_indices_df['treatment'] = cytokine_max_indices_df['cell-stim'].apply(lambda x: x[1])
cytokine_max_indices_df['mean cpm'] = cytokine_max_indices_df['mean cpm']
cytokine_max_indices_df['mean cpm PBS'] = None
cytokine_max_indices_df['produced cytokine'] = cytokine_max_indices_df['produced cytokine']
for index, row in cytokine_max_indices_df.iterrows():
    treatment = row['treatment']
    cell_type = row['cell type']
    cytokine = row['produced cytokine']
    cytokine_max_indices_df.loc[index, 'mean cpm PBS'] = cytokine_expression_df_all_PBS.loc[cell_type, cytokine]

    genes = cytokine_info.loc[cytokine, "gene"]
    genes = [gene.strip() for gene in genes.split(",")]
    if not all([gene in edger_df_log2FC.index for gene in genes]):
        cytokine_max_indices_df.loc[index, 'EdgeR FC'] = np.nan
        continue
    log2fcs = edger_df_log2FC.loc[genes, (cell_type, treatment)].values
    if all(fc > 0 for fc in log2fcs):
        cytokine_max_indices_df.loc[index, 'EdgeR FC'] = 2**min(log2fcs)
    elif all(fc < 0 for fc in log2fcs):
        cytokine_max_indices_df.loc[index, 'EdgeR FC'] = 2**max(log2fcs)
    else:
        cytokine_max_indices_df.loc[index, 'EdgeR FC'] = np.nan

cytokine_max_indices_df['produced cytokine'] = cytokine_max_indices_df['produced cytokine'].map(cytokine_info["gene_to_cytokine"])
cytokine_max_indices_df['FC to PBS'] = cytokine_max_indices_df['mean cpm'] / cytokine_max_indices_df['mean cpm PBS']
cytokine_max_indices_df['FC to PBS'] = cytokine_max_indices_df['FC to PBS'].astype(float).round(1)

# replace 'FC to PBS' values by EdgeR FC values if available
for index, row in cytokine_max_indices_df.iterrows():
    if not pd.isna(row['EdgeR FC']):
        cytokine_max_indices_df.loc[index, 'FC to PBS'] = float(f"{row['EdgeR FC']:.1f}")

# Only use max_expression, cell_type, and treatment columns, and split into two tables with at most 40 entries each
cols_to_show = ['produced cytokine', 'cell type', 'treatment', 'mean cpm', 'FC to PBS']
df_to_show = cytokine_max_indices_df[cols_to_show]
df_to_show = df_to_show.sort_values(by='produced cytokine')
df_to_show['produced cytokine'] = df_to_show['produced cytokine'].apply(lambda x: introduce_greek_letters(x))
df_to_show['treatment'] = df_to_show['treatment'].apply(lambda x: introduce_greek_letters(x))
df_to_show['mean cpm'] = df_to_show['mean cpm'].astype(int)

max_rows_per_table = 40
num_rows = len(df_to_show)
num_tables = (num_rows // max_rows_per_table) + int(num_rows % max_rows_per_table > 0)

for i in range(num_tables):
    start_idx = i * max_rows_per_table
    end_idx = min((i + 1) * max_rows_per_table, num_rows)
    sub_df = df_to_show.iloc[start_idx:end_idx]
    
    fig, ax = plt.subplots(figsize=cm_to_inches((7, num_rows / 22)))
    ax.axis('off')
    table = ax.table(cellText=sub_df.values, colLabels=sub_df.columns, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(6)
    table.auto_set_column_width(col=list(range(len(sub_df.columns))))

    # Set header row (row=0) to bold and light gray background
    header_color = "#ededed"
    for col in range(len(sub_df.columns)):
        cell = table[0, col]
        cell.set_fontsize(6)
        cell.set_text_props(weight='bold')
        cell.set_facecolor(header_color)
    
    # Set all data cells to light gray background
    for row in range(1, len(sub_df) + 1):
        for col in range(len(sub_df.columns)):
            cell = table[row, col]
            cell.set_facecolor(header_color)
    
    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, f"5_max_cytokine_production_table_{i}.svg"))
    plt.show()

#### Calculate family counts

In [ ]:
cytokines_expressed_anywhere = []
for l in expressed_cytokines_by_cell_type_all.values():
    cytokines_expressed_anywhere.extend(l)
cytokines_expressed_anywhere = list(set(cytokines_expressed_anywhere))
len(cytokines_expressed_anywhere)

# ------- CRITICAL: SETS WHAT IS USED FOR DOWNSTREAM COMPARISONS ----------
cytokine_expression_df_filtered = cytokine_expression_df_top5[cytokines_expressed_anywhere]

In [ ]:
maximum_cell_type_count = {cell_type: 0 for cell_type in ALL_CELL_TYPES}
maximum_cell_type_identity = {}
maximum_cell_type_list = []
for column in cytokine_expression_df_filtered.columns:
    maximum_cell_type = cytokine_expression_df_filtered[column].idxmax()
    row = {
        "cell_type": maximum_cell_type,
        "cytokine": column
    }
    maximum_cell_type_list.append(row)
    maximum_cell_type_count[maximum_cell_type] += 1
    maximum_cell_type_identity[column] = maximum_cell_type

maximum_cell_type_df = pd.DataFrame(maximum_cell_type_list)
maximum_cell_type_df["family"] = maximum_cell_type_df["cytokine"].map(cytokine_name_to_level_2_family)
family_value_counts_df = maximum_cell_type_df.groupby("cell_type")["family"].value_counts().unstack().fillna(0)

# Order the family_value_counts_df by ALL_CELL_TYPES
family_value_counts_df = family_value_counts_df.reindex(ALL_CELL_TYPES).dropna()

available_families = family_value_counts_df.columns
ordered_available_families = [family for family in cytokine_level_2_order if family in available_families]
family_value_counts_df = family_value_counts_df[ordered_available_families]
family_value_counts_df = family_value_counts_df.iloc[:, ::-1]

In [ ]:
max_value_counts = maximum_cell_type_df["cell_type"].value_counts()
for cell_type in ALL_CELL_TYPES:
    if not cell_type in max_value_counts.index:
        max_value_counts[cell_type] = 0
max_value_counts = max_value_counts[ALL_CELL_TYPES]

plt.figure(figsize=cm_to_inches((4.2,3.2)))
plt.bar(max_value_counts.index, max_value_counts.values, color="black", width=0.65)
plt.title("strongest production", fontsize=7)
plt.xticks(rotation=90)
plt.yticks(rotation=90)
plt.xlim(-0.5, len(max_value_counts)-0.5)
plt.ylim(0, 30)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"5_max_cell_type_count.{format}"), dpi=300)
plt.show()

In [ ]:
any_value_counts = expressed_cytokines_by_cell_type_all_df.unstack().to_frame().reset_index()
any_value_counts = any_value_counts.rename(columns={0: "expressed", "level_0": "cytokine"})
any_value_counts["level1_family"] = any_value_counts["cytokine"].map(cytokine_name_to_level_1_family)

any_level1_family_count = any_value_counts.groupby(["cell_type", "level1_family"])["expressed"].sum().to_frame().reset_index()
relevant_families = any_level1_family_count.groupby("level1_family")["expressed"].sum()
relevant_families = relevant_families[relevant_families > 0].index
any_level1_family_count = any_level1_family_count[any_level1_family_count["level1_family"].isin(relevant_families)]

# create a pivot table to get counts by cell type and level1_family
pivot_df = any_level1_family_count.pivot(index="cell_type", columns="level1_family", values="expressed").fillna(0)
pivot_df = pivot_df.reindex(ALL_CELL_TYPES).fillna(0)
ordered_families = sorted(pivot_df.columns)

# plotting
ax = pivot_df[ordered_families[::-1]].plot(kind='bar', stacked=True, figsize=cm_to_inches((4.2,4)), width=0.65)
plt.xlabel("")
plt.title("expressed cytokines", fontsize=7)
plt.xticks(rotation=90)
plt.yticks(rotation=90)
plt.tick_params(axis='x', which='both', length=0)

plt.tight_layout()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"5_cytokine_family_by_cell_type.{format}"), dpi=300)
plt.show()

In [ ]:
# Transpose the dataframe to plot by cytokine family instead of cell type
family_counts_by_cell_type = family_value_counts_df.T

# Plot each cell type as a separate bar segment
fig, ax = plt.subplots(figsize=cm_to_inches((8, 8)))
bottom = np.zeros(len(family_counts_by_cell_type.index))
for cell_type in family_counts_by_cell_type.columns:
    ax.barh(family_counts_by_cell_type.index, family_counts_by_cell_type[cell_type], 
           left=bottom, label=cell_type, color=CELL_TYPE_COLORS.get(cell_type, 'gray'))
    bottom += family_counts_by_cell_type[cell_type]

ax.set_xlabel('Maximum production per family')
ax.set_ylabel('')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlim(0, 16)
plt.tight_layout()

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"5_maximum_expression_by_family.{format}"), dpi=300)
plt.show()

In [ ]:
# Calculate the relative cpm across cell types (again for the mean expression)
cytokine_expression_df_rel = cytokine_expression_df_filtered.div(cytokine_expression_df_filtered.max(axis=0), axis=1)

# sort the columns by the maximum cell type identity
maximum_cell_type_identity = pd.DataFrame([(k,v) for k,v in maximum_cell_type_identity.items()], columns=['gene', 'cell_type'])

# sort cell type
maximum_cell_type_identity["cell_type"] = pd.Categorical(maximum_cell_type_identity["cell_type"], categories=ALL_CELL_TYPES, ordered=True)
maximum_cell_type_identity = maximum_cell_type_identity.sort_values(by="cell_type")
cytokine_expression_df_rel = cytokine_expression_df_rel.loc[ALL_CELL_TYPES, maximum_cell_type_identity["gene"]]

In [ ]:
# plot in two separate groups (whole plot is too big)
cytokine_groups_here = [['interleukin', 'interferon', 'TNF family', 'complement', 'growth factor', 'other'], ["chemokine"]]
for group_index, family in enumerate(cytokine_groups_here):
    cytokine_expression_df_rel_family = cytokine_expression_df_rel.loc[:, cytokine_info["family_level_1"].isin(family)]
    # order = [cyto for cyto in cytokine_info.index[cytokine_info["gene_to_cytokine"].notna()] if cyto in cytokine_expression_df_rel_family.columns]
    # cytokine_expression_df_rel_family = cytokine_expression_df_rel_family[order]
    keep = cytokine_info.index[cytokine_info["gene_to_cytokine"].notna()]
    cytokine_expression_df_rel_family = cytokine_expression_df_rel_family.loc[:, cytokine_expression_df_rel_family.columns.isin(keep)]
    if group_index == 0:
        plt.figure(figsize=cm_to_inches((6, 16)))
    else:
        plt.figure(figsize=cm_to_inches((6, 10)))

    ax = sns.heatmap(cytokine_expression_df_rel_family.T, cmap="Grays", vmin=0, vmax=1,
        cbar_kws={"ticks": [0, 0.2, 0.4, 0.6, 0.8, 1],  "label": "relative expression (mean of top five conditions)",  "shrink": 0.25}, square=False)

    plt.yticks(rotation=0)
    ax.tick_params(axis='y', labelsize=6)
    tick_positions = np.arange(len(cytokine_expression_df_rel_family.columns))+0.5
    labels = [introduce_greek_letters(cytokine_info.loc[label, "gene_to_cytokine"]) for label in cytokine_expression_df_rel_family.columns]
    ax.set_yticks(tick_positions)
    ax.set_yticklabels(labels)
    
    ax.tick_params(axis='y', labelsize=6, length=0)
    ax.tick_params(axis='x', length=0)

    plt.xlabel("")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, f"5_average_cytokine_production_by_cell_type_{group_index}.svg"), bbox_inches="tight", dpi=300)
    plt.show()

In [ ]:
# small sample plot for the main figure
cytokine_expression_df_rel_family = cytokine_expression_df_rel.loc[:, ["IL-32-beta", "CD40L", "IFN-gamma", "TNF-alpha", "IL-10"]]
plt.figure(figsize=cm_to_inches((4.525, 3)))

ax = sns.heatmap(cytokine_expression_df_rel_family.T, cmap="Grays", vmin=0, vmax=1,
    cbar=False, square=True, linewidths=0, linecolor='black')
plt.yticks(rotation=0)
ax.tick_params(axis='y', labelsize=6)
tick_positions = np.arange(len(cytokine_expression_df_rel_family.columns))+0.5
labels = [introduce_greek_letters(cytokine_info.loc[label, "gene_to_cytokine"]) for label in cytokine_expression_df_rel_family.columns]
ax.set_yticks(tick_positions)
ax.set_yticklabels(labels)
ax.tick_params(axis='y', labelsize=6, length=0)
ax.tick_params(axis='x', length=0)
plt.xlabel("")

# black box
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.spines['left'].set_visible(True)
for spine in ax.spines.values():
    spine.set_color('black')
    spine.set_linewidth(1)

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"5_average_cytokine_production_by_cell_type_particular.{format}"), bbox_inches="tight",dpi=300)
plt.show()

In [ ]:
# Compare a few closely related cell types
def mathrm_label(value):
    return str(value).replace("\\", "").replace(" ", r"\ ")

pairings = [
    ('NK CD56hi', 'NK CD56low', (1.5, 4.2), 'max'),
    ('cDC', 'CD14 Mono', (1.5, 4.2), 'max'),
    ('CD8 T cell', 'CD4 T cell', (1.5, 4.2), 'expressed')
]
for cell1, cell2, figsize, filter_type in pairings:
    relative = (cytokine_expression_df_top5.loc[cell1] + 1) / (cytokine_expression_df_top5.loc[cell2] + 1)
    if filter_type == 'max':
        used_filter = maximum_cell_type_df[maximum_cell_type_df['cell_type'].isin([cell1, cell2])]['cytokine']
    elif filter_type == 'expressed':
        used_filter = list(set(expressed_cytokines_by_cell_type_all[cell1] + expressed_cytokines_by_cell_type_all[cell2]))
    relative = relative[used_filter]
    relative = relative.sort_values(ascending=True)
    relative = relative.reset_index()
    relative.columns = ['cytokine', 'value']
    relative['color'] = relative['value'].apply(lambda x: CELL_TYPE_COLORS[cell1] if x>=1 else CELL_TYPE_COLORS[cell2])
    
    fig, ax = plt.subplots(1, 1, figsize=cm_to_inches(figsize))
    relative = pd.concat([relative.head(7), relative.tail(7)])
    relative['value'] = np.log2(relative['value'])

    plt.barh(relative['cytokine'], relative['value'], edgecolor='black', color=relative['color'])
    plt.tick_params(axis='y', length=0)
    plt.rcParams["text.usetex"] = False

    c1_tex = mathrm_label(cell1)
    c2_tex = mathrm_label(cell2)
    ax.set_xlabel(
        rf"$\log_2\!\left(\;"
        rf"\dfrac{{\mathrm{{{c1_tex}}}}}"
        rf"{{\mathrm{{{c2_tex}}}}}"
        rf"\right)$"
    )
    
    plt.ylim(-0.75, len(relative)-0.25)
    apply_greek_labels_heatmap(ax)
    plt.tick_params(axis='x', rotation=0)
    plt.tick_params(axis='both', labelsize=6)
    min_x, max_x = relative['value'].min(), relative['value'].max()
    plt.xlim(min_x-0.2, max_x+0.2)
    plt.savefig(os.path.join(plot_folder, f"5_relative_{cell1}_{cell2}.svg"))
    plt.show()

In [ ]:
raise Exception("What's with type 2?")
specificity_cutoff = 0.35

comparisons = [
    ("level 1", "Mixed", {
        "Myeloid": ["CD14 Mono", "CD16 Mono", "cDC"],
        "Lymphoid": ["B cell", "CD4 T cell", "CD8 T cell", "Treg", "NKT","MAIT", "NK CD56low", "NK CD56hi"],
    }),
    ("level 2", "Myeloid", {
        "Monocyte": ["CD14 Mono", "CD16 Mono"],
        "cDC": ["cDC"],
    }),
    ("level 2", "Lymphoid", {
        "B cell": ["B cell"],
        "T cell": ["CD4 T cell", "CD8 T cell", "Treg", "NKT", "MAIT"],
        "NK cell": ["NK CD56low", "NK CD56hi"],
    }),
    ("level 3", "Monocyte", {
        "CD14 Mono": ["CD14 Mono"],
        "CD16 Mono": ["CD16 Mono"],
    }),
    ("level 3", "T cell", {
        "CD4 T cell": ["CD4 T cell"],
        "CD8 T cell": ["CD8 T cell"],
        "Treg": ["Treg"],
        "NKT": ["NKT"],
        "MAIT": ["MAIT"],
    }),
    ("level 3", "NK cell", {
        "NK CD56low": ["NK CD56low"],
        "NK CD56hi": ["NK CD56hi"],
    }),
]

# Prepare relative expression
expression_rel = cytokine_expression_df_rel.drop(index="HSPC", errors="ignore",).copy()
expression_rel = expression_rel.div(expression_rel.max(axis=0), axis=1)

comparison_parents = {parent for _, parent, _ in comparisons}
incoming = {"Mixed": expression_rel.columns,}
paths = {cytokine: ["Mixed"] for cytokine in expression_rel.columns}
flow_rows = []
terminal_rows = []
for level, parent, lineage_groupings in comparisons:
    parent_cytokines = pd.Index(incoming.get(parent, []))
    if len(parent_cytokines) == 0:
        continue

    df_rel_lineage = pd.DataFrame({
        lineage: expression_rel.loc[cell_types, parent_cytokines].max(axis=0) for lineage, cell_types in lineage_groupings.items()
    })

    highest_lineage = df_rel_lineage.idxmax(axis=1)
    highest_expression = df_rel_lineage.max(axis=1)

    second_highest = df_rel_lineage.apply(lambda row: row.nlargest(2).iloc[-1], axis=1)
    non_specific = second_highest.index[second_highest >= specificity_cutoff]

    for cytokine in non_specific:
        target = f"{parent}"
        paths[cytokine].append(target)

        flow_rows.append({
            "level": level,
            "source": parent,
            "target": target,
            "cytokine": cytokine,
        })

        terminal_rows.append({
            "cytokine": cytokine,
            "terminal_assignment": target,
            "terminal_level": int(level[-1]),
            "assignment_type": "shared",
            "path": " > ".join(paths[cytokine]),
            "relative_expression": highest_expression[cytokine],
            "second_highest_expression": second_highest[cytokine],
        })

    for lineage in lineage_groupings:
        specific = df_rel_lineage.index[(highest_lineage == lineage) & ~(df_rel_lineage.index.isin(non_specific))]
        incoming[lineage] = specific

        for cytokine in specific:
            paths[cytokine].append(lineage)
            flow_rows.append({
                "level": level,
                "source": parent,
                "target": lineage,
                "cytokine": cytokine,
            })

            # stop if there is no finer comparison for this lineage
            if lineage not in comparison_parents:
                terminal_rows.append({
                    "cytokine": cytokine,
                    "terminal_assignment": lineage,
                    "terminal_level": int(level[-1]),
                    "assignment_type": "specific",
                    "path": " > ".join(paths[cytokine]),
                    "relative_expression": df_rel_lineage.loc[cytokine, lineage],
                    "second_highest_expression": second_highest[cytokine],
                })

flow_df = pd.DataFrame(flow_rows)
cytokine_terminal_assignments = pd.DataFrame(terminal_rows)

assert (len(cytokine_terminal_assignments) == expression_rel.shape[1])
assert (cytokine_terminal_assignments["cytokine"].nunique() == expression_rel.shape[1])

# Add information used to select examples
dominant_cell_type = expression_rel.idxmax(axis=0)
cytokine_terminal_assignments["dominant_cell_type"] = cytokine_terminal_assignments["cytokine"].map(dominant_cell_type)
cytokine_terminal_assignments["specificity_margin"] = cytokine_terminal_assignments["relative_expression"] - cytokine_terminal_assignments["second_highest_expression"]
cytokine_terminal_assignments["mean_expression"] = [cytokine_expression_df_filtered.loc[cell_type, cytokine] \
    for cell_type, cytokine in zip(cytokine_terminal_assignments["dominant_cell_type"], cytokine_terminal_assignments["cytokine"])]
cytokine_terminal_assignments = cytokine_terminal_assignments.sort_values(["terminal_assignment","mean_expression","specificity_margin",],ascending=[True, False, False]).reset_index(drop=True)
cytokine_terminal_assignments["example_rank"] = cytokine_terminal_assignments.groupby("terminal_assignment").cumcount()+ 1

In [ ]:
terminal_matrix_plot = cytokine_terminal_assignments.copy()
terminal_matrix_plot["functional_grouping"] = terminal_matrix_plot["cytokine"].map(cytokine_name_to_functional)
terminal_matrix_plot["cytokine"] = terminal_matrix_plot["cytokine"].str.replace("IL-32-beta", "IL-32")
assert terminal_matrix_plot["functional_grouping"].notna().all()

row_order = ["Mixed", "Myeloid", "Monocyte", "CD14 Mono", "CD16 Mono", "cDC", "Lymphoid", "T cell", "NK cell", "NK CD56hi"]
# make sure this covers everything
assert set(row_order) == set(terminal_matrix_plot['terminal_assignment'].unique())

functional_column_order = {
    "inflammatory": "Inflammatory",
    "type 1": "Type 1",
    "type 17": "Type 17",
    "regulatory": "Regulatory",
    "lymphocyte growth": "Lymph. growth",
    "myeloid growth": "Myeloid growth",
    "tissue repair": "Tissue repair",
    "tissue remodeling": "Tissue remodel.",
    "vascular growth": "Vascular growth",
    "B-cell support": "B-cell support",
    "costimulation": "Costimulation",
    "cell death": "Cell death",
    "chemo: mixed": "Mixed",
    "chemo: lymph/DC": "Lymph./DC",
    "chemo: neutrophil": "Neutrophil",
    "chemo: monocyte": "Monocyte",
    "chemo: eosinophil": "Eosinophil",
    "chemo: cDC1": "cDC1",
    "chemo: Th2/Treg": "Th2/Treg",
    "chemo: B cell": "B cell",
    "complement": "Complement",
}
functional_blocks = {
    "Immune\ncytokines": ["inflammatory", "type 1", "type 17", "regulatory", "lymphocyte growth", "myeloid growth"],
    "Growth/\nremodel": ["tissue repair", "tissue remodeling", "vascular growth"],
    "TNF\nligands": ["B-cell support", "costimulation", "cell death"],
    "Chemokines": ["chemo: mixed", "chemo: lymph/DC", "chemo: neutrophil", "chemo: monocyte", "chemo: eosinophil", "chemo: cDC1", "chemo: Th2/Treg", "chemo: B cell"],
    "Complement": ["complement"],
}
functional_block_colors = {
    "Immune\ncytokines": "#9c6ade",
    "Growth/\nremodel": "#e9c46a",
    "TNF\nligands": "#d76b55",
    "Chemokines": "#2a9d8f",
    "Complement": "#9aa0a6",
}
functional_to_block = {group: block for block, groups in functional_blocks.items() for group in groups}
assert set(functional_to_block) == set(functional_column_order)

unexpected_groups = set(terminal_matrix_plot["functional_grouping"].unique()) - set(functional_column_order)
assert not unexpected_groups, unexpected_groups

functional_matrix = pd.crosstab(terminal_matrix_plot["terminal_assignment"], terminal_matrix_plot["functional_grouping"])
functional_matrix = functional_matrix.reindex(index=row_order, columns=list(functional_column_order), fill_value=0)
row_labels = [f"{row} ({int(functional_matrix.loc[row].sum())})" for row in row_order]

fig, ax = plt.subplots(figsize=cm_to_inches((10, 4.75)))
for row_index, endpoint in enumerate(row_order):
    for column_index, group in enumerate(functional_column_order):
        count = int(functional_matrix.loc[endpoint, group])
        if count == 0:
            continue
        width = 0.9
        height = 0.9
        ax.add_patch(plt.Rectangle(
            (column_index - width / 2, row_index - height / 2), width, height,
            facecolor=functional_block_colors[functional_to_block[group]], edgecolor="none",
            alpha=0.38 + 0.09 * min(count, 6), zorder=2,
        ))
        ax.text(column_index, row_index, str(count), ha="center", va="center", fontsize=6, color="black", zorder=3)

example_x = len(functional_column_order)
ax.set_xlim(-2.2, example_x + 4)
ax.set_ylim(len(row_order) - 0.5, -0.5)
ax.set_xticks(range(len(functional_column_order)), list(functional_column_order.values()), rotation=90, ha="left", va="center", rotation_mode="anchor")
ax.xaxis.tick_top()
ax.tick_params(axis="x", length=0, pad=1, labelsize=6)
ax.set_yticks(range(len(row_order)), row_labels)
ax.tick_params(axis="y", length=0, pad=2, labelsize=6)

for block, groups in functional_blocks.items():
    block_positions = [list(functional_column_order).index(group) for group in groups]
    if not block_positions:
        continue
    x0 = min(block_positions) - 0.45
    x1 = max(block_positions) + 0.45
    ax.text((x0 + x1) / 2, 1.85, block, transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=6, fontweight="bold", color=functional_block_colors[block], clip_on=False)
    ax.plot([x0, x1], [1.8, 1.8], transform=ax.get_xaxis_transform(), color=functional_block_colors[block], linewidth=0.75, clip_on=False)
    if x1 < len(functional_column_order) - 0.5:
        ax.axvline(x1, color="#777777", linewidth=0.45, zorder=0)

for y in range(len(row_order) + 2):
    line_color = "#504b4b" if y in [0, 1, 6, 10] else "#e2e2e2"
    ax.plot([-0.5, example_x + 4], [y - 0.5, y - 0.5], color=line_color, linewidth=0.5, zorder=0)

hierarchy_brackets = [
    (-2.25, 0, 9),   # all
    (-1.7, 1, 5),    # Myeloid > Monocyte, cDC
    (-1.7, 6, 9),    # Lymphoid > T cell, NK CD56hi
    (-1.15, 2, 4),   # Monocyte > CD14 Mono, CD16 Mono
    (-1.15, 8, 9),   # NK cell > NK CD56hi
]
for x, y0, y1 in hierarchy_brackets:
    ax.plot([x, x], [y0 - 0.25, y1 + 0.25], color="#777777", linewidth=1, clip_on=False)
    ax.plot([x, x + 0.25], [y0 - 0.25, y0 - 0.25], color="#777777", linewidth=1, clip_on=False)
    ax.plot([x, x + 0.25], [y1 + 0.25, y1 + 0.25], color="#777777", linewidth=1, clip_on=False)

selected_cytokines = [
    "TNF-alpha", "TGF-beta1", # Mixed
    "CXCL9", "CXCL11", # Myeloid
    "IL-6", "APRIL", # Monocyte
    "IL-10", "IL-8", # CD14 Mono
    "C1q", # CD16 Mono
    "CCL19", "CCL22", # cDC
    "LIGHT", "PDGF-DD", # Lymphoid
    "EDA", "FLT3L", # T cell
    "IFN-gamma", # NK cell
    "RANKL", "XCL1", # NK CD56hi
]
selected_rows = terminal_matrix_plot.set_index("cytokine").loc[selected_cytokines].reset_index()
selected_rows["label"] = selected_rows["cytokine"].map(introduce_greek_letters)

# plotting w/0 color
# selected_examples = selected_rows.groupby("terminal_assignment", sort=False)["label"].agg(", ".join).to_dict()
# for endpoint, label in selected_examples.items():
#     ax.text(example_x, row_order.index(endpoint), label, ha="left", va="center", fontsize=6, color="#222222")

# plotting w/ color
renderer = fig.canvas.get_renderer()
for endpoint, rows in selected_rows.groupby("terminal_assignment", sort=False):
    x_offset = 0  # points to the right of example_x
    for i, (_, r) in enumerate(rows.iterrows()):
        color = functional_block_colors[functional_to_block[r["functional_grouping"]]]
        text = ax.annotate(r["label"] + ("," if i < len(rows) - 1 else ""), (example_x, row_order.index(endpoint)),
                           xytext=(x_offset, 0), textcoords="offset points", ha="left", va="center", fontsize=6, color=color)
        x_offset += text.get_window_extent(renderer).width * 72 / fig.dpi + 2

shift = matplotlib.transforms.ScaledTranslation(0, -0.5 / 72, fig.dpi_scale_trans)
for text in fig.findobj(matplotlib.text.Text):
    if isinstance(text, matplotlib.text.Annotation):
        x, y = text.get_position()
        text.set_position((x, y - 1))  # annotations already use offset points
    else:
        text.set_transform(text.get_transform() + shift)

for spine in ax.spines.values():
    spine.set_visible(False)
    
fig.subplots_adjust(left=0.225, right=0.995, top=0.48, bottom=0.04)
hierarchy_matrix_path = os.path.join(plot_folder, "cytokine_lineage_function_block_matrix.svg")
fig.savefig(hierarchy_matrix_path, format="svg", transparent=False)
plt.show()

## 5.1 - Cytokine to cytokine signaling

In [ ]:
plot_folder = os.path.join(base_plot_folder, "5.1_cytokine_to_cytokine_production")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
# constrain to all cytokines that have a calculated fold change for all subunits (here as gene lists per cytokine)
all_cytokines_in_measured_genes = [cytokine for cytokine in cytokine_info["gene"] if all([subunit in edger_df_padj.index for subunit in cytokine.split(", ")])]
cytokines_not_in_measured_genes = [cytokine for cytokine in cytokine_info["gene"] if cytokine not in all_cytokines_in_measured_genes] # for debugging

# get the padj and log2FoldChange values for all necessary genes
all_cytokines_in_measured_genes_flat = [subunit for cytokine in all_cytokines_in_measured_genes for subunit in cytokine.split(", ")]
edger_df_padj_cytokine = edger_df_padj.loc[edger_df_padj.index.isin(all_cytokines_in_measured_genes_flat), :].fillna(1)
edger_df_log2FC_cytokine = edger_df_log2FC.loc[edger_df_padj.index.isin(all_cytokines_in_measured_genes_flat), :].fillna(0)

In [ ]:
cytokine_response_df_all = pd.DataFrame(columns=["cell_type", "treatment", "response", "p_value", "fold_changes"])

for cell_type in ALL_CELL_TYPES:
    for cytokine_treatment in CYTOKINE_ORDER:
        for cytokine_response in all_cytokines_in_measured_genes:
            individual_cytokines = cytokine_response.split(", ")
            
            # skip this response if not all subunits are present (should already have been filtered?)
            skip = False
            for individual_cytokine in individual_cytokines:
                if not individual_cytokine in edger_df_padj_cytokine.index:
                    skip = True
            if skip:
                continue
            
            # max for p value (conservative)
            p_value = max([edger_df_padj_cytokine.loc[individual_cytokine, (cell_type, cytokine_treatment)].astype(float) for individual_cytokine in individual_cytokines])

            # list for fold changes
            fold_changes = [edger_df_log2FC_cytokine.loc[individual_cytokine, (cell_type, cytokine_treatment)].astype(float) for individual_cytokine in individual_cytokines]
            if p_value > 0.05: # only keep significant responses (regardless of fc for now)
                continue
            
            if all(fc < 0 for fc in fold_changes): # if all fold changes are negative, take the maximum (least negative) as the overall fold change for the cytokine response
                single_fc = max(fold_changes)
            elif all(fc > 0 for fc in fold_changes): # if all fold changes are positive, take the minimum (least positive) as the overall fold change for the cytokine response
                single_fc = min(fold_changes)
            else: # mixed response
                single_fc = 0.0 
            
            new_row = pd.DataFrame({
                "cell_type": [cell_type], 
                "treatment": [cytokine_treatment], 
                "response": [cytokine_response],
                "p_value": [p_value],
                "fold_changes": [fold_changes],
                "single_fc": [single_fc]
            })
            cytokine_response_df_all = pd.concat([cytokine_response_df_all, new_row], axis=0)

In [ ]:
# look at the correlation of IL15 and IL15RA responses across conditions
df = cytokine_response_df_all[cytokine_response_df_all["response"] == "IL15, IL15RA"].copy()
df = df.sort_values(by="single_fc", ascending=False)
r = np.corrcoef(df['fold_changes'].apply(lambda x: x[0]), df['fold_changes'].apply(lambda x: x[1]))[0, 1]
r

In [ ]:
# convert back to actual cytokine names
cytokine_response_df_all.loc[:, "response"] = cytokine_response_df_all["response"].map(gene_to_cytokine)

# remove duplicates (basically just LTA, LTB because they have the same genes)
print(len(cytokine_response_df_all))
cytokine_response_df_all = cytokine_response_df_all.drop_duplicates(ignore_index=True, subset=["cell_type", "treatment", "response"])
print(len(cytokine_response_df_all))

# add the direction of the regulation
direction = []
for index, row in cytokine_response_df_all.iterrows():
    if all([fc > 1 for fc in row["fold_changes"]]):
        direction.append("upregulated")
    elif all([fc < -1 for fc in row["fold_changes"]]):
        direction.append("downregulated")
    else:
        direction.append("neither")
        
cytokine_response_df_all["direction"] = direction

In [ ]:
# save the results
cytokine_response_df_all.to_csv(os.path.join(output_tables_folder, "Table_4.2_cytokine-cytokine-crosstalk-all.csv"), index=False)

# filter to parse cytokines and save that subset
cytokine_response_df_all_filter = cytokine_response_df_all[cytokine_response_df_all["response"].isin(cytokine_info_parse["gene_to_cytokine"])]
cytokine_response_df_all_filter.to_csv(os.path.join(output_tables_folder, "5.1_cytokine-cytokine-crosstalk-parse.csv"), index=False)

In [ ]:
if write_SI_tables_flag:
    df = cytokine_response_df_all.copy()
    df['treatment'] = df['treatment'].apply(lambda x: introduce_greek_letters(x))
    df['response'] = df['response'].apply(lambda x: introduce_greek_letters(x))
    df = df.rename(columns={
        'cell_type': 'Cell Type',
        'treatment': 'Cytokine Treatment',
        'response': 'Response Cytokine',
        'fold_changes': 'Log2 Fold Change by subunit',
        'p_value': 'Adjusted P-Value',
        'direction': 'Direction',
        'single_fc': 'Overall Log2 Fold Change'
    }
    )
    write_to_SI_table(df, "cytokine-cytokine-crosstalk")

In [ ]:
cytokine_response_df_all_filter = cytokine_response_df_all[cytokine_response_df_all['direction'].isin(['upregulated', 'downregulated'])]
counts = cytokine_response_df_all_filter.groupby(['treatment'])['direction'].value_counts().unstack().fillna(0)
counts['max'] = counts.max(axis=1)
counts = counts.sort_values(by='max', ascending=False).drop(columns=['max'])

up = counts['upregulated']
down = -counts['downregulated']

fig, ax = plt.subplots(figsize=(6, 3))

x = np.arange(len(counts))
ax.bar(x, up, color='red', label='Upregulated')
ax.bar(x, down, color='blue', label='Downregulated')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(counts.index, rotation=90, fontsize=6)
ax.set_ylabel('Number of responses')
ax.legend(frameon=False, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
import plotly.io as pio
pio.defaults.headers = None

# special IL-32-beta plot
treatment_order = CYTOKINE_ORDER
response_order = ALL_CYTOKINES_ORDER

treatment_order = [introduce_greek_letters(treatment) for treatment in treatment_order]
response_order = [introduce_greek_letters(response) for response in response_order]

# make sure the fold-change is sufficiently large for all gene products
cytokine_response_df_all_curr = cytokine_response_df_all[cytokine_response_df_all["direction"] != "neither"]
cytokine_response_df_all_curr = cytokine_response_df_all_curr[cytokine_response_df_all_curr["single_fc"].abs() >= 2]
cytokine_response_df_all_curr.loc[:, "response"] = cytokine_response_df_all_curr["response"].apply(lambda x: introduce_greek_letters(x))
cytokine_response_df_all_curr.loc[:, "treatment"] = cytokine_response_df_all_curr["treatment"].apply(lambda x: introduce_greek_letters(x))

current_df = cytokine_response_df_all_curr[cytokine_response_df_all_curr["treatment"] == "IL-32-β"].copy() 
current_df = current_df[current_df['cell_type'].isin(["CD14 Mono", "CD16 Mono", "cDC"])].copy() # myeloid only

link_color_df = []
for index, row in current_df.iterrows():
    if row['direction'] == 'upregulated':
        link_color_df.append({
            'cell_type': row['cell_type'],
            'response': row['response'],
            'color': 'rgba(220, 50, 50, 0.45)'  # soft red
        })
    elif row['direction'] == 'downregulated':
        link_color_df.append({
            'cell_type': row['cell_type'],
            'response': row['response'],
            'color': 'rgba(50, 100, 220, 0.45)'  # soft blue
        })
link_color_df = pd.DataFrame(link_color_df)

current_df['metadata'] = current_df['direction']
fig, height, width = create_sankey_diagram_strength(
    df=current_df,
    column1_name="treatment",
    column1_order=treatment_order,
    column2_name="cell_type",
    column3_name="response",
    link_color_df=link_color_df,
    column3_order=response_order,
    color1 = "rgba(44, 160, 44, 0.8)",
    color2 = "rgba(31, 119, 180, 0.8)",
    color3 = "rgba(44, 160, 44, 0.8)",
    title=None,
    height_add=-15
)

fig.write_image(
    os.path.join(plot_folder, f"5.1_cytokine_{unidecode('IL-32-beta')}_regulates_others.svg"),
    width=width,
    height=height
    )

In [ ]:
"""Compact, fixed-6-point Matplotlib Sankey for manuscript panels."""

import matplotlib.pyplot as plt
import warnings
from matplotlib.patches import PathPatch, Rectangle
from matplotlib.path import Path
import numpy as np
import pandas as pd


def create_sankey_diagram_strength_publication(
    df, functional_groupings, functional_order,
    column1_name='treatment', column2_name='cell_type', column3_name='response',
    column1_order=None, column2_order=None, column3_order=None,
    figsize_cm=(4.1, 8.6), node_colors=('#2ca02c', '#1f77b4', '#2ca02c'),
    functional_blocks=None,
    label_abbreviations=None,
    show_categories=True,
    cell_type_rotation=None,
    column_positions=None,
):
    """Return a Matplotlib figure and axes; all text is exactly 6 pt.

    Links count rows, as in create_sankey_diagram_strength. Input/output nodes
    are sorted by function, then the supplied cytokine order. Up/down links
    are separate ribbons, so opposing events are not cancelled. Cytokine labels
    occupy one column outside the ribbons. Functional labels use the supplied
    display names and broad functional categories in staggered vertical lanes.
    Every output group is labelled,
    including singletons; input groups are labelled when they have >1 nodes.
    Group mappings must use the same cytokine names as df (Greek aliases are
    fine). Incomplete metadata/orders and layout limits produce warnings,
    without changing the requested panel dimensions or 6 pt fonts.
    Dimensions are actual centimetres, independent of export resolution.
    Incoming plots (one output, multiple inputs) automatically put horizontal
    input/cell-type labels on the left and a vertical target on the right.
    show_categories can be False, True, or (input_side, output_side); a side
    with only one cytokine is never annotated. cell_type_rotation overrides
    the automatic 0 degrees (incoming) / 90 degrees (outgoing). column_positions
    optionally sets the three node x coordinates as fractions of panel width.
    """
    columns = [column1_name, column2_name, column3_name]
    d = df.copy()
    missing_columns = [c for c in columns if c not in d]
    if not missing_columns:
        incomplete = d[columns].isna().any(axis=1)
        if incomplete.any():
            warnings.warn(f'Omitting {incomplete.sum()} rows with missing node names.', stacklevel=2)
            d = d.loc[~incomplete].copy()
    if d.empty or missing_columns:
        message = 'No rows to plot.' if not missing_columns else f'Missing columns: {missing_columns}'
        warnings.warn(message, stacklevel=2)
        fig, ax = plt.subplots(figsize=np.array(figsize_cm)/2.54)
        ax.axis('off')
        ax.text(.5, .5, 'No Sankey data', ha='center', va='center', fontsize=6)
        return fig, ax
    functional_groupings = dict(functional_groupings)
    group_labels = dict(functional_order) if isinstance(functional_order, dict) else {}
    abbreviations = {
        'Immune cytokines': 'Immune cytokine', 'Growth/remodel': 'Growth/remod.',
        'TNF ligands': 'TNF lig.', 'Chemokines': 'Chemokine', 'Complement': 'Compl.',
        'Inflammatory': 'Inflamm.', 'Regulatory': 'Reg.',
        'Tissue repair': 'Repair', 'Tissue remodel.': 'Remodel.',
        'Vascular growth': 'Vasc. growth', 'B-cell support': 'B-cell supp.',
        'Cell death': 'Death', 'Neutrophil': 'Neutrophil', 'Monocyte': 'Mono.',
        'Eosinophil': 'Eosino.',
    }
    abbreviations.update(label_abbreviations or {})
    group_order = list(functional_order)
    if functional_blocks is None:
        functional_blocks = {
            'Immune cytokines': ['inflammatory', 'type 1', 'type 2', 'type 17',
                                 'regulatory', 'lymphocyte growth', 'myeloid growth'],
            'Growth/remodel': ['tissue repair', 'tissue remodeling', 'vascular growth'],
            'TNF ligands': ['B-cell support', 'costimulation', 'cell death'],
            'Chemokines': [g for g in group_order if g.startswith('chemo:')],
            'Complement': ['complement'],
        }
    functional_blocks = {category: list(groups) for category, groups in functional_blocks.items()}
    # The production matrix omits type 2 because none of its plotted producers
    # fall there; incoming panels can nevertheless contain type-2 stimuli.
    if ('type 2' in functional_groupings.values()
            and not any('type 2' in groups for groups in functional_blocks.values())):
        immune_key = next((k for k in functional_blocks if k.replace('\n', ' ') == 'Immune cytokines'),
                          'Immune cytokines')
        functional_blocks.setdefault(immune_key, []).append('type 2')
    category_of = {g: category.replace('\n', '') if '/\n' in category
                   else category.replace('\n', ' ')
                   for category, groups in functional_blocks.items() for g in groups}
    missing = [v for v in pd.unique(d[[column1_name, column3_name]].to_numpy().ravel())
               if pd.isna(functional_groupings.get(v))]
    if missing:
        warnings.warn(f'Missing functional grouping; labelled Unclassified: {missing}', stacklevel=2)
        functional_groupings.update({v: 'Unclassified' for v in missing})
    for column in (column1_name, column3_name):
        for value in pd.unique(d[column]):
            group = functional_groupings[value]
            if group not in group_order:
                group_order.append(group)
            if group not in category_of:
                warnings.warn(f'Missing broad category for {group}; labelled Other.', stacklevel=2)
                category_of[group] = 'Other'
                functional_blocks.setdefault('Other', []).append(group)
    ranks = {g: i for i, g in enumerate(group_order)}
    orders = []
    for side, (column, preferred) in enumerate(zip(
            columns, (column1_order, column2_order, column3_order))):
        present = pd.unique(d[column]).tolist()
        if preferred is not None:
            absent = set(present) - set(preferred)
            if absent:
                warnings.warn(f'{column} order omits {absent}; appended after listed nodes.', stacklevel=2)
            present = [v for v in dict.fromkeys(preferred) if v in present] + [v for v in present if v not in preferred]
        if side != 1:
            present.sort(key=lambda v: ranks[functional_groupings[v]])
        orders.append(present)

    incoming = len(orders[2]) == 1 and len(orders[0]) > 1
    if cell_type_rotation is None:
        cell_type_rotation = 0 if incoming else 90
    category_sides = (show_categories, show_categories) if isinstance(show_categories, bool) else show_categories

    width, height = np.array(figsize_cm) * 10  # geometry in mm
    fig, ax = plt.subplots(figsize=np.array(figsize_cm) / 2.54)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    ax.set(xlim=(0, width), ylim=(height, 0))
    ax.axis('off')
    top, bottom = 3, height - 3
    usable = bottom - top
    xs = ([width-24, width-9, width-3.5] if incoming
          else [3.5, width*.23, width-14])
    if column_positions is not None:
        xs = np.asarray(column_positions)*width
    node_width = .9
    counts = [d[c].value_counts().reindex(o) for c, o in zip(columns, orders)]
    largest = max(map(len, orders))
    # Use the largest common ribbon scale that keeps label centres >=2.35 mm
    # apart. Gaps depend on neighbouring node heights rather than consuming
    # most of the panel even when those nodes are already tall.
    minimum_gap, minimum_spacing = .65, 2.35
    def gaps_for(sizes):
        return np.maximum(minimum_gap, minimum_spacing - (sizes[:-1]+sizes[1:])/2)

    low, high = 0., usable / len(d)
    for _ in range(60):
        trial = (low+high)/2
        fits = all((n.to_numpy()*trial).sum() + gaps_for(n.to_numpy()*trial).sum() <= usable
                   for n in counts)
        if fits:
            low = trial
        else:
            high = trial
    scale = low
    if scale <= 1e-6:
        warnings.warn('Panel too short for collision-free 6 pt labels; drawing compressed '
                      'node spacing. Increase figsize_cm height for publication.', stacklevel=2)
        minimum_gap, minimum_spacing = usable / largest * .45, 0
        scale = usable / len(d) * .55
    bounds = []
    for order, n in zip(orders, counts):
        sizes = n.to_numpy() * scale
        gaps = gaps_for(sizes)
        if len(gaps):
            gaps += (usable - sizes.sum() - gaps.sum()) / len(gaps)
        y = top if len(order) > 1 else top + (usable - sizes.sum()) / 2
        side_bounds = {}
        for i, (value, size) in enumerate(zip(order, sizes)):
            side_bounds[value] = (y, y + size)
            y += size + (gaps[i] if i < len(gaps) else 0)
        bounds.append(side_bounds)

    def ribbon(x0, y0, y1, x1, z0, z1, color):
        bend = (x1 - x0) * .48
        path = Path([(x0, y0), (x0+bend, y0), (x1-bend, z0), (x1, z0),
                     (x1, z1), (x1-bend, z1), (x0+bend, y1), (x0, y1), (x0, y0)],
                    [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4,
                     Path.LINETO, Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.CLOSEPOLY])
        ax.add_patch(PathPatch(path, facecolor=color, edgecolor='none', zorder=1))

    for side in (0, 1):
        starts = {v: b[0] for v, b in bounds[side].items()}
        ends = {v: b[0] for v, b in bounds[side+1].items()}
        direction = 'direction' if 'direction' in d else 'metadata' if 'metadata' in d else None
        for source in orders[side]:
            for target in orders[side+1]:
                sub = d[d[columns[side]].eq(source) & d[columns[side+1]].eq(target)]
                chunks = sub.groupby(direction, sort=False) if side == 1 and direction else [(None, sub)]
                for state, rows in chunks:
                    if not len(rows):
                        continue
                    size = len(rows) * scale
                    color = ((.86, .20, .20, .40) if state in ('upregulated', 'upregulation') else
                             (.20, .39, .86, .40) if state in ('downregulated', 'downregulation') else
                             (0, 0, 0, .15))
                    ribbon(xs[side]+node_width/2, starts[source], starts[source]+size,
                           xs[side+1]-node_width/2, ends[target], ends[target]+size, color)
                    starts[source] += size
                    ends[target] += size

    text_artists = []
    def text(x, y, label, **kwargs):
        artist = ax.text(x, y, label, fontsize=6, va='center', **kwargs)
        text_artists.append(artist)
        return artist

    for side, order in enumerate(orders):
        for i, value in enumerate(order):
            lo, hi = bounds[side][value]
            centre = (lo+hi)/2
            ax.add_patch(Rectangle((xs[side]-node_width/2, lo), node_width, hi-lo,
                                   facecolor=node_colors[side], edgecolor='black', lw=.3, zorder=3))
            if side == 2:
                if incoming:
                    text(xs[side]+1.9, centre, value, ha='center', rotation=90, zorder=4)
                else:
                    text(xs[side]-1, centre, value, ha='right', zorder=4)
            elif side == 0:
                text(xs[side]-1, centre, value, ha='right', rotation=0 if incoming else 90, zorder=4)
            else:
                text(xs[side]-1.6, centre, value,
                     ha='right' if cell_type_rotation == 0 else 'center',
                     rotation=cell_type_rotation, zorder=4)

    # Nested brackets identify every subgroup's broad category without
    # repeating long category names for each member.
    group_text_lanes = {}
    for side in (0, 2):
        # No redundant category annotation on a single-cytokine side.
        if len(orders[side]) <= 1 or not category_sides[side//2]:
            continue
        grouped = []
        for group in group_order:
            members = [v for v in orders[side] if functional_groupings[v] == group]
            if members:
                lo = bounds[side][members[0]][0]
                hi = bounds[side][members[-1]][1]
                grouped.append((group, lo, hi))
        for i, (group, lo, hi) in enumerate(grouped):
            lane = i % 2
            x = xs[side]+1.5+lane*3.1 if side == 2 else xs[side]-10-lane*3.1
            label = group_labels.get(group, group)
            category = category_of.get(group)
            if category.lower() == label.lower():
                continue
            label = abbreviations.get(label, label)
            ax.plot([x, x], [lo, hi], color='#444444', lw=.45)
            artist = text(x+1.3 if side == 2 else x-1.3, (lo+hi)/2, label,
                          ha='center', rotation=90, multialignment='center', zorder=4)
            group_text_lanes.setdefault((side, lane), []).append(artist)

        for category, groups in functional_blocks.items():
            members = [v for v in orders[side] if functional_groupings[v] in groups]
            if not members or (side == 0 and len(orders[side]) == 1):
                continue
            lo = bounds[side][members[0]][0]
            hi = bounds[side][members[-1]][1]
            x = xs[side]+8.5 if side == 2 else xs[side]-17.5
            inward = -.5 if side == 2 else .5
            ax.plot([x+inward, x, x, x+inward], [lo, lo, hi, hi], color='#222222', lw=.55)
            label = category.replace('\n', '') if '/\n' in category else category.replace('\n', ' ')
            label = abbreviations.get(label, label)
            artist = text(x+1.3 if side == 2 else x-1.3, (lo+hi)/2,
                          label, ha='center', rotation=90, zorder=4)
            group_text_lanes.setdefault((side, 2), []).append(artist)

    # Measure labels in physical units, then move them within their
    # own lanes to avoid collisions and page clipping. Brackets retain the
    # exact group spans. Neither font size nor panel dimensions change.
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    pixels_per_mm = fig.dpi / 25.4
    for lane in group_text_lanes.values():
        half = np.array([a.get_window_extent(renderer).height / pixels_per_mm / 2 for a in lane])
        positions = np.array([a.get_position()[1] for a in lane])
        if 2*half.sum() + (len(lane)-1)*1.5 > height-2:
            warnings.warn('Functional labels cannot all fit at 6 pt in this panel; '
                          'drawing without resizing. Increase figsize_cm height.', stacklevel=2)
        positions[0] = max(positions[0], 1+half[0])
        for i in range(1, len(lane)):
            positions[i] = max(positions[i], positions[i-1]+half[i-1]+half[i]+1.5)
        positions[-1] = min(positions[-1], height-1-half[-1])
        for i in range(len(lane)-2, -1, -1):
            positions[i] = min(positions[i], positions[i+1]-half[i+1]-half[i]-1.5)
        for artist, y in zip(lane, positions):
            artist.set_y(y)

    # Expose artists for physical-size overlap checks in the rendering script.
    fig._sankey_text_artists = text_artists
    fig._sankey_bounds = bounds
    fig._sankey_category_sides = tuple(side for side in (0, 2)
                                      if len(orders[side]) > 1 and category_sides[side//2])
    fig._sankey_output_groups = [g for g in group_order
                                 if any(functional_groupings[v] == g for v in orders[2])]
    return fig, ax


In [ ]:
functional_groups_plot = {
    introduce_greek_letters(k): v
    for k, v in cytokine_name_to_functional.items()
}
fig, ax = create_sankey_diagram_strength_publication(
    current_df,
    functional_groupings=functional_groups_plot,
    functional_order=functional_column_order,
    functional_blocks=functional_blocks,
    column1_order=treatment_order,
    column2_order=["CD14 Mono", "CD16 Mono", "cDC"],
    column3_order=response_order,
)
fig.savefig(os.path.join(plot_folder, f"5.1_cytokine_{unidecode('IL-32-beta')}_regulates_others_test.svg"))
# fig.write_image(
#     ,
#     width=width,
#     height=height
#     )

In [ ]:
treatment_order = CYTOKINE_ORDER
response_order = ALL_CYTOKINES_ORDER

treatment_order = [introduce_greek_letters(treatment) for treatment in treatment_order]
response_order = [introduce_greek_letters(response) for response in response_order]

# make sure the fold-change is sufficiently large for all gene products
cytokine_response_df_all_curr = cytokine_response_df_all[cytokine_response_df_all["direction"] != "neither"]
cytokine_response_df_all_curr.loc[:, "response"] = cytokine_response_df_all_curr["response"].apply(lambda x: introduce_greek_letters(x))
cytokine_response_df_all_curr.loc[:, "treatment"] = cytokine_response_df_all_curr["treatment"].apply(lambda x: introduce_greek_letters(x))

# # now, do this separately for every cytokine
# current_plot_folder = os.path.join(plot_folder, "5.1_regulates_others")
# os.makedirs(current_plot_folder, exist_ok=True)
# for treat_index, current_treatment in enumerate(treatment_order):
#     current_df = cytokine_response_df_all_curr[cytokine_response_df_all_curr["treatment"] == current_treatment]
#     if len(current_df) < 4:
#         continue
    
#     path = os.path.join(current_plot_folder, f"5.1_cytokine_{treat_index}_{unidecode(current_treatment)}_regulates_others.svg")
#     if os.path.exists(path):
#         continue
    
#     link_color_df = []
#     for index, row in current_df.iterrows():
#         if row['direction'] == 'upregulated':
#             link_color_df.append({
#                 'cell_type': row['cell_type'],
#                 'response': row['response'],
#                 'color': 'rgba(220, 50, 50, 0.45)' # red
#             })
#         elif row['direction'] == 'downregulated':
#             link_color_df.append({
#                 'cell_type': row['cell_type'],
#                 'response': row['response'],
#                 'color': 'rgba(50, 100, 220, 0.45)' # blue
#             })
#     link_color_df = pd.DataFrame(link_color_df)
    
#     fig, height, width = create_sankey_diagram_strength(
#         df=current_df,
#         column1_name="treatment",
#         column1_order=treatment_order,
#         column2_name="cell_type",
#         column3_name="response",
#         column3_order=response_order,
#         link_color_df=link_color_df,
#         color1 = "rgba(44, 160, 44, 0.8)",
#         color2 = "rgba(31, 119, 180, 0.8)",
#         color3 = "rgba(44, 160, 44, 0.8)",
#         title=None
#     )
    
#     treat_index_str = str(treat_index).zfill(2)
#     fig.write_image(
#         path,
#         width=width,
#         height=height
#         )

sizes = {
    'IFN-γ': (4.5, 3.5),
    'TNF-α': (4.5, 5)
}

current_plot_folder = os.path.join(plot_folder, "5.1_is_upregulated_by_others")
os.makedirs(current_plot_folder, exist_ok=True)
for current_response in cytokine_response_df_all_curr["response"].unique():
    path = os.path.join(current_plot_folder, f"5.1_cytokine_{unidecode(current_response)}_is_upregulated_by_others.svg")
    # if os.path.exists(path):
    #     continue
    
    current_df = cytokine_response_df_all_curr[cytokine_response_df_all_curr["response"] == current_response]
    current_df = current_df[current_df['direction'].isin(['upregulated'])]
    if len(current_df) < 6:
        continue
    
    if not current_response in ['IFN-γ', 'TNF-α']:
        continue
    
    fig, ax = create_sankey_diagram_strength_publication(
        current_df,
        functional_groupings=functional_groups_plot,
        functional_order=functional_column_order,
        functional_blocks=functional_blocks,
        column1_order=treatment_order,
        column2_order=ALL_CELL_TYPES,
        column3_order=response_order,
        cell_type_rotation=0,
        figsize_cm=sizes[current_response],
    )
    fig.savefig(path)
    
    # fig, height, width = create_sankey_diagram_strength(
    #     df=current_df,
    #     column1_name="treatment",
    #     column1_order=treatment_order,
    #     column2_name="cell_type",
    #     column3_name="response",
    #     column3_order=response_order,
    #     color1 = "rgba(44, 160, 44, 0.8)",
    #     color2 = "rgba(31, 119, 180, 0.8)",
    #     color3 = "rgba(44, 160, 44, 0.8)",
    #     title=None
    # )
    # fig.write_image(
    #     path,
    #     width=width,
    #     height=height
    # )

In [ ]:
cytokine_response_df_all_curr["response"].unique()

In [ ]:
cytokine_response_df_all_curr[cytokine_response_df_all_curr["response"] == 'TNF-α']

In [ ]:
current_response

In [ ]:
current_df['response']

In [ ]:
cytokine_response_df_all_curr["response"].unique()

In [ ]:
# overview plots
cytokine_response_df_all_current = cytokine_response_df_all[cytokine_response_df_all["direction"].isin(["upregulated", "downregulated"])]

# treatments
treatment_counts = cytokine_response_df_all_current["treatment"].value_counts()
treatment_counts = treatment_counts[treatment_counts > 10]

plt.figure(figsize=cm_to_inches((7, 2.4)))
plt.bar(treatment_counts.index, treatment_counts.values, color='skyblue', edgecolor='black')
plt.xticks(rotation=90)
plt.ylabel('outgoing\nsignaling\nresponses')
apply_greek_labels_heatmap(plt.gca())
plt.margins(x=0.02)
plt.xticks(fontsize=5.9)
plt.tick_params(axis='x', which='both', length=0)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "5.1_stimulation_counts.svg"), bbox_inches="tight")

# responses
response_counts = cytokine_response_df_all_current["response"].value_counts()
response_counts = response_counts[response_counts > 4]

plt.figure(figsize=cm_to_inches((12, 3)))
plt.bar(response_counts.index, response_counts.values, color='skyblue', edgecolor='black')
plt.xticks(rotation=90)
plt.ylabel('incoming\nsignaling responses')
apply_greek_labels_heatmap(plt.gca())
plt.margins(x=0.02)
plt.xticks(fontsize=5.5)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "5.1_response_counts.svg"), bbox_inches="tight")
plt.show()

## 5.2 - Supernatant simulation

In [ ]:
plot_folder = os.path.join(base_plot_folder, "5.2_supernatant_simulation")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
from utils.proteomics_utils import calculate_supernatant
from utils.helper_functions import calculate_cell_fractions

def wrapper_supernatants(counts, norm_df, mean_UMIs_per_cell):
    all_supernatants = []
    all_contributions = []
    
    cell_fraction_vector = calculate_cell_fractions(counts, norm_df.columns)
    cell_fraction_vector_scaled = calculate_cell_fractions(counts, norm_df.columns, mean_UMIs_per_cell)

    # used to have some weird errors, making sure
    assert cell_fraction_vector_scaled.index.equals(norm_df.columns)
    assert not cell_fraction_vector_scaled.isna().any()

    for index, row in cytokine_info.iterrows():
        if pd.isna(row["gene_to_cytokine"]):  # LT-alpha2-beta1 duplicates LT-alpha1-beta2
            continue
        genes = ';'.join([s.strip() for s in row['gene'].split(',')])

        all_present=True
        for gene in genes.split(";"):
            if not gene in norm_df.index:
                all_present=False
                print(f"Missing gene: {gene}")
        if not all_present:
            continue

        metadata_row = pd.Series({
            "protein": index,
            "usable_genes": genes,
            "target_genes": genes
        })
        supernatant, contrib = calculate_supernatant(
            metadata_row,
            norm_df,
            cell_fraction_vector_scaled,
        )
        all_supernatants.append(supernatant)
        all_contributions.append(contrib)
        
    all_supernatants = pd.concat(all_supernatants)
    all_contributions = pd.concat(all_contributions)
    
    all_supernatants = all_supernatants.pivot_table(columns='protein', index=['treatment', 'donor'], values='sim_supernatant')
    all_supernatants.index = pd.MultiIndex.from_tuples([(i[0], i[1].replace("Donor", "D")) for i in all_supernatants.index])
    all_supernatants.index.names = ['treatment', 'donor']
    
    return cell_fraction_vector, all_supernatants, all_contributions
    
cell_fraction_vector, all_supernatants, all_contributions = wrapper_supernatants(cell_count_agg, count_df_norm_agg, mean_UMIs_per_cell)

detailed_supernatant_cpm = normalize_df_to_rpm(sum_gene_count_df_raw_by_donor, along_axis=0, pseudocount_initial=0, pseudocount_after=0.5)
cell_fraction_vector_detailed, all_supernatants_detailed, all_contributions_detailed = wrapper_supernatants(all_counts_by_condition, detailed_supernatant_cpm, mean_UMIs_per_cell_detailed)
del detailed_supernatant_cpm

In [ ]:
def assert_supernatant_agreement(aggregate, detailed, min_correlation=0.95, max_median_abs_log2_error=0.25, max_p95_abs_log2_error=1):
    # Make sure that calculations generate congruent results
    a, d = aggregate.align(detailed, join="inner")
    assert a.size > 0, "No overlapping conditions/proteins"
    a, d = a.to_numpy(float).ravel(), d.to_numpy(float).ravel()
    valid = np.isfinite(a) & np.isfinite(d) & (a > 0) & (d > 0)
    assert valid.mean() > 0.95, f"Only {valid.mean():.1%} of aligned values are finite and positive"
    log_a, log_d = np.log2(a[valid]), np.log2(d[valid])
    correlation = np.corrcoef(log_a, log_d)[0, 1]
    errors = np.abs(log_d - log_a)
    report = {"correlation": correlation, "median_abs_log2_error": np.median(errors), "p95_abs_log2_error": np.quantile(errors, 0.95), "n_compared": valid.sum()}
    assert correlation >= min_correlation, f"Poor aggregate–detailed correlation: {correlation:.3f}"
    assert report["median_abs_log2_error"] <= max_median_abs_log2_error, f"Median discrepancy too large: {report['median_abs_log2_error']:.3f} log2"
    assert report["p95_abs_log2_error"] <= max_p95_abs_log2_error, f"Large tail discrepancy: {report['p95_abs_log2_error']:.3f} log2"
    return report

assert_supernatant_agreement(all_supernatants, all_supernatants_detailed)

In [ ]:
def get_supernatant_fcs(df, baseline_label):
    # Split baseline and non-baseline
    is_baseline = df.index.get_level_values("treatment") == baseline_label
    df_treat = df.loc[~is_baseline]
    baseline = df.loc[is_baseline]

    # Align indices
    baseline = baseline.droplevel('treatment')
    treat_match = df_treat.index.droplevel('treatment')
    baseline = baseline.reindex(treat_match)
    baseline.index = df_treat.index

    # Calc log2fcs
    log2fc_by_donor = np.log2(df_treat) - np.log2(baseline)
    return log2fc_by_donor

all_supernatants_main = all_supernatants[~all_supernatants.index.get_level_values('treatment').isin(review_replace_list)]
all_supernatants_review = all_supernatants[all_supernatants.index.get_level_values('treatment').isin(full_review_list)]
supernatant_log2fc_main = get_supernatant_fcs(all_supernatants_main, "PBS")
supernatant_log2fc_review = get_supernatant_fcs(all_supernatants_review, "PBS_review")
supernatant_log2fc_by_donor = pd.concat([supernatant_log2fc_main, supernatant_log2fc_review]).T

In [ ]:
all_supernatants = all_supernatants.loc[all_supernatants.index.get_level_values('treatment').isin(CYTOKINE_ORDER_WITH_PBS)]
supernatant_log2fc_by_donor = supernatant_log2fc_by_donor.loc[:, supernatant_log2fc_by_donor.columns.get_level_values('treatment').isin(CYTOKINE_ORDER_WITH_PBS)]

In [ ]:
supernatant_log2fc = supernatant_log2fc_by_donor.T.groupby(level='treatment').mean().T

# restrict to cytokines expressed in at least one cell type
supernatant_log2fc = supernatant_log2fc.loc[supernatant_log2fc.index.isin(cytokines_expressed_anywhere)]

# filter to those where the maximum expression is substantial
expression_filter = all_supernatants.groupby(level='treatment').mean().max(axis=0) >= 8
expression_filter = expression_filter[expression_filter].index
expression_filter = [c for c in expression_filter if c in supernatant_log2fc.index]
supernatant_log2fc = supernatant_log2fc.loc[expression_filter]

In [ ]:
# Cell-type contribution per condition
contrib = all_contributions.pivot_table(index=["treatment", "donor", "protein"], columns=["cell_type"], values="cell_type_contribution")

def log2fc_attribution(df, baseline):
    is_baseline = df.index.get_level_values("treatment") == baseline
    treated = df.loc[~is_baseline]

    control = df.loc[is_baseline].droplevel("treatment")
    control = control.reindex(treated.index.droplevel("treatment"))
    control.index = treated.index

    delta = treated - control
    total_treated = treated.sum(axis=1)
    total_control = control.sum(axis=1)
    total_delta = total_treated - total_control
    total_log2fc = np.log2(total_treated / total_control)

    # Exact additive decomposition: rows sum to total_log2fc
    scale = total_log2fc.div(total_delta).where(~np.isclose(total_delta, 0), 0)
    return delta.mul(scale, axis=0)

# get the contributions
contrib_log2fc = pd.concat([
    log2fc_attribution(contrib.loc[~contrib.index.get_level_values("treatment").isin(review_replace_list)], "PBS"),
    log2fc_attribution(contrib.loc[contrib.index.get_level_values("treatment").isin(full_review_list)], "PBS_review")
]).groupby(level=["treatment", "protein"]).mean()

# make sure this gives the correct result
expected = supernatant_log2fc.T.stack()
expected.index = expected.index.set_names(["treatment", "protein"])
assert np.allclose(contrib_log2fc.sum(axis=1).reindex(expected.index), expected, equal_nan=True)

In [ ]:
# Significant supernatant changes among the displayed treatments

# get the most important cytokines
supernatant_changes = (abs(supernatant_log2fc) > 1).sum().sort_values(ascending=False).head(20)
supernatant_changes = supernatant_changes.iloc[::-1]

# what was changed?
events = supernatant_log2fc.loc[:, supernatant_changes.index].T.stack().loc[lambda x: x.abs() > 1]
events.index = events.index.set_names(["treatment", "protein"])

# Cell type contributing most in the direction of each total change
directional_contrib = contrib_log2fc.reindex(events.index).mul(np.sign(events), axis=0)
dominant_cell = directional_contrib.idxmax(axis=1)
dominant_val = directional_contrib.max(axis=1).div(directional_contrib.clip(lower=0).sum(axis=1))

# Classify dominant and mixed contributions
contribution_fraction = directional_contrib.clip(lower=0).div(directional_contrib.clip(lower=0).sum(axis=1), axis=0)
source_class = dominant_cell.copy()
source_class.loc[(dominant_val <= 0.75) & (contribution_fraction["Mono/cDC"] >= 0.25)] = "mixed\nmyeloid–lymphoid"
source_class.loc[(dominant_val <= 0.75) & (contribution_fraction["Mono/cDC"] < 0.25)] = "mixed lymphoid"

source_order = ["B cell", "T cell", "NK cell", "Mono/cDC", "mixed lymphoid", "mixed\nmyeloid–lymphoid"]
stacked_counts = pd.crosstab(source_class.index.get_level_values("treatment"), source_class)
stacked_counts = stacked_counts.reindex(index=supernatant_changes.index, columns=source_order, fill_value=0)

source_colors = {
    "B cell": CELL_TYPE_COLORS["B cell"],
    "T cell": CELL_TYPE_COLORS["CD4 T cell"],
    "NK cell": CELL_TYPE_COLORS["NK CD56low"],
    "Mono/cDC": CELL_TYPE_COLORS["CD14 Mono"],
    "mixed lymphoid": "lightgray",
    "mixed\nmyeloid–lymphoid": "dimgray"
}

fig, ax = plt.subplots(figsize=cm_to_inches((3, 6)))
stacked_counts.plot.barh(stacked=True, ax=ax, color=[source_colors[x] for x in stacked_counts.columns], edgecolor="black", linewidth=0.35)
ax.set_xlabel(">2-fold cytokine changes\nin sim. supernatant")
ax.set_ylabel("")
apply_greek_labels_heatmap(ax)
ax.margins(x=0.02)
ax.tick_params(axis="y", which="both", length=0, labelsize=5.9)
ax.spines[["top", "right"]].set_visible(False)
ax.set_ylim(-1, len(supernatant_changes))
ax.legend(title="dominant source", frameon=False, fontsize=5, title_fontsize=5)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "5.2_prot_supernatant_changes_by_cell_type.svg"), bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=cm_to_inches((2.5, 6)))
plt.barh(supernatant_changes.index, supernatant_changes.values, color='skyblue', edgecolor='black')
plt.xticks(rotation=0)
plt.xlabel('>2-fold cytokine changes\nin sim. supernatant')
apply_greek_labels_heatmap(plt.gca())
plt.margins(x=0.02)
plt.yticks(fontsize=5.9)
plt.tick_params(axis='y', which='both', length=0)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.ylim(-1, len(supernatant_changes))
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "5.2_prot_supernatant_changes.svg"), bbox_inches="tight")
plt.show()

In [ ]:
# IL-1 type cytokine release filter
il1_release = count_df_norm_agg['Mono/cDC'].loc['CASP5'].groupby(level=0).median() > 30

In [ ]:
from matplotlib.patches import Rectangle, Patch
LOG2FC_FILTER = 1

sub = supernatant_log2fc.loc[:, abs(supernatant_log2fc).max(axis=0) > LOG2FC_FILTER]
sub = sub[abs(sub).max(axis=1) > LOG2FC_FILTER]
cytokines_present = [cyto for cyto in CYTOKINE_ORDER if cyto in sub.columns]
sub = sub[cytokines_present].T

order = [cyto for cyto in cytokine_info.index[cytokine_info["gene_to_cytokine"].notna()] if cyto in sub.columns]
sub = sub[order]

fig, ax = plt.subplots(1, 1, figsize=cm_to_inches((20, 14)))
sub_plot = sub.rename(columns=cytokine_info["gene_to_cytokine"])
sns.heatmap(sub_plot, square=True, cmap='coolwarm', center=0, cbar_kws={'shrink': 0.3}, xticklabels=1, yticklabels=1)

for i in range(sub.shape[0]):
    for j in range(sub.shape[1]):
        if abs(sub.iloc[i, j]) > LOG2FC_FILTER:
            treatment = sub.index[i]
            protein = sub.columns[j]
            if sub.iloc[i, j] < 0:
                treatment = 'PBS_review' if treatment in review_replace_list else 'PBS'
            if (not il1_release.loc[treatment]) and protein in ["IL-1-alpha", 'IL-1-beta', 'IL-18']:
                ls = '--'
            else:
                ls = '-'
            ax.add_patch(Rectangle((j, i), 1, 1, fill=False, edgecolor='black' if ls == '-' else 'gray', linewidth=0.7, clip_on=False, ls=ls))

plt.tick_params(axis='both', labelsize=6)
apply_greek_labels_heatmap(ax)
plt.xlabel('response cytokine')
plt.ylabel('cytokine treatment')
plt.tick_params(axis='both', length=0)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, '5.2_supernatant_all.svg'))
plt.show()

# -------------------------------------------------

# filter to those with potentially strong effects
any_strong = strong_impact_by_cell_type[['B cell', 'CD4 T cell', 'NK CD56low', 'CD14 Mono']].any(axis=1)
any_strong = any_strong[~any_strong.index.isin(["C3a"])]
any_strong = any_strong[any_strong].index
sub = sub.loc[:, sub.columns.isin(any_strong)]

sub = sub.loc[:, abs(sub).max(axis=0) > LOG2FC_FILTER]
sub = sub[abs(sub).max(axis=1) > LOG2FC_FILTER]

order = [cyto for cyto in CYTOKINE_ORDER if cyto in sub.columns]
sub = sub[order]

fig, ax = plt.subplots(1, 1, figsize=cm_to_inches((6, 6)))
sub_plot = sub.rename(columns=cytokine_info["gene_to_cytokine"])
sns.heatmap(sub_plot, square=True, cmap='coolwarm', center=0, cbar_kws={'shrink': 0.5, 'label': 'log2FC(simulated supernatant)'})

for i in range(sub.shape[0]):
    for j in range(sub.shape[1]):
        if abs(sub.iloc[i, j]) > LOG2FC_FILTER:
            treatment = sub.index[i]
            protein = sub.columns[j]
            if sub.iloc[i, j] < 0:
                treatment = 'PBS_review' if treatment in review_replace_list else 'PBS'
            if (not il1_release.loc[treatment]) and protein in ["IL-1-alpha", 'IL-1-beta', 'IL-18']:
                ls = '--'
            else:
                ls = '-'

            ax.add_patch(Rectangle((j, i), 1, 1, fill=False, edgecolor='black' if ls == '-' else 'gray', linewidth=0.7, clip_on=False, ls=ls))

apply_greek_labels_heatmap(ax)
plt.tick_params(axis='both', length=0)
plt.xlabel('response cytokine')
plt.ylabel('cytokine treatment')
plt.tight_layout()

legend_handles = [
    Patch(facecolor='none', edgecolor='black', linewidth=0.7, label=r'$|\mathrm{log_2FC}| > 1$'),
    Patch(facecolor='none', edgecolor='gray', linewidth=0.7, linestyle='--', label='CASP5-constrained')
]

ax.legend(handles=legend_handles, frameon=False, loc='upper left')
plt.savefig(os.path.join(plot_folder, '5.2_supernatant_salient.svg'))
plt.show()

In [ ]:
cyto = 'IFN-gamma'
vals = supernatant_log2fc.loc[cyto].sort_values(ascending=False)
top5 = vals.head(5)
bottom5 = vals.tail(5)
plot_vals = pd.concat([top5, bottom5]).sort_values()
fig, ax = plt.subplots(figsize=cm_to_inches((3, 4)))
ax.barh(
    plot_vals.index,
    plot_vals.values,
    height=0.65,
)
ax.axvline(0, color='black', lw=0.7)
# visually separate bottom/top 5
ax.axhline(4.5, color='0.8', lw=0.7)

ax.set_xlabel(r'log$_2$FC')
ax.set_title(introduce_greek_letters(cyto), fontsize=7)
ax.tick_params(axis='both', labelsize=7)
ax.spines[['top', 'right']].set_visible(False)
apply_greek_labels_heatmap(ax)

plt.tight_layout()
plt.show()

## 5.3 - A more detailed IL-32-beta signaling analysis

In [ ]:
plot_folder = os.path.join(base_plot_folder, "5.3_il32_analysis")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
def robust_z_score(df):
    median = df.median(axis=1)
    mad = (df.sub(median, axis=0)).abs().median(axis=1)
    robust_z = df.sub(median, axis=0).div(1.4826 * mad.replace(0, np.nan), axis=0)
    return robust_z

monocyte_diff_marker_panels = {
    "moDC markers": [
        "CD1A",    # lipid antigen presentation
        "CD1C",    # classical moDC marker
        "ITGAM",   # Cd11b
        "HLA-DRA", # MHC-II expansion
        "CIITA",   # MHC-II transactivator
    ],
    "macrophage markers": [ 
        "CD14",    # monocyte/macrophage marker
        "MERTK",   # 
        "MAFB",    # transcription factor for macrophage differentiation
        "CD68",    # macrophage marker
        "CD163",   # macrophage marker
    ],
}

robust_vals_by_panel = {}
for panel_name, genes in monocyte_diff_marker_panels.items():
    # we here use raw values because the filters for edgeR means that some genes are missing for some conditions
    # this makes it hard to compute z-scores properly
    cd14_lfc = raw_log2fc_values.loc[genes]["CD14 Mono"]
    cd14_lfc = cd14_lfc[[c for c in CYTOKINE_ORDER if c in cd14_lfc.columns]]
    robust_vals_by_panel[panel_name] = robust_z_score(cd14_lfc)
    vals = raw_log2fc_values.loc[genes]["CD14 Mono"][
        ["IL-4", "GM-CSF", "M-CSF", "IL-10", "C5a", "IFN-beta", "IL-32-beta", "VEGF"]
    ]
    
    plt.figure(figsize=cm_to_inches((6, 3.5)))
    sns.heatmap(vals, center=0, cmap="vlag", annot=True, fmt=".1f", cbar_kws={"label": "log2 Fold Change"})
    plt.ylabel(panel_name)
    plt.xlabel("")
    plt.tick_params(axis='x', rotation=90, length=0)
    plt.tick_params(axis='y', rotation=0, length=0)
    apply_greek_labels_heatmap(plt.gca())
    plt.savefig(os.path.join(plot_folder, f"heatmap_{panel_name.replace(' ', '_')}.svg"))
    plt.show()

for panel in robust_vals_by_panel:
    vals = robust_vals_by_panel[panel]
    score = vals.mean(axis=0).sort_values(ascending=False)
    top5 = score.head(5)
    bottom5 = score.tail(5)
    
    plt.figure(figsize=cm_to_inches((4, 4.5)))
    indices = np.arange(len(top5) + len(bottom5))
    plt.barh(indices[:5], top5.values, color="tab:green",)
    plt.barh(indices[5:], bottom5.values, color="tab:red")
    plt.axvline(0, color="black", linewidth=0.8)
    plt.gca().invert_yaxis()
    plt.yticks(indices, list(top5.index) + list(bottom5.index))
    plt.xlabel("robust Z-score")
    plt.title(f"{panel}\n{', '.join(monocyte_diff_marker_panels[panel])}", fontsize=6.5)
    apply_greek_labels_heatmap(plt.gca())
    plt.savefig(os.path.join(plot_folder, f"5.3_barplot_{panel.replace(' ', '_')}.svg"))
    plt.show()

In [ ]:
IL32b_response = cytokine_response_df_all[cytokine_response_df_all["response"] == "IL-32"]
IL32b_response.to_csv(os.path.join(output_tables_folder, "Table_5.3_IL-32-beta_response.csv"), index=False)
IL32b_treatment = cytokine_response_df_all[cytokine_response_df_all["treatment"] == "IL-32-beta"]
IL32b_treatment.to_csv(os.path.join(output_tables_folder, "Table_5.3_IL-32-beta_treatment.csv"), index=False)

In [ ]:
# have a look at all chemokines
chemokines = cytokine_info[cytokine_info['family_level_1'] == "chemokine"].index
detected_in_CD14_Mono = edger_df_log2FC['CD14 Mono'].notna().any(axis=1)
detected_in_CD14_Mono = detected_in_CD14_Mono[detected_in_CD14_Mono].index
chemokines_detected_in_CD14Mono = [c for c in chemokines if c in detected_in_CD14_Mono]
edger_df_log2FC.loc[chemokines_detected_in_CD14Mono]["CD14 Mono"]["IL-32-beta"].sort_values()

In [ ]:
# get median fold-change for specific cytokines (called out in main text)
il32_treat = cytokine_response_df_all[cytokine_response_df_all['treatment'] == 'IL-32-beta']
il32_treat = il32_treat[il32_treat['cell_type'] == 'CD14 Mono']

print(il32_treat[il32_treat['response'].isin(["CXCL1", "CXCL2", "CXCL3", "CXCL5", "CXCL6", "IL-8", "IL-1-alpha", "IL-1-beta"])]['single_fc'].median())
print(il32_treat[il32_treat['response'].isin(["CXCL9", "CXCL10", "CXCL11", "IL-18"])]['single_fc'].median())

In [ ]:
# analyze consensus signaling responses induced by IL-32-beta
chemokine_panels = {
    "Neutrophil": ["CXCL1", "CXCL2", "CXCL3", "CXCL5", "IL-8"], # CXCR1/2 ligands (direct type III response)
    "Th1": ["CXCL9", "CXCL10", "CXCL11"], # CXCR3 ligands (Th1, CD8+ T cells, NK cells)
    "Th2": ["CCL17", "CCL22"], # CCR4 ligands (Th2 cells, Tregs)
    "type I": ["CCL3", "CCL3L1", "CCL4", "CCL5"],
    "mixed": ["CCL7", "CCL8"],
    "type II": ["CCL13"]
}
other_cytokines = {
    "Survival factors": ["APRIL", "TRAIL"],
    "Growth factors": ["HGF", "PDGF-CC", "BMP-6", "VEGF", "TGF-alpha"],
    "IL-1 family": ["IL-1-alpha", "IL-1-beta", "IL-18"],
    "IL-6 family": ["IL-6", "OSM"],
    "IL-10 family": ["IL-10", "IL-19", "IL-24"]
}

for plot_index, response_categories in enumerate([chemokine_panels, other_cytokines]):
    response_to_category = {}
    for cat, responses in response_categories.items():
        for resp in responses:
            response_to_category[resp] = cat

    # Filter IL32b_treatment to only those responses in our categories
    plot_df = cytokine_response_df_all[cytokine_response_df_all['treatment'] == 'IL-32-beta']
    plot_df = plot_df[plot_df['response'].isin(response_to_category.keys())].copy()
    plot_df['category'] = plot_df['response'].map(response_to_category)

    # Prepare x axis
    category_order = list(response_categories.keys())
    response_order = []
    response_to_x = {}
    x_tick_labels = []
    x_tick_positions = []
    x_pos = 0
    for cat in category_order:
        responses = response_categories[cat]
        for resp in responses:
            response_order.append(resp)
            response_to_x[resp] = x_pos
            x_tick_labels.append(resp)
            x_tick_positions.append(x_pos)
            x_pos += 1
        x_pos += 1

    # Plot
    plt.figure(figsize=cm_to_inches((8, 4)))
    offset_width = 0.12
    
    cell_types = ['cDC', 'CD14 Mono', 'CD16 Mono']
    for i, cell_type in enumerate(cell_types):
        cell_type_df = plot_df[plot_df['cell_type'] == cell_type]
        for resp in cell_type_df['response'].unique():
            resp_df = cell_type_df[cell_type_df['response'] == resp]
            x = response_to_x[resp] + (i - (len(cell_types)-1)/2)*offset_width
            plt.scatter(
                np.full(len(resp_df), x),
                resp_df['single_fc'],
                color=CELL_TYPE_COLORS[cell_type],
                label=cell_type if resp == cell_type_df['response'].unique()[0] else "",
                alpha=1,
                s=10,
                edgecolor='none',
                linewidth=0.5
            )
    plt.axhline(0, color='gray', linestyle='--', linewidth=0.8, zorder=0)
    
    # draw category labels
    category_centers = []
    category_labels = []
    last = 0
    for cat in category_order:
        responses = response_categories[cat]
        if len(responses) == 0:
            continue
        first_x = response_to_x[responses[0]]
        last_x = response_to_x[responses[-1]]
        center = (first_x + last_x) / 2
        category_centers.append(center)
        category_labels.append(cat)
        plt.axvspan(first_x-0.5, last_x+0.5, color='lightgrey', alpha=0.08, zorder=-1)

    plt.xticks(x_tick_positions, x_tick_labels, rotation=90, ha='center')
    apply_greek_labels_heatmap(plt.gca())
    plt.ylabel("log2FC")
    plt.ylim(-8,8)

    # Add category labels above the x-axis
    for i, (center, label) in enumerate(zip(category_centers, category_labels)):
        plt.text(center, plt.ylim()[1] + 1 + 1.5*(i%2), label,
                ha='center', va='bottom', fontsize=6, fontweight='bold', color='black')

    # Legend (one entry per cell type)
    handles = [plt.Line2D([0], [0], marker='o', color='w', label=cell_type,
                        markerfacecolor=CELL_TYPE_COLORS[cell_type], markersize=5, markeredgecolor='black')
            for cell_type in cell_types]
    
    plt.legend(handles=handles, bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0.)
    plt.savefig(os.path.join(plot_folder, f"5.3_scatter_IL-32-beta_responses_by_category_{plot_index+1}.svg"))
    plt.show()

In [ ]:
# get donor-specific values for the response to IL-32-beta in CD14 Monocytes
donor_set_1 = DONOR_GROUPS['interferon']
donor_set_2 = DONOR_GROUPS['non-interferon']
donor_sets = donor_set_1 + donor_set_2

for plot_index, response_categories in enumerate([chemokine_panels, other_cytokines]):
    # Calculate means and standard deviations for each gene group
    means_set_1 = []
    stds_set_1 = []
    means_set_2 = []
    stds_set_2 = []
    
    for gene_group in response_categories.values():
        # Get values for all genes in the group
        group_vals = []
        all_vals = raw_log2fc_values_by_donor["CD14 Mono", 'IL-32-beta']
        for gene in gene_group:
            gene = cytokine_to_gene.get(gene, gene)
            vals = all_vals.loc[gene]
            group_vals.append(vals)
        
        # Calculate mean and std for each donor set across all genes in group
        group_vals_df = pd.DataFrame(group_vals, index=gene_group, columns=DONOR_NAMES)
        
        # Average over donors for each gene individually
        donor_set_1_means = group_vals_df[donor_set_1].mean(axis=1, skipna=True)
        donor_set_2_means = group_vals_df[donor_set_2].mean(axis=1, skipna=True)
        donor_set_1_stds = group_vals_df[donor_set_1].std(axis=1, skipna=True)
        donor_set_2_stds = group_vals_df[donor_set_2].std(axis=1, skipna=True)
        
        # Store individual gene values instead of group averages
        means_set_1.extend(donor_set_1_means.values)
        stds_set_1.extend(donor_set_1_stds.values)
        means_set_2.extend(donor_set_2_means.values)
        stds_set_2.extend(donor_set_2_stds.values)

    # Create labels for individual genes
    all_gene_names = []
    for gene_group in response_categories.values():
        all_gene_names.extend(gene_group)

    # Create bar plot
    x = []
    last = 0
    for group in response_categories.values():
        for _ in group:
            x.append(last)
            last += 1
        last += 1  # add gap between groups
    x = np.array(x)
    width = 0.35

    fig, ax = plt.subplots(figsize=cm_to_inches((10, 5)))

    bars1 = ax.bar(x - width/2, means_set_1, width, yerr=stds_set_1, 
                label='interferon group', capsize=0, alpha=1, color='royalblue')
    bars2 = ax.bar(x + width/2, means_set_2, width, yerr=stds_set_2, 
                label='non-interferon group', capsize=0, alpha=1, color='forestgreen')

    ax.set_ylabel('log2FC')
    ax.set_xticks(x)
    ax.set_xticklabels(all_gene_names, rotation=90)
    apply_greek_labels_heatmap(ax)
    ax.legend()

    # add group labels at the top
    y_max = max(max(means_set_1), max(means_set_2)) + max(max(stds_set_1), max(stds_set_2))
    y_label_height = y_max * 1.1

    # calculate group positions
    group_positions = []
    bars_so_far = 0
    for gene_group in response_categories.values():
        group_size = len(gene_group)
        group_center = (x[bars_so_far] + x[bars_so_far+group_size-1]) / 2
        group_positions.append((group_center, x[bars_so_far], x[bars_so_far + group_size - 1]))
        bars_so_far += group_size

    # add group labels
    names = list(response_categories.keys())
    for i, (center_pos, start_pos, end_pos) in enumerate(group_positions):
        ax.text(center_pos, y_label_height, names[i], 
                ha='center', va='bottom', fontweight='bold', fontsize=7)
        # Add horizontal line to show group span
        ax.plot([start_pos - 0.4, end_pos + 0.4], [y_label_height * 0.95, y_label_height * 0.95], 
                'k-', linewidth=1)
        plt.axvspan(start_pos-0.5, end_pos+0.5, color='lightgrey', alpha=0.08, zorder=-1)
        
    ax.set_ylim(bottom=ax.get_ylim()[0], top=y_label_height * 1.2)
    ax.axhline(y=0, color='black', linewidth=1, linestyle='-')

    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, f"5.3_cytokine_il32_by_donor_group_{plot_index+1}.svg"))
    plt.show()

### Compare IL-32 with LPS values

In [ ]:
ref_dataset = "Lawlor_2021"  # Options: "Lawlor_2021", "Babbe_2023"

rename = {
    "Monocyte": "CD14 Mono",
    "natural killer cell": "NK CD56low"
}

ref_log2fcs_lawlor = pd.read_parquet("../../data/lps_datasets/lawlor_log2fc_values_mean.parquet")
ref_log2fcs_std_lawlor = pd.read_parquet("../../data/lps_datasets/lawlor_log2fc_values_std.parquet")

ref_log2fcs_babbe = pd.read_parquet("../../data/lps_datasets/babbe_log2fc_values_mean.parquet")
ref_log2fcs_std_babbe = pd.read_parquet("../../data/lps_datasets/babbe_log2fc_values_std.parquet")
ref_log2fcs_babbe = ref_log2fcs_babbe.xs("T24", level="time", axis=1)
ref_log2fcs_std_babbe = ref_log2fcs_std_babbe.xs("T24", level="time", axis=1)

ref_log2fcs_lawlor = ref_log2fcs_lawlor.rename(rename, axis=1)
ref_log2fcs_std_lawlor = ref_log2fcs_std_lawlor.rename(rename, axis=1)
ref_log2fcs_babbe = ref_log2fcs_babbe.rename(rename, axis=1)
ref_log2fcs_std_babbe = ref_log2fcs_std_babbe.rename(rename, axis=1)

if ref_dataset == "Lawlor_2021":
    ref_log2fcs = ref_log2fcs_lawlor
    ref_log2fcs_std = ref_log2fcs_std_lawlor
elif ref_dataset == "Babbe_2023":
    ref_log2fcs = ref_log2fcs_babbe
    ref_log2fcs_std = ref_log2fcs_std_babbe

shared_cell_types = set(raw_log2fc_values.columns.get_level_values(0).unique()).intersection(
    set(ref_log2fcs.columns.get_level_values("cell_type").unique()))
print(f"Shared cell types between human cytokine dict and {ref_dataset}: {shared_cell_types}")

In [ ]:
stims = ref_log2fcs.columns.get_level_values("condition").unique()

corr_df = []
for stim in stims:
    for cell_type in shared_cell_types:
        if not stim in ref_log2fcs[cell_type].columns:
            print(f"Stim {stim} not found in reference dataset for cell type {cell_type}. Skipping.")
            continue
        for cytokine in raw_log2fc_values[cell_type].columns:
            hcd_subset = raw_log2fc_values[cell_type][cytokine].dropna()
            study_vals = ref_log2fcs[cell_type][stim].dropna()
            hcd_subset, study_vals = get_common_index(hcd_subset, study_vals)
            assert len(hcd_subset) == len(study_vals), "Mismatch in lengths after aligning indices"

            r = np.corrcoef(hcd_subset, study_vals)[0, 1]
            if cytokine == 'IL-32-beta_review':
                cytokine = 'IL-32-beta\n(2nd screen)'
            elif cytokine not in CYTOKINE_ORDER:
                continue
            else:
                pass

            corr_df.append({
                "cell_type": cell_type,
                "stim": stim,
                "cytokine": cytokine,
                "correlation": r,
            })
corr_df = pd.DataFrame(corr_df)

In [ ]:
stim = 'LPS'
cell_types = ['CD14 Mono', 'CD4 T cell', 'CD8 T cell', 'NK CD56low', 'B cell']

fig, axes = plt.subplots(1, len(cell_types), figsize=cm_to_inches((18, 4)), sharey=True)
for cell_i, cell_type in enumerate(cell_types):
    ax = axes[cell_i]
    sub = corr_df[(corr_df['cell_type'] == cell_type) & (corr_df['stim'] == stim)]
    sub_sorted = sub.sort_values(by='correlation', ascending=False).head(10)
    
    ax.bar(x=sub_sorted['cytokine'], height=sub_sorted['correlation'], color='skyblue')
    ax.set_title(f"{cell_type} - {stim}")
    if cell_i == 0:
        ax.set_ylabel("Pearson r")
    else:
        ax.set_ylabel('')
    ax.tick_params(axis="x", rotation=90)
    apply_greek_labels(ax)
plt.tight_layout()
plt.show()

In [ ]:
rename = {
    "LPS_babbe": "LPS, Babbe (2023)",
    "LPS_lawlor": "LPS, Lawlor (2021)",
    "IL-32-beta": introduce_greek_letters("IL-32-beta (main screen)"),
    "IL-32-beta_review": introduce_greek_letters("IL-32-beta (2nd screen)")
}

cell_type = "CD14 Mono"
cytokine = "IL-32-beta"
cytokine2 = "IL-32-beta_review"
stim = "LPS"

hcd_subset = raw_log2fc_values[cell_type][cytokine].dropna()
if cytokine2 is not None:
    hcd_subset2 = raw_log2fc_values[cell_type][cytokine2].dropna()
study_vals_lawlor = ref_log2fcs_lawlor[cell_type]["LPS"].dropna()
study_vals_babbe = ref_log2fcs_babbe[cell_type]["LPS-5"].dropna()

universal_index = hcd_subset.index.intersection(study_vals_lawlor.index).intersection(study_vals_babbe.index)
if cytokine2 is not None:
    universal_index = universal_index.intersection(hcd_subset2.index)

hcd_subset = hcd_subset.loc[universal_index]
if cytokine2 is not None:
    hcd_subset2 = hcd_subset2.loc[universal_index]
study_vals_lawlor = study_vals_lawlor.loc[universal_index]
study_vals_babbe = study_vals_babbe.loc[universal_index]

hcd_subset_std = raw_log2fc_values_std[cell_type][cytokine].loc[hcd_subset.index]
if cytokine2 is not None:
    hcd_subset2_std = raw_log2fc_values_std[cell_type][cytokine2].loc[hcd_subset.index]
study_vals_std_lawlor = ref_log2fcs_std_lawlor[cell_type]["LPS"].loc[study_vals_lawlor.index]
study_vals_std_babbe = ref_log2fcs_std_babbe[cell_type]["LPS-5"].loc[study_vals_babbe.index]

integrated = pd.DataFrame({
    f"{cytokine}": hcd_subset,
    f"{cytokine2}": hcd_subset2,
    f"{stim}_lawlor": study_vals_lawlor,
    f"{stim}_babbe": study_vals_babbe,
    f"{cytokine}_std": hcd_subset_std,
    f"{cytokine2}_std": hcd_subset2_std,
    f"{stim}_lawlor_std": study_vals_std_lawlor,
    f"{stim}_babbe_std": study_vals_std_babbe
})

type1_response = ['CXCL9', 'CXCL10', 'CXCL11', 'IL12A', 'IL12B']
neutrophil_response = ['IL1A', 'IL1B', 'IL6', 'CXCL1', 'CXCL2', 'CXCL3', 'CXCL5', 'CXCL8']
anti_inflammatory_response = ['IL10', 'IL19', 'IL24']
isgs = ['IFIT1', 'IFIT2', 'IFIT3']
groups = {
    "Type 1 Response": type1_response,
    "Neutrophil Response": neutrophil_response,
    "Anti-inflammatory": anti_inflammatory_response,
    "ISGs": isgs
}

if cytokine2 is not None:
    conditions = [cytokine, cytokine2, f"{stim}_lawlor", f"{stim}_babbe"]
else:
    conditions = [cytokine, f"{stim}_lawlor", f"{stim}_babbe"]
plot_df = []
for group_name, genes in groups.items():
    for gene in genes:
        if gene in integrated.index:
            for condition in conditions:
                val = integrated.loc[gene, condition]
                std = integrated.loc[gene, f"{condition}_std"]

                if pd.notna(val):
                    plot_df.append({"group": group_name, "gene": gene, "condition": condition, "log2fc": val, "std": std, "condition_name": rename.get(condition, condition)})
plot_df = pd.DataFrame(plot_df)

plt.figure(figsize=cm_to_inches((18, 5)))

gene_order = [gene for genes in groups.values() for gene in genes if gene in integrated.index]
ax = sns.barplot(data=plot_df, x="gene", y="log2fc", hue="condition_name", order=gene_order, errorbar=None)
ax.axhline(0, color="black", linewidth=0.8)

# add std error bars
hue_order = conditions
gene_to_x = {g: i for i, g in enumerate(gene_order)}
n_hue = len(hue_order)
bar_width = 0.8 / n_hue

for _, row in plot_df.iterrows():
    x_base = gene_to_x[row["gene"]]
    hue_i = hue_order.index(row["condition"])
    x = x_base - 0.4 + bar_width / 2 + hue_i * bar_width
    ax.errorbar(x=x, y=row["log2fc"], yerr=row["std"], fmt="none", color="black", capsize=2, linewidth=0.8,)

# group labels / stretches
ymin, ymax = ax.get_ylim()
yrange = ymax - ymin
label_y = ymin - 0.12 * yrange
line_y = ymin - 0.1 * yrange

start = 0
for group_name, genes in groups.items():
    present_genes = [g for g in genes if g in gene_order]
    if len(present_genes) == 0:
        continue

    end = start + len(present_genes) - 1
    center = (start + end) / 2

    ax.plot([start - 0.4, end + 0.4], [line_y, line_y], color="black", clip_on=False, linewidth=1)
    ax.text(center, label_y, group_name, ha="center", va="top", fontsize=6, clip_on=False)

    start = end + 1

ax.set_ylim(ymin - 0.22 * yrange, ymax)
ax.set_xlabel("")
ax.set_ylabel("log2FC")
overall_correlation_lawlor = np.corrcoef(hcd_subset, study_vals_lawlor)[0, 1]
overall_correlation_babbe = np.corrcoef(hcd_subset, study_vals_babbe)[0, 1]
ax.set_title(f"{cell_type}: {cytokine} vs {stim} (r = {overall_correlation_lawlor:.2f} Lawlor, {overall_correlation_babbe:.2f} Babbe)")
ax.tick_params(axis="x", rotation=90)
ax.legend(title="")

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"response_type_{cell_type}_{cytokine}_vs_{stim}.svg"), dpi=300)
plt.show()

## 5.4 - Family - family communication

In [ ]:
plot_folder = os.path.join(base_plot_folder, "5.4_family_counts")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
cytokine_response_df_all["family"] = cytokine_response_df_all["response"].map(cytokine_name_to_functional)
cytokine_response_df_all["response_plus_cell_type"] = cytokine_response_df_all["response"] + "_" + cytokine_response_df_all["cell_type"]

In [ ]:
family_value_counts_positive = {}
for treatment in cytokine_response_df_all["treatment"].unique():
    response_df_filter = cytokine_response_df_all[cytokine_response_df_all["treatment"] == treatment]
    response_df_filter = response_df_filter[response_df_filter["fold_changes"].apply(lambda x: all(fc > 1 for fc in x))]
    family_value_counts_positive[treatment] = response_df_filter["family"].value_counts()
    
family_value_counts_positive = pd.DataFrame(family_value_counts_positive)
family_value_counts_positive.fillna(0, inplace=True)
family_value_counts_positive = family_value_counts_positive.astype(int)

family_value_counts_negative = {}
for treatment in cytokine_response_df_all["treatment"].unique():
    response_df_filter = cytokine_response_df_all[cytokine_response_df_all["treatment"] == treatment]
    response_df_filter = response_df_filter[response_df_filter["fold_changes"].apply(lambda x: all(fc < -1 for fc in x))]
    family_value_counts_negative[treatment] = response_df_filter["family"].value_counts()
    
family_value_counts_negative = pd.DataFrame(family_value_counts_negative)
family_value_counts_negative.fillna(0, inplace=True)
family_value_counts_negative = family_value_counts_negative.astype(int)

In [ ]:
# inspect the results manually
subset = cytokine_response_df_all[cytokine_response_df_all["family"] == "regulatory"]
subset = subset[subset["direction"] == "upregulated"]
subset["treatment"].value_counts()

In [ ]:
# add 0 counts for all missing families
family_value_counts_positive = family_value_counts_positive.reindex(functional_column_order, fill_value=0)
family_value_counts_negative = family_value_counts_negative.reindex(functional_column_order, fill_value=0)

In [ ]:
# total counts by cytokine treatment
total_significant_interactions_cytokine = family_value_counts_positive + family_value_counts_negative
total_significant_interactions_cytokine = total_significant_interactions_cytokine.sum(axis=0)

plt.figure(figsize=cm_to_inches((4, 3)))
plt.hist(total_significant_interactions_cytokine, bins=np.arange(0, 250, 10))
plt.xlabel("significant changes")
plt.ylabel("stimulation conditions")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.4_total_significant_interactions_cytokine.svg"), bbox_inches="tight")
plt.show()

In [ ]:
# total counts by cytokine response family
total_significant_interactions_family = family_value_counts_positive + family_value_counts_negative
total_significant_interactions_family = total_significant_interactions_family.sum(axis=1)

plt.figure(figsize=cm_to_inches((4, 3)))
plt.hist(total_significant_interactions_family, bins=np.arange(0, 260, 10))
plt.xlabel("significant changes")
plt.ylabel("family response")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.4_total_significant_interactions_family.svg"), bbox_inches="tight")
plt.show()

In [ ]:
# filter to those with more than 3 interactions for plotting purposes
filter_cytokine_interactions = total_significant_interactions_cytokine[total_significant_interactions_cytokine > 3]
filter_family_interactions = total_significant_interactions_family[total_significant_interactions_family > 3]
family_value_counts_positive = family_value_counts_positive.loc[filter_family_interactions.index, filter_cytokine_interactions.index]
family_value_counts_negative = family_value_counts_negative.loc[filter_family_interactions.index, filter_cytokine_interactions.index]

# order columns by all_known_cytokine_and_chemokine_names
family_value_counts_positive = family_value_counts_positive.reindex(columns=[col for col in CYTOKINE_ORDER if col in family_value_counts_positive.columns])
family_value_counts_negative = family_value_counts_negative.reindex(columns=[col for col in CYTOKINE_ORDER if col in family_value_counts_negative.columns])
family_value_counts_both = family_value_counts_positive - family_value_counts_negative
family_value_counts_abs = family_value_counts_positive + family_value_counts_negative

family_value_counts_positive_rel = family_value_counts_positive.divide(family_value_counts_positive.max(axis=1), axis=0)
family_value_counts_negative_rel = family_value_counts_negative.divide(family_value_counts_negative.max(axis=1), axis=0)
family_value_counts_both_rel = family_value_counts_both.divide(abs(family_value_counts_both).max(axis=1), axis=0)

In [ ]:
# This is done to plot overview labels on the y-axis
counts_per_block = {}
for family in family_value_counts_positive_rel.index:
    block = functional_to_block.get(family)
    if block is not None:
        counts_per_block[block] = counts_per_block.get(block, 0) + 1
        
# invert the order of the dictionary
counts_per_block = dict(reversed(counts_per_block.items())) 

In [ ]:
counts_per_block = {key.replace("\n", " "):item for key, item in counts_per_block.items()}
functional_block_colors = {key.replace("\n", " "):item for key, item in functional_block_colors.items()}

In [ ]:
fig = plt.figure(figsize=cm_to_inches((14, 8)))

ax = sns.heatmap(family_value_counts_positive_rel, vmin=0, vmax=1, cmap="Reds",
            cbar_kws={"label": "relative number of upregulated cytokines", "shrink": 0.6}, square=True)

plot_overview_labels_along_y_axis(
    fig, ax, counts_per_block.keys(), counts_per_block,
    -8, len(family_value_counts_positive_rel.index), stagger=2, color=functional_block_colors)

plt.xlabel("cytokine treatment")
plt.ylabel("")
plt.gca().tick_params(axis='both', length=0)
apply_greek_labels_heatmap(plt.gca())

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.4_family_value_counts_positive.svg"), bbox_inches="tight")
plt.show()

In [ ]:
fig = plt.figure(figsize=cm_to_inches((14, 8)))

ax = sns.heatmap(family_value_counts_negative_rel, vmin=0, vmax=1, cmap="Blues",
            cbar_kws={"label": "relative number of downregulated cytokines", "shrink": 0.6}, square=True)
plot_overview_labels_along_y_axis(
    fig, ax, counts_per_block.keys(), counts_per_block,
    -8, len(family_value_counts_positive_rel.index), stagger=2, color=functional_block_colors)
plt.ylabel("")
apply_greek_labels_heatmap(plt.gca())
plt.xlabel("cytokine treatment")
plt.gca().tick_params(axis='both', length=0)

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.4_family_value_counts_negative.svg"), bbox_inches="tight")
plt.show()

In [ ]:
fig = plt.figure(figsize=cm_to_inches((11, 7)))

df = family_value_counts_both_rel.copy()
df.index = df.index.map(functional_column_order)
ax = sns.heatmap(df, vmin=df.min().min(), vmax=df.max().max(), center=0, cmap="coolwarm",
            cbar_kws={"label": "relative number of regulated cytokines", "shrink": 0.6}, square=True)
plot_overview_labels_along_y_axis(
    fig, ax, counts_per_block.keys(), counts_per_block,
    -9, len(family_value_counts_positive_rel.index), stagger=1.7, color=functional_block_colors)

plt.xticks(fontsize=6)
plt.yticks(fontsize=6)
plt.ylabel("")
apply_greek_labels_heatmap(plt.gca())

# colour family labels by functional block
family_label_to_block = {functional_column_order[f]: functional_to_block[f].replace("\n", " ") for f in functional_column_order}
for label in ax.get_yticklabels():
    label.set_color(functional_block_colors[family_label_to_block[label.get_text()]])

plt.xlabel("cytokine treatment")
plt.gca().tick_params(axis='both', length=0)

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.4_family_value_counts_both.svg"), bbox_inches="tight")
plt.show()

# 6 - Cytokine similarity and grouping

In [ ]:
plot_folder = os.path.join(base_plot_folder, "6.1_within_cell_similarity")
os.makedirs(plot_folder, exist_ok=True)

from scipy.spatial.distance import pdist, squareform
use_spatial_gene_set = False
spatial_gene_set = {}

In [ ]:
# can verify that spatial gene sets do not change the results significantly despite having a much smaller number of genes
# (not relevant for the main publication; internal validation)
# spatial_gene_set = pd.read_csv("../annotations/spatial_genes.csv")["gene_name"].tolist()
# use_spatial_gene_set = False

# cytokine_genes = cytokine_info_parse["gene"]
# cytokine_genes_in_spatial = []
# for gene in cytokine_genes:
#     gene = gene.split(", ")
#     present = all([g in spatial_gene_set for g in gene])
#     if present:
#         cytokine_genes_in_spatial.append(', '.join(gene) if len(gene) > 1 else gene[0])
# print(len(cytokine_genes_in_spatial))

In [ ]:
# FOR DEBUGGING/TESTING THE SIMILARITY CODE
cell_type = "CD14 Mono"
all_inclusive_filter = expressed_genes_by_cell_type_dict[cell_type]
existing_fold_change_vals = edger_df_log2FC[cell_type].dropna(how='all').index
all_inclusive_filter = [gene for gene in all_inclusive_filter if gene in existing_fold_change_vals]
if use_spatial_gene_set:
    all_inclusive_filter = [gene for gene in all_inclusive_filter if gene in spatial_gene_set]
cell_type_log2 = edger_df_log2FC.loc[all_inclusive_filter, cell_type]

cytokine1 = "GM-CSF"
cytokine2 = "IL-3"

cell_data_1 = cell_type_log2[cytokine1].dropna()
cell_data_2 = cell_type_log2[cytokine2].dropna()

common_index = cell_data_1.index.intersection(cell_data_2.index)
cell_data_1 = cell_data_1[common_index]
cell_data_2 = cell_data_2[common_index]

r = np.corrcoef(cell_data_1, cell_data_2)[0, 1]
r

In [ ]:
cytokine_combs = list(combinations_with_replacement(CYTOKINE_ORDER, 2))

if os.path.exists(os.path.join(intermediate_output_folder, f"6.1_r_vals_by_cell_type{'_spatial' if use_spatial_gene_set else ''}.pkl")):
    with open(os.path.join(intermediate_output_folder, f"6.1_r_vals_by_cell_type{'_spatial' if use_spatial_gene_set else ''}.pkl"), "rb") as f:
        r_vals_by_cell_type = pickle.load(f)
else:
    r_vals_by_cell_type = {}
    for cell_type in ALL_CELL_TYPES:
        r_vals_by_cytokine = pd.DataFrame(index=[cytokine for cytokine in CYTOKINE_ORDER],
                                             columns=[cytokine for cytokine in CYTOKINE_ORDER])
        print(cell_type)
        i = 0
        
        # first, use the pre-calculated filter based on expression values
        all_inclusive_filter = expressed_genes_by_cell_type_dict[cell_type]
        
        # then, subset to those where we actually have fold change values for this cell type
        existing_fold_change_vals = edger_df_log2FC[cell_type].dropna(how='all').index
        all_inclusive_filter = [gene for gene in all_inclusive_filter if gene in existing_fold_change_vals]

        if use_spatial_gene_set:
            all_inclusive_filter = [gene for gene in all_inclusive_filter if gene in spatial_gene_set]
        
        print(len(all_inclusive_filter))
        cell_type_log2 = edger_df_log2FC.loc[all_inclusive_filter, cell_type]
        for cytokine1, cytokine2 in cytokine_combs:
            # get values for both cytokines and drop na
            cell_data_1 = cell_type_log2[cytokine1].dropna()
            cell_data_2 = cell_type_log2[cytokine2].dropna()

            if len(cell_data_1) == 0 or len(cell_data_2) == 0:
                r_vals_by_cytokine.loc[cytokine1, cytokine2] = np.nan
                continue
            
            # get the intersection of the two cytokines
            common_index = cell_data_1.index.intersection(cell_data_2.index)
            cell_data_1 = cell_data_1[common_index]
            cell_data_2 = cell_data_2[common_index]
            
            r = np.corrcoef(cell_data_1, cell_data_2)[0, 1]
            r_vals_by_cytokine.loc[cytokine1, cytokine2] = r
            
        # fill in the other half of the correlation matrix
        r_vals_by_cytokine = r_vals_by_cytokine.fillna(r_vals_by_cytokine.T).astype(float)
        r_vals_by_cell_type[cell_type] = r_vals_by_cytokine.copy()

    os.makedirs(intermediate_output_folder, exist_ok=True)
    with open(os.path.join(intermediate_output_folder, f"6.1_r_vals_by_cell_type{'_spatial' if use_spatial_gene_set else ''}.pkl"), "wb") as f:
        pickle.dump(r_vals_by_cell_type, f)

In [ ]:
# Create multi-index DataFrame
index = pd.MultiIndex.from_product([CYTOKINE_ORDER, CYTOKINE_ORDER], names=['cytokine1', 'cytokine2'])
r_vals_by_cell_type_df = pd.DataFrame(index=index, columns=ALL_CELL_TYPES)
for cell_type in ALL_CELL_TYPES:
    r_matrix = r_vals_by_cell_type[cell_type]
    for cyt1 in CYTOKINE_ORDER:
        for cyt2 in CYTOKINE_ORDER:
            r_vals_by_cell_type_df.loc[(cyt1, cyt2), cell_type] = r_matrix.loc[cyt1, cyt2]

### Get correlations for spatial gene subset (Here, we can verify the spatial genes as being meaningful)

In [ ]:
# with open(os.path.join("output", f"6.1_r_vals_by_cell_type.pkl"), "rb") as f:
#     r_vals_by_cell_type_full = pickle.load(f)
# with open(os.path.join("output", f"6.1_r_vals_by_cell_type_spatial.pkl"), "rb") as f:
#     r_vals_by_cell_type_spatial = pickle.load(f)

# # Create figure with subplots
# fig, axes = plt.subplots(3, 5, figsize=cm_to_inches((16, 10)))
# axes = axes.flatten()

# for i, cell_type in enumerate(ALL_CELL_TYPES):
#     df_full = r_vals_by_cell_type_full[cell_type]
#     df_spatial = r_vals_by_cell_type_spatial[cell_type]

#     df_full = df_full.reset_index().melt(id_vars=["index"]).rename(columns={"index": "cytokine_1", "variable": "cytokine_2", "value": "r"})
#     df_spatial = df_spatial.reset_index().melt(id_vars=["index"]).rename(columns={"index": "cytokine_1", "variable": "cytokine_2", "value": "r"})

#     # drop diagonal
#     df_full = df_full[df_full["cytokine_1"] != df_full["cytokine_2"]]
#     df_spatial = df_spatial[df_spatial["cytokine_1"] != df_spatial["cytokine_2"]]

#     # make identifier
#     df_full["identifier"] = df_full["cytokine_1"] + "_" + df_full["cytokine_2"]
#     df_spatial["identifier"] = df_spatial["cytokine_1"] + "_" + df_spatial["cytokine_2"]

#     # set as index
#     df_full.set_index("identifier", inplace=True)
#     df_spatial.set_index("identifier", inplace=True)

#     # drop na
#     df_full = df_full.dropna()
#     df_spatial = df_spatial.dropna()

#     # harmonize index
#     df_spatial = df_spatial.loc[df_full.index]

#     r = np.corrcoef(df_full["r"], df_spatial["r"])[0, 1]
    
#     # Plot on appropriate subplot
#     axes[i].scatter(df_full["r"], df_spatial["r"], rasterized=True, color="skyblue", s=5, alpha=0.5)
#     axes[i].plot([-1,1], [-1,1], color="black")
#     axes[i].text(0.05, 0.95, f'r = {r:.2f}', transform=axes[i].transAxes, 
#                 verticalalignment='top')
#     axes[i].set_title(cell_type)
#     axes[i].set_xlabel("Pearson r\nall genes")
#     axes[i].set_ylabel("Pearson r\nspatial genes")
#     axes[i].label_outer()
#     axes[i].set_xlim(-1,1)
#     axes[i].set_ylim(-1,1)

# # Remove empty subplots
# for i in range(len(ALL_CELL_TYPES), len(axes)):
#     fig.delaxes(axes[i])

# plt.tight_layout()
# for format in ["png", "svg"]:
#     plt.savefig(os.path.join(plot_folder, f"6.1_full_vs_spatial_correlation.{format}"), dpi=400)

### Explore how correlation values depend on the impact classification

In [ ]:
# Apply logical AND across the index to find cell types where both IL-3 and GM-CSF have significant impact
both_strong = strong_impact_by_cell_type.loc["IL-3", ALL_CELL_TYPES] & strong_impact_by_cell_type.loc["GM-CSF", ALL_CELL_TYPES]
both_strong_columns = both_strong[both_strong].index
not_both_columns = both_strong[~both_strong].index

In [ ]:
print(r_vals_by_cell_type_df.loc[("IL-3", "GM-CSF"), both_strong_columns].mean())
print(r_vals_by_cell_type_df.loc[("IL-3", "GM-CSF"), both_strong_columns].std())

In [ ]:
print(r_vals_by_cell_type_df.loc[("IL-3", "GM-CSF"), not_both_columns].mean())
print(r_vals_by_cell_type_df.loc[("IL-3", "GM-CSF"), not_both_columns].std())

### create scatter plots

In [ ]:
def get_lower_triangle(df):
    """Get values from the lower triangle of a DataFrame (excluding the diagonal)."""
    return df.where(np.tril(np.ones(df.shape), k=-1).astype(bool))

def find_pairs_thresh(r_vals_by_cytokine, threshold):
    pairs = []
    for cytokine1, cytokine2 in cytokine_combs:
        if threshold > 0:
            if r_vals_by_cytokine.loc[cytokine1, cytokine2] > threshold:
                if cytokine1 != cytokine2:
                    pairs.append((cytokine1, cytokine2))
        if threshold < 0:
            if r_vals_by_cytokine.loc[cytokine1, cytokine2] < threshold:
                if cytokine1 != cytokine2:
                    pairs.append((cytokine1, cytokine2))
    return pairs

In [ ]:
all_pairs_df_high = []
for cell_type in ALL_CELL_TYPES:
    r_vals = r_vals_by_cell_type[cell_type]
    pairs = find_pairs_thresh(r_vals, threshold=0.8)
    # sort the pairs by name
    pairs = sorted(pairs, key=lambda x: (x[0], x[1]))
    for cytokine1, cytokine2 in pairs:
        new_row = {"cell_type": cell_type, "cytokine1": cytokine1, "cytokine2": cytokine2, "r": r_vals.loc[cytokine1, cytokine2]}
        new_row = pd.DataFrame([new_row])
        if len(new_row) > 0:
            all_pairs_df_high.append(new_row)
all_pairs_df_high = pd.concat(all_pairs_df_high, ignore_index=True)

all_pairs_df_low = []
for cell_type in ALL_CELL_TYPES:
    r_vals = r_vals_by_cell_type[cell_type]
    pairs = find_pairs_thresh(r_vals, threshold=-0.4)
    # sort the pairs by name
    pairs = sorted(pairs, key=lambda x: (x[0], x[1]))
    for cytokine1, cytokine2 in pairs:
        new_row = {"cell_type": cell_type, "cytokine1": cytokine1, "cytokine2": cytokine2, "r": r_vals.loc[cytokine1, cytokine2]}
        new_row = pd.DataFrame([new_row])
        if len(new_row) > 0:
            all_pairs_df_low.append(new_row)
all_pairs_df_low = pd.concat(all_pairs_df_low, ignore_index=True)
    
all_pairs_df_high["cytokine_pair"] = all_pairs_df_high["cytokine1"] + "_" + all_pairs_df_high["cytokine2"]
all_pairs_df_low["cytokine_pair"] = all_pairs_df_low["cytokine1"] + "_" + all_pairs_df_low["cytokine2"]       

In [ ]:
pairs_to_plot = [
    ("IL-15", "IL-2"),
    ("IL-4", "IL-13"),
    ("IL-15", "IL-10"),
    ("IL-3", "GM-CSF"),
    ("IL-15", "APRIL"),
    ("IL-15", "VEGF")
]

from scipy.stats import gaussian_kde
corr_plot_folder = os.path.join(plot_folder, "correlation_plots")
os.makedirs(corr_plot_folder, exist_ok=True)

density_plot_flag = False
        
# ------------------------- Large plot -------------------------
fig, axs = plt.subplots(nrows=4, ncols=6, figsize=cm_to_inches((14, 8)), sharex=True, sharey=True)
axs = axs.flatten()
counter = 0

for cell_type in ALL_CELL_TYPES:
    if not cell_type in ["B cell", "CD14 Mono", "cDC", "CD4 T cell"]:
        continue
    print(cell_type)
    
    r_vals = r_vals_by_cell_type[cell_type]
    #pairs = find_pairs_thresh(r_vals, threshold=-0.4)
    #print(cell_type, pairs)
    pairs = pairs_to_plot
    
    all_inclusive_filter = expressed_genes_by_cell_type_dict[cell_type]
    existing_fold_change_vals = edger_df_log2FC[cell_type].dropna(how='all').index
    all_inclusive_filter = [gene for gene in all_inclusive_filter if gene in existing_fold_change_vals]
    print(len(all_inclusive_filter))
    cell_type_log2 = edger_df_log2FC.loc[all_inclusive_filter, cell_type].dropna()
    
    if use_spatial_gene_set:
        all_inclusive_filter = [gene for gene in cell_type_log2.index if gene in spatial_gene_set]
        cell_type_log2_old = cell_type_log2.copy()
        cell_type_log2 = cell_type_log2.loc[all_inclusive_filter]
    
    for cytokine1, cytokine2 in pairs:
        x = cell_type_log2.loc[:, cytokine1]
        y = cell_type_log2.loc[:, cytokine2]
        r = np.corrcoef(x, y)[0, 1]
        if use_spatial_gene_set: 
            x_old = cell_type_log2_old.loc[:, cytokine1]
            y_old = cell_type_log2_old.loc[:, cytokine2]
            r_old = np.corrcoef(x_old, y_old)[0, 1]
        
        # ------------------------- Small plot -------------------------
        plt.figure(figsize=cm_to_inches((4, 4)))
        plt.scatter(x, y, s = 1, edgecolor="none", alpha=0.5, rasterized=True, zorder=2, label=f"r={r:.2f}")

        plt.ylim(-4, 4)
        plt.xticks([-4, 0, 4], fontsize=6)
        plt.yticks([-4, 0, 4], fontsize=6) 

        plt.xlabel(f"log$_{2}$FC", fontsize=6)
        plt.ylabel(f"log$_{2}$FC", fontsize=6)
        
        plt.title(f'{cytokine1} vs {cytokine2}', fontsize=6.5)
        plt.legend(loc="upper left", fontsize=6)
        
        plt.tight_layout()
        for format in ["png", "svg"]:
            plt.savefig(os.path.join(corr_plot_folder, f"6.1_small_corr_{cell_type}_{cytokine1}_vs_{cytokine2}{'_spatial' if use_spatial_gene_set else ''}.{format}"),
                            bbox_inches="tight", dpi=400)
        plt.close()
        
        # ---- subplots of large plot ----
        ax = axs[counter]
        
        # create a cross at the origin
        ax.plot([-5, 5], [0, 0], color="black", linewidth=1, zorder=0, alpha=0.5)
        ax.plot([0, 0], [-5, 5], color="black", linewidth=1, zorder=0, alpha=0.5)
        ax.scatter(x, y, s = 1, edgecolor="none", alpha=0.5, rasterized=True, zorder=2, color="red" if use_spatial_gene_set else "tab:blue")
        if use_spatial_gene_set:
            ax.scatter(x_old, y_old, s = 1, edgecolor="none", alpha=0.5, rasterized=True, zorder=1, color="gray")
        
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.set_xticks([-4, -2, 0, 2, 4])
        ax.set_yticks([-4, -2, 0, 2, 4]) 
        ax.set_xlabel(f"{introduce_greek_letters(cytokine1)}", fontsize=6)
        ax.set_ylabel(f"{introduce_greek_letters(cytokine2)}", fontsize=6)

        if use_spatial_gene_set:
            correlation_text = f"r$_\\mathrm{{spatial}}$={r:.2f}\nr$_\\mathrm{{full}}$={r_old:.2f}"
        else:
            correlation_text = f"r={r:.2f}"
        ax.text(0.05 if "IL-10" not in [cytokine1, cytokine2] else 0.4, 0.95, correlation_text, transform=ax.transAxes, 
                fontsize=5, verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'))
        ax.tick_params(axis='both', which='both', length=0)
        
        counter += 1
        if not ax.get_subplotspec().is_first_col():
            plt.setp(ax.get_yticklabels(), visible=False)
        
        if not ax.get_subplotspec().is_last_row():
            plt.setp(ax.get_xticklabels(), visible=False)

plt.tight_layout(pad=0.5, h_pad=0.5, w_pad=0.5)
for fmt in ["png", "svg"]:
    plt.savefig(os.path.join(corr_plot_folder, f"6.1_large_corr_panel{'_spatial' if use_spatial_gene_set else ''}.{fmt}"),
                    bbox_inches="tight", dpi=400)
plt.show()
plt.close()

In [ ]:
if use_spatial_gene_set:
    raise Exception("Deactivate spatial gene set to continue!")

In [ ]:
if write_SI_tables_flag:
    r_vals_by_cell_type_df.to_excel(os.path.join(output_tables_folder, "6.1_r_vals_by_cell_type_df.xlsx"))
    df = r_vals_by_cell_type_df.copy()
    df = df.reset_index()
    df['cytokine1'] = df['cytokine1'].apply(introduce_greek_letters)
    df['cytokine2'] = df['cytokine2'].apply(introduce_greek_letters)
    write_to_SI_table(df, "cytokine_similarity_pearson_r")

In [ ]:
# Create a subplot grid based on number of cell types
n_cell_types = len(r_vals_by_cell_type.keys())
n_rows = 3
n_cols = 5

fig, axes = plt.subplots(n_rows, n_cols, figsize=cm_to_inches((16, 8)))
axes = axes.flatten()

# Plot histogram for each cell type
for idx, (cell_type, ax) in enumerate(zip(r_vals_by_cell_type.keys(), axes)):
    df_subset = r_vals_by_cell_type[cell_type].copy()

    for i in range(min(df_subset.shape)):
        df_subset.iat[i, i] = np.nan
        
    flattened = pd.Series(df_subset.values.flatten()).dropna()
    ax.hist(flattened, bins=np.arange(-1, 1.01, 0.1), density=True, color="skyblue")
    mean_val = flattened.mean()
    
    ax.axvline(mean_val, color='black', linestyle='dashed', linewidth=1)
    ax.text(mean_val+0.3, 3, f'{mean_val:.2f}', color='black', ha='center')
    ax.set_title(cell_type)
    ax.set_ylim(0, 3.5)
    ax.set_xlim(-1, 1)
    
    ax.set_ylabel("density")
    ax.set_xlabel("Pearson r")
    ax.label_outer()

for idx in range(len(r_vals_by_cell_type.keys()), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
for fmt in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"6.1_corr_histograms.{fmt}"), dpi=300, bbox_inches="tight")
plt.show()

### Dendrogram clustering

#### This isn't actually being used in the paper, plotting functions are used below

In [ ]:
def plot_heatmap_with_clustering(distance_matrix, cell_type, plot_name, heatmap_label, communities=None):
    """This plots a heatmap for the Supplementary Information."""
    if communities is not None:
        communities_flat = [item for sublist in communities for item in sublist]
        distance_matrix = distance_matrix.loc[communities_flat, communities_flat]

    plt.figure(figsize=(len(distance_matrix)*0.1+1, len(distance_matrix)*0.1+1))
    ax = sns.heatmap(distance_matrix, cmap="coolwarm", center=0, annot=False, fmt=".2f", vmin=-1, vmax=1, cbar=True, square=True,
                cbar_kws={'label': heatmap_label, 'shrink': 0.6})
    
    # plot black boxes around the communities
    if communities is not None:
        current_pos = 0
        for community in communities:
            num_entries = len(community)
            rect = plt.Rectangle((current_pos, current_pos), num_entries, num_entries, fill=False, color='black', linewidth=1.5, clip_on=False)
            plt.gca().add_patch(rect)
            current_pos += num_entries
    
    # set font sizes to 5.5
    ax.set_xticks(np.arange(0.5, len(distance_matrix)+0.5))
    ax.set_yticks(np.arange(0.5, len(distance_matrix)+0.5))
    ax.set_xticklabels(distance_matrix.index, fontsize=5.5, rotation=90)  # Rotate for better visibility
    ax.set_yticklabels(distance_matrix.index, fontsize=5.5)
    # set the tick length to 0
    ax.tick_params(length=0)
    
    apply_greek_labels_heatmap(ax)

    plt.title(f"{cell_type}", fontsize=7)
    plt.tight_layout()

    for format in ["png", "svg"]:
        plt.savefig(f"{plot_folder}/6_{plot_name}_{cell_type}.{format}", bbox_inches="tight")
    plt.show()
        
        
def plot_heatmap_with_clustering_small(distance_matrix, cell_type, plot_name, heatmap_label, communities=None):
    """This plots a heatmap for the Main Text."""
    if communities is not None:
        communities_flat = [item for sublist in communities for item in sublist]
        distance_matrix = distance_matrix.loc[communities_flat, communities_flat]

    plt.figure(figsize=cm_to_inches((8.5,8.5)))
    ax = sns.heatmap(
        distance_matrix, cmap="coolwarm", center=0, annot=False, fmt=".2f",
        vmin=-1, vmax=1, cbar=True, square=True,
        cbar_kws={'label': heatmap_label, 'shrink': 0.45}
    )
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=5.5)  # Reduce the fontsize of the colorbar labels
    
    # plot black boxes around the communities
    if communities is not None:
        current_pos = 0
        for community in communities:
            num_entries = len(community)
            rect = plt.Rectangle((current_pos, current_pos), num_entries, num_entries,
                                 fill=False, color='black', linewidth=1.5, clip_on=False)
            plt.gca().add_patch(rect)
            current_pos += num_entries
    
    # Instead of the default y-axis labels, we will stagger them in 3 positions.
    # First, remove the default labels:
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    # Define 3 staggered horizontal offsets (in axis coordinates: 0=left, 1=right).
    # Adjust these values as needed for your plot.
    stagger_offsets = [-0.01, -0.15, -0.19]
    
    # Manually add the y-axis labels at staggered positions.
    for i, label in enumerate(distance_matrix.index):
        offset = stagger_offsets[i % 2]
        ax.text(
            offset, i + 0.2, introduce_greek_letters(label),
            ha='right', va='center', fontsize=5.5,
            transform=ax.get_yaxis_transform()  # x in axis fraction, y in data coordinates
        )
    
    # remove tick marks
    ax.tick_params(length=0)
    
    apply_greek_labels_heatmap(ax)
    
    plt.title(f"{cell_type}", fontsize=7)
    plt.tight_layout()

    for format in ["png", "svg"]:
        plt.savefig(f"{plot_folder}/6_small_{plot_name}_{cell_type}.{format}", bbox_inches="tight", dpi=400)
    plt.show()

In [ ]:
%%capture output
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform

def dendrogram_clustering(distance_matrix):
    columns = distance_matrix.columns
    distance_matrix_original = distance_matrix.copy()
    
    # distance matrix
    distance_matrix = np.nan_to_num(distance_matrix)

    # condense the distance matrix
    condensed_distance = squareform(distance_matrix)

    # hierarchical clustering 
    linkage = sch.linkage(condensed_distance, method='ward')

    # get the dendrogram and ordered indices
    dendro = sch.dendrogram(linkage, labels=columns, no_plot=True)
    ordered_indices = dendro['leaves']

    # compute cophenetic distances between leaves
    _, cophenetic_distances = sch.cophenet(linkage, condensed_distance)
    # convert to a square matrix
    cophenetic_matrix = squareform(cophenetic_distances)
    cophenetic_df = pd.DataFrame(cophenetic_matrix, index=columns, columns=columns)
    cophenetic_df = cophenetic_df.iloc[ordered_indices, ordered_indices]
    cophenetic_df = cophenetic_df/cophenetic_df.max().max()
    
    return distance_matrix_original.iloc[ordered_indices, ordered_indices], cophenetic_df, linkage

# for cell_type in r_vals_by_cell_type.keys():
#     cytokine_filter = strong_impact_by_cell_type[cell_type][strong_impact_by_cell_type[cell_type]].index
#     distance_matrix = 1 - r_vals_by_cell_type[cell_type].loc[cytokine_filter, cytokine_filter].astype(float)
#     ordered_distance_vals, _, _ = dendrogram_clustering(distance_matrix)
#     plot_heatmap_with_clustering(1-ordered_distance_vals, cell_type, plot_name="cosine_dendrogram", heatmap_label="cosine similarity")

## Group cytokines by highly similar correlation values

In [ ]:
# get the correlation values in long format
all_r_vals_long = r_vals_by_cell_type_df.stack().reset_index().rename(columns={"level_2": "cell_type", 0: "r_val"})

# check if both cytokines have a strong impact in a given cell type
all_r_vals_long["strong_impact"] = False
for index, row in all_r_vals_long.iterrows():
    cytokine_1 = row["cytokine1"]
    cytokine_2 = row["cytokine2"]
    cell_type = row["cell_type"]
    strong_impact_1 = strong_impact_by_cell_type.loc[cytokine_1, cell_type]
    strong_impact_2 = strong_impact_by_cell_type.loc[cytokine_2, cell_type]
    if strong_impact_1 and strong_impact_2:
        all_r_vals_long.loc[index, "strong_impact"] = True
        
# subset to strong impact and exclude unwanted cell types
all_r_vals_long = all_r_vals_long[all_r_vals_long["strong_impact"]]

# Create a unique identifier and drop duplicates (Pearson r is symmetric)
all_r_vals_long["unique_identifier"] = all_r_vals_long.apply(
    lambda x: "_".join(sorted([str(x.get("cytokine1", "")), str(x.get("cytokine2", "")), str(x.get("cell_type", ""))])), 
    axis=1
)
all_r_vals_long = all_r_vals_long.drop_duplicates(subset=["unique_identifier"])

In [ ]:
# subset to those that have values in at least three cell types
all_r_vals_long["cyto1_cyto2"] = all_r_vals_long["cytokine1"] + "_" + all_r_vals_long["cytokine2"]
val_counts = all_r_vals_long["cyto1_cyto2"].value_counts()

# subset to those measured in at least three cell types (otherwise an overall consensus value is not meaningful)
val_counts = val_counts[val_counts > 2]
all_r_vals_long = all_r_vals_long[all_r_vals_long["cyto1_cyto2"].isin(val_counts.index)]

# calculate the mean value
mean_r = all_r_vals_long.groupby(["cytokine1", "cytokine2"])["r_val"].mean()
mean_r = mean_r.unstack()

In [ ]:
# threshold at mean values
group_by_cytokine = pd.DataFrame(columns=["cytokine1", "cytokine2"])
threshold = 3 / 4
for cytokine_1 in mean_r.index:
    cytokine_r = mean_r.loc[cytokine_1]
    cytokine_r = cytokine_r[cytokine_r > threshold]
    for cytokine_2 in cytokine_r.index:
        group_by_cytokine.loc[len(group_by_cytokine)] = [cytokine_1, cytokine_2]

group_by_cytokine = group_by_cytokine[group_by_cytokine["cytokine1"] != group_by_cytokine["cytokine2"]]

In [ ]:
# Create pair column by sorting cytokine names alphabetically and joining with underscore
group_by_cytokine["pair"] = group_by_cytokine.apply(lambda x: f"{min(x['cytokine1'], x['cytokine2'])}_{max(x['cytokine1'], x['cytokine2'])}", axis=1)

# Remove duplicate pairs
group_by_cytokine = group_by_cytokine.drop_duplicates(subset=["pair"])

In [ ]:
def sort_list_by_reference(input_list, reference):
    """
    Sort a list based on the order of elements in a reference list.
    
    Parameters:
    -----------
    input_list : list
        The list to be sorted. Should be a subset of the reference list.
    reference : list
        The reference list that defines the desired order.
        
    Returns:
    --------
    list
        A new list with elements from input_list sorted according to their order in reference.
    """
    # Create a dictionary mapping each element in reference to its position
    reference_positions = {element: i for i, element in enumerate(reference)}
    
    # Sort the input list based on the positions in the reference list
    sorted_list = sorted(input_list, key=lambda x: reference_positions.get(x, float('inf')))
    
    return sorted_list

In [ ]:
import igraph as ig

# Create an undirected graph
G = ig.Graph()

# Add vertices (unique cytokines)
unique_cytokines = set(group_by_cytokine['cytokine1'].unique()) | set(group_by_cytokine['cytokine2'].unique())
G.add_vertices(list(unique_cytokines))

# Add edges from the dataframe
edges = list(zip(group_by_cytokine['cytokine1'], group_by_cytokine['cytokine2']))
G.add_edges(edges)

# Find connected components
components = G.components()

# Print each group
all_close_groups = []
for i, component in enumerate(components, 1):
    # Get the vertex names for this component
    group = [G.vs[idx]['name'] for idx in component]
    sorted_group = sort_list_by_reference(group, list(cytokine_info.index))
    print(f"Group {i}: {sorted_group}")
    all_close_groups.append(sorted_group)

In [ ]:
with open(os.path.join(intermediate_output_folder, "6_highly_similar_groups.pkl"), "wb") as f:
    pickle.dump(all_close_groups, f)
    
if write_SI_tables_flag:
    df = all_close_groups.copy()
    df = pd.Series([", ".join(group) for group in df])
    df = df.apply(introduce_greek_letters)
    df = df.to_frame(name="Highly similar cytokine groups")
    df['Group ID'] = df.index + 1
    df = df[['Group ID', 'Highly similar cytokine groups']]
    write_to_SI_table(df, "highly_similar_cytokine_groups")    

## Leiden clustering


#### Clustering in each cell type separately

In [ ]:
%%capture output
all_communities_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    # subset to cytokines with strong impact in this cell type
    # could deactivate this but it makes the plots too large and largely includes noise
    cytokine_filter = strong_impact_by_cell_type[cell_type][strong_impact_by_cell_type[cell_type]].index
    rvals = r_vals_by_cell_type[cell_type].loc[cytokine_filter, cytokine_filter].astype(float).dropna(how='all', axis=0).dropna(how='all', axis=1)
    adjacency_matrix = rvals
    
    # Get the distance matrix.
    df = adjacency_matrix.copy()
    
    for i in range(min(df.shape)):
        df.iat[i, i] = np.nan

    all_communities, _ = leiden_clustering(
        adjacency_matrix,
        0.5,
        False,
        invert_distance=False
        )
    
    mean_r_to_first_community = []
    for community in all_communities:
        r_vals = adjacency_matrix.loc[all_communities[0], community]
        mean_r = r_vals.stack().mean()
        mean_r_to_first_community.append(mean_r)

    sort_index = np.argsort(mean_r_to_first_community)[::-1]
    all_communities = [all_communities[i] for i in sort_index]
    all_communities_by_cell_type[cell_type] = all_communities

    plot_heatmap_with_clustering(adjacency_matrix, cell_type, plot_name="pearson_leiden", heatmap_label="Pearson r", communities=all_communities)
    if cell_type in ["CD14 Mono", "CD8 T cell"]:
        plot_heatmap_with_clustering_small(adjacency_matrix, cell_type, plot_name="pearson_leiden", heatmap_label="Pearson r", communities=all_communities)

In [ ]:
# all_communities_by_cell_type_df = {}
# for cell_type in ALL_CELL_TYPES:
#     comms = all_communities_by_cell_type[cell_type]
#     cytokine_filter = strong_impact_by_cell_type[cell_type][strong_impact_by_cell_type[cell_type]].index
#     grouped = pd.Series(index=pd.MultiIndex.from_product([cytokine_filter, cytokine_filter]), dtype=int, data=0)
#     for comm_id, community in enumerate(comms):
#         for cytokine1 in community:
#             for cytokine2 in community:
#                 grouped.loc[(cytokine1, cytokine2)] = 1         
#     all_communities_by_cell_type_df[cell_type] = grouped
# all_communities_by_cell_type_df = pd.DataFrame(all_communities_by_cell_type_df)

In [ ]:
# all_communities_by_cell_type_df.to_excel(os.path.join(output_tables_folder, "DATE_6.1_community_assignments_by_cell_type.xlsx"))

### Per-lineage clustering

In [ ]:
myeloid_cells = ["CD14 Mono", "CD16 Mono", "cDC"]
lymphoid_cells = ["B cell", "CD4 T cell", "CD8 T cell", "NK CD56low", "NK CD56hi"]

In [ ]:
def get_combined_communities(cell_types, all_communities_by_cell_type):
    # create a distance matrix by normalizing clustering counts
    grouped_together_total = pd.DataFrame(0, index=CYTOKINE_ORDER, columns=CYTOKINE_ORDER)

    for cell_type in cell_types:
        communities = all_communities_by_cell_type[cell_type]
        for community in communities:
            for cytokine_1 in community:
                for cytokine_2 in community:
                    grouped_together_total.loc[cytokine_1, cytokine_2] += 1
                    
    # discard those with 0 in the diagonal
    # Keep only cytokines that have at least 2 interactions
    non_zero_diag = grouped_together_total.index[grouped_together_total.values.diagonal() > 1]
    grouped_together_total = grouped_together_total.loc[non_zero_diag, non_zero_diag]

    # Compute the row sums
    row_sums = grouped_together_total.sum(axis=1)

    # Compute the inverse square roots of the row sums
    eps = 1e-10
    inv_sqrt = 1 / np.sqrt(row_sums + eps)

    # Get the normalization factors
    norm_factors = pd.Series(inv_sqrt, index=grouped_together_total.index)

    # Perform the symmetric normalization
    # Multiply rows by the normalization factors, then columns.
    normalized_matrix = grouped_together_total.mul(norm_factors, axis=0)\
                                            .mul(norm_factors, axis=1)

    # round to 5 decimal places
    normalized_matrix = normalized_matrix.round(5)
    cosine_distances = squareform(pdist(normalized_matrix.values, metric='cosine'))
    cosine_distances = pd.DataFrame(cosine_distances, index=normalized_matrix.index, columns=normalized_matrix.index)
    
    # perform Leiden clustering (resolution = 0.5)
    all_communities, _ = leiden_clustering(cosine_distances, resolution_parameter=0.5, print_communities=False, invert_distance=True)

    # reorder communities internally to group highly similar cytokines
    reordered_communities = []

    for community in all_communities:
        community_list = list(community)
        
        # track which elements were moved to maintain order of non-subgroup elements
        moved_elements = set()
        
        # process each group that's a subset of this community in all_close_groups
        for group in all_close_groups: # this was calculated above by grouping on correlation
            if set(group).issubset(set(community)):
                # find the earliest index of any group member in the community
                earliest_idx = min(community_list.index(member) for member in group)
                
                # remove group members from their current positions
                # (if they haven't been moved by a previously)
                for member in group:
                    if member not in moved_elements:
                        community_list.remove(member)
                        moved_elements.add(member)
                
                # insert group members together at the earliest possible index
                for i, member in enumerate(group):
                    if member not in community_list:  # avoid duplicates (do I need this?)
                        community_list.insert(earliest_idx + i, member)
        
        reordered_communities.append(community_list)

    # replace all_communities with the reordered version
    all_communities = reordered_communities
    communities_flat = [item for sublist in all_communities for item in sublist]      
    
    return all_communities, communities_flat, grouped_together_total, normalized_matrix


cell_subset_labels = ['myeloid', 'lymphoid', 'all']
cell_subsets = [myeloid_cells, lymphoid_cells, ALL_CELL_TYPES]

for i in range(len(cell_subset_labels)):
    cell_subset_label = cell_subset_labels[i]
    cell_types = cell_subsets[i]
    
    all_communities, communities_flat, grouped_together_total, normalized_matrix = get_combined_communities(cell_types, all_communities_by_cell_type)

    # -------------------- Plotting --------------------
    df_to_plot = grouped_together_total.loc[communities_flat, communities_flat]
    plt.figure(figsize=cm_to_inches((12, 12)))

    ax = sns.heatmap(df_to_plot, square=True, vmin=0, vmax=df_to_plot.max().max(), cbar=True,
                cbar_kws={'label': 'grouping count', 'shrink': 0.6})

    # plot white boxes around the communities
    current_pos = 0
    for community in all_communities:
        num_entries = len(community)
        rect = plt.Rectangle((current_pos, current_pos), num_entries, num_entries, fill=False, color='white', linewidth=1.5, clip_on=False)
        plt.gca().add_patch(rect)
        current_pos += num_entries

    # Plot additional boxes around highly similar groups
    if cell_subset_label == 'all':
        for group in all_close_groups:
            group_indices = [communities_flat.index(member) for member in group if member in communities_flat]
            if group_indices:
                min_idx = min(group_indices)
                max_idx = max(group_indices)
                rect = plt.Rectangle((min_idx, min_idx), max_idx - min_idx + 1, max_idx - min_idx + 1, fill=False, color='skyblue', ls="--", linewidth=1, clip_on=False)
                plt.gca().add_patch(rect)

    # set font sizes to 5.5
    ax.set_xticks(np.arange(0.5, len(df_to_plot)+0.5))
    ax.set_yticks(np.arange(0.5, len(df_to_plot)+0.5))
    ax.set_xticklabels(df_to_plot.index, fontsize=5.5, rotation=90)  # Rotate for better visibility
    ax.set_yticklabels(df_to_plot.index, fontsize=5.5)
    # set the tick length to 0
    ax.tick_params(length=0)

    apply_greek_labels_heatmap(ax)

    plt.tight_layout()
    for format in ["png", "svg"]:
        plt.savefig(f"{plot_folder}/6_grouped_together_total_{cell_subset_label}.{format}", bbox_inches="tight")
    plt.show()

    # -------------------- Plotting ALT --------------------
    df_to_plot = grouped_together_total.loc[communities_flat, communities_flat]
    plt.figure(figsize=cm_to_inches((8.5, 8.5)))

    ax = sns.heatmap(df_to_plot, square=True, vmin=0, vmax=df_to_plot.max().max(), cbar=True,
                cbar_kws={'label': 'grouping count', 'shrink': 0.45})

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=5.5)  # Reduce the fontsize of the colorbar labels

    # plot white boxes around the communities
    current_pos = 0
    for community in all_communities:
        num_entries = len(community)
        rect = plt.Rectangle((current_pos, current_pos), num_entries, num_entries, fill=False, color='white', linewidth=1, clip_on=False)
        plt.gca().add_patch(rect)
        current_pos += num_entries

    # Plot additional boxes around highly similar groups
    if cell_subset_label == 'all':
        for group in all_close_groups:
            group_indices = [communities_flat.index(member) for member in group if member in communities_flat]
            if group_indices:
                min_idx = min(group_indices)
                max_idx = max(group_indices)
                rect = plt.Rectangle((min_idx, min_idx), max_idx - min_idx + 1, max_idx - min_idx + 1, fill=False, color='skyblue', ls="--", linewidth=1, clip_on=False)
                plt.gca().add_patch(rect)


    # Instead of the default y-axis labels, we will stagger them in 3 positions.
    # First, remove the default labels:
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    # Define 3 staggered horizontal offsets (in axis coordinates: 0=left, 1=right).
    # Adjust these values as needed for your plot.
    stagger_offsets = [-0.01, -0.15, -0.19]

    # Manually add the y-axis labels at staggered positions.
    # Here, the y coordinate is the same as the tick (i + 0.5) and the x position is determined by the stagger.
    for i, label in enumerate(df_to_plot.index):
        offset = stagger_offsets[i % 2]
        ax.text(
            offset, i + 0.2, introduce_greek_letters(label),
            ha='right', va='center', fontsize=5.5,
            transform=ax.get_yaxis_transform()  # x in axis fraction, y in data coordinates
        )

    # remove tick marks
    ax.tick_params(length=0)
    
    if cell_subset_label == "all":
        ax.set_title(f"grouping across cell types", fontsize=7)
    else:
        ax.set_title(f"{cell_subset_label} cells ({', '.join(cell_types)})", fontsize=7)
    apply_greek_labels_heatmap(ax)

    plt.tight_layout()
    for format in ["png", "svg"]:
        plt.savefig(f"{plot_folder}/6_grouped_together_total_alt_{cell_subset_label}.{format}", bbox_inches="tight")
    plt.show()

In [ ]:
if write_SI_tables_flag:
    df = all_communities.copy()
    df = pd.Series([", ".join(group) for group in df])
    df = df.apply(introduce_greek_letters)
    df = df.to_frame(name="Cytokine Groups by Leiden Clustering")
    df['Group ID'] = df.index + 1
    df = df[['Group ID', 'Cytokine Groups by Leiden Clustering']]
    write_to_SI_table(df, "cytokine_groups")    

In [ ]:
# # save the results (do I need these anywhere?)
# with open(os.path.join(intermediate_output_folder, "6_all_communities_cyto_cyto.pkl"), "wb") as f:
#     pickle.dump(all_communities, f)

In [ ]:
all_communities_str = [", ".join(l) for l in all_communities]
all_communities_df = pd.DataFrame(all_communities_str).rename(columns={0: "communities_across_cells"})
all_communities_df["close_groups_across_cells"] = ""

for l in all_close_groups:
    mapping_index = [i for i, community in enumerate(all_communities) if all(cyto in community for cyto in l)]
    if mapping_index:
        if len(all_communities_df.loc[mapping_index[0], "close_groups_across_cells"]) == 0:
            all_communities_df.loc[mapping_index[0], "close_groups_across_cells"] = ", ".join(l)
        else:
            all_communities_df.loc[mapping_index[0], "close_groups_across_cells"] = \
                all_communities_df.loc[mapping_index[0], "close_groups_across_cells"] + "; " + ", ".join(l)
all_communities_df.to_csv(os.path.join(output_tables_folder, "6.1_across_cell_communities.csv"))

In [ ]:
cytokine_to_across_cell_group = {}
for index, row in all_communities_df.iterrows():
    community = row["communities_across_cells"].split(", ")
    for cytokine in community:
        cytokine_to_across_cell_group[cytokine] = f"Group {'0' if index+1<10 else ''}{index+1}" if len(community)>1 else "Ungrouped"

In [ ]:
all_communities_by_cell_type_df = {}
cytokine_to_group_by_cell_type = {}
group_to_cytokines_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    cytokine_to_group = {}
    group_to_cytokines = {}
    communities = all_communities_by_cell_type[cell_type]
    output_df = pd.DataFrame(index=CYTOKINE_ORDER, columns=CYTOKINE_ORDER, data=np.nan)
    
    # set everything with strong impact to 0
    strong_impact = strong_impact_by_cell_type[cell_type]
    strong_impact = strong_impact[strong_impact].index
    output_df.loc[strong_impact, strong_impact] = 0

    for i, community in enumerate(communities):
        combinations = list(itertools.product(community, community))
        for c1, c2 in combinations:
            output_df.loc[c1, c2] = 1
        for entry in community:
            cytokine_to_group[entry] = f"Group {i + 1:02d}" if len(community) > 1 else "Ungrouped"
        group_to_cytokines[f"Group {i + 1:02d}"] = community
    group_to_cytokines_by_cell_type[cell_type] = group_to_cytokines
    cytokine_to_group_by_cell_type[cell_type] = cytokine_to_group
    all_communities_by_cell_type_df[cell_type] = output_df
all_communities_by_cell_type_df = pd.concat(all_communities_by_cell_type_df, axis=1, keys=ALL_CELL_TYPES)
all_communities_by_cell_type_df.to_csv(os.path.join(output_tables_folder, "6.1_within_cell_communities.csv"))

In [ ]:
# # save the results (do I need these anywhere?)
# os.makedirs("group_exploration", exist_ok=True)
# with open("group_exploration/group_to_cytokines_by_cell_type.pkl", 'wb') as f:
#     pickle.dump(group_to_cytokines_by_cell_type, f)
# with open("group_exploration/cytokine_to_group_by_cell_type.pkl", 'wb') as f:
#     pickle.dump(cytokine_to_group_by_cell_type, f)
# with open("group_exploration/cytokine_to_across_cell_group.pkl", 'wb') as f:
#     pickle.dump(cytokine_to_across_cell_group, f)

## 7 - Secondary responses

#### Algorithmic explanations

To identify cytokines that may mediate secondary responses, we combined changes in the simulated supernatant with the similarity of transcriptional responses. Candidate filters were deliberately permissive; candidates were then ranked by a priority score and classified by the evidence for a secondary mechanism.

Candidate pairs. For each input cytokine q and secondary cytokine s (s ≠ q), we considered pairs with a simulated supernatant change |log2FC(s)| > 0.5 under stimulation with q, for which both cytokines had a strong impact on at least one of B cells, CD4 T cells, NK CD56low cells or CD14 monocytes.

Recipient evidence. In each cell type c, we used the Pearson correlation r_c between the consensus log2FC values of stimulation with q and with s (see Grouping cytokine responses), multiplied by the sign of the supernatant change, such that an increase in s requires a positive and a decrease a negative correlation. Only cell types that express the receptor for s (≥8 cpm) and on which s has a strong impact were considered. Within each lineage (B cells; T cells: CD4 T, CD8 T, Treg, NKT, MAIT; NK cells: NK CD56low, NK CD56hi; myeloid cells: CD14 Mono, CD16 Mono, cDC), we calculated the lineage evidence as the mean of the positive part of these signed correlations, weighted by the median abundance of each cell type across samples. A lineage supported the pair if its evidence was at least 0.3, and the pair evidence was the mean evidence across evaluable lineages. Pairs were classified as having cross-lineage support (at least two supporting lineages), single-lineage support (one evaluable lineage, which supported the pair), inconsistent support (several evaluable lineages, of which only one supported the pair) or no support. The priority of a pair was defined as min(|log2FC(s)|/2, 1) × pair evidence.

Requirement for a secondary cytokine. A cell type with a signed correlation of at least 0.3 was considered a supported recipient. If all supported recipients expressed the receptor for q (≥8 cpm), the similarity may arise from direct stimulation with q ("possibly incidental similarity"). If at least one supported recipient lacked the receptor for q, the response in this cell type requires a secondary cytokine.

Higher-order routes. If more than 2/3 of the production of s came from cells lacking the receptor for q, we searched for intermediate cytokines m such that both q→m and m→s were consistently supported immediate production edges (supported recipient, direct producer share > 2/3) whose combined direction matched the observed change in s. We required at least four donors responsive in both pairs and quantified the overlap between the cells producing s indirectly under q (lacking the receptor for q) and those producing it directly under m (expressing the receptor for m). For each donor, the overlap is the sum over these cell types of the smaller of the two production shares (each renormalized to the indirect and direct production, respectively); routes with a median overlap of at least 0.5 were accepted, and the route with the highest product of edge strengths (priority × direct producer share) and overlap was reported ("probable tertiary"). Pairs with a direct producer share and an indirect share of at most 2/3 were classified as shared direct/indirect, and indirect pairs without an accepted route as unresolved. IL-32-β, for which no receptor is known, could not be classified.

In [ ]:
plot_folder = os.path.join(base_plot_folder, "7_secondary_responses")
os.makedirs(plot_folder, exist_ok=True)

setup_plotting_environment()

In [ ]:
# Median PBMC composition across treatment/donor samples.
counts = all_counts_by_condition.copy()
counts.index = counts.index.set_names(["cell_type", "treatment", "donor"])
counts = counts[counts.index.get_level_values("cell_type").isin(ALL_CELL_TYPES)]
fractions = counts / counts.groupby(level=["treatment", "donor"]).transform("sum")
cell_abundance = fractions.groupby(level="cell_type").median().reindex(ALL_CELL_TYPES).fillna(0)
cell_abundance = cell_abundance / cell_abundance.sum()

In [ ]:
# FILTER SUPERNATANT CHANGES
def get_substantial_supernatant_change(supernatant_log2fc, config, use_anchors=True, use_strong_filter=True):
    # subset to relevant supernatant changes
    substantial_sup_change = supernatant_log2fc.copy()

    # we can only do analysis for cytokines where we have the primary stimulation
    cytokines_present = [cyto for cyto in CYTOKINE_ORDER if cyto in substantial_sup_change.index]
    substantial_sup_change = substantial_sup_change.loc[cytokines_present]

    # we only care about cytokines that have major impact on a major lineage
    if use_strong_filter:
        if use_anchors:
            strong = strong_impact_by_cell_type[list(config.anchor_cell_types)].any(axis=1)
        else:
            strong = strong_impact_by_cell_type.any(axis=1)
        strong = strong[strong].index
        substantial_sup_change = substantial_sup_change[[col for col in substantial_sup_change.columns if col in strong]]
        substantial_sup_change = substantial_sup_change.loc[[i for i in substantial_sup_change.index if i in strong]]

    substantial_sup_change = substantial_sup_change.loc[:, abs(substantial_sup_change).max(axis=0) > config.supernatant_min_log2fc]
    substantial_sup_change = substantial_sup_change[abs(substantial_sup_change).max(axis=1) > config.supernatant_min_log2fc]
    
    return substantial_sup_change.T

# PAIR-LEVEL SECONDARY-RESPONSE PRIORITIZATION
def calculate_pair_priority(substantial_sup_change, config):
    pair_rows = []
    similarity_by_pair = {}
    informative_by_pair = {}
    support_by_pair = {}
    query_receptor_by_pair = {}
    pair_details_by_pair = {}
    for query_cyto in substantial_sup_change.index:
        for secondary_cyto, medium_change in substantial_sup_change.loc[query_cyto].items():
            if pd.isna(medium_change) or abs(medium_change) <= config.supernatant_min_log2fc or query_cyto == secondary_cyto:
                continue
            key = (query_cyto, secondary_cyto)

            # response similarity profile
            similarity = pd.Series({
                cell: r_vals_by_cell_type_df.loc[query_cyto, cell][secondary_cyto]
                for cell in ALL_CELL_TYPES
            }, dtype=float)
            
            # higher supernatant -> more similarity
            # lower supernatant -> less similarity
            direction_sign = np.sign(medium_change)
            aligned_similarity = direction_sign * similarity
            # if aligned_similarity.isna().any(): # maybe not correct but let's see
            #     raise Exception(f"NaN value in aligned similarity for {query_cyto}, {secondary_cyto}")

            # Is the cell type expected to respond to the proposed secondary cytokine?
            secondary_receptor = receptor_expression_by_cytokine.reindex(index=[secondary_cyto], columns=ALL_CELL_TYPES).iloc[0]
            secondary_impact = strong_impact_by_cell_type.reindex(index=[secondary_cyto], columns=ALL_CELL_TYPES).iloc[0]
            eligible = (secondary_receptor >= config.receptor_min) & secondary_impact

            # Is the response similar to the initial cytokine
            in_lineage = pd.Series({c: config.lineage_by_cell.get(c) in config.major_lineages for c in ALL_CELL_TYPES})
            informative = eligible & aligned_similarity.notna() & in_lineage
            support = informative & aligned_similarity.ge(config.similarity_min)

            # aggregate evidence
            lineage_evidence = {}
            lineage_contradiction = {}
            lineage_support = {}
            for lineage in config.major_lineages:
                cells = [cell for cell in ALL_CELL_TYPES if config.lineage_by_cell.get(cell) == lineage and informative[cell]]
                if not cells:
                    continue

                # renormalize within eligible lineages
                weights = cell_abundance.loc[cells]
                weights = weights / weights.sum()

                # over eligible cells, what is the weighted similarity?
                lineage_evidence[lineage] = float((weights * aligned_similarity.loc[cells].clip(lower=0)).sum())
                # over eligible cells, what is the weighted disimilarity?
                lineage_contradiction[lineage] = float((weights * (-aligned_similarity.loc[cells]).clip(lower=0)).sum())
                # does the weighted similarity for this lineage clear the threshold?
                lineage_support[lineage] = lineage_evidence[lineage] >= config.similarity_min
            
            # aggregate evidence
            eligible_lineages = len(lineage_evidence)
            supporting_lineages = sum(lineage_support.values())
            pair_evidence = np.mean(list(lineage_evidence.values())) if eligible_lineages else np.nan
            pair_contradiction = np.mean(list(lineage_contradiction.values())) if eligible_lineages else np.nan
            support_fraction = supporting_lineages / eligible_lineages if eligible_lineages else np.nan
            
            # classify
            if eligible_lineages == 0:
                confidence = "not evaluable"
            elif eligible_lineages > 0 and supporting_lineages == 0:
                confidence = "no support"
            elif eligible_lineages == 1 and supporting_lineages == 1:
                confidence = "single-lineage supported"
            elif eligible_lineages >= 2 and supporting_lineages == 1:
                confidence = "inconsistent support"
            elif eligible_lineages >= 2 and supporting_lineages >= 2:
                confidence = "cross-lineage support"
            else:
                confidence = "error"
            
            # multiply out supernatant change with pair evidence for prioritization
            magnitude_score = min(abs(medium_change) / config.magnitude_saturation, 1)
            priority = magnitude_score * pair_evidence if pd.notna(pair_evidence) else np.nan
            
            # Population coverage (does not affect evidence weighting)
            supporting_cells = list(support[support].index)
            eligible_cells = list(informative[informative].index)
            population_support_mass = float(cell_abundance.reindex(supporting_cells).sum())
            
            # Initial query-receptor expression for context
            if query_cyto in receptor_expression_by_cytokine.index:
                query_receptor = receptor_expression_by_cytokine.loc[query_cyto, ALL_CELL_TYPES].astype(float)
            else:
                query_receptor = pd.Series(np.nan, index=ALL_CELL_TYPES)
            
            # save detailed vals
            pair_details_by_pair[key] = {"similarity": similarity, "aligned_similarity": aligned_similarity,
            "secondary_receptor": secondary_receptor, "secondary_impact": secondary_impact, "eligible": eligible,
            "informative": informative, "support": support, "query_receptor": query_receptor, "lineage_evidence": lineage_evidence,
            "lineage_contradiction": lineage_contradiction, "lineage_support": lineage_support}
            
            pair_rows.append({
                "query_cyto": query_cyto,
                "secondary": secondary_cyto,
                "direction": "secondary increased" if medium_change > 0 else "secondary reduced",
                "medium_log2fc": medium_change,
                "magnitude_score": magnitude_score,
                "pair_evidence": pair_evidence,
                "pair_contradiction": pair_contradiction,
                "priority": priority,
                "confidence": confidence,
                "eligible_lineages": eligible_lineages,
                "supporting_lineages": supporting_lineages,
                "eligible_lineages_detail": list(lineage_evidence.keys()),
                "supporting_lineages_detail": [key for key, val in lineage_support.items() if val],
                "support_fraction": support_fraction,
                "eligible_cells": ", ".join(eligible_cells),
                "supporting_cells": ", ".join(supporting_cells),
                "population_support_mass": population_support_mass
            })

            similarity_by_pair[key] = similarity
            informative_by_pair[key] = informative
            support_by_pair[key] = support
            query_receptor_by_pair[key] = query_receptor
    pair_priority = pd.DataFrame(pair_rows)

    confidence_order = {
        "cross-lineage support": 0,
        "single-lineage supported": 1,
        "inconsistent support": 2,
        "no support": 3,
        "not evaluable": 4,
    }
    pair_priority["confidence_order"] = pair_priority["confidence"].map(confidence_order)
    pair_priority["evidence_rank"] = pair_priority.groupby("direction")["priority"].rank(ascending=False, method="min")
    pair_priority = pair_priority.sort_values(["direction", "confidence_order", "priority"], ascending=[True, True, False]).reset_index(drop=True)
    
    return pair_priority, similarity_by_pair, informative_by_pair, support_by_pair, query_receptor_by_pair, pair_details_by_pair

### CLASSIFY
def add_secondary_path_classes(
    pair_priority, all_contributions, receptor_expression_by_cytokine, support_by_pair, baseline_name_by_treatment,
    config, default_baseline="PBS", print_warnings=True,
    ):
    """
    baseline_name_by_treatment: dict that specifies a different baseline as comparison (necessary for review perturbations)
    """
    result = pair_priority.copy()
    direction_to_sign = {"secondary increased": 1, "secondary reduced": -1}

    # catch renaming of directions
    unknown_directions = set(result["direction"].dropna()) - set(direction_to_sign)
    if unknown_directions:
        raise ValueError(f"Unknown direction labels: {sorted(unknown_directions)}")

    # catch missing columns
    required_columns = {"cell_type", "treatment", "donor", "protein", "cell_type_contribution"}
    missing = required_columns - set(all_contributions.columns)
    if missing:
        raise ValueError(f"all_contributions is missing: {sorted(missing)}")

    contribution_table = all_contributions.pivot_table(
        index=["treatment", "donor", "protein"], columns="cell_type", values="cell_type_contribution", aggfunc="sum")

    receptor_class = pd.DataFrame(index=receptor_expression_by_cytokine.index, columns=receptor_expression_by_cytokine.columns, dtype="object")
    receptor_class[receptor_expression_by_cytokine >= config.receptor_min] = "direct"
    receptor_class[receptor_expression_by_cytokine < config.receptor_min] = "indirect"
    if receptor_class.isna().any().any():
        raise ValueError("Some receptor values were not classified")
    receptor_class.loc['IL-32-beta'] = 'missing'

    def recipient_requirement(query, secondary):
        # check whether there's support (corr) and receptor expression
        key = (query, secondary)
        if key not in support_by_pair:
            raise Exception(f"Missing support information for {query}, {secondary}")

        # which cells are supported (have high corr)
        supported = support_by_pair[key]
        invalid_support = ~supported.map(lambda value: isinstance(value, (bool, np.bool_)))
        if invalid_support.any():
            raise ValueError(f"Non-boolean support values for {query}, {secondary}")
        supported_cells = list(supported[supported].index)
        if len(supported_cells) == 0:
            return False, [], []

        receptor_classes = receptor_class.loc[query, supported_cells]
        if (receptor_classes == 'missing').any():
            # should only trigger for IL-32-beta
            if print_warnings:
                print(f"Missing receptor expression for {query}, {secondary}")
            return np.nan, supported_cells, []
        receptor_low_cells = receptor_classes.index[receptor_classes.eq("indirect")].tolist()
        secondary_required = len(receptor_low_cells) > 0
        return secondary_required, supported_cells, receptor_low_cells

    def producer_attribution(query, secondary, direction_sign):
        # shape: columns: cell types, index: donors
        baseline_treatment = baseline_name_by_treatment.get(query, default_baseline)
        try:
            treated = contribution_table.xs((query, secondary), level=("treatment", "protein"))
            baseline = contribution_table.xs((baseline_treatment, secondary), level=("treatment", "protein"))
        except KeyError:
            raise Exception(f"PRODUCER_ATTRIBUTION: Missing entry in contribution_table for {query} and {secondary}")

        treated_donors = treated.index.tolist()
        baseline_donors = baseline.index.tolist()
        shape_treatment, shape_baseline= treated.shape, baseline.shape
        treated, baseline = treated.align(baseline, join="inner", axis=0) # intersection over donors
        treated, baseline = treated.align(baseline, join="outer", axis=1, fill_value=0) # union over cell types
        shape_treatment_after, shape_baseline_after = treated.shape, baseline.shape
        if shape_baseline != shape_baseline_after or shape_treatment != shape_treatment_after:
            if print_warnings:
                print(f"PRODUCER_ATTRIBUTION: Change in shape for for {query} and {secondary}:" + \
                f"baseline: {shape_baseline}, {shape_baseline_after}; treatment: {shape_treatment}, {shape_treatment_after}")
        if treated.isna().any().any() or baseline.isna().any().any():
            if print_warnings:
                print(f"Missing contribution values for {query}, {secondary}")
        if treated.empty:
            raise Exception(f"PRODUCER_ATTRIBUTION: Alignment messes up producer attribution for {query} and {secondary}.")
        treated = treated.fillna(0)
        baseline = baseline.fillna(0)

        # filter to responsive donors with a sufficient response that cell types can be properly attributed
        per_donor_log2fc = np.log2(treated.sum(axis=1) / baseline.sum(axis=1))
        responsive_donors = (per_donor_log2fc * direction_sign) > config.supernatant_min_log2fc_per_donor
        responsive_donors = responsive_donors[responsive_donors].index
        if len(responsive_donors) < config.warn_below_retained_donors:
            if print_warnings:
                print(f"Warning: inconsistent donor behavior for {query} and {secondary}: {per_donor_log2fc * direction_sign}")
        
        # attribute changes to cell types and donors
        attributed_change_all = ((treated - baseline) * direction_sign).clip(lower=0)
        donor_shares_all = attributed_change_all.div(attributed_change_all.sum(axis=1), axis=0)
        donor_shares = donor_shares_all.loc[responsive_donors]

        # what is the production fraction in known target cells?
        producer_receptor_classes = receptor_class.loc[query, donor_shares.columns]
        missing_producers = producer_receptor_classes.index[producer_receptor_classes.eq("missing")].tolist()
        
        # save details for detailed inspection
        producer_details = {"treated_donors": treated_donors, "baseline_donors": baseline_donors,
        "signed_log2fc": per_donor_log2fc * direction_sign, "responsive_donors": responsive_donors.tolist(),
        "donor_shares_all": donor_shares_all, "producer_receptor_classes": producer_receptor_classes}
        
        if len(missing_producers) > 0:
            if print_warnings:
                print(f"Missing producer receptor expression for {query}, {secondary}: {missing_producers}")
            return np.nan, np.nan, len(donor_shares), donor_shares, producer_details

        direct_producers = producer_receptor_classes.index[producer_receptor_classes.eq("direct")].tolist()
        donor_direct_share = donor_shares.loc[:, direct_producers].sum(axis=1)
        direct_share = float(donor_direct_share.median())
        indirect_share = float((1 - donor_direct_share).median())
        
        return direct_share, indirect_share, len(donor_shares), donor_shares, producer_details

    # --------------- Initial search for direct connections ---------------
    secondary_required, supported_cells, required_cells = [], [], []
    direct_shares, indirect_shares, attribution_donors = [], [], []
    producer_shares = {}
    producer_details_by_pair = {}
    for _, row in result.iterrows():
        query, secondary = row["query_cyto"], row["secondary"]
        direction_sign = direction_to_sign[row["direction"]]
        required, supported, receptor_low = recipient_requirement(query, secondary)
        direct_share, indirect_share, n_donors, donor_shares, producer_details = producer_attribution(query, secondary, direction_sign)

        secondary_required.append(required)
        supported_cells.append(supported)
        required_cells.append(receptor_low)
        direct_shares.append(direct_share)
        indirect_shares.append(indirect_share)
        attribution_donors.append(n_donors)
        producer_shares[(query, secondary)] = donor_shares
        producer_details_by_pair[(query, secondary)] = producer_details
        
    result["has_supported_recipient"] = [bool(cells) for cells in supported_cells]
    result["secondary_required"] = secondary_required
    result["supported_recipient_cells"] = supported_cells
    result["query_receptor_low_recipient_cells"] = required_cells
    result["direct_producer_share"] = direct_shares
    result["indirect_producer_share"] = indirect_shares
    result["producer_attribution_donors"] = attribution_donors
    result["consistent_support"] = result["confidence"].isin(config.consistent_confidence)
    result["causal_effect"] = result["direction"].map(
        {"secondary increased": "gain of secondary signal", "secondary reduced": "loss of secondary signal"})
    
    production_edges = {
        (row["query_cyto"], row["secondary"]): direction_to_sign[row["direction"]]
        for _, row in result.iterrows()
        if row["consistent_support"] and row["has_supported_recipient"] and pd.notna(row["direct_producer_share"]) and row["direct_producer_share"] > config.direct_share_min
    }
    edge_strength = {
        (row["query_cyto"], row["secondary"]): row["priority"] * row["direct_producer_share"]
        for _, row in result.iterrows()
        if pd.notna(row["priority"]) and pd.notna(row["direct_producer_share"])
    }

    # --------------- Search for plausible higher-order connections ---------------
    all_cytokines = sorted(set(result["query_cyto"].dropna()) | set(result["secondary"].dropna()))

    def evaluate_intermediate_route(query, secondary, intermediate):
        target_key = (query, secondary)
        second_edge = (intermediate, secondary)
        target_donor_shares = producer_shares[target_key]
        second_donor_shares = producer_shares[second_edge]
        
        # are there enough shared donors?
        # donors in donor_shares were subset to responsive donors, so if different donors are responsive, this might not fire
        common_donors = target_donor_shares.index.intersection(second_donor_shares.index)
        if len(common_donors) < config.route_min_common_donors:
            return {"rejection": f"shared donors {len(common_donors)} < {config.route_min_common_donors}"}
        target_donor_shares = target_donor_shares.loc[common_donors]
        second_donor_shares = second_donor_shares.loc[common_donors]
        
        # are there any cell types where the secondary cytokine we want to explain is direct under intermediate stimulation?
        target_receptor_classes = receptor_class.loc[query, target_donor_shares.columns]
        second_receptor_classes = receptor_class.loc[intermediate, second_donor_shares.columns]
        target_indirect_producers = target_receptor_classes.index[target_receptor_classes.eq("indirect")].tolist()
        second_direct_producers = second_receptor_classes.index[second_receptor_classes.eq("direct")].tolist()
        overlap_producers = [producer for producer in target_indirect_producers if producer in second_direct_producers]
        if len(overlap_producers) == 0:
            return {"rejection": "no producer cell type is input-receptor-low and intermediate-receptor-high"}

        # what fraction of the production of secondary under query is indirect for each donor?
        target_indirect_total = target_donor_shares.loc[:, target_indirect_producers].sum(axis=1)
        # what fraction of the production of secondary under intermediate is direct for each donor?
        second_direct_total = second_donor_shares.loc[:, second_direct_producers].sum(axis=1)
        # this works because signed changes are taken care of upstream
        valid_donors = common_donors[(target_indirect_total > 0) & (second_direct_total > 0)]
        if len(valid_donors) < config.route_min_common_donors:
            return {"rejection": f"donors with positive relevant production {len(valid_donors)} < {config.route_min_common_donors}"}

        # what fraction of the total indirect production under query is in cell types that are direct under intermediate?
        target_overlap_distribution = target_donor_shares.loc[valid_donors, overlap_producers].div(target_indirect_total.loc[valid_donors], axis=0)
        # what fraction of the total direct production under intermediate is in cell types that are indirect under query?
        second_overlap_distribution = second_donor_shares.loc[valid_donors, overlap_producers].div(second_direct_total.loc[valid_donors], axis=0)
        # take the minimum of the two to get an overlap score per cell type and donor
        overlap_by_producer_and_donor = target_overlap_distribution.where(target_overlap_distribution <= second_overlap_distribution, second_overlap_distribution)
        
        # sum to get an overlap score per donor, then get an across-donor value
        overlap_by_donor = overlap_by_producer_and_donor.sum(axis=1)
        producer_overlap_share = float(overlap_by_donor.median())
        
        # take the mean across donors to get a value per cell type
        overlap_by_producer = overlap_by_producer_and_donor.mean(axis=0).sort_values(ascending=False)
        overlap_producers = overlap_by_producer.index[overlap_by_producer > 0].tolist()

        return {"producer_overlap_share": producer_overlap_share, "overlap_producers": overlap_producers, "overlap_by_producer": overlap_by_producer.to_dict(),
                "overlap_by_donor": overlap_by_donor.to_dict(), "valid_donors": valid_donors.tolist(), "n_valid_donors": len(valid_donors)}

    # Record failed gates as well as accepted routes; classification is unchanged.
    pair_lookup = {(r["query_cyto"], r["secondary"]): r for _, r in result.iterrows()}

    def edge_failure(edge):
        if edge in production_edges:
            return "pass"
        if edge not in pair_lookup:
            return "pair absent from candidate table"
        r = pair_lookup[edge]
        failures = []
        if not r["consistent_support"]:
            failures.append(f"confidence: {r['confidence']}")
        if not r["has_supported_recipient"]:
            failures.append("no supported recipient")
        share = r["direct_producer_share"]
        if pd.isna(share):
            failures.append("direct share unavailable")
        elif share <= config.direct_share_min:
            failures.append(f"direct share {share:.1%} <= {config.direct_share_min:.1%}")
        return "; ".join(failures)

    route_search_audits = []
    candidate_intermediates = []
    candidate_intermediate_details = []
    best_intermediates = []
    best_intermediate_scores = []
    best_intermediate_overlap_shares = []
    best_intermediate_overlap_producers = []
    best_intermediate_common_donors = []

    for _, row in result.iterrows():
        query, secondary = row["query_cyto"], row["secondary"]
        target_sign = direction_to_sign[row["direction"]]
        candidates = []
        route_details = {}
        route_audit = []

        if row["secondary_required"] == True and row["indirect_producer_share"] > config.indirect_share_min:
            for intermediate in all_cytokines:
                # the connection itself
                if intermediate in {query, secondary}:
                    continue
                
                first_edge = (query, intermediate)
                second_edge = (intermediate, secondary)
                check = {"intermediate": intermediate,
                         "first_edge": edge_failure(first_edge),
                         "second_edge": edge_failure(second_edge),
                         "result": "edge gate failed"}
                route_audit.append(check)
                # do both edges exist?
                if first_edge not in production_edges or second_edge not in production_edges:
                    continue
                # does their overall direction work out?
                if production_edges[first_edge] * production_edges[second_edge] != target_sign:
                    check["result"] = "edge directions do not match target direction"
                    continue
                
                # evaluate
                details = evaluate_intermediate_route(query, secondary, intermediate)
                if "rejection" in details:
                    check["result"] = details["rejection"]
                    continue
                if details["producer_overlap_share"] < config.route_overlap_min:
                    check["result"] = f"producer overlap {details['producer_overlap_share']:.1%} < {config.route_overlap_min:.1%}"
                    continue

                score = edge_strength[first_edge] * edge_strength[second_edge] * details["producer_overlap_share"]
                details["score"] = float(score)
                details["first_edge"] = first_edge
                details["second_edge"] = second_edge
                details["first_edge_sign"] = production_edges[first_edge]
                details["second_edge_sign"] = production_edges[second_edge]
                check["result"] = f"accepted: overlap {details['producer_overlap_share']:.1%}, {details['n_valid_donors']} donors"
                candidates.append(intermediate)
                route_details[intermediate] = details
            candidates.sort(key=lambda intermediate: (-route_details[intermediate]["score"], intermediate))

        if len(candidates) > 0:
            best_intermediate = candidates[0]
            best_details = route_details[best_intermediate]
            best_score = best_details["score"]
            best_overlap_share = best_details["producer_overlap_share"]
            best_overlap_producers = best_details["overlap_producers"]
            best_common_donors = best_details["n_valid_donors"]
        else:
            best_intermediate = np.nan
            best_score = np.nan
            best_overlap_share = np.nan
            best_overlap_producers = []
            best_common_donors = 0

        route_search_audits.append(route_audit)
        candidate_intermediates.append(candidates)
        candidate_intermediate_details.append(route_details)
        best_intermediates.append(best_intermediate)
        best_intermediate_scores.append(best_score)
        best_intermediate_overlap_shares.append(best_overlap_share)
        best_intermediate_overlap_producers.append(best_overlap_producers)
        best_intermediate_common_donors.append(best_common_donors)

    result["route_search_audit"] = route_search_audits
    result["candidate_intermediates"] = candidate_intermediates
    result["candidate_intermediate_details"] = candidate_intermediate_details
    result["best_intermediate"] = best_intermediates
    result["best_intermediate_score"] = best_intermediate_scores
    result["best_intermediate_producer_overlap"] = best_intermediate_overlap_shares
    result["best_intermediate_overlap_producers"] = best_intermediate_overlap_producers
    result["best_intermediate_common_donors"] = best_intermediate_common_donors

    causal_classes = []
    for _, row in result.iterrows():
        if pd.isna(row["priority"]):
            causal_class = "not evaluable"
        elif not row["has_supported_recipient"]:
            causal_class = "unsupported similarity"
        elif pd.isna(row["secondary_required"]):
            causal_class = "unresolved receptor coverage"
        elif not row["secondary_required"]:
            causal_class = "possibly incidental similarity"
        elif (row["direct_producer_share"] <= config.direct_share_min
              and row["indirect_producer_share"] <= config.indirect_share_min):
            causal_class = "shared direct/indirect"
        elif row["direct_producer_share"] > config.direct_share_min:
            causal_class = "immediate secondary"
        elif row["indirect_producer_share"] > config.indirect_share_min and pd.notna(row["best_intermediate"]):
            causal_class = "probable tertiary"
        else:
            causal_class = "unresolved indirect"
        causal_classes.append(causal_class)

    result["causal_class"] = causal_classes
    return result, producer_shares, producer_details_by_pair

In [ ]:
import importlib
import utils.parameters
importlib.reload(utils.parameters)
from utils.parameters import SecondaryResponseConfig
config = SecondaryResponseConfig()

In [ ]:
substantial_sup_change = get_substantial_supernatant_change(
    supernatant_log2fc=supernatant_log2fc,
    config=config
    )

pair_priority, similarity_by_pair, informative_by_pair, support_by_pair, query_receptor_by_pair, pair_details_by_pair = calculate_pair_priority(
    substantial_sup_change=substantial_sup_change,
    config=config
)

baseline_by_treatment = {treatment: "PBS_review" for treatment in review_replace_list}
causal_pair_priority, producer_shares_by_pair, producer_details_by_pair = add_secondary_path_classes(
    pair_priority, all_contributions_detailed, receptor_expression_by_cytokine, support_by_pair,
    config=config, baseline_name_by_treatment=baseline_by_treatment, print_warnings=False
    )

In [ ]:
def plot_secondary_pair(
    query_cyto,
    pair_priority,
    n_pairs=10,
    direction=None,
    plot_lineage_support=False,
    confidence_levels=(
        "cross-lineage support",
        "single-lineage supported",
        "inconsistent support",
        "no support",
        ),
    ):
    """Plot secondary-response similarity with primary-receptor and lineage support."""

    selected = pair_priority.loc[pair_priority["query_cyto"].eq(query_cyto)].copy()
    if direction is not None:
        selected = selected.loc[selected["direction"].eq(direction)]
    if confidence_levels is not None:
        selected = selected.loc[selected["confidence"].isin(confidence_levels)]

    selected = selected.sort_values(["priority", "confidence_order"], ascending=[False, True]).head(n_pairs).reset_index(drop=True)
    if selected.empty:
        raise ValueError(f"No candidate secondary responses found for {query_cyto!r}.")

    labels = [introduce_greek_letters(cytokine) for cytokine in selected["secondary"]]
    keys = list(zip(selected["query_cyto"], selected["secondary"]))

    similarity_matrix = pd.DataFrame([similarity_by_pair[key] for key in keys],index=labels,columns=ALL_CELL_TYPES)
    informative_matrix = pd.DataFrame([informative_by_pair[key] for key in keys],index=labels,columns=ALL_CELL_TYPES,)
    support_matrix = pd.DataFrame([support_by_pair[key] for key in keys],index=labels,columns=ALL_CELL_TYPES,)

    # One shared column annotation: where can the primary cytokine act directly?
    query_receptor = np.log2(receptor_expression_by_cytokine.reindex(index=[query_cyto], columns=ALL_CELL_TYPES).iloc[0].astype(float)).clip(lower=0)
    # receptor_matrix = np.log2(query_receptor.clip(lower=0).add(1)).to_frame().T

    # -1: not evaluable in this lineage; 0: evaluable but unsupported; 1: supported.
    lineage_order = ["B", "T", "NK", "myeloid"]

    def _as_set(value):
        if isinstance(value, (list, tuple, set, np.ndarray, pd.Index)):
            return set(value)
        return set() if pd.isna(value) else {value}

    lineage_state = pd.DataFrame(-1,index=labels,columns=lineage_order,dtype=int,)
    for row_i, row in selected.iterrows():
        eligible = _as_set(row["eligible_lineages_detail"])
        supporting = _as_set(row["supporting_lineages_detail"])
        lineage_state.loc[labels[row_i], list(eligible)] = 0
        lineage_state.loc[labels[row_i], list(supporting)] = 1

    n_rows = len(selected)
    fig = plt.figure(figsize=cm_to_inches((5.8, max(3, 0.32 * n_rows + 0.3))))
    grid = fig.add_gridspec(2, 3, height_ratios=[1, n_rows],width_ratios=[8, 1.3, 1.3],hspace=0.05,wspace=0.3)

    ax_receptor = fig.add_subplot(grid[0, 0])
    ax_similarity = fig.add_subplot(grid[1, 0])
    ax_medium = fig.add_subplot(grid[1, 1])
    ax_priority = fig.add_subplot(grid[1, 2])

    # Receptor expression
    if len(query_receptor.dropna()) > 0:
        colors = ['tab:blue' if val > np.log2(config.receptor_min) else 'tab:red' for val in query_receptor.values]
        ax_receptor.bar(query_receptor.index, query_receptor.values, edgecolor='black', color=colors, linewidth=0.5)
        ax_receptor.tick_params(axis="y", rotation=0, length=0, labelsize=6)
        ax_receptor.set_xlim(-0.5, len(query_receptor)-0.5)
        ax_receptor.set_xlabel("")
        ax_receptor.set_yticks([1, 8], ["1", "8"])
        ax_receptor.set_ylim(0, max(query_receptor.max(), 8))
        ax_receptor.set_ylabel('log2 cpm\nreceptor', rotation=0,ha='right',va='center')
        ax_receptor.set_xticklabels("")
        ax_receptor.tick_params(length=0)
        ax_receptor.spines[["top", "right"]].set_visible(False)

    # Similarity heatmap; black outlines retain cell-level support.
    ax_similarity.set_facecolor("0.86")
    sns.heatmap(similarity_matrix.where(informative_matrix),vmin=-1,vmax=1,center=0,cmap="vlag",ax=ax_similarity, cbar=False)

    # Inset colorbar: positioned relative to the heatmap, without affecting GridSpec.
    ax_similarity_cbar = ax_similarity.inset_axes([1.035, 0, 0.028, 1],transform=ax_similarity.transAxes,)
    similarity_cbar = fig.colorbar(ax_similarity.collections[0], cax=ax_similarity_cbar)
    similarity_cbar.set_label(f"Pearson r to {query_cyto}",fontsize=6)
    similarity_cbar.ax.tick_params(length=2, pad=2, labelsize=6)

    for row_i in range(n_rows):
        for cell_i in range(len(ALL_CELL_TYPES)):
            if support_matrix.iloc[row_i, cell_i]:
                ax_similarity.add_patch(plt.Rectangle((cell_i, row_i),1,1,fill=False,edgecolor="black",linewidth=0.75,clip_on=False,zorder=5))

    cell_lineages = [config.lineage_by_cell[cell] for cell in ALL_CELL_TYPES]
    lineage_columns = {lineage: [] for lineage in lineage_order}
    for cell_i, lineage in enumerate(cell_lineages):
        lineage_columns.setdefault(lineage, []).append(cell_i)

    if plot_lineage_support:
        for row_i, label in enumerate(labels):
            for lineage in lineage_order:
                state = lineage_state.loc[label, lineage]
                columns = lineage_columns.get(lineage, [])
                if state < 0 or not columns:
                    continue
                edgecolor = config.support_color if state == 1 else "0.5"
                inset = 0.05
                ax_similarity.add_patch(plt.Rectangle(
                    (min(columns) + inset, row_i + inset),
                    max(columns) - min(columns) + 1 - 2*inset,
                    1 - 2*inset,
                    fill=False, edgecolor=edgecolor, linewidth=0.5, linestyle='-',
                    clip_on=False, zorder=6,
                ))
    for cell_i in range(1, len(ALL_CELL_TYPES)):
        if cell_lineages[cell_i] != cell_lineages[cell_i - 1]:
            ax_similarity.axvline(cell_i, color="white", linewidth=0.75)

    ax_similarity.set_xlabel("")
    ax_similarity.set_ylabel("Secondary cytokine")
    ax_similarity.tick_params(axis="x", rotation=90, length=0, labelsize=6)
    ax_similarity.tick_params(axis="y", rotation=0, length=0, labelsize=6)

    # Medium supernatant change.
    sns.heatmap(selected[["medium_log2fc"]].to_numpy(),vmin=-config.magnitude_saturation,vmax=config.magnitude_saturation,center=0,cmap="coolwarm",annot=True,
        fmt="+.1f",xticklabels=[""],yticklabels=False,cbar=False,ax=ax_medium,annot_kws={'size': 6})
    #ax_medium.set_title("sim.\nsuper.", fontsize=7)
    ax_medium.set_title(r"$\log_2$FC" + "\n(medium)", fontsize=6)
    ax_medium.tick_params(axis="x", length=0)
    ax_medium.text(0.5,-0.12,"Lineage:",transform=ax_medium.transAxes,ha="center",va="top",fontsize=6)
    ax_medium.text(0.5,-0.24,"supported",transform=ax_medium.transAxes,ha="center",va="top",fontsize=6,color="#168a8a")
    ax_medium.text(0.5,-0.36,"unsupported",transform=ax_medium.transAxes,ha="center",va="top",fontsize=6,color="0.6")

    y = np.arange(n_rows) + 0.5
    supported_confidence = {"cross-lineage support", "single-lineage supported"}
    priority_colors = np.where(selected["confidence"].isin(supported_confidence), config.support_color, "0.6")
    ax_priority.barh(y,selected["priority"],height=0.62,color=priority_colors)
    ax_priority.set(ylim=(n_rows, 0), yticks=[])
    ax_priority.set_xticks([0, 0.5], ['0', '0.5'])
    ax_priority.set_xlabel("priority")
    ax_priority.spines[["top", "right"]].set_visible(False)

    fig.suptitle(f"Secondary responses for {introduce_greek_letters(query_cyto)}",y=1.01,fontsize=6.5)
    for f in ['svg', 'png']:
        fig.savefig(os.path.join(plot_folder, f"7_secondary_{query_cyto}.{f}"), dpi=400,bbox_inches="tight")
    plt.show()

plot_secondary_pair("IL-15", pair_priority=pair_priority, n_pairs=20, direction=None, plot_lineage_support=False)
plot_secondary_pair("IL-18", pair_priority=pair_priority, n_pairs=20, direction=None, plot_lineage_support=False)
# plot_secondary_pair("IL-12", n_pairs=20)
# plot_secondary_pair("IL-18", n_pairs=20)
# plot_secondary_pair("IL-1-beta", n_pairs=20)
# plot_secondary_pair("IFN-beta", n_pairs=20)
# plot_secondary_pair("IL-10", n_pairs=20)

In [ ]:
setup_plotting_environment()

import textwrap
def wrap_legend_label(label, width=22):
    return "\n".join(textwrap.wrap(label, width=width))

def plot_secondary_overview(
    pair_priority, figsize, savepath=None, cytokine_order=CYTOKINE_ORDER,
    consistent_confidence=("cross-lineage support",), subset_to_consistent=False,
    flip_orientation=False, use_numbers=False, simplified=False, legend_style='linear'
):
    p = pair_priority.copy()
    finite = p[p["priority"].notna()]

    if subset_to_consistent:
        axis_source = finite[finite["confidence"].isin(consistent_confidence)]
        if axis_source.empty:
            raise ValueError(f"No pairs found with confidence in {consistent_confidence}.")
    else:
        axis_source = finite

    query_present = set(axis_source["query_cyto"].dropna())
    secondary_present = set(axis_source["secondary"].dropna())
    query_order = [c for c in cytokine_order if c in query_present] + sorted(query_present - set(cytokine_order))
    secondary_order = [c for c in cytokine_order if c in secondary_present] + sorted(secondary_present - set(cytokine_order))

    if flip_orientation:
        row_field, column_field = "query_cyto", "secondary"
        row_order, column_order = query_order, secondary_order
        xlabel, ylabel = "Altered supernatant cytokine", "Input cytokine"
    else:
        row_field, column_field = "secondary", "query_cyto"
        row_order, column_order = secondary_order, query_order
        xlabel, ylabel = "Input cytokine", "Altered supernatant cytokine"

    n_rows, n_cols = len(row_order), len(column_order)

    def make_matrix(value):
        return p.pivot_table(index=row_field, columns=column_field, values=value, aggfunc="first").reindex(index=row_order, columns=column_order)

    priority = make_matrix("priority")
    medium = make_matrix("medium_log2fc")
    confidence = make_matrix("confidence")
    causal_class = make_matrix("causal_class")

    collapse_map = {
        "shared direct/indirect": "unresolved/higher order",
        "probable tertiary": "unresolved/higher order",
        "unresolved indirect": "unresolved/higher order",
    }
    if simplified:
        causal_class = causal_class.replace(collapse_map)
    signed_priority = priority * np.sign(medium)
    consistent_support = confidence.isin(consistent_confidence)

    fig, ax = plt.subplots(figsize=figsize)
    cmap = sns.color_palette("vlag", as_cmap=True)
    cmap.set_bad("white")
    values = np.ma.masked_invalid(signed_priority.to_numpy(dtype=float))
    image = ax.imshow(values, cmap=cmap, vmin=-0.5, vmax=0.5, interpolation="none", aspect="equal")

    def inset_color(row_i, col_i):
        value = signed_priority.iat[row_i, col_i]
        if not np.isfinite(value):
            return "#333333"
        red, green, blue, _ = cmap(image.norm(value))
        luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
        return "white" if luminance < 0.45 else "#333333"

    for row_i, col_i in np.argwhere(consistent_support.to_numpy()):
        ax.add_patch(plt.Rectangle((col_i - 0.5, row_i - 0.5), 1, 1, fill=False, edgecolor=config.support_color, linewidth=0.65))

    causal_styles = dict(config.causal_styles)
    if simplified:
        causal_styles = {
            "immediate secondary": causal_styles["immediate secondary"],
            "possibly incidental similarity": causal_styles["possibly incidental similarity"],
            "unresolved/higher order": {"marker": "D", "label": "Unresolved/\nhigher order"},
            "unresolved receptor coverage": causal_styles["unresolved receptor coverage"],
            "unsupported similarity": causal_styles["unsupported similarity"],
        }

    special_glyphs = {
        "unresolved receptor coverage": "?",
        "unsupported similarity": "−",
    }
    numbered_classes = [label for label in causal_styles if label not in special_glyphs]
    number_by_class = {label: str(i + 1) for i, label in enumerate(numbered_classes)}
    for causal_label, style in causal_styles.items():
        rows, cols = np.where(causal_class.eq(causal_label).to_numpy())
        if use_numbers:
            glyph = special_glyphs[causal_label] if causal_label in special_glyphs else number_by_class[causal_label]
            for row_i, col_i in zip(rows, cols):
                shift = 0.05 if glyph != "−" else 0
                ax.text(col_i, row_i+shift, glyph, ha="center", va="center", fontsize=5.5, color=inset_color(row_i, col_i), zorder=6)
        elif style["marker"] in config.white_inner_set:
            for row_i, col_i in zip(rows, cols):
                color = inset_color(row_i, col_i)
                ax.scatter(col_i, row_i, s=9, marker=style["marker"], facecolor="none" if color == "white" else "white", edgecolor=color, linewidth=0.65, zorder=6)
        else:
            for row_i, col_i in zip(rows, cols):
                ax.scatter(col_i, row_i, s=9 if style['marker'] != 's' else 6, marker=style["marker"], color=inset_color(row_i, col_i), linewidth=0.65 if not style["marker"] in ['X', '$?$'] else 0.3, zorder=6)

    xlabels = [introduce_greek_letters(c) for c in column_order]
    ylabels = [introduce_greek_letters(c) for c in row_order]
    ax.set_xticks(np.arange(n_cols), labels=xlabels, rotation=90, fontsize=6)
    ax.set_yticks(np.arange(n_rows), labels=ylabels, fontsize=6)
    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(n_rows - 0.5, -0.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.tick_params(length=0)

    cbar = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("Secondary-response priority", fontsize=6)
    cbar.set_ticks([-0.5, 0, 0.5], labels=["decreased", "0", "increased"], rotation=90, va="center")
    cbar.ax.tick_params(labelsize=6, length=2)

    from matplotlib.lines import Line2D
    legend_handles = [plt.Rectangle((0, 0), 1, 1, fill=False, edgecolor=config.support_color,linewidth=0.7, label="Consistent support"),]
    for causal_label, style in causal_styles.items():
        if use_numbers:
            glyph = special_glyphs[causal_label] if causal_label in special_glyphs else number_by_class[causal_label]
            marker = "_" if glyph == "−" else f"${glyph}$"
            legend_handles.append(Line2D(
                [0], [0], marker=marker, linestyle="", color="#333333",
                markeredgewidth=0.8 if glyph == "−" else 0, markersize=4 if glyph in special_glyphs.values() else 4,
                label=wrap_legend_label(style["label"], width=100), #width=15),
            ))
        else:
            white_inner = style["marker"] in config.white_inner_set
            legend_handles.append(
                Line2D([0], [0], marker=style["marker"], linestyle="", color="#333333", markerfacecolor="white" if white_inner else "#333333",
                markeredgecolor="#333333" if style["marker"] != 'X' else 'none', markeredgewidth=0.65 if not style["marker"] in ['X', '$?$'] else 0.3, markersize=4,
                label=wrap_legend_label(style["label"], width=15)))

    fig.tight_layout()
    legend = ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.3, 1), frameon=False, fontsize=6, ncol=2)
    if savepath is not None:
        fig.savefig(savepath, dpi=300, bbox_inches="tight")

    plt.show()
    return fig


causal_pair_priority_sub = causal_pair_priority[
    (causal_pair_priority['query_cyto'].isin(
        ['IL-1-alpha', 'IL-1-beta', 'IL-18', "IL-2", "IL-7", "IL-15", "IL-3", "GM-CSF", "IL-6", "IL-12", 'IFN-beta', 'IFN-gamma', 'IL-32-beta', 'TNF-alpha', 'C5a']
    ))
]
fig = plot_secondary_overview(
    causal_pair_priority_sub,
    savepath=os.path.join(plot_folder, '7_secondary_overview_main.svg'),
    figsize=cm_to_inches((6.5, 5)),
    consistent_confidence=("cross-lineage support","single-lineage supported"),
    subset_to_consistent=True,
    flip_orientation=True,
    use_numbers=True,
    simplified=True,
    legend_style='linear'
)

fig = plot_secondary_overview(
    causal_pair_priority,
    savepath=os.path.join(plot_folder, '7_secondary_overview_supp.svg'),
    figsize=cm_to_inches((8, 7)),
    consistent_confidence=("cross-lineage support","single-lineage supported"),
    subset_to_consistent=True,
    flip_orientation=True,
    use_numbers=True,
    simplified=False
)

In [ ]:
# export vals
secondary_names = [
    "pair_priority", "causal_pair_priority", "similarity_by_pair",
    "informative_by_pair", "support_by_pair", "query_receptor_by_pair",
    "pair_details_by_pair", "producer_shares_by_pair", "producer_details_by_pair",
]
secondary_effect_analysis = {name: globals()[name] for name in secondary_names}
pd.to_pickle(secondary_effect_analysis, os.path.join(output_tables_folder, "secondary_effect_analysis.pkl"))

secondary_effect_analysis = pd.read_pickle(
    os.path.join(output_tables_folder, "secondary_effect_analysis.pkl")
)

pair_priority = secondary_effect_analysis["pair_priority"]
causal_pair_priority = secondary_effect_analysis["causal_pair_priority"]
similarity_by_pair = secondary_effect_analysis["similarity_by_pair"]
informative_by_pair = secondary_effect_analysis["informative_by_pair"]
support_by_pair = secondary_effect_analysis["support_by_pair"]
query_receptor_by_pair = secondary_effect_analysis["query_receptor_by_pair"]
pair_details_by_pair = secondary_effect_analysis["pair_details_by_pair"]
producer_shares_by_pair = secondary_effect_analysis["producer_shares_by_pair"]
producer_details_by_pair = secondary_effect_analysis["producer_details_by_pair"]

In [ ]:
from utils.secondary_overview_explainer import plot_secondary_overview_explainer

query = "IL-18"
secondary = "TNF-alpha"
fig, audit = plot_secondary_overview_explainer(
    query_cyto=query,
    secondary_cyto=secondary,
    pair_table=causal_pair_priority,
    pair_details_by_pair=pair_details_by_pair,
    producer_details_by_pair=producer_details_by_pair,
    config=config,
    strong_impact_by_cell_type=strong_impact_by_cell_type,
    label_formatter=introduce_greek_letters,
    figsize_cm=(18, 23),
    savepath=os.path.join(plot_folder, f"7_secondary_route_{query}_{secondary}.svg")
    )
display(fig)
plt.close()

In [ ]:
from utils.secondary_cell_flow_network import plot_secondary_cell_flow_network
plot_pairs = [
    ("IL-15", ["IFN-gamma", "TNF-alpha", "IL-6"], (5.3, 3.5)),
    ("IL-1-beta", ["IL-15", "IFN-gamma", "TNF-alpha"], (5.3, 3)),
    ("IL-18", ['IL-15', 'IFN-gamma', 'TNF-alpha'], (5.3, 3)),
    ("IL-12", ['IL-15', 'IFN-gamma'], (5.3, 3)),
    ("IFN-gamma", ["IL-15", "IL-10"], (5, 3))
]

for input_cyto, downstream_cytos, figsize in plot_pairs:
    fig, pairs, displayed_paths = plot_secondary_cell_flow_network(
        input_cyto,
        pair_table=causal_pair_priority,
        producer_shares_by_pair=producer_shares_by_pair,
        producer_details_by_pair=producer_details_by_pair,
        pair_details_by_pair=pair_details_by_pair,
        support_by_pair=support_by_pair,
        similarity_by_pair=similarity_by_pair,
        config=config,
        cell_type_order=ALL_CELL_TYPES,
        label_formatter=introduce_greek_letters,
        downstream_cytokines=downstream_cytos,
        figsize_cm=figsize,
        font_size=6,
        minimum_producer_share=0.1,
        minimum_recipient_similarity=0.30,
        simplified=True,

        # Purely aesthetic—does not encode magnitude.
        path_width=0.85,
        path_alpha=0.62,
        port_spacing=0.018,
        left_bar_height=0.08,
        right_bar_height=0.05,
    )
    display(fig)
    plt.savefig(os.path.join(plot_folder, f'7_sankey_{input_cyto}.svg'))
    
    test = displayed_paths.loc[
        displayed_paths["view_side"].eq("production")
        & displayed_paths["secondary"].eq("IL-15")
        & displayed_paths["cell"].eq("CD14 Mono"),
        ["route", "intermediate", "value"],
    ]